# Cardio-JEPA v3 — corrected and extended framework (package v0.2.0)

**Latent forward-prediction JEPA for ECG**: masked latent infilling (morphology, Kim's
ECG-JEPA) + horizon-conditioned forward latent prediction (dynamics), compared against
SimCLR, CPC, ST-MEM, supervised and random-init baselines on the same encoder.

### What changed from the v0.1 run
The v0.1 run had test-set leakage (PTB-XL was pretrained on), forecasting leakage (the
forward predictor was trained on the records it was tested on), untrained horizon
embeddings at test time, a mis-read 128 Hz corpus, a missing EMA-only arm, a baseline that
never trained, a buggy VQ arm, over-confident statistics and an OOM that lost all results.
All 30 issues are listed in the **Appendix (bug report)** at the end, with the fix and the
regression test for each. **v0.1 numbers are not valid evidence; re-run with this notebook.**

### How to run (multi-session — one Kaggle T4 session cannot fit the study)
1. Attach datasets (table below). **Detach** MEETI (no readable files, cost ~most of the
   51-min discovery) and *ECG Heartbeat Categorization* (single beats; leakage risk).
2. Set `SESSION = "A"` in the configuration cell → **Save Version → Run All**.
3. For session B (then C, D): **Add Input → your previous notebook version's output**, set
   `SESSION = "B"`, run again. Checkpoints and results from earlier sessions are found and
   reused automatically.
4. Every session writes `/kaggle/working/cardio_jepa_v3_outputs.zip` (results ledger,
   tables, figures, docs, package source). The last session has the complete paper tables.

`RUN_MODE = "smoke"` runs every code path on synthetic data in a few minutes (CPU ok).

The framework source is **written by this notebook** (section 1) — no model upload needed.


## Datasets

| Role | Kaggle dataset | Notes |
|---|---|---|
| Pretraining (12-lead) | `bjoernjostein/shaoxing-and-ningbo-first-hospital-database` | all 45,152 recordings (cap raised) |
| Pretraining (optional) | CODE-15 (`ccf002/code-15-ecg-complete`) | needs `h5py` (present on Kaggle) |
| **Probe only** | PTB-XL (`omar1863/ptb-xl` or `khyeh0719/ptb-xl-dataset`) | official folds 1–8 / 9 / 10; **never pretrained on** |
| Probe only (external site) | CPSC2018 (`physionet/china-physiological-signal-challenge-in-2018`) | recommended for cross-dataset results |
| Forward pretraining + anticipation | Long-Term AF (`sahilshahare12/long-term-af-database`) | 30 % of records held out before pretraining |
| Forward pretraining | MIT-BIH Long-Term ECG (`protobioengineering/mit-bih-long-term-ecg-database`) | now read at 128 Hz (was 360 Hz) |
| Robustness noise | MIT-BIH Noise Stress Test (ships inside `omar1863/ptb-xl`) | `bw` / `em` / `ma` noise records |

Not used: **MEETI** (no reader for its format — detach or add a reader), **ECG Heartbeat
Categorization** (single 125 Hz beats derived from MIT-BIH/PTB — detach). The separate
Chapman-Shaoxing mirror is redundant with the Shaoxing+Ningbo mirror (indexed once).


In [ ]:
# --- 0. environment: must run before torch initialises CUDA -----------------------------
import os, sys, json, time, math, glob, shutil, warnings, platform, datetime, copy, gc, traceback
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")       # fragmentation fix
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # older torch name
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")                   # deterministic cuBLAS
# NOTE (v0.1 notebook bug): do NOT set CUDA_LAUNCH_BLOCKING=1 -- it serialises every kernel.

import numpy as np
warnings.filterwarnings("ignore", category=FutureWarning)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(os.getcwd())
os.chdir(WORK)
RUNS_DIR = os.path.join(WORK, "runs")
RESULTS_DIR = os.path.join(WORK, "results")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
SCORE_DIR = os.path.join(RESULTS_DIR, "scores")
for d in (RUNS_DIR, RESULTS_DIR, FIG_DIR, SCORE_DIR, "cardio_jepa_v3_src/cardio_jepa/models",
          "cardio_jepa_v3_src/cardio_jepa/data", "cardio_jepa_v3_src/cardio_jepa/eval",
          "cardio_jepa_v3_src/tests", "cardio_jepa_v3_src/docs"):
    os.makedirs(d, exist_ok=True)

import torch
print("python ", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only)")
print("workdir", WORK)

## 1. Framework source (cardio_jepa v0.2.0)

The next cells write the corrected package to `/kaggle/working/cardio_jepa_v3_src/`. They
are the single source of truth used by the rest of the notebook (and they are saved in the
output zip, so every result is traceable to the exact code). Collapse this section.


In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/__init__.py
"""Cardio-JEPA: latent forward-prediction JEPA for cardiac world modelling from ECG.

Reference implementation of the research proposal
"Cardio-JEPA: Latent Forward-Prediction Joint-Embedding Architecture for Cardiac
World Modeling from ECG".

Built on top of, and attributing, these public codebases:
  * Kim, ECG-JEPA (arXiv:2410.08559)  https://github.com/sehunfromdaegu/ECG_JEPA
      -> CroPA encoder, masked lead-wise predictor, sin-cos 2-D positions,
         EMA target schedule, linear-probe protocol.
  * Assran et al., I-JEPA (CVPR 2023) https://github.com/facebookresearch/ijepa
      -> ViT blocks, context/target asymmetry, predictor design.
  * Na et al., ST-MEM (ICLR 2024)     -> masked raw-reconstruction baseline.

New here (the proposal's contributions): the forward latent predictor with
horizon tokens, the two-level hierarchical rhythm stack, the four-arm
anti-collapse study, and the anticipation benchmarks.
"""

from .config import (ANTI_COLLAPSE_DEFAULTS, DataConfig, EvalConfig, ExperimentConfig,
                     ForwardSamplingConfig, HierarchicalConfig, HORIZON_BUCKETS_S,
                     LossConfig, ModelConfig, OptimConfig, REDUCED_LEAD_IDX,
                     REDUCED_LEAD_NAMES, SignalConfig, configure_arm, full_config,
                     kaggle_config, smoke_config)
from .checkpoint import (describe_model_input, find_checkpoints, find_package_root,
                         load_weights)
from .diagnostics import CollapseMonitor, detect_collapse_signature, effective_rank
from .engine import (InterleavedLoader, free_cuda, load_train_result, pick_device,
                     pretrain, set_seed)
from .models.baselines import Inception1d, STMEMBaseline, XResNet1d
from .models.ssl_baselines import CPCBaseline, SimCLRBaseline
from .models.cardio_jepa import CardioJEPA

__version__ = "0.2.0"

__all__ = [
    "CardioJEPA", "STMEMBaseline", "XResNet1d", "Inception1d",
    "SimCLRBaseline", "CPCBaseline", "load_train_result", "free_cuda",
    "ExperimentConfig", "SignalConfig", "ModelConfig", "LossConfig", "OptimConfig",
    "ForwardSamplingConfig", "HierarchicalConfig", "DataConfig", "EvalConfig",
    "HORIZON_BUCKETS_S", "REDUCED_LEAD_IDX", "REDUCED_LEAD_NAMES",
    "smoke_config", "kaggle_config", "full_config", "configure_arm",
    "ANTI_COLLAPSE_DEFAULTS",
    "pretrain", "pick_device", "set_seed", "InterleavedLoader",
    "CollapseMonitor", "effective_rank", "detect_collapse_signature",
    "load_weights", "find_package_root", "find_checkpoints", "describe_model_input",
    "__version__",
]

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/config.py
"""Configuration objects for Cardio-JEPA.

All experiment knobs live here so that the Kaggle notebook can switch between a
fast smoke run (minutes) and a full pretraining run (hours) by swapping a single
object.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import List, Optional, Sequence, Tuple

# --------------------------------------------------------------------------------------
# Signal geometry
# --------------------------------------------------------------------------------------
# Following Kim's ECG-JEPA (arXiv:2410.08559) we work with the 8 independent leads
# (I, II, V1..V6); leads III, aVR, aVL, aVF are exact linear combinations of I and II
# (Einthoven / Goldberger) and therefore carry no extra information.
REDUCED_LEAD_NAMES: Tuple[str, ...] = ("I", "II", "V1", "V2", "V3", "V4", "V5", "V6")
# Index of those leads inside the standard 12-lead ordering
# [I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6]
REDUCED_LEAD_IDX: Tuple[int, ...] = (0, 1, 6, 7, 8, 9, 10, 11)

# Horizon buckets used by the forward predictor's learned horizon token (Section 5.3b of
# the proposal). Values are in seconds.
HORIZON_BUCKETS_S: Tuple[float, ...] = (
    2.0, 5.0, 10.0, 30.0, 60.0, 300.0, 600.0, 1800.0, 3600.0)


@dataclass
class SignalConfig:
    """Tokenisation / resampling geometry for a single ECG window."""

    fs: int = 250                 # working sampling rate (Hz)
    window_s: float = 10.0        # seconds per window
    patch_samples: int = 50       # samples per patch  (0.2 s @ 250 Hz)
    n_leads: int = 8              # 8 reduced leads
    reduced_lead: bool = True

    @property
    def window_samples(self) -> int:
        return int(round(self.fs * self.window_s))

    @property
    def n_time_patches(self) -> int:
        assert self.window_samples % self.patch_samples == 0, (
            f"window_samples ({self.window_samples}) must be divisible by "
            f"patch_samples ({self.patch_samples})"
        )
        return self.window_samples // self.patch_samples

    @property
    def n_tokens(self) -> int:
        return self.n_leads * self.n_time_patches


@dataclass
class ModelConfig:
    """Encoder / predictor sizes."""

    encoder_embed_dim: int = 384
    encoder_depth: int = 8
    encoder_num_heads: int = 6
    predictor_embed_dim: int = 192
    predictor_depth: int = 4
    predictor_num_heads: int = 6
    mlp_ratio: float = 4.0
    qkv_bias: bool = True
    drop_rate: float = 0.0
    attn_drop_rate: float = 0.0
    drop_path_rate: float = 0.0
    init_std: float = 0.02
    pos_type: str = "sincos"          # 'sincos' | 'learnable'
    use_cropa: bool = True            # Cross-Pattern Attention in the encoder
    # masked-infilling branch
    mask_type: str = "block"          # 'block' | 'random'
    mask_scale: Tuple[float, float] = (0.175, 0.225)
    # forward branch
    forward_target: str = "tokens"    # 'tokens' (per-patch future latents) | 'pooled'
    #: v0.2: target used by the *cross-window* forward objective (horizons >= 10 s).
    #: Predicting the per-patch token grid of a window minutes ahead requires knowing
    #: the beat phase at an unknown future time; the optimal L1/L2 answer is then the
    #: per-position mean, i.e. noise for the predictor to fit. The pooled window latent
    #: is phase-invariant. 'tokens' reproduces v0.1 behaviour (ablation flag).
    forward_target_long: str = "pooled"
    predictor_cropa: bool = False     # apply CroPA inside the predictor too


@dataclass
class LossConfig:
    """Dual objective weights + anti-collapse arm (RQ3)."""

    lambda_mask: float = 1.0          # weight on L_mask (morphology)
    weight_forward: float = 1.0       # weight on L_fwd (dynamics)
    beta_reg: float = 1.0             # weight on L_reg (anti-collapse)
    loss_fn: str = "smooth_l1"        # 'smooth_l1' | 'l1' | 'l2'
    # RQ3 arm: 'ema' | 'vicreg' | 'vq' | 'sigreg'
    anti_collapse: str = "ema"
    #: What L_reg is measured on: 'pooled' (per-sample representation, what every
    #: downstream probe consumes), 'tokens' (flattened patch latents), or 'both'.
    #: 'tokens' is misleading on its own -- positional variance satisfies the penalty
    #: while every sample still maps to the same pooled vector.
    reg_target: str = "pooled"
    use_ema_target: bool = True       # sigreg arm may set this False (no target network)
    # VICReg
    vicreg_var_gamma: float = 1.0
    vicreg_var_weight: float = 1.0
    vicreg_cov_weight: float = 0.04
    # soft vector-quantised codebook
    vq_codebook_size: int = 512
    #: v0.2 FIX: 0.5 on *cosine* logits (range [-1, 1]) gives a near-uniform soft
    #: assignment over 512 codes, so every target becomes ~the same codebook average
    #: -> constant target -> the loss collapse seen in the v0.1 RQ3 run. 0.1 gives
    #: logits in [-10, 10], i.e. genuinely selective assignments.
    vq_temperature: float = 0.1
    vq_entropy_weight: float = 0.1
    #: v0.2 FIX: weight of the per-sample entropy (commitment) term. v0.1 used 0.1,
    #: which let a uniform per-sample assignment (= constant target) minimise L_reg.
    vq_commit_weight: float = 1.0
    # SIGReg / LeJEPA
    sigreg_n_slices: int = 64
    sigreg_n_freqs: int = 16


@dataclass
class OptimConfig:
    epochs: int = 20
    batch_size: int = 32
    lr: float = 5e-4
    min_lr: float = 1e-6
    warmup_epochs: int = 2
    weight_decay: float = 0.05
    grad_clip: float = 1.0
    amp: bool = True
    ema_start: float = 0.996
    ema_end: float = 1.0
    num_workers: int = 2
    seed: int = 0
    #: v0.2: cudnn.deterministic + no benchmark autotuning, so a fixed seed gives the
    #: same run twice (needed for the reproducibility exit criterion).
    deterministic: bool = True

    #: Cap the number of optimiser steps per epoch. Without this an epoch is one pass
    #: over the corpus, so attaching a large corpus silently multiplies wall-clock:
    #: ~1900 steps/epoch x 30 epochs x ~0.5 s/step is ~8 h *per model*, and the
    #: notebook trains up to 9. Epochs then stop meaning "a pass over the data" and
    #: start meaning "a fixed compute budget", which is what you actually want when
    #: comparing arms on a fixed session.
    max_steps_per_epoch: Optional[int] = None

    # ---- collapse guard -------------------------------------------------------------
    #: How often (in steps) to test for representation collapse. 0 disables the check.
    collapse_check_every: int = 200
    #: 'stop'   -- abandon this run as soon as collapse is detected (default: a
    #:             collapsed model is worthless, and continuing burns GPU hours)
    #: 'warn'   -- report once and keep going (what RQ3 wants: collapse is the
    #:             measurement, not a failure)
    #: 'ignore' -- no check
    on_collapse: str = "stop"
    #: A latent-prediction loss this far below its own starting value is the trivial
    #: constant solution; legitimate training does not get there in a few hundred steps.
    collapse_loss_frac: float = 0.02
    #: Companion embedding test: near-identical directions for every sample.
    collapse_cos_sim: float = 0.9995


@dataclass
class ForwardSamplingConfig:
    """How (context, future) pairs are drawn from long recordings (Section 5.3b)."""

    context_windows: int = 1          # number of consecutive 10 s windows in the context
    # horizons (seconds) sampled during pretraining; each maps to a horizon-token bucket.
    # v0.2 FIX: v0.1 trained only (10, 30, 60, 300) but evaluated at 900 s and 1800 s,
    # which map to the 600 s / 1800 s buckets whose horizon embeddings had never
    # received a gradient (random init). Every evaluated horizon must be trained.
    horizons_s: Tuple[float, ...] = (10.0, 30.0, 60.0, 300.0, 600.0, 1800.0)
    #: v0.2: draw a fresh context start on every access (more distinct pairs per
    #: epoch) instead of a fixed plan of 200 starts per record.
    resample_starts: bool = True
    # allow "future" windows drawn from a different recording of the same patient (RQ4)
    patient_aware: bool = False
    patient_aware_prob: float = 0.25
    # Intra-window forward prediction. A 10 s clip has no "next window", so for the
    # short corpora (MIMIC/CODE-15/Chapman/Ningbo/CPSC/PTB-XL) we instead predict the
    # latents of the *later* time patches from the earlier ones inside the same clip.
    # This is the "short intra-recording horizons" de-risking step of Stage 1 and it
    # makes L_fwd trainable on all ~800k short recordings, not only on Holter data.
    intra_window: bool = True
    intra_window_frac: float = 0.6    # fraction of time patches used as context


@dataclass
class HierarchicalConfig:
    """Level-2 (rhythm) stack for RQ2."""

    enabled: bool = False
    l2_embed_dim: int = 384
    l2_depth: int = 4
    l2_num_heads: int = 6
    l2_predictor_dim: int = 192
    l2_predictor_depth: int = 2
    l2_context_windows: int = 12      # 12 x 10 s = 2 min of context
    l2_horizons_s: Tuple[float, ...] = (60.0, 300.0, 600.0, 1800.0)
    l2_weight: float = 1.0
    l2_mask_weight: float = 0.5


@dataclass
class DataConfig:
    input_root: str = "/kaggle/input"
    #: v0.2: preprocessed probe caches go to /tmp, NOT /kaggle/working (which counts
    #: against the 20 GB output limit and is zipped at the end).
    cache_dir: str = "/tmp/cardio_jepa_cache"
    # hard caps so a Kaggle session never OOMs on the ~800k-recording corpora
    max_records_per_dataset: int = 20000
    max_long_records: int = 40
    #: v0.2 FIX (leakage): PTB-XL and CPSC2018 are the *downstream* benchmarks, so they
    #: are no longer pretraining corpora. v0.1 pretrained on PTB-XL folds 9-10 and then
    #: reported the fold-10 linear probe (transductive test-set leakage). Kim's
    #: ECG-JEPA protocol also pretrains on Chapman/Ningbo/CODE-15 only.
    pretrain_datasets: Sequence[str] = (
        "meeti", "mimic", "code15", "chapman", "ningbo",
    )
    probe_datasets: Sequence[str] = ("ptbxl", "cpsc")
    #: Set True only for an explicit transductive ablation.
    allow_probe_in_pretrain: bool = False
    #: v0.2: MIT-BIH Long-Term ECG (`mitbih_lt`) is its own corpus (128 Hz), no longer
    #: mislabelled as MIT-BIH Arrhythmia (360 Hz).
    long_datasets: Sequence[str] = ("ltaf", "mitbih", "mitbih_lt")
    #: v0.2: corpus keys that discovery must not even walk (e.g. 'meeti' when its
    #: format has no reader yet -- walking it cost most of the 51 min v0.1 spent in
    #: discovery).
    skip_datasets: Sequence[str] = ()
    #: v0.2: share of training batches drawn from long (2-lead, tiled) Holter pairs.
    #: v0.1 hard-coded weight 2.0 vs 1.0, i.e. 2/3 of every arm's updates -- including
    #: the masked-only "Kim" arm -- came from tiled 2-channel Holter data, a large
    #: domain shift from the 12-lead downstream tasks. Weight relative to short = 1.0.
    long_loader_weight: float = 0.5
    #: v0.2: PTB-XL at 500 Hz (falls back to 100 Hz if the hr files are absent).
    ptbxl_use_hr: bool = True
    #: v0.2: the probe benchmark is never subsampled (v0.1 indexed a random 20k of
    #: 21.8k PTB-XL records, which also thinned the official test fold).
    max_probe_records: int = 10 ** 9
    # The MIT-BIH *Noise Stress Test* database is deliberately noise-corrupted and
    # ships three pure-noise records (bw/em/ma). Off by default: it is fine as
    # augmentation-like pretraining data but must never be read as an ECG corpus.
    use_noise_corpora: bool = False
    allow_synthetic_fallback: bool = True
    synthetic_n_short: int = 512
    synthetic_n_long: int = 12
    synthetic_long_minutes: float = 40.0


@dataclass
class EvalConfig:
    linear_probe_epochs: int = 40
    linear_probe_lr: float = 1e-3
    linear_probe_batch_size: int = 128
    #: v0.2: learning rates tried for the linear head; chosen on the validation split
    #: (PTB-XL fold 9 / record-level inner split), never on the test split.
    linear_probe_lr_grid: Tuple[float, ...] = (1e-2, 1e-3, 1e-4)
    linear_probe_patience: int = 8
    bootstrap_n: int = 2000
    label_fractions: Tuple[float, ...] = (0.01, 0.1, 0.5, 1.0)
    #: v0.2: repeat every sub-100% label fraction with different random subsets and
    #: report mean +/- std (a single 1% draw is mostly noise).
    label_fraction_repeats: int = 3
    # anticipation benchmark. v0.2: every horizon is a *trained* horizon bucket.
    anticipation_horizons_s: Tuple[float, ...] = (300.0, 600.0, 1800.0)
    #: v0.2: fraction of long records held out from *pretraining* and used only as
    #: the anticipation / burden / forecast-skill test set.
    long_test_frac: float = 0.3
    anticipation_context_s: float = 10.0
    anticipation_stride_s: float = 30.0
    # exclusion margin: windows within this many seconds *before* an event are dropped
    # from the negative pool to avoid label leakage on the boundary
    anticipation_guard_s: float = 5.0


@dataclass
class ExperimentConfig:
    name: str = "cardio_jepa"
    signal: SignalConfig = field(default_factory=SignalConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    loss: LossConfig = field(default_factory=LossConfig)
    optim: OptimConfig = field(default_factory=OptimConfig)
    forward: ForwardSamplingConfig = field(default_factory=ForwardSamplingConfig)
    hierarchical: HierarchicalConfig = field(default_factory=HierarchicalConfig)
    data: DataConfig = field(default_factory=DataConfig)
    evaluation: EvalConfig = field(default_factory=EvalConfig)
    out_dir: str = "/kaggle/working/runs"

    def to_dict(self) -> dict:
        return asdict(self)


# --------------------------------------------------------------------------------------
# Presets
# --------------------------------------------------------------------------------------
#: Per-arm regulariser strengths for RQ3.
#:
#: `vicreg` beta was measured on the failing case (forward-only, which collapses
#: outright under EMA-only). Sweeping beta on the *pooled* target:
#:
#:     beta    pooled std   cos_sim   L_reg share of total loss
#:     ema     0.00057      1.00000   -           <- collapsed
#:     0.1     0.38693      0.309     22%
#:     1.0     0.39229      0.636     55%
#:     5.0     0.38802      0.174     68%
#:     20.0    0.40832      0.203     69%
#:
#: The variance floor saturates by 0.1 (0.387 of an eventual 0.408); larger beta buys
#: almost no extra spread and progressively replaces the prediction objective, which
#: is the regime where downstream accuracy degrades. Hence 0.1.
#:
#: Caveat worth knowing: this was chosen on embedding geometry and loss balance, NOT
#: on downstream accuracy. A linear probe over beta in {0.1, 1.0, 5.0} on the
#: synthetic AF task returned 0.855 / 0.842 / 0.856 -- and the *collapsed* EMA model
#: scored 0.873, the highest of all. That task separates on heart rate alone, so a
#: near-degenerate representation still carries enough residual to win it; the probe
#: simply cannot rank representation quality here. Re-validate beta on PTB-XL or
#: CPSC2018 before treating it as tuned.
ANTI_COLLAPSE_DEFAULTS = {
    "ema":    {"beta_reg": 0.0,  "use_ema_target": True},
    "vicreg": {"beta_reg": 0.1, "use_ema_target": True},
    "vq":     {"beta_reg": 0.5,  "use_ema_target": True},   # see LossConfig.vq_* fixes
    "sigreg": {"beta_reg": 0.5,  "use_ema_target": False},
}


def configure_arm(cfg: "ExperimentConfig", arm: str) -> "ExperimentConfig":
    """Return a copy of ``cfg`` set up for one RQ3 anti-collapse arm."""
    import copy as _copy
    if arm not in ANTI_COLLAPSE_DEFAULTS:
        raise ValueError(f"unknown arm {arm!r}; expected one of "
                         f"{sorted(ANTI_COLLAPSE_DEFAULTS)}")
    out = _copy.deepcopy(cfg)
    out.name = f"{cfg.name}_{arm}"
    out.loss.anti_collapse = arm
    for k, v in ANTI_COLLAPSE_DEFAULTS[arm].items():
        setattr(out.loss, k, v)
    return out


def smoke_config() -> ExperimentConfig:
    """Tiny end-to-end run: verifies every code path in a couple of minutes on CPU."""
    cfg = ExperimentConfig(name="smoke")
    cfg.model.encoder_embed_dim = 96
    cfg.model.encoder_depth = 2
    cfg.model.encoder_num_heads = 4
    cfg.model.predictor_embed_dim = 64
    cfg.model.predictor_depth = 2
    cfg.model.predictor_num_heads = 4
    cfg.optim.epochs = 2
    cfg.optim.batch_size = 4
    cfg.optim.warmup_epochs = 0
    cfg.optim.amp = False
    cfg.optim.num_workers = 0
    cfg.data.max_records_per_dataset = 64
    cfg.data.max_long_records = 6
    cfg.data.synthetic_n_short = 64
    cfg.data.synthetic_n_long = 6
    cfg.data.synthetic_long_minutes = 30.0
    cfg.evaluation.linear_probe_epochs = 3
    cfg.evaluation.bootstrap_n = 100
    cfg.hierarchical.l2_context_windows = 4
    cfg.data.max_probe_records = 400
    cfg.evaluation.label_fraction_repeats = 2
    cfg.evaluation.linear_probe_lr_grid = (1e-2, 1e-3)
    cfg.evaluation.anticipation_horizons_s = (60.0, 300.0)
    cfg.forward.horizons_s = (10.0, 30.0, 60.0, 300.0)
    cfg.optim.deterministic = False
    cfg.out_dir = "/tmp/cardio_jepa_runs"
    return cfg


def kaggle_config() -> ExperimentConfig:
    """Fits a P100 (16 GB) / 2xT4 Kaggle session with time to spare for evaluation."""
    cfg = ExperimentConfig(name="cardio_jepa_kaggle")
    cfg.model.encoder_embed_dim = 384
    cfg.model.encoder_depth = 8
    cfg.model.encoder_num_heads = 6
    cfg.model.predictor_embed_dim = 192
    cfg.model.predictor_depth = 4
    cfg.optim.epochs = 30
    cfg.optim.batch_size = 32
    cfg.optim.lr = 5e-4
    # An epoch here is a fixed compute budget, not a pass over the corpus. With a real
    # corpus attached a full pass is ~1900 steps, so 30 unbounded epochs is ~8 h for
    # ONE model and the notebook trains up to nine. 250 x 30 = 7500 steps/model keeps
    # the default all-stages run inside a single Kaggle session; raise it if you run
    # fewer stages. The estimator cell in the notebook computes the value for the
    # machine you actually get.
    cfg.optim.max_steps_per_epoch = 250
    # L_fwd has a trivial constant solution, so the pooled representation needs an
    # explicit variance floor -- see ANTI_COLLAPSE_DEFAULTS and Stage 3.
    cfg.loss.anti_collapse = "vicreg"
    cfg.loss.reg_target = "pooled"
    cfg.loss.beta_reg = ANTI_COLLAPSE_DEFAULTS["vicreg"]["beta_reg"]
    # v0.2: PTB-XL left the pretraining pool, so take all of Chapman+Ningbo (45,152).
    cfg.data.max_records_per_dataset = 46000
    # LTAFDB ships 84 records and MIT-BIH Arrhythmia 48; the cap applies per corpus,
    # so 100 takes all of both. Long records cost almost nothing to hold -- only
    # annotations live in RAM, signals are read from disk per window (~145 MB for all
    # 84 LTAFDB beat-annotation sets) -- and the anticipation benchmark is
    # patient-level, so every extra record is an extra independent split unit.
    cfg.data.max_long_records = 100
    return cfg


def full_config() -> ExperimentConfig:
    """Paper-scale settings (multi-GPU / multi-day; will not finish in one Kaggle run)."""
    cfg = ExperimentConfig(name="cardio_jepa_full")
    cfg.model.encoder_embed_dim = 768
    cfg.model.encoder_depth = 12
    cfg.model.encoder_num_heads = 16
    cfg.model.predictor_embed_dim = 384
    cfg.model.predictor_depth = 6
    cfg.model.predictor_num_heads = 12
    cfg.model.drop_path_rate = 0.1
    cfg.optim.epochs = 100
    cfg.optim.batch_size = 64
    cfg.optim.lr = 5e-5
    cfg.data.max_records_per_dataset = 10 ** 9
    cfg.data.max_long_records = 10 ** 9
    return cfg

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/regularizers.py
"""Anti-collapse mechanisms compared under RQ3 (proposal Section 5.5).

Four arms:
  1. ``ema``    -- encoder asymmetry + stop-gradient only (Kim / Weimann & Conrad).
  2. ``vicreg`` -- variance hinge + covariance penalty (BRo-JEPA style).
  3. ``vq``     -- soft vector-quantised codebook bottleneck on the target
                   embeddings (MTS-JEPA style).
  4. ``sigreg`` -- LeJEPA / Sketched Isotropic Gaussian Regularization: push the
                   embedding distribution towards an isotropic Gaussian using
                   sliced univariate normality tests, which removes the need for a
                   second (target) network altogether.

Every arm exposes the same interface::

    reg = build_regularizer(loss_cfg, embed_dim)
    penalty, stats = reg(embeddings)          # embeddings: (B, D) or (B, N, D)
    target = reg.transform_target(target)     # identity except for the VQ arm
"""

from __future__ import annotations

import math
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


def _flatten(z: torch.Tensor) -> torch.Tensor:
    """(B, D) or (B, N, D) -> (B*N, D) in float32 (penalties are scale sensitive)."""
    if z.dim() == 3:
        z = z.reshape(-1, z.size(-1))
    return z.float()


class BaseRegularizer(nn.Module):
    name = "none"

    def forward(self, z: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
        return z.new_zeros(()), {}

    def transform_target(self, h: torch.Tensor) -> torch.Tensor:
        return h


class EMAOnly(BaseRegularizer):
    """Arm 1 -- no explicit penalty; collapse avoidance relies on EMA + stop-grad."""
    name = "ema"


class VICReg(BaseRegularizer):
    """Arm 2 -- variance/covariance regularisation.

        L_reg = w_v * mean_k max(0, gamma - std(z_k))
              + w_c * (1/d) * sum_{i != j} Cov(z)_ij^2
    """

    name = "vicreg"

    def __init__(self, gamma: float = 1.0, var_weight: float = 1.0,
                 cov_weight: float = 0.04):
        super().__init__()
        self.gamma = gamma
        self.var_weight = var_weight
        self.cov_weight = cov_weight

    def forward(self, z: torch.Tensor):
        z = _flatten(z)
        n, d = z.shape
        if n < 2:
            return z.new_zeros(()), {}
        z = z - z.mean(dim=0, keepdim=True)

        std = torch.sqrt(z.var(dim=0, unbiased=False) + 1e-4)
        var_loss = F.relu(self.gamma - std).mean()

        cov = (z.T @ z) / (n - 1)
        off_diag = cov - torch.diag_embed(torch.diagonal(cov))
        cov_loss = off_diag.pow(2).sum() / d

        loss = self.var_weight * var_loss + self.cov_weight * cov_loss
        return loss, {"vicreg_var": float(var_loss.detach()),
                      "vicreg_cov": float(cov_loss.detach()),
                      "vicreg_std_mean": float(std.mean().detach())}


class SoftVQ(BaseRegularizer):
    """Arm 3 -- differentiable soft codebook bottleneck on the *target* embeddings.

    Targets are replaced by a temperature-softmaxed convex combination of codebook
    entries before the prediction loss is taken. A constant target then requires a
    constant code assignment, which the batch-level entropy bonus penalises, so the
    trivial solution stops being free.
    """

    name = "vq"

    def __init__(self, embed_dim: int, codebook_size: int = 512,
                 temperature: float = 0.1, entropy_weight: float = 0.1,
                 commit_weight: float = 1.0):
        super().__init__()
        self.temperature = temperature
        self.entropy_weight = entropy_weight
        self.commit_weight = commit_weight
        self.codebook_size = codebook_size
        self.codebook = nn.Parameter(F.normalize(torch.randn(codebook_size, embed_dim),
                                                 dim=-1))
        # v0.2 FIX: keep *every* assignment made during a step. v0.1 kept only the
        # last one, so when both L_mask and L_fwd transformed targets, the masked
        # branch's codes were never regularised.
        self._assigns = []
        # v0.2 FIX: data-dependent codebook initialisation. With random codes in
        # d=384, target-code cosines are ~N(0, 0.05): every soft assignment is
        # near-uniform, the quantised target is the (rescaled) codebook mean for every
        # input, i.e. a constant target -- the mechanism behind the v0.1 VQ collapse
        # (loss -> 1 % of its start in 200 steps). Codes are seeded from real targets
        # on the first training call.
        self.register_buffer("initialized", torch.zeros((), dtype=torch.long))

    def _assign(self, h: torch.Tensor) -> torch.Tensor:
        # Cosine logits rather than raw euclidean distance: with d=384 embeddings the
        # squared distances are O(d), so a temperature of 0.5 on them saturates the
        # softmax and kills the gradient that is supposed to spread codebook usage.
        flat = F.normalize(h.reshape(-1, h.size(-1)).float(), dim=-1)
        book = F.normalize(self.codebook.float(), dim=-1)
        logits = (flat @ book.T) / max(self.temperature, 1e-4)
        return F.softmax(logits, dim=-1)

    @torch.no_grad()
    def _init_from(self, h: torch.Tensor):
        flat = h.reshape(-1, h.size(-1)).float()
        idx = torch.randint(0, flat.size(0), (self.codebook_size,), device=flat.device)
        codes = F.normalize(flat[idx] + 0.01 * torch.randn_like(flat[idx]), dim=-1)
        self.codebook.data.copy_(codes.to(self.codebook.dtype))
        self.initialized.fill_(1)

    def transform_target(self, h: torch.Tensor) -> torch.Tensor:
        if self.training and int(self.initialized) == 0:
            self._init_from(h)
        assign = self._assign(h)
        self._assigns.append(assign)        # kept live: the entropy term trains the codebook
        quant = assign @ self.codebook.to(assign.dtype)
        # keep the target's original scale so the prediction loss stays comparable
        # across arms (the codebook only reshapes *where* targets can live)
        quant = quant * (h.reshape(-1, h.size(-1)).float().norm(dim=-1, keepdim=True)
                         / (quant.norm(dim=-1, keepdim=True) + 1e-6))
        # Detach the *target*: if the prediction loss could reach the codebook, the
        # cheapest way to cut it is to move the targets towards whatever the predictor
        # already outputs -- a target-chases-prediction shortcut that is collapse by
        # another name. The codebook is shaped only by the spread/commitment penalty
        # in `forward`, which still sees the live assignments in `_assigns`.
        return quant.reshape(h.shape).to(h.dtype).detach()

    def forward(self, z: torch.Tensor):
        if not self._assigns:
            return z.new_zeros(()), {}
        assign = torch.cat(self._assigns, dim=0)
        log_k = math.log(self.codebook_size)

        # (a) usage spread: penalise *low* marginal entropy. The previous
        # `ent_sample - ent_batch` form is exactly 0 both at full collapse (every
        # sample on one code: 0 - 0) and cannot punish it, which is the failure this
        # arm exists to prevent. `log K - ent_batch` is >= 0 and maximal at collapse.
        marginal = assign.mean(dim=0)
        ent_batch = -(marginal * torch.log(marginal + 1e-8)).sum()
        spread = (log_k - ent_batch) / log_k

        # (b) commitment: each individual target should still pick a few codes
        ent_sample = -(assign * torch.log(assign + 1e-8)).sum(dim=-1).mean()
        commit = ent_sample / log_k

        # v0.2 FIX: information-maximisation form, spread (high marginal entropy) +
        # commitment (low per-sample entropy) with equal weight. With v0.1's 0.1
        # weight, a *uniform* per-sample assignment -- which makes every target the
        # same codebook average, i.e. a constant target -- had near-zero penalty.
        loss = self.entropy_weight * (spread + self.commit_weight * commit)
        perplexity = float(torch.exp(ent_batch).detach())
        self._assigns = []
        return loss, {"vq_perplexity": perplexity,
                      "vq_spread": float(spread.detach()),
                      "vq_ent_sample": float(ent_sample.detach())}


class SIGReg(BaseRegularizer):
    """Arm 4 -- Sketched Isotropic Gaussian Regularization (LeJEPA).

    Sketch the embedding distribution onto ``n_slices`` random unit directions and
    penalise the deviation of each 1-D projection from a standard normal using an
    Epps--Pulley style characteristic-function statistic:

        T(u) = sum_t w_t * | (1/n) sum_i exp(i t u_i) - exp(-t^2/2) |^2

    Zero only when every projection is N(0, 1); by Cramer--Wold that pins the joint
    distribution to an isotropic Gaussian, which is full rank by construction. This
    is the arm that lets us drop the EMA target network entirely.
    """

    name = "sigreg"

    def __init__(self, embed_dim: int, n_slices: int = 64, n_freqs: int = 16):
        super().__init__()
        self.n_slices = n_slices
        self.embed_dim = embed_dim
        freqs = torch.linspace(0.25, 3.0, n_freqs)
        self.register_buffer("freqs", freqs, persistent=False)

    def forward(self, z: torch.Tensor):
        z = _flatten(z)
        n, d = z.shape
        if n < 4:
            return z.new_zeros(()), {}

        # sketch: fresh random directions every step (this is the "sketched" part)
        dirs = torch.randn(d, self.n_slices, device=z.device, dtype=z.dtype)
        dirs = F.normalize(dirs, dim=0)
        proj = z @ dirs                                     # (n, n_slices)

        # standardise each projection so the test targets shape, not scale/offset,
        # then separately penalise the scale so variance cannot shrink to zero
        mu = proj.mean(dim=0, keepdim=True)
        sd = proj.std(dim=0, keepdim=True) + 1e-5
        u = (proj - mu) / sd

        t = self.freqs.to(z.dtype).view(1, 1, -1)           # (1, 1, F)
        arg = u.unsqueeze(-1) * t                           # (n, S, F)
        emp_re = torch.cos(arg).mean(dim=0)                 # (S, F)
        emp_im = torch.sin(arg).mean(dim=0)
        target_re = torch.exp(-0.5 * self.freqs.to(z.dtype) ** 2).view(1, -1)
        stat = ((emp_re - target_re) ** 2 + emp_im ** 2).mean()

        # scale term: log-variance should sit at 0 (unit variance)
        scale_pen = (torch.log(sd.squeeze(0) ** 2 + 1e-8) ** 2).mean()

        loss = stat + 0.1 * scale_pen
        return loss, {"sigreg_stat": float(stat.detach()),
                      "sigreg_scale": float(scale_pen.detach())}


def build_regularizer(loss_cfg, embed_dim: int) -> BaseRegularizer:
    arm = loss_cfg.anti_collapse
    if arm == "ema":
        return EMAOnly()
    if arm == "vicreg":
        return VICReg(gamma=loss_cfg.vicreg_var_gamma,
                      var_weight=loss_cfg.vicreg_var_weight,
                      cov_weight=loss_cfg.vicreg_cov_weight)
    if arm == "vq":
        return SoftVQ(embed_dim=embed_dim,
                      codebook_size=loss_cfg.vq_codebook_size,
                      temperature=loss_cfg.vq_temperature,
                      entropy_weight=loss_cfg.vq_entropy_weight,
                      commit_weight=getattr(loss_cfg, "vq_commit_weight", 1.0))
    if arm == "sigreg":
        return SIGReg(embed_dim=embed_dim,
                      n_slices=loss_cfg.sigreg_n_slices,
                      n_freqs=loss_cfg.sigreg_n_freqs)
    raise ValueError(f"unknown anti_collapse arm: {arm!r} "
                     f"(expected one of: ema, vicreg, vq, sigreg)")

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/diagnostics.py
"""Representation-collapse diagnostics (RQ3, proposal Section 5.5).

Logged every epoch across all four anti-collapse arms:

  * **effective rank** via singular-value entropy (RankMe, Garrido et al. 2023):
        p_i = sigma_i / sum_j sigma_j ,   erank = exp(-sum_i p_i log p_i)
    A collapsed representation concentrates all spectral mass on one direction and
    drops to erank ~ 1; a healthy one sits at a decent fraction of ``d``.
  * **per-dimension variance** (mean / min / fraction of near-dead dimensions).
  * **loss-curve shape** -- ER-JEPA (arXiv:2607.01145) reports a specific collapse
    signature where the loss falls to ~0 and later partially recovers. We detect it
    explicitly rather than eyeballing the curve.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence

import numpy as np
import torch


#: v0.2 collapse thresholds (on a >= 512-sample probe batch; ratios are normalised by
#: min(n_samples, dim)). See CollapseMonitor.summary.
COLLAPSE_COS = 0.90
COLLAPSE_RANK = 0.25
DIM_COLLAPSE_RANK = 0.10


@torch.no_grad()
def effective_rank(z: torch.Tensor, eps: float = 1e-12, center: bool = False) -> float:
    """Singular-value-entropy effective rank (RankMe) of an (N, D) embedding matrix.

    ``center=False`` (the default, and what RankMe specifies) is what detects
    *complete* collapse: if every sample maps to the same vector, the matrix is
    rank 1 and this returns ~1. Mean-centring first would subtract exactly that
    constant direction away and report the residual noise as full rank, hiding the
    failure we are trying to measure.

    ``center=True`` gives the covariance spectrum instead, which is the right lens
    for *dimensional* collapse (some directions dying while others survive). Both
    are logged; see ``embedding_report``.
    """
    if z.dim() == 3:
        z = z.reshape(-1, z.size(-1))
    z = z.detach().float()
    if z.size(0) < 2:
        return float("nan")
    if center:
        z = z - z.mean(dim=0, keepdim=True)
    try:
        sv = torch.linalg.svdvals(z)
    except Exception:                                  # pragma: no cover - rare LAPACK fail
        return float("nan")
    total = sv.sum()
    if float(total) <= eps:
        return 1.0
    p = sv / total
    entropy = -(p * torch.log(p + eps)).sum()
    return float(torch.exp(entropy))


@torch.no_grad()
def variance_stats(z: torch.Tensor, dead_threshold: float = 1e-3) -> Dict[str, float]:
    if z.dim() == 3:
        z = z.reshape(-1, z.size(-1))
    z = z.detach().float()
    if z.size(0) < 2:
        return {"var_mean": float("nan"), "var_min": float("nan"),
                "std_mean": float("nan"), "dead_dim_frac": float("nan")}
    var = z.var(dim=0, unbiased=False)
    std = var.sqrt()
    return {
        "var_mean": float(var.mean()),
        "var_min": float(var.min()),
        "std_mean": float(std.mean()),
        "dead_dim_frac": float((std < dead_threshold).float().mean()),
    }


@torch.no_grad()
def pairwise_cosine(z: torch.Tensor, max_n: int = 512) -> float:
    """Mean off-diagonal cosine similarity. -> 1.0 means every sample maps to the
    same direction (the textbook complete-collapse signature)."""
    if z.dim() == 3:
        z = z.mean(dim=1)
    z = z.detach().float()[:max_n]
    if z.size(0) < 2:
        return float("nan")
    zn = torch.nn.functional.normalize(z, dim=-1)
    sim = zn @ zn.T
    n = sim.size(0)
    off = (sim.sum() - torch.diagonal(sim).sum()) / (n * (n - 1))
    return float(off)


@torch.no_grad()
def embedding_report(z: torch.Tensor) -> Dict[str, float]:
    """All RQ3 diagnostics for one batch of embeddings.

    The ratios are normalised by ``min(n_samples, d)``, not by ``d``: the rank of an
    (N, D) matrix cannot exceed ``min(N, D)``, so dividing by ``d`` alone makes every
    arm look collapsed whenever the probe batch is smaller than the embedding width.
    Use a probe batch of at least ``embed_dim`` samples for a well-conditioned estimate.
    """
    flat = z.reshape(-1, z.size(-1)) if z.dim() == 3 else z
    n, d = flat.shape
    denom = max(min(n, d), 1)

    out = {
        "effective_rank": effective_rank(z, center=False),      # complete collapse
        "cov_effective_rank": effective_rank(z, center=True),   # dimensional collapse
        "cos_sim": pairwise_cosine(z),
        "rank_denominator": float(denom),
    }
    out.update(variance_stats(z))
    out["erank_ratio"] = out["effective_rank"] / denom
    out["cov_erank_ratio"] = out["cov_effective_rank"] / denom
    return out


def detect_collapse_signature(losses: Sequence[float],
                              near_zero_frac: float = 0.05,
                              recovery_frac: float = 0.5) -> Dict[str, object]:
    """Detect ER-JEPA's reported 'loss -> ~0 then partial recovery' signature.

    Args:
        losses: per-epoch training loss.
        near_zero_frac: a loss below this fraction of the initial loss counts as
            "collapsed to ~0".
        recovery_frac: a subsequent rise above this fraction of the initial loss
            counts as "recovered".
    """
    losses = [float(x) for x in losses]
    out: Dict[str, object] = {"collapsed_to_zero": False, "recovered": False,
                              "collapse_epoch": None, "recovery_epoch": None}
    if len(losses) < 3:
        return out
    l0 = max(abs(losses[0]), 1e-8)
    for i, l in enumerate(losses):
        if l < near_zero_frac * l0:
            out["collapsed_to_zero"] = True
            out["collapse_epoch"] = i
            for j in range(i + 1, len(losses)):
                if losses[j] > recovery_frac * l0:
                    out["recovered"] = True
                    out["recovery_epoch"] = j
                    break
            break
    return out


@dataclass
class CollapseMonitor:
    """Accumulates per-epoch diagnostics for one training arm."""

    arm: str = "ema"
    history: List[Dict[str, float]] = field(default_factory=list)

    def log(self, epoch: int, loss: float, z: Optional[torch.Tensor] = None,
            extra: Optional[Dict[str, float]] = None) -> Dict[str, float]:
        rec: Dict[str, float] = {"epoch": float(epoch), "loss": float(loss)}
        if z is not None:
            rec.update(embedding_report(z))
        if extra:
            rec.update({k: float(v) for k, v in extra.items()})
        self.history.append(rec)
        return rec

    @property
    def losses(self) -> List[float]:
        return [r["loss"] for r in self.history]

    def summary(self) -> Dict[str, object]:
        if not self.history:
            return {"arm": self.arm}
        last = self.history[-1]
        first = self.history[0]
        out: Dict[str, object] = {
            "arm": self.arm,
            "final_loss": last.get("loss"),
            "final_effective_rank": last.get("effective_rank"),
            "final_erank_ratio": last.get("erank_ratio"),
            "final_cov_effective_rank": last.get("cov_effective_rank"),
            "final_cov_erank_ratio": last.get("cov_erank_ratio"),
            "final_std_mean": last.get("std_mean"),
            "final_cos_sim": last.get("cos_sim"),
            # Complete collapse = the *centred* spectrum is degenerate too, i.e. there
            # is essentially nothing left for a downstream head. A near-1 cosine
            # similarity on its own is NOT sufficient evidence: mean-pooling 400
            # LayerNormed tokens always leaves a large shared component, and a
            # representation can sit at cos ~ 0.9999 and still linearly separate
            # classes from the residual. That case is reported as `offset_dominated`.
            "complete_collapse": bool(last.get("cov_erank_ratio", 1.0) < 0.02
                                      and last.get("std_mean", 1.0) < 1e-5),
            # v0.2 FIX: thresholds. v0.1 flagged nothing in its own RQ3 run, even the
            # VQ arm whose loss fell to 1% of its start with mean pairwise cosine 0.94
            # and per-dim std 0.07. Trained healthy encoders here sit at cos ~0.1-0.2,
            # so "most samples point the same way AND few directions" is collapse.
            "representation_collapse": bool(
                (last.get("cos_sim", 0.0) > COLLAPSE_COS
                 and last.get("cov_erank_ratio", 1.0) < COLLAPSE_RANK)
                or last.get("std_mean", 1.0) < 1e-3),
            "dimensional_collapse": bool(last.get("cov_erank_ratio", 1.0) < DIM_COLLAPSE_RANK),
            "offset_dominated": bool(last.get("cos_sim", 0.0) > 0.99),
            "erank_delta": (last.get("effective_rank", float("nan"))
                            - first.get("effective_rank", float("nan"))),
        }
        out.update(detect_collapse_signature(self.losses))
        out["verdict"] = ("collapsed" if (out["representation_collapse"]
                                          or out["complete_collapse"]
                                          or out.get("collapsed_to_zero"))
                          else "dimensional" if out["dimensional_collapse"]
                          else "healthy")
        # a shrinking effective rank *and* a rising cosine similarity is the
        # dimensional-collapse pattern H3 predicts for the EMA-only arm
        eranks = [r.get("effective_rank", float("nan")) for r in self.history]
        eranks = [e for e in eranks if not np.isnan(e)]
        out["erank_monotone_decreasing"] = bool(
            len(eranks) >= 3 and all(b <= a + 1e-6 for a, b in zip(eranks, eranks[1:])))
        return out

    def to_frame(self):
        import pandas as pd
        return pd.DataFrame(self.history)

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/engine.py
"""Pretraining loop: cosine LR + warmup, EMA target schedule, AMP, collapse logging."""

from __future__ import annotations

import json
import math
import os
import time
from dataclasses import dataclass
from typing import Dict, Iterable, Iterator, List, Optional, Sequence

import numpy as np
import torch
from torch.utils.data import DataLoader

from .config import ExperimentConfig, OptimConfig
from .diagnostics import CollapseMonitor


def set_seed(seed: int, deterministic: Optional[bool] = None):
    """Seed python/numpy/torch. ``deterministic=True`` (v0.2) also pins cuDNN.

    Note: full bit-for-bit determinism on GPU additionally needs
    ``CUBLAS_WORKSPACE_CONFIG=:4096:8`` set before CUDA initialises (the notebook does
    this in its first cell); SDPA's memory-efficient backward is deterministic.
    """
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic is not None:
        torch.backends.cudnn.deterministic = bool(deterministic)
        torch.backends.cudnn.benchmark = not bool(deterministic)


def pick_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def cosine_lr(step: int, total: int, warmup: int, base_lr: float,
              min_lr: float) -> float:
    if total <= 0:
        return base_lr
    if warmup > 0 and step < warmup:
        return min_lr + (base_lr - min_lr) * step / max(warmup, 1)
    p = (step - warmup) / max(total - warmup, 1)
    p = min(max(p, 0.0), 1.0)
    return min_lr + 0.5 * (base_lr - min_lr) * (1.0 + math.cos(math.pi * p))


def ema_momentum(step: int, total: int, start: float, end: float) -> float:
    if total <= 0:
        return start
    return start + (end - start) * min(step / total, 1.0)


def param_groups(model: torch.nn.Module, weight_decay: float) -> List[dict]:
    """No weight decay on biases, norms and other 1-D parameters (I-JEPA convention)."""
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if (p.ndim <= 1 or n.endswith(".bias")) else decay).append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


class InterleavedLoader:
    """Round-robin over several loaders, keeping each *batch* homogeneous.

    Short 10 s clips and long-recording forward pairs need different loss branches,
    so mixing them inside one batch would mean two forward passes per step. Instead
    we alternate whole batches in proportion to ``weights``.
    """

    def __init__(self, loaders: Sequence[DataLoader], weights: Optional[Sequence[float]] = None,
                 steps_per_epoch: Optional[int] = None):
        self.loaders = [l for l in loaders if l is not None and len(l) > 0]
        if not self.loaders:
            raise ValueError("InterleavedLoader needs at least one non-empty loader")
        w = list(weights) if weights else [1.0] * len(self.loaders)
        w = w[:len(self.loaders)]
        total = sum(w) or 1.0
        self.weights = [x / total for x in w]
        self.steps = steps_per_epoch or sum(len(l) for l in self.loaders)
        self._epoch = 0

    def reset(self):
        """v0.2: restart the interleaving schedule.

        The schedule RNG is seeded by an epoch counter that v0.1 never reset, so the
        arms trained one after another on the same loader object saw *different*
        short/long batch orders (masked_only used schedules 0-29, forward_only 30-59,
        ...). ``pretrain`` now calls this, so every arm sees the same schedule.
        """
        self._epoch = 0

    def __len__(self) -> int:
        return self.steps

    def __iter__(self) -> Iterator[dict]:
        iters = [iter(l) for l in self.loaders]
        # vary the interleaving pattern across epochs (each sub-loader still shuffles
        # internally, but a fixed pattern would pair the same short/long batches up)
        rng = np.random.default_rng(self._epoch)
        self._epoch += 1
        for _ in range(self.steps):
            k = int(rng.choice(len(iters), p=self.weights))
            try:
                yield next(iters[k])
            except StopIteration:
                iters[k] = iter(self.loaders[k])
                yield next(iters[k])


def _to_device(batch: Dict[str, torch.Tensor], device) -> Dict[str, torch.Tensor]:
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v)
            for k, v in batch.items()}


@dataclass
class TrainResult:
    monitor: CollapseMonitor
    history: List[Dict[str, float]]
    seconds: float
    checkpoint: Optional[str] = None
    collapsed: bool = False
    collapse_step: Optional[int] = None
    collapse_reason: str = ""
    stopped_early: bool = False


@torch.no_grad()
def check_collapse(model, probe_batch, device, loss_now: float, loss_ref: float,
                   cfg: OptimConfig) -> Optional[str]:
    """Return a reason string if the run has collapsed, else None.

    **The loss is the primary signal.** Targets are LayerNormed encoder outputs, so a
    latent-prediction error at ~2 % of where it started is the constant solution: the
    encoder emits one vector, the predictor memorises it, and the loss goes to zero
    carrying no information. Measured: an EMA-only forward-only run went 0.39 -> 2e-5,
    while the same setup with a variance floor sat at 0.12.

    Embedding statistics are deliberately *not* used as the main gate. On this
    architecture the pooled representation is offset-dominated whether or not it has
    collapsed -- healthy and collapsed runs both sit at cosine similarity ~0.9999, and
    their covariance effective ranks (0.29 vs 0.20 of full) overlap. Gating on those
    would fire on healthy models. The embedding trigger below is kept only for the
    unambiguous case where the representation is numerically dead, which the loss
    signal can miss if training never had a meaningful starting reference.
    """
    if loss_ref > 0 and loss_now < cfg.collapse_loss_frac * loss_ref:
        return (f"loss {loss_now:.6f} fell to "
                f"{100.0 * loss_now / loss_ref:.2f}% of its initial value "
                f"({loss_ref:.6f}) - trivial constant solution")

    if probe_batch is None:
        return None
    z = _probe_embeddings(model, probe_batch, device)
    if z is None or z.size(0) < 4:
        return None
    from .diagnostics import embedding_report
    rep = embedding_report(z)
    # Numerically dead: one direction, no spread. Uses the *absolute* effective rank
    # rather than the ratio -- the ratio is normalised by min(probe_batch, dim), so a
    # ratio threshold silently changes meaning with the probe batch size.
    if (rep["cos_sim"] > 0.99999 and rep["std_mean"] < 1e-5
            and rep["effective_rank"] < 1.5):
        return (f"embeddings numerically dead: cos_sim={rep['cos_sim']:.6f}, "
                f"std={rep['std_mean']:.2e}, erank={rep['effective_rank']:.2f}")
    return None


@torch.no_grad()
def estimate_runtime(model, loader, cfg: ExperimentConfig, device=None,
                     n_steps: int = 10, n_models: int = 1) -> Dict[str, float]:
    """Time a few real steps and project the total, before committing a GPU session.

    Reports seconds/step, per-model and total projections. A Kaggle session is capped
    at ~12 h, so a projection past that is the signal to cut `STAGES`,
    `optim.epochs`, or `optim.max_steps_per_epoch` *before* starting rather than
    discovering it eight hours in.
    """
    device = device or pick_device()
    model.to(device).train()
    opt = torch.optim.AdamW(param_groups(model, cfg.optim.weight_decay), lr=cfg.optim.lr)
    use_amp = bool(cfg.optim.amp and device.type == "cuda")

    it = iter(loader)
    timings: List[float] = []
    for i in range(n_steps + 2):                    # first two steps warm up kernels
        try:
            batch = next(it)
        except StopIteration:
            it = iter(loader)
            batch = next(it)
        batch = _to_device(batch, device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        with torch.enable_grad():
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                has_future = bool(batch.get("has_future", torch.tensor(False)).any())
                if hasattr(model, "forward_predictor"):
                    out = model(batch["ctx"],
                                future_window=batch.get("future") if has_future else None,
                                horizon_s=batch.get("horizon_s") if has_future else None,
                                l2_future=batch.get("l2_future"),
                                l2_horizon_s=batch.get("l2_horizon_s"))
                else:
                    out = model(batch["ctx"])
            out["loss"].backward()
            opt.step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        if i >= 2:
            timings.append(time.time() - t0)

    s_per_step = float(np.median(timings)) if timings else float("nan")
    steps_per_epoch = max(1, len(loader))
    # v0.2 FIX: honour the step cap (v0.1 projected the uncapped 1818 steps/epoch
    # even after applying max_steps_per_epoch, so the "fixed" estimate never changed)
    if cfg.optim.max_steps_per_epoch:
        steps_per_epoch = min(steps_per_epoch, int(cfg.optim.max_steps_per_epoch))
    per_model = s_per_step * steps_per_epoch * cfg.optim.epochs
    if hasattr(loader, "reset"):
        loader.reset()
    return {"s_per_step": s_per_step, "steps_per_epoch": float(steps_per_epoch),
            "steps_per_model": float(steps_per_epoch * cfg.optim.epochs),
            "hours_per_model": per_model / 3600.0,
            "hours_total": per_model * n_models / 3600.0,
            "n_models": float(n_models)}


def budget_steps_per_epoch(s_per_step: float, epochs: int, n_models: int,
                           budget_hours: float = 11.0, minimum: int = 25) -> int:
    """Largest steps/epoch that keeps `n_models` runs inside `budget_hours`."""
    if s_per_step <= 0 or epochs <= 0 or n_models <= 0:
        return minimum
    total_steps = (budget_hours * 3600.0) / s_per_step
    return max(minimum, int(total_steps / (epochs * n_models)))


def report_runtime(est: Dict[str, float], budget_hours: float = 11.0) -> str:
    lines = [
        f"  {est['s_per_step']:.3f} s/step x {est['steps_per_epoch']:.0f} steps/epoch "
        f"x epochs = {est['steps_per_model']:.0f} steps/model",
        f"  ~{est['hours_per_model']:.2f} h per model, "
        f"~{est['hours_total']:.2f} h for {est['n_models']:.0f} model(s)",
    ]
    if est["hours_total"] > budget_hours:
        over = est["hours_total"] / budget_hours
        lines.append(
            f"  >>> {over:.1f}x over a ~{budget_hours:.0f} h session. This will not "
            f"finish; reduce the budget before starting, not eight hours in.")
    return "\n".join(lines)


def pretrain(model: torch.nn.Module,
             loader,
             cfg: ExperimentConfig,
             device: Optional[torch.device] = None,
             arm_name: str = "ema",
             log_every: int = 50,
             probe_batch: Optional[torch.Tensor] = None,
             save_path: Optional[str] = None,
             verbose: bool = True) -> TrainResult:
    """Train any model exposing ``forward(...) -> {'loss': ...}`` and ``update_target``."""
    device = device or pick_device()
    opt_cfg: OptimConfig = cfg.optim
    set_seed(opt_cfg.seed, getattr(opt_cfg, "deterministic", None))
    if hasattr(loader, "reset"):
        loader.reset()                       # v0.2: same batch schedule for every arm
    model.to(device)

    steps_per_epoch = max(1, len(loader))
    if opt_cfg.max_steps_per_epoch:
        steps_per_epoch = min(steps_per_epoch, opt_cfg.max_steps_per_epoch)
    total_steps = steps_per_epoch * opt_cfg.epochs
    warmup_steps = steps_per_epoch * opt_cfg.warmup_epochs

    optimizer = torch.optim.AdamW(param_groups(model, opt_cfg.weight_decay),
                                  lr=opt_cfg.lr, betas=(0.9, 0.95))
    use_amp = bool(opt_cfg.amp and device.type == "cuda")
    try:                                             # torch >= 2.3
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError):              # pragma: no cover - older torch
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    monitor = CollapseMonitor(arm=arm_name)
    history: List[Dict[str, float]] = []
    step = 0
    lr = opt_cfg.lr
    t_start = time.time()

    loss_ref: Optional[float] = None          # smoothed loss over the first few steps
    collapsed = False
    collapse_step: Optional[int] = None
    collapse_reason = ""
    guard_on = (opt_cfg.on_collapse in ("stop", "warn")
                and opt_cfg.collapse_check_every > 0)

    for epoch in range(opt_cfg.epochs):
        if collapsed and opt_cfg.on_collapse == "stop":
            break
        model.train()
        agg: Dict[str, float] = {}
        n_batches = 0
        t_epoch = time.time()

        for batch in loader:
            lr = cosine_lr(step, total_steps, warmup_steps, opt_cfg.lr, opt_cfg.min_lr)
            for g in optimizer.param_groups:
                g["lr"] = lr

            batch = _to_device(batch, device)
            ctx = batch["ctx"]
            has_future = bool(batch.get("has_future", torch.tensor(False)).any())
            future = batch.get("future") if has_future else None
            horizon = batch.get("horizon_s") if has_future else None
            l2_future = batch.get("l2_future")
            l2_h = batch.get("l2_horizon_s")

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                if hasattr(model, "forward_predictor"):
                    out = model(ctx, future_window=future, horizon_s=horizon,
                                l2_future=l2_future, l2_horizon_s=l2_h)
                else:                                    # ST-MEM style baseline
                    out = model(ctx)
                loss = out["loss"]

            if not torch.isfinite(loss):
                if verbose:
                    print(f"  [warn] non-finite loss at step {step}, skipping batch")
                step += 1
                continue

            scaler.scale(loss).backward()
            if opt_cfg.grad_clip:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), opt_cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()

            m = ema_momentum(step, total_steps, opt_cfg.ema_start, opt_cfg.ema_end)
            if hasattr(model, "update_target"):
                model.update_target(m)

            for k, v in out.items():
                if k in ("z_ctx",):
                    continue
                val = float(v.detach()) if torch.is_tensor(v) else float(v)
                agg[k] = agg.get(k, 0.0) + val
            n_batches += 1
            step += 1

            # reference loss from the first few steps, once warmup has settled
            if loss_ref is None and step >= 5:
                loss_ref = float(loss.detach())
            elif step < 5:
                loss_ref = None

            if verbose and log_every and step % log_every == 0:
                print(f"  step {step:6d}/{total_steps}  loss={float(loss):.4f}  "
                      f"lr={lr:.2e}  ema={m:.5f}")

            # ---- collapse guard ----------------------------------------------------
            if (guard_on and not collapsed and loss_ref is not None
                    and step % opt_cfg.collapse_check_every == 0):
                reason = check_collapse(model, probe_batch, device, float(loss),
                                        loss_ref, opt_cfg)
                if reason:
                    collapsed = True
                    collapse_step = step
                    collapse_reason = reason
                    print(f"\n  *** COLLAPSE DETECTED at step {step} [{arm_name}] ***")
                    print(f"      {reason}")
                    if opt_cfg.on_collapse == "stop":
                        print(f"      Stopping this run. A collapsed encoder cannot "
                              f"support any downstream probe, so the remaining "
                              f"{total_steps - step} steps would be wasted.")
                        print(f"      If the objective includes L_fwd, it has a trivial "
                              f"constant solution unless L_reg constrains the *pooled* "
                              f"representation: set loss.anti_collapse='vicreg' with "
                              f"loss.reg_target='pooled'.")
                        print(f"      To observe collapse instead of aborting (RQ3), "
                              f"set optim.on_collapse='warn'.\n")
                        break
                    print("      Continuing (on_collapse='warn').\n")

            if opt_cfg.max_steps_per_epoch and n_batches >= opt_cfg.max_steps_per_epoch:
                break

        means = {k: v / max(n_batches, 1) for k, v in agg.items()}
        means["epoch"] = epoch
        means["lr"] = lr
        means["seconds"] = time.time() - t_epoch
        history.append(means)

        z = _probe_embeddings(model, probe_batch, device) if probe_batch is not None else None
        rec = monitor.log(epoch, means.get("loss", float("nan")), z,
                          extra={k: v for k, v in means.items()
                                 if k.startswith(("loss_", "vicreg", "vq_", "sigreg"))})
        if verbose:
            er = rec.get("effective_rank", float("nan"))
            print(f"epoch {epoch:03d}/{opt_cfg.epochs}  loss={means.get('loss', 0):.4f}  "
                  f"fwd={means.get('loss_fwd', 0):.4f}  mask={means.get('loss_mask', 0):.4f}  "
                  f"reg={means.get('loss_reg', 0):.4f}  erank={er:.1f}  "
                  f"{means['seconds']:.1f}s")

    ckpt = None
    if save_path:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        torch.save({"model": model.state_dict(),
                    "config": cfg.to_dict(),
                    "arm": arm_name,
                    "history": history}, save_path)
        ckpt = save_path
        with open(os.path.splitext(save_path)[0] + "_diagnostics.json", "w") as fh:
            json.dump({"history": monitor.history, "summary": monitor.summary(),
                       "train_history": history, "arm": arm_name,
                       "seconds": time.time() - t_start,
                       "collapsed": collapsed, "collapse_step": collapse_step,
                       "collapse_reason": collapse_reason},
                      fh, indent=2, default=float)

    return TrainResult(monitor=monitor, history=history,
                       seconds=time.time() - t_start, checkpoint=ckpt,
                       collapsed=collapsed, collapse_step=collapse_step,
                       collapse_reason=collapse_reason,
                       stopped_early=bool(collapsed and opt_cfg.on_collapse == "stop"))


@torch.no_grad()
def _probe_embeddings(model, probe_batch: torch.Tensor, device) -> Optional[torch.Tensor]:
    """Encoder outputs on a fixed held-out batch -- the substrate for RQ3 diagnostics."""
    if probe_batch is None:
        return None
    was_training = model.training
    model.eval()
    x = probe_batch.to(device)
    if x.dim() == 4:
        x = x[:, -1]
    try:
        z = model.representation(x)
    except Exception:
        z = None
    if was_training:
        model.train()
    return None if z is None else z.detach().float().cpu()


def load_train_result(checkpoint_path: str) -> TrainResult:
    """v0.2: rebuild a ``TrainResult`` from the ``*_diagnostics.json`` written next to
    a checkpoint, so a later Kaggle session can reuse a model trained in an earlier
    one (multi-session runs) without losing its collapse diagnostics."""
    diag = os.path.splitext(checkpoint_path)[0] + "_diagnostics.json"
    with open(diag) as fh:
        d = json.load(fh)
    mon = CollapseMonitor(arm=d.get("arm", "?"))
    mon.history = list(d.get("history", []))
    return TrainResult(monitor=mon, history=list(d.get("train_history", [])),
                       seconds=float(d.get("seconds", 0.0)),
                       checkpoint=checkpoint_path,
                       collapsed=bool(d.get("collapsed", False)),
                       collapse_step=d.get("collapse_step"),
                       collapse_reason=d.get("collapse_reason", ""),
                       stopped_early=False)


def free_cuda(*models):
    """Move models to CPU and release cached GPU memory (between stages)."""
    import gc
    for m in models:
        try:
            m.to("cpu")
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/checkpoint.py
"""Loading the package and pretrained weights from a Kaggle Model input.

Kaggle mounts a Model at::

    /kaggle/input/models/<owner>/<model>/<framework>/<variation>/<version>/

which can hold the ``cardio_jepa`` package source, saved checkpoints, or both.
This module covers the two things a notebook needs:

  * ``find_package_root``   -- locate an importable ``cardio_jepa`` package so the
    notebook can ``sys.path`` to it instead of writing the source out itself.
  * ``load_weights``        -- warm-start a model from a previous run's checkpoint,
    tolerating shape and key mismatches so a run at a different width still loads
    whatever genuinely matches.

The bootstrap half of ``find_package_root`` is duplicated inline in the notebook's
first cell -- it has to be, because it runs *before* the package is importable.
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

DEFAULT_MODEL_ROOTS = ("/kaggle/input/models", "/kaggle/input")
CHECKPOINT_EXTS = (".pth", ".pt", ".ckpt", ".bin", ".safetensors")


def find_package_root(search_roots: Sequence[str] = DEFAULT_MODEL_ROOTS,
                      max_depth: int = 8) -> Optional[str]:
    """Return a directory D such that ``D/cardio_jepa/__init__.py`` exists."""
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        stack: List[Tuple[str, int]] = [(root, 0)]
        while stack:
            path, depth = stack.pop()
            if depth > max_depth:
                continue
            cand = os.path.join(path, "cardio_jepa", "__init__.py")
            if os.path.isfile(cand):
                return path
            try:
                for e in os.scandir(path):
                    if e.is_dir(follow_symlinks=False) and not e.name.startswith("."):
                        stack.append((e.path, depth + 1))
            except (PermissionError, FileNotFoundError, NotADirectoryError):
                continue
    return None


def find_checkpoints(search_roots: Sequence[str] = DEFAULT_MODEL_ROOTS,
                     max_depth: int = 8) -> List[str]:
    """Checkpoint-looking files under the given roots, **in root order**.

    Root order is the priority: callers pass the curated model input first and
    scratch directories like ``runs/`` last, so a checkpoint deliberately shipped
    with the model always outranks whatever an earlier session happened to leave
    lying around. Sorting globally by size instead -- as this used to -- let a
    mid-run scratch checkpoint that is a few KB larger silently win, which is the
    opposite of what "auto" should mean.

    Within one root, newest first (most recent training run), then largest.
    """
    paths: List[str] = []
    seen: set = set()

    for root in search_roots:
        if not root or not os.path.isdir(root):
            continue
        here: List[Tuple[float, int, str]] = []
        stack: List[Tuple[str, int]] = [(root, 0)]
        while stack:
            path, depth = stack.pop()
            if depth > max_depth:
                continue
            try:
                entries = list(os.scandir(path))
            except (PermissionError, FileNotFoundError, NotADirectoryError):
                continue
            for e in entries:
                if e.is_dir(follow_symlinks=False) and not e.name.startswith("."):
                    stack.append((e.path, depth + 1))
                elif e.is_file() and e.name.lower().endswith(CHECKPOINT_EXTS):
                    try:
                        st = e.stat()
                        here.append((st.st_mtime, st.st_size, e.path))
                    except OSError:
                        here.append((0.0, 0, e.path))
        here.sort(reverse=True)
        for _mtime, _size, p in here:
            rp = os.path.realpath(p)
            if rp not in seen:
                seen.add(rp)
                paths.append(p)
    return paths


def _extract_state_dict(obj) -> Dict[str, object]:
    """Unwrap the several shapes a torch checkpoint can arrive in."""
    if not isinstance(obj, dict):
        return {}
    for key in ("model", "state_dict", "encoder", "module"):
        inner = obj.get(key)
        if isinstance(inner, dict) and inner:
            # a nested wrapper, or already the tensors themselves
            if all(hasattr(v, "shape") for v in list(inner.values())[:4]):
                return inner
            return _extract_state_dict(inner)
    if all(hasattr(v, "shape") for v in list(obj.values())[:4] or [None]):
        return obj
    return {}


@dataclass
class LoadReport:
    path: str = ""
    loaded: List[str] = field(default_factory=list)
    shape_mismatch: List[str] = field(default_factory=list)
    missing: List[str] = field(default_factory=list)
    unexpected: List[str] = field(default_factory=list)
    checkpoint_config: Optional[dict] = None

    @property
    def n_loaded(self) -> int:
        return len(self.loaded)

    def summary(self) -> str:
        total = len(self.loaded) + len(self.missing)
        pct = 100.0 * len(self.loaded) / max(total, 1)
        lines = [f"loaded {len(self.loaded)}/{total} tensors ({pct:.1f}%) from "
                 f"{os.path.basename(self.path)}"]
        if self.shape_mismatch:
            lines.append(f"  {len(self.shape_mismatch)} skipped on shape mismatch "
                         f"(e.g. {self.shape_mismatch[0]})")
        if self.missing:
            lines.append(f"  {len(self.missing)} left at initialisation "
                         f"(e.g. {self.missing[0]})")
        if self.unexpected:
            lines.append(f"  {len(self.unexpected)} in checkpoint but not in model "
                         f"(e.g. {self.unexpected[0]})")
        if pct < 50.0:
            lines.append("  WARNING: under half the model was restored - the "
                         "checkpoint's architecture probably differs from this config, "
                         "so this is closer to a fresh start than a warm one.")
        return "\n".join(lines)


def load_weights(model, path: str, strict: bool = False,
                 map_location: str = "cpu", verbose: bool = True) -> LoadReport:
    """Warm-start ``model`` from a checkpoint, skipping anything that does not fit.

    Returns a report rather than raising, because the common real case -- resuming a
    run at a different encoder width -- legitimately restores only part of the model,
    and silently loading 3% of the tensors while calling it a warm start would be
    worse than saying so.
    """
    import torch

    try:
        obj = torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:                                   # torch < 2.0 has no weights_only
        obj = torch.load(path, map_location=map_location)

    report = LoadReport(path=path)
    if isinstance(obj, dict):
        report.checkpoint_config = obj.get("config") if isinstance(
            obj.get("config"), dict) else None

    src = _extract_state_dict(obj)
    if not src:
        if verbose:
            print(f"[checkpoint] {path}: no tensors found inside")
        return report

    # tolerate a `module.` prefix from DataParallel-saved checkpoints
    if all(k.startswith("module.") for k in src):
        src = {k[len("module."):]: v for k, v in src.items()}

    dst = model.state_dict()
    to_load: Dict[str, object] = {}
    for k, v in dst.items():
        if k not in src:
            report.missing.append(k)
            continue
        sv = src[k]
        if getattr(sv, "shape", None) == getattr(v, "shape", None):
            to_load[k] = sv
            report.loaded.append(k)
        else:
            report.shape_mismatch.append(f"{k}: {tuple(sv.shape)} vs {tuple(v.shape)}")
            report.missing.append(k)
    report.unexpected = [k for k in src if k not in dst]

    if strict and (report.missing or report.unexpected):
        raise RuntimeError(
            f"strict load failed: {len(report.missing)} missing, "
            f"{len(report.unexpected)} unexpected")

    dst.update(to_load)
    model.load_state_dict(dst)
    if verbose:
        print("[checkpoint] " + report.summary())
    return report


def describe_model_input(root: str) -> str:
    """Human-readable inventory of a Kaggle Model directory."""
    if not os.path.isdir(root):
        return f"{root}: not mounted"
    lines = [f"{root}:"]
    n_py = 0
    pkg = None
    ckpts: List[Tuple[str, float]] = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if not d.startswith(".")
                       and d != "__pycache__"]
        if os.path.basename(dirpath) == "cardio_jepa" and "__init__.py" in filenames:
            pkg = dirpath
        for fn in filenames:
            if fn.endswith(".py"):
                n_py += 1
            elif fn.lower().endswith(CHECKPOINT_EXTS):
                full = os.path.join(dirpath, fn)
                try:
                    ckpts.append((full, os.path.getsize(full) / 1e6))
                except OSError:
                    ckpts.append((full, 0.0))
    lines.append(f"  cardio_jepa package : {pkg or 'not found'}")
    lines.append(f"  python files        : {n_py}")
    lines.append(f"  checkpoints         : {len(ckpts)}")
    for p, mb in sorted(ckpts, key=lambda x: -x[1])[:8]:
        lines.append(f"      {mb:8.1f} MB  {os.path.relpath(p, root)}")
    return "\n".join(lines)

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/models/__init__.py
from .baselines import Inception1d, STMEMBaseline, XResNet1d
from .cardio_jepa import CardioJEPA, Level2Stack, normalize_target, prediction_loss
from .encoder import CroPAEncoder, MaskedPredictor, make_block_mask, make_random_mask
from .forward_predictor import ForwardPredictor, SequencePredictor, horizon_to_bucket
from .ssl_baselines import CPCBaseline, SimCLRBaseline, ecg_augment

__all__ = [
    "CardioJEPA", "Level2Stack", "prediction_loss", "normalize_target",
    "CroPAEncoder", "MaskedPredictor", "make_block_mask", "make_random_mask",
    "ForwardPredictor", "SequencePredictor", "horizon_to_bucket",
    "STMEMBaseline", "XResNet1d", "Inception1d",
    "SimCLRBaseline", "CPCBaseline", "ecg_augment",
]

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/models/pos_encoding.py
"""Sin-cos positional embeddings.

Vendored from Kim's ECG-JEPA (https://github.com/sehunfromdaegu/ECG_JEPA,
`pos_encoding.py`), which in turn follows the MAE / ST-MEM implementations.
Reused verbatim so that our reproduction of the masked-infilling baseline is
faithful to the published model.
"""

from __future__ import annotations

import numpy as np


def get_1d_sincos_pos_embed_from_grid(embed_dim: int, pos: np.ndarray) -> np.ndarray:
    """embed_dim: output dimension per position; pos: (M,) -> (M, embed_dim)."""
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=float)
    omega /= embed_dim / 2.0
    omega = 1.0 / 10000 ** omega  # (D/2,)

    pos = pos.reshape(-1)  # (M,)
    out = np.einsum("m,d->md", pos, omega)  # (M, D/2)

    emb_sin = np.sin(out)
    emb_cos = np.cos(out)
    return np.concatenate([emb_sin, emb_cos], axis=1)  # (M, D)


def get_2d_sincos_pos_embed_from_grid(embed_dim: int, grid: np.ndarray) -> np.ndarray:
    assert embed_dim % 2 == 0
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])  # (H*W, D/2)
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])  # (H*W, D/2)
    return np.concatenate([emb_h, emb_w], axis=1)  # (H*W, D)


def get_2d_sincos_pos_embed(embed_dim: int, grid_size_h: int, grid_size_w: int,
                            cls_token: bool = False) -> np.ndarray:
    """Return (grid_h*grid_w, embed_dim) laid out lead-major, time-minor."""
    grid_h = np.arange(grid_size_h, dtype=float)
    grid_w = np.arange(grid_size_w, dtype=float)
    grid = np.meshgrid(grid_w, grid_h)  # w goes first
    grid = np.stack(grid, axis=0)
    grid = grid.reshape([2, 1, grid_size_h, grid_size_w])
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    if cls_token:
        pos_embed = np.concatenate([np.zeros([1, embed_dim]), pos_embed], axis=0)
    return pos_embed


def get_1d_sincos_pos_embed(embed_dim: int, length: int) -> np.ndarray:
    """1-D sin-cos embedding, used by the Level-2 (rhythm) transformer."""
    return get_1d_sincos_pos_embed_from_grid(embed_dim, np.arange(length, dtype=float))

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/models/transformer.py
"""Transformer primitives.

Adapted from Kim's ECG-JEPA (https://github.com/sehunfromdaegu/ECG_JEPA,
`ecg_jepa.py`), which itself derives from facebookresearch/ijepa's
`src/models/vision_transformer.py`.

Changes vs. the upstream ECG-JEPA code:
  * `Attention` accepts an additive/boolean attention mask and fills with
    `torch.finfo(dtype).min` instead of `-inf`, which is NaN-safe under AMP.
  * `Block.forward` takes an optional key-padding-free mask of shape
    (N, N) or (B, 1, N, N).
  * `DropPath` is implemented locally so the package has no `timm` dependency
    (timm is not installed in the default Kaggle image).
"""

from __future__ import annotations

import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

#: v0.2: use torch's fused scaled-dot-product attention when available (torch >= 2.1
#: accepts the ``scale`` keyword). Set to False to reproduce v0.1 numerics exactly.
USE_SDPA: bool = hasattr(F, "scaled_dot_product_attention")
try:  # the `scale` kwarg arrived in torch 2.1
    import inspect as _inspect
    if USE_SDPA and "scale" not in _inspect.signature(F.scaled_dot_product_attention).parameters:
        USE_SDPA = False
except (TypeError, ValueError):  # pragma: no cover - builtins without signatures
    pass


# --------------------------------------------------------------------------------------
# init helpers (cut & paste from PyTorch / timm, as in the upstream repos)
# --------------------------------------------------------------------------------------
def _no_grad_trunc_normal_(tensor, mean, std, a, b):
    def norm_cdf(x):
        return (1.0 + math.erf(x / math.sqrt(2.0))) / 2.0

    with torch.no_grad():
        low = norm_cdf((a - mean) / std)
        up = norm_cdf((b - mean) / std)
        tensor.uniform_(2 * low - 1, 2 * up - 1)
        tensor.erfinv_()
        tensor.mul_(std * math.sqrt(2.0))
        tensor.add_(mean)
        tensor.clamp_(min=a, max=b)
        return tensor


def trunc_normal_(tensor, mean=0.0, std=1.0, a=-2.0, b=2.0):
    return _no_grad_trunc_normal_(tensor, mean, std, a, b)


def drop_path(x: torch.Tensor, drop_prob: float = 0.0, training: bool = False) -> torch.Tensor:
    if drop_prob == 0.0 or not training:
        return x
    keep_prob = 1.0 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    return x.div(keep_prob) * random_tensor


class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


# --------------------------------------------------------------------------------------
# blocks
# --------------------------------------------------------------------------------------
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        return self.drop(self.fc2(x))


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None,
                 attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, attention_mask: Optional[torch.Tensor] = None):
        B, N, C = x.shape
        qkv = (self.qkv(x)
               .reshape(B, N, 3, self.num_heads, C // self.num_heads)
               .permute(2, 0, 3, 1, 4))
        q, k, v = qkv[0], qkv[1], qkv[2]

        if USE_SDPA:
            # v0.2 FIX (memory): the explicit (B, H, N, N) softmax kept one fp32 score
            # matrix per layer alive for backward -- 1.38 GiB per layer for a
            # 12-window Stage-4 batch, which is the OOM in the v0.1 run. SDPA uses the
            # memory-efficient kernel (works on T4/sm75 with a boolean mask) and never
            # materialises the full score matrix. Numerically equivalent.
            mask = None
            if attention_mask is not None:
                mask = attention_mask.to(device=q.device) != 0
            out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=mask,
                dropout_p=self.attn_drop.p if self.training else 0.0,
                scale=self.scale)
            x = out.transpose(1, 2).reshape(B, N, C)
            return self.proj_drop(self.proj(x))

        attn = (q @ k.transpose(-2, -1)) * self.scale
        if attention_mask is not None:
            # NaN-safe masking under fp16 autocast: use dtype min rather than -inf.
            neg = torch.finfo(attn.dtype).min
            attn = attn.masked_fill(attention_mask == 0, neg)

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=False, qk_scale=None,
                 drop=0.0, attn_drop=0.0, drop_path_rate=0.0,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                              qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0.0 else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio),
                       act_layer=act_layer, drop=drop)

    def forward(self, x, attention_mask=None):
        x = x + self.drop_path(self.attn(self.norm1(x), attention_mask))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class CrossAttention(nn.Module):
    """Queries attend to an external memory sequence."""

    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None,
                 attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.q = nn.Linear(dim, dim, bias=qkv_bias)
        self.kv = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, memory):
        B, N, C = x.shape
        M = memory.size(1)
        q = self.q(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        kv = (self.kv(memory)
              .reshape(B, M, 2, self.num_heads, C // self.num_heads)
              .permute(2, 0, 3, 1, 4))
        k, v = kv[0], kv[1]

        if USE_SDPA:
            out = F.scaled_dot_product_attention(
                q, k, v, dropout_p=self.attn_drop.p if self.training else 0.0,
                scale=self.scale)
            x = out.transpose(1, 2).reshape(B, N, C)
            return self.proj_drop(self.proj(x))

        attn = ((q @ k.transpose(-2, -1)) * self.scale).softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))


class DecoderBlock(nn.Module):
    """Pre-norm decoder block: self-attention over queries + cross-attention to context.

    Used by the forward predictor. Keeping the context as a cross-attention memory
    (rather than concatenating it into one long sequence) keeps the attention matrix
    at O(N_query x N_ctx) instead of O((N_query + N_ctx)^2), which matters at
    C*P = 400 tokens per window on a 16 GB Kaggle GPU.
    """

    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=False, qk_scale=None,
                 drop=0.0, attn_drop=0.0, drop_path_rate=0.0,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.self_attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                   qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)
        self.norm_q = norm_layer(dim)
        self.norm_m = norm_layer(dim)
        self.cross_attn = CrossAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                         qk_scale=qk_scale, attn_drop=attn_drop,
                                         proj_drop=drop)
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0.0 else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio),
                       act_layer=act_layer, drop=drop)

    def forward(self, x, memory, attention_mask=None):
        x = x + self.drop_path(self.self_attn(self.norm1(x), attention_mask))
        x = x + self.drop_path(self.cross_attn(self.norm_q(x), self.norm_m(memory)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class DecoderStack(nn.Module):
    def __init__(self, embed_dim=192, depth=4, num_heads=6, mlp_ratio=4.0,
                 qkv_bias=False, qk_scale=None, drop_rate=0.0, attn_drop_rate=0.0,
                 drop_path_rate=0.0, norm_layer=nn.LayerNorm):
        super().__init__()
        dpr = (drop_path_rate if isinstance(drop_path_rate, list)
               else [x.item() for x in torch.linspace(0, drop_path_rate, depth)])
        self.blocks = nn.ModuleList([
            DecoderBlock(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio,
                         qkv_bias=qkv_bias, qk_scale=qk_scale, drop=drop_rate,
                         attn_drop=attn_drop_rate, drop_path_rate=dpr[i],
                         norm_layer=norm_layer)
            for i in range(depth)
        ])

    def forward(self, x, memory, pos=None, attention_mask=None):
        for block in self.blocks:
            block_in = x if pos is None else x + pos
            x = block(block_in, memory, attention_mask)
        return x


class TransformerStack(nn.Module):
    """A stack of Blocks that adds the positional embedding at *every* layer.

    Re-adding `pos` inside each block is the upstream ECG-JEPA behaviour
    (`Encoder_Block.forward`), kept for fidelity with the published baseline.
    """

    def __init__(self, embed_dim=384, depth=12, num_heads=6, mlp_ratio=4.0,
                 qkv_bias=False, qk_scale=None, drop_rate=0.0, attn_drop_rate=0.0,
                 drop_path_rate=0.0, norm_layer=nn.LayerNorm):
        super().__init__()
        dpr = (drop_path_rate if isinstance(drop_path_rate, list)
               else [x.item() for x in torch.linspace(0, drop_path_rate, depth)])
        self.blocks = nn.ModuleList([
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio,
                  qkv_bias=qkv_bias, qk_scale=qk_scale, drop=drop_rate,
                  attn_drop=attn_drop_rate, drop_path_rate=dpr[i], norm_layer=norm_layer)
            for i in range(depth)
        ])

    def forward(self, x, pos=None, attention_mask=None):
        for block in self.blocks:
            block_in = x if pos is None else x + pos
            x = block(block_in, attention_mask)
        return x


def init_weights_(module: nn.Module, init_std: float = 0.02):
    if isinstance(module, nn.Linear):
        trunc_normal_(module.weight, std=init_std)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0)
    elif isinstance(module, nn.LayerNorm):
        nn.init.constant_(module.bias, 0)
        nn.init.constant_(module.weight, 1.0)
    elif isinstance(module, nn.Conv1d):
        trunc_normal_(module.weight, std=init_std)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0)


def rescale_blocks_(blocks: nn.ModuleList):
    """Layer-id dependent rescaling of residual projections (BEiT / ECG-JEPA trick).

    Handles both `Block` (attr `attn`) and `DecoderBlock` (attrs `self_attn`,
    `cross_attn`).
    """
    for layer_id, layer in enumerate(blocks):
        scale = math.sqrt(2.0 * (layer_id + 1))
        for name in ("attn", "self_attn", "cross_attn"):
            sub = getattr(layer, name, None)
            if sub is not None:
                sub.proj.weight.data.div_(scale)
        layer.mlp.fc2.weight.data.div_(scale)

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/models/encoder.py
"""CroPA context/target encoder.

This is Kim's ECG-JEPA `MaskTransformer` (Cross-Pattern Attention over a
lead x time patch grid) refactored into an explicit API:

    tokens = encoder(x_win, time_mask=None)     # (B, n_visible_tokens, D)
    z      = encoder.representation(x_win)      # (B, D)  mean-pooled, no masking

Cross-Pattern Attention (CroPA): a patch may attend only to patches that share
its lead (temporal context) or share its time column (cross-lead context). This
is the structural prior from arXiv:2410.08559 -- single-lead morphology plus
inter-lead physics -- and it cuts attention cost relative to dense attention over
the full C x P grid.

Masking is *synchronised across leads* (a boolean mask over the P time patches is
tiled over all C leads), exactly as in the upstream implementation.
"""

from __future__ import annotations

from typing import Optional, Tuple

import torch
import torch.nn as nn

from .pos_encoding import get_2d_sincos_pos_embed
from .transformer import TransformerStack, init_weights_, rescale_blocks_, trunc_normal_


def make_block_mask(n_patches: int, mask_scale: Tuple[float, float],
                    n_blocks: int = 4, generator: Optional[torch.Generator] = None
                    ) -> torch.Tensor:
    """Contiguous multi-block temporal mask (ECG-JEPA `_make_block_mask`)."""
    mask = torch.zeros(n_patches, dtype=torch.bool)
    for _ in range(n_blocks):
        ratio = mask_scale[0] + (mask_scale[1] - mask_scale[0]) * torch.rand(
            1, generator=generator).item()
        num = max(1, int(n_patches * ratio))
        num = min(num, n_patches - 1)
        start = torch.randint(0, n_patches - num, (1,), generator=generator).item()
        mask[start:start + num] = True
    # never mask everything -- the context encoder needs at least one visible patch
    if bool(mask.all()):
        mask[0] = False
    return mask


def make_random_mask(n_patches: int, mask_scale: Tuple[float, float],
                     generator: Optional[torch.Generator] = None) -> torch.Tensor:
    """i.i.d. random temporal mask (ECG-JEPA `_make_rand_mask`)."""
    mask = torch.zeros(n_patches, dtype=torch.bool)
    ratio = mask_scale[0] + (mask_scale[1] - mask_scale[0]) * torch.rand(
        1, generator=generator).item()
    num = max(1, min(int(n_patches * ratio), n_patches - 1))
    idx = torch.randperm(n_patches, generator=generator)[:num]
    mask[idx] = True
    return mask


class CroPAEncoder(nn.Module):
    def __init__(self,
                 embed_dim: int = 384,
                 depth: int = 12,
                 num_heads: int = 6,
                 mlp_ratio: float = 4.0,
                 qkv_bias: bool = True,
                 qk_scale=None,
                 drop_rate: float = 0.0,
                 attn_drop_rate: float = 0.0,
                 drop_path_rate: float = 0.0,
                 norm_layer=nn.LayerNorm,
                 init_std: float = 0.02,
                 pos_type: str = "sincos",
                 use_cropa: bool = True,
                 n_leads: int = 8,
                 n_time_patches: int = 50,
                 patch_samples: int = 50):
        super().__init__()
        self.c = n_leads
        self.p = n_time_patches
        self.t = patch_samples
        self.embed_dim = embed_dim
        self.init_std = init_std
        self.use_cropa = use_cropa

        self.patch_embed = nn.Linear(patch_samples, embed_dim)
        self.blocks = TransformerStack(
            embed_dim=embed_dim, depth=depth, num_heads=num_heads, mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias, qk_scale=qk_scale, drop_rate=drop_rate,
            attn_drop_rate=attn_drop_rate, drop_path_rate=drop_path_rate,
            norm_layer=norm_layer)
        self.norm = norm_layer(embed_dim)

        if pos_type == "learnable":
            pos = torch.empty(self.c * self.p, embed_dim)
            nn.init.uniform_(pos, -0.02, 0.02)
            self.pos_embed = nn.Parameter(pos, requires_grad=True)
        else:
            self.pos_embed = nn.Parameter(torch.zeros(self.c * self.p, embed_dim),
                                          requires_grad=False)
            sincos = get_2d_sincos_pos_embed(embed_dim, self.c, self.p)
            self.pos_embed.data.copy_(torch.from_numpy(sincos).float())

        # CroPA attention mask is constant -> register as a buffer (not a parameter)
        self.register_buffer("cropa_mask", self._build_cropa_mask(), persistent=False)

        self.apply(lambda m: init_weights_(m, self.init_std))
        rescale_blocks_(self.blocks.blocks)

    # ------------------------------------------------------------------ masks
    def _build_cropa_mask(self) -> torch.Tensor:
        """(C*P, C*P) binary mask: 1 where attention is allowed."""
        size = self.c * self.p
        m = torch.zeros(size, size)
        for i in range(self.c):                       # same lead  (temporal context)
            m[i * self.p:(i + 1) * self.p, i * self.p:(i + 1) * self.p] = 1
        for j in range(self.p):                       # same time column (cross-lead)
            m[j::self.p, j::self.p] = 1
        return m.clamp(max=1)

    def attention_mask(self, keep: Optional[torch.Tensor] = None) -> Optional[torch.Tensor]:
        if not self.use_cropa:
            return None
        m = self.cropa_mask
        if keep is not None:
            m = m[keep][:, keep]
        return m

    # --------------------------------------------------------------- forward
    def tokenize(self, x: torch.Tensor) -> torch.Tensor:
        """(B, C, T) -> (B, C*P, patch_samples)."""
        B, C, T = x.shape
        assert C == self.c, f"expected {self.c} leads, got {C}"
        assert T == self.p * self.t, f"expected {self.p * self.t} samples, got {T}"
        return x.reshape(B, self.c * self.p, self.t)

    def forward(self, x: torch.Tensor, time_mask: Optional[torch.Tensor] = None
                ) -> torch.Tensor:
        """Encode a window.

        Args:
            x: (B, C, T) raw window.
            time_mask: (P,) bool over time patches. True = masked (hidden from the
                context encoder). If None the full grid is encoded.
        Returns:
            (B, N, D) where N = C*P if `time_mask` is None else C*(P - n_masked).
        """
        x = self.tokenize(x)
        pos = self.pos_embed.unsqueeze(0).expand(x.size(0), -1, -1)
        attn = self.attention_mask()

        if time_mask is not None:
            # masks are sampled on CPU; indexing a CUDA tensor with a CPU boolean mask
            # raises "indices should be either on cpu or on the same device"
            time_mask = time_mask.to(x.device)
            keep = (~time_mask).repeat(self.c)        # lead-major tiling
            x = x[:, keep]
            pos = pos[:, keep]
            attn = self.attention_mask(keep)

        x = self.patch_embed(x)
        x = self.blocks(x, pos, attn)
        return self.norm(x)

    def full_tokens(self, x: torch.Tensor) -> torch.Tensor:
        """Encode the full grid: (B, C, T) -> (B, C*P, D)."""
        return self.forward(x, time_mask=None)

    def representation(self, x: torch.Tensor) -> torch.Tensor:
        """Mean-pooled embedding used by every downstream probe: (B, C, T) -> (B, D)."""
        return self.full_tokens(x).mean(dim=1)


class MaskedPredictor(nn.Module):
    """Predicts masked target latents from visible context latents.

    Port of ECG-JEPA's `MaskTransformerPredictor`. The predictor runs *per lead*
    (the sequence is reshaped to (B*C, P, D)) with a shared 1-D positional
    embedding over time patches and a learned mask token, which is the
    "lead-wise predictor" described in arXiv:2410.08559.
    """

    def __init__(self,
                 embed_dim: int = 384,
                 predictor_embed_dim: int = 192,
                 depth: int = 4,
                 num_heads: int = 6,
                 mlp_ratio: float = 4.0,
                 qkv_bias: bool = True,
                 qk_scale=None,
                 drop_rate: float = 0.0,
                 attn_drop_rate: float = 0.0,
                 drop_path_rate: float = 0.0,
                 norm_layer=nn.LayerNorm,
                 init_std: float = 0.02,
                 pos_type: str = "sincos",
                 n_leads: int = 8,
                 n_time_patches: int = 50):
        super().__init__()
        self.c = n_leads
        self.p = n_time_patches
        self.embed_dim = embed_dim
        self.init_std = init_std

        self.predictor_embed = nn.Linear(embed_dim, predictor_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_embed_dim))

        if pos_type == "learnable":
            pos = torch.empty(self.p, predictor_embed_dim)
            nn.init.uniform_(pos, -0.02, 0.02)
            self.pos_embed = nn.Parameter(pos, requires_grad=True)
        else:
            self.pos_embed = nn.Parameter(torch.zeros(self.p, predictor_embed_dim),
                                          requires_grad=False)
            sincos = get_2d_sincos_pos_embed(predictor_embed_dim, 1, self.p)
            self.pos_embed.data.copy_(torch.from_numpy(sincos).float())

        self.blocks = TransformerStack(
            embed_dim=predictor_embed_dim, depth=depth, num_heads=num_heads,
            mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
            drop_rate=drop_rate, attn_drop_rate=attn_drop_rate,
            drop_path_rate=drop_path_rate, norm_layer=norm_layer)
        self.norm = norm_layer(predictor_embed_dim)
        self.proj = nn.Linear(predictor_embed_dim, embed_dim, bias=True)

        trunc_normal_(self.mask_token, std=init_std)
        self.apply(lambda m: init_weights_(m, self.init_std))
        rescale_blocks_(self.blocks.blocks)

    def forward(self, ctx_tokens: torch.Tensor, time_mask: torch.Tensor) -> torch.Tensor:
        """
        Args:
            ctx_tokens: (B, C*P_vis, D) output of the context encoder.
            time_mask:  (P,) bool, True = masked.
        Returns:
            (B, C*n_masked, D) predictions for the masked patches, in lead-major order.
        """
        n_mask = int(time_mask.sum().item())
        B = ctx_tokens.size(0)

        x = self.predictor_embed(ctx_tokens)
        pdim = x.size(-1)
        x = x.reshape(B * self.c, -1, pdim)                    # (B*C, P_vis, D')
        mask_tok = self.mask_token.expand(x.size(0), n_mask, pdim)
        x = torch.cat([x, mask_tok], dim=1)                    # (B*C, P_vis+n_mask, D')

        # positional embeddings reordered to [visible..., masked...]
        vis_idx = (~time_mask).nonzero(as_tuple=True)[0]
        msk_idx = time_mask.nonzero(as_tuple=True)[0]
        order = torch.cat([vis_idx, msk_idx]).to(self.pos_embed.device)
        pos = self.pos_embed[order].unsqueeze(0).expand(x.size(0), -1, -1)

        x = self.blocks(x, pos)
        x = self.proj(self.norm(x))
        x = x.reshape(B, -1, self.embed_dim)                   # (B, C*(P_vis+n_mask), D)

        # keep only the predicted (masked) slots, per lead
        x = x.reshape(B, self.c, -1, self.embed_dim)[:, :, -n_mask:, :]
        return x.reshape(B, -1, self.embed_dim)

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/models/forward_predictor.py
"""Forward (future) latent predictor -- the novel component of Cardio-JEPA.

Section 5.3(b) of the proposal:

    z_ctx = f_theta(x_[t-L, t])            # context encoder, gradients flow
    z_fut = sg(fbar_theta(x_[t, t+H]))     # EMA target encoder, stop-gradient
    L_fwd = || g_phi(z_ctx, h) - z_fut ||_1

`h` is a learned *horizon token*, indexed by a discretised horizon bucket
({10 s, 30 s, 1 min, 5 min, 10 min, 30 min, 1 hr} by default), so a single
predictor can be queried at multiple forecast horizons -- this is what makes the
model a queryable cardiac world model rather than a fixed-lag forecaster.

Two output modes:
  * ``tokens``  -- predict the full (C*P, D) future patch grid. Richer signal,
                   what we use by default.
  * ``pooled``  -- predict a single (D,) window-level future latent. Cheaper;
                   useful for long horizons and for the Level-2 rhythm stack.
"""

from __future__ import annotations

from typing import Optional, Sequence

import numpy as np
import torch
import torch.nn as nn

from .pos_encoding import get_2d_sincos_pos_embed
from .transformer import DecoderStack, init_weights_, rescale_blocks_, trunc_normal_


def horizon_to_bucket(horizon_s, buckets: Sequence[float]) -> torch.Tensor:
    """Map horizons in seconds to the index of the nearest bucket (in log-space).

    Works on floats, lists, numpy arrays and tensors; always returns a long tensor.
    """
    h = torch.as_tensor(horizon_s, dtype=torch.float32).reshape(-1)
    b = torch.as_tensor(list(buckets), dtype=torch.float32)
    # nearest in log space, so 10 s vs 30 s is as distinguishable as 10 min vs 30 min
    d = (torch.log(h.clamp_min(1e-3)).unsqueeze(1) - torch.log(b).unsqueeze(0)).abs()
    return d.argmin(dim=1).long()


class ForwardPredictor(nn.Module):
    """g_phi: (context latents, horizon) -> predicted future latents."""

    def __init__(self,
                 embed_dim: int = 384,
                 predictor_embed_dim: int = 192,
                 depth: int = 4,
                 num_heads: int = 6,
                 mlp_ratio: float = 4.0,
                 qkv_bias: bool = True,
                 qk_scale=None,
                 drop_rate: float = 0.0,
                 attn_drop_rate: float = 0.0,
                 drop_path_rate: float = 0.0,
                 norm_layer=nn.LayerNorm,
                 init_std: float = 0.02,
                 n_leads: int = 8,
                 n_time_patches: int = 50,
                 n_horizon_buckets: int = 7,
                 max_context_windows: int = 8,
                 output_mode: str = "tokens",
                 use_cropa_queries: bool = False):
        super().__init__()
        assert output_mode in ("tokens", "pooled")
        self.c = n_leads
        self.p = n_time_patches
        self.embed_dim = embed_dim
        self.output_mode = output_mode
        self.init_std = init_std
        self.use_cropa_queries = use_cropa_queries

        pdim = predictor_embed_dim
        self.ctx_proj = nn.Linear(embed_dim, pdim, bias=True)

        # --- context side: 2-D (lead, time) grid position + window-index embedding ---
        grid = get_2d_sincos_pos_embed(pdim, self.c, self.p)
        self.register_buffer("grid_pos", torch.from_numpy(grid).float(), persistent=False)
        self.window_embed = nn.Parameter(torch.zeros(max_context_windows, pdim))

        # --- horizon conditioning ---
        self.horizon_embed = nn.Embedding(n_horizon_buckets, pdim)

        # --- query side ---
        n_query = self.c * self.p if output_mode == "tokens" else 1
        self.n_query = n_query
        self.query_token = nn.Parameter(torch.zeros(1, 1, pdim))
        # v0.2: a dedicated query slot for a *pooled* (window-level) forecast, used by
        # the cross-window objective even when the intra-window path predicts tokens.
        self.pooled_query_pos = nn.Parameter(torch.zeros(1, pdim))
        if output_mode == "tokens":
            self.register_buffer("query_pos", torch.from_numpy(grid).float(),
                                 persistent=False)
        else:
            self.register_buffer("query_pos", torch.zeros(1, pdim), persistent=False)

        self.blocks = DecoderStack(
            embed_dim=pdim, depth=depth, num_heads=num_heads, mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias, qk_scale=qk_scale, drop_rate=drop_rate,
            attn_drop_rate=attn_drop_rate, drop_path_rate=drop_path_rate,
            norm_layer=norm_layer)
        self.norm = norm_layer(pdim)
        self.proj = nn.Linear(pdim, embed_dim, bias=True)

        if use_cropa_queries and output_mode == "tokens":
            self.register_buffer("query_attn_mask", self._build_cropa_mask(),
                                 persistent=False)
        else:
            self.query_attn_mask = None

        trunc_normal_(self.query_token, std=init_std)
        trunc_normal_(self.window_embed, std=init_std)
        trunc_normal_(self.pooled_query_pos, std=init_std)
        self.apply(lambda m: init_weights_(m, self.init_std))
        rescale_blocks_(self.blocks.blocks)

    def _build_cropa_mask(self) -> torch.Tensor:
        size = self.c * self.p
        m = torch.zeros(size, size)
        for i in range(self.c):
            m[i * self.p:(i + 1) * self.p, i * self.p:(i + 1) * self.p] = 1
        for j in range(self.p):
            m[j::self.p, j::self.p] = 1
        return m.clamp(max=1)

    def forward(self, ctx_tokens: torch.Tensor, horizon_idx: torch.Tensor,
                n_context_windows: int = 1,
                ctx_pos_index: Optional[torch.Tensor] = None,
                query_pos_index: Optional[torch.Tensor] = None,
                pooled: Optional[bool] = None) -> torch.Tensor:
        """
        Args:
            ctx_tokens:  (B, n_context_windows * n_ctx, D) context-encoder output.
            horizon_idx: (B,) long tensor of horizon-bucket indices.
            n_context_windows: how many windows are stacked inside ctx_tokens.
            ctx_pos_index: (n_ctx,) indices into the C*P grid that ``ctx_tokens``
                occupy within each window. ``None`` means the full grid. This is what
                lets the *intra-window* forward objective feed only the past patches
                of a single 10 s recording as context.
            query_pos_index: (n_q,) grid indices to predict. ``None`` means the full
                grid (tokens mode) or a single pooled slot (pooled mode).
            pooled: v0.2 -- force a single window-level forecast (B, D) regardless of
                ``output_mode``; ``None`` keeps the constructor's ``output_mode``.
        Returns:
            ``tokens`` mode: (B, n_q, D) predicted future patch latents.
            ``pooled`` mode: (B, D)      predicted future window latent.
        """
        B = ctx_tokens.size(0)
        device = ctx_tokens.device
        mem = self.ctx_proj(ctx_tokens)

        # context positional embedding: grid position + which window it came from
        base = self.grid_pos
        if ctx_pos_index is not None:
            base = base[ctx_pos_index.to(base.device)]
        tiles = [base + self.window_embed[w] for w in range(n_context_windows)]
        ctx_pos = torch.cat(tiles, dim=0).unsqueeze(0)                 # (1, W*n_ctx, D')
        if ctx_pos.size(1) != mem.size(1):
            raise ValueError(
                f"context has {mem.size(1)} tokens but positional layout expects "
                f"{ctx_pos.size(1)} (n_context_windows={n_context_windows}, "
                f"n_ctx_per_window={base.size(0)})")
        mem = mem + ctx_pos

        # horizon token is appended to the memory so every query can attend to it
        h_tok = self.horizon_embed(horizon_idx.to(device)).unsqueeze(1)
        mem = torch.cat([mem, h_tok], dim=1)                           # (B, W*n_ctx+1, D')

        # queries: learned token + target position + the same horizon embedding,
        # so the horizon is available both additively and through attention
        want_pooled = (self.output_mode == "pooled") if pooled is None else bool(pooled)
        if want_pooled:
            q_pos = (self.query_pos if self.output_mode == "pooled"
                     else self.pooled_query_pos)
            attn_mask = None
        elif query_pos_index is None:
            q_pos = self.query_pos
            attn_mask = self.query_attn_mask
        else:
            qi = query_pos_index.to(self.query_pos.device)
            q_pos = self.query_pos[qi]
            attn_mask = (self.query_attn_mask[qi][:, qi]
                         if self.query_attn_mask is not None else None)
        n_q = q_pos.size(0)

        q = self.query_token.expand(B, n_q, -1)
        q = q + q_pos.unsqueeze(0) + h_tok

        q = self.blocks(q, mem, attention_mask=attn_mask)
        q = self.proj(self.norm(q))                                    # (B, n_q, D)

        if want_pooled:
            return q.squeeze(1)
        return q


class SequencePredictor(nn.Module):
    """Level-2 forward predictor over a coarse sequence of window summaries.

    Section 5.4: Level 1 works on the patch grid inside a 10 s window; Level 2
    pools per-window summary tokens into a minutes-long sequence and forecasts
    future *window* latents. Same JEPA recipe, one time-scale up.
    """

    def __init__(self,
                 embed_dim: int = 384,
                 predictor_embed_dim: int = 192,
                 depth: int = 2,
                 num_heads: int = 6,
                 mlp_ratio: float = 4.0,
                 qkv_bias: bool = True,
                 drop_rate: float = 0.0,
                 attn_drop_rate: float = 0.0,
                 drop_path_rate: float = 0.0,
                 norm_layer=nn.LayerNorm,
                 init_std: float = 0.02,
                 max_context_len: int = 64,
                 n_query: int = 1,
                 n_horizon_buckets: int = 7):
        super().__init__()
        pdim = predictor_embed_dim
        self.embed_dim = embed_dim
        self.n_query = n_query
        self.init_std = init_std

        self.ctx_proj = nn.Linear(embed_dim, pdim, bias=True)
        self.horizon_embed = nn.Embedding(n_horizon_buckets, pdim)

        from .pos_encoding import get_1d_sincos_pos_embed
        self.register_buffer(
            "ctx_pos",
            torch.from_numpy(get_1d_sincos_pos_embed(pdim, max_context_len)).float(),
            persistent=False)
        self.register_buffer(
            "query_pos",
            torch.from_numpy(get_1d_sincos_pos_embed(pdim, max(n_query, 1))).float(),
            persistent=False)

        self.query_token = nn.Parameter(torch.zeros(1, 1, pdim))
        self.blocks = DecoderStack(
            embed_dim=pdim, depth=depth, num_heads=num_heads, mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias, drop_rate=drop_rate, attn_drop_rate=attn_drop_rate,
            drop_path_rate=drop_path_rate, norm_layer=norm_layer)
        self.norm = norm_layer(pdim)
        self.proj = nn.Linear(pdim, embed_dim, bias=True)

        trunc_normal_(self.query_token, std=init_std)
        self.apply(lambda m: init_weights_(m, self.init_std))
        rescale_blocks_(self.blocks.blocks)

    def forward(self, ctx_seq: torch.Tensor, horizon_idx: Optional[torch.Tensor] = None,
                ctx_pos_index: Optional[torch.Tensor] = None,
                query_pos_index: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            ctx_seq: (B, L, D) sequence of Level-1 window summaries.
            horizon_idx: (B,) long, or None (masked infilling has no horizon).
            ctx_pos_index: (L,) positions the context entries occupy in the coarse
                sequence. v0.2 FIX: v0.1 always used 0..L-1, so after dropping masked
                windows every later window was given the wrong position.
            query_pos_index: (n_query,) positions to predict. v0.2 FIX: v0.1 used
                fixed slots 0..n_query-1, so the Level-2 masked predictor was never
                told *which* windows were masked and could only learn their mean.
        Returns:
            (B, n_query, D) predicted window summaries.
        """
        B, L, _ = ctx_seq.shape
        if L > self.ctx_pos.size(0):
            raise ValueError(f"context length {L} exceeds max_context_len "
                             f"{self.ctx_pos.size(0)}")
        if ctx_pos_index is None:
            cpos = self.ctx_pos[:L]
        else:
            cpos = self.ctx_pos[ctx_pos_index.to(self.ctx_pos.device)]
        mem = self.ctx_proj(ctx_seq) + cpos.unsqueeze(0)
        if horizon_idx is not None:
            h_tok = self.horizon_embed(horizon_idx.to(ctx_seq.device)).unsqueeze(1)
            mem = torch.cat([mem, h_tok], dim=1)
        else:
            h_tok = 0.0

        if query_pos_index is None:
            n_q = self.n_query
            qpos = self.query_pos[:n_q]
        else:
            n_q = int(query_pos_index.numel())
            qpos = self.ctx_pos[query_pos_index.to(self.ctx_pos.device)]
        q = self.query_token.expand(B, n_q, -1)
        q = q + qpos.unsqueeze(0) + h_tok
        q = self.blocks(q, mem)
        return self.proj(self.norm(q))

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/models/cardio_jepa.py
"""Cardio-JEPA: dual-objective (masked infilling + forward latent prediction) JEPA.

    L = w_fwd * L_fwd  +  lambda * L_mask  +  beta * L_reg

  * ``L_mask`` -- Kim's masked cross-lead/cross-time infilling (morphology term).
    Reproduces arXiv:2410.08559 exactly when ``w_fwd = 0``, which is the
    masked-infilling baseline arm.
  * ``L_fwd``  -- forward latent prediction (dynamics term). Predicts the target
    encoder's latents for a *future* window at a queryable horizon. This is the
    component no published ECG-JEPA has.
  * ``L_reg``  -- the RQ3 anti-collapse regulariser.

Optionally a Level-2 "rhythm" stack (Section 5.4) forecasts future *window*
latents over a minutes-long coarse sequence built from Level-1 window summaries.
"""

from __future__ import annotations

import copy
import warnings
from typing import Dict, List, Optional, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F

from ..config import (HORIZON_BUCKETS_S, ForwardSamplingConfig, HierarchicalConfig,
                      LossConfig, ModelConfig, SignalConfig)
from ..regularizers import build_regularizer
from .encoder import CroPAEncoder, MaskedPredictor, make_block_mask, make_random_mask
from .forward_predictor import ForwardPredictor, SequencePredictor, horizon_to_bucket
from .pos_encoding import get_1d_sincos_pos_embed
from .transformer import TransformerStack, init_weights_, rescale_blocks_


def prediction_loss(pred: torch.Tensor, target: torch.Tensor, kind: str = "smooth_l1"
                    ) -> torch.Tensor:
    if kind == "l1":
        return F.l1_loss(pred, target)
    if kind == "l2":
        return F.mse_loss(pred, target)
    if kind == "smooth_l1":
        return F.smooth_l1_loss(pred, target)
    raise ValueError(f"unknown loss_fn: {kind!r}")


def normalize_target(h: torch.Tensor) -> torch.Tensor:
    """LayerNorm over the feature dimension, as in I-JEPA / ECG-JEPA target processing."""
    return F.layer_norm(h.float(), (h.size(-1),)).to(h.dtype)


class Level2Stack(nn.Module):
    """Rhythm-scale (Level-2) context encoder + EMA target + forward predictor."""

    def __init__(self, embed_dim: int, cfg: HierarchicalConfig, n_horizon_buckets: int,
                 max_context_len: int = 64, n_mask_slots: int = 2,
                 use_ema_target: bool = True):
        super().__init__()
        self.cfg = cfg
        self.n_mask_slots = n_mask_slots
        self.use_ema_target = use_ema_target

        self.in_proj = nn.Linear(embed_dim, cfg.l2_embed_dim)
        self.encoder = TransformerStack(
            embed_dim=cfg.l2_embed_dim, depth=cfg.l2_depth, num_heads=cfg.l2_num_heads,
            qkv_bias=True)
        self.norm = nn.LayerNorm(cfg.l2_embed_dim)
        self.register_buffer(
            "pos", torch.from_numpy(
                get_1d_sincos_pos_embed(cfg.l2_embed_dim, max_context_len)).float(),
            persistent=False)

        self.fwd_predictor = SequencePredictor(
            embed_dim=cfg.l2_embed_dim, predictor_embed_dim=cfg.l2_predictor_dim,
            depth=cfg.l2_predictor_depth, num_heads=cfg.l2_num_heads,
            max_context_len=max_context_len, n_query=1,
            n_horizon_buckets=n_horizon_buckets)
        self.mask_predictor = SequencePredictor(
            embed_dim=cfg.l2_embed_dim, predictor_embed_dim=cfg.l2_predictor_dim,
            depth=cfg.l2_predictor_depth, num_heads=cfg.l2_num_heads,
            max_context_len=max_context_len, n_query=n_mask_slots,
            n_horizon_buckets=n_horizon_buckets)

        self.apply(lambda m: init_weights_(m, 0.02))
        rescale_blocks_(self.encoder.blocks)

        if use_ema_target:
            self.target_in_proj = copy.deepcopy(self.in_proj)
            self.target_encoder = copy.deepcopy(self.encoder)
            self.target_norm = copy.deepcopy(self.norm)
            for p in self.target_parameters():
                p.requires_grad = False
        else:
            self.target_in_proj = None
            self.target_encoder = None
            self.target_norm = None

    def target_parameters(self):
        if self.target_encoder is None:
            return []
        return (list(self.target_in_proj.parameters())
                + list(self.target_encoder.parameters())
                + list(self.target_norm.parameters()))

    def online_parameters(self):
        return (list(self.in_proj.parameters())
                + list(self.encoder.parameters())
                + list(self.norm.parameters()))

    def encode(self, summaries: torch.Tensor, target: bool = False,
               keep: Optional[torch.Tensor] = None) -> torch.Tensor:
        """(B, L, D_l1) -> (B, L, D_l2)."""
        in_proj = self.target_in_proj if (target and self.target_in_proj is not None) else self.in_proj
        enc = self.target_encoder if (target and self.target_encoder is not None) else self.encoder
        norm = self.target_norm if (target and self.target_norm is not None) else self.norm

        x = in_proj(summaries)
        pos = self.pos[:summaries.size(1)].unsqueeze(0)
        if keep is not None:
            x = x[:, keep]
            pos = pos[:, keep]
        return norm(enc(x, pos))


class CardioJEPA(nn.Module):
    def __init__(self,
                 signal: SignalConfig,
                 model: ModelConfig,
                 loss: LossConfig,
                 hierarchical: Optional[HierarchicalConfig] = None,
                 forward_sampling: Optional[ForwardSamplingConfig] = None,
                 horizon_buckets: Sequence[float] = HORIZON_BUCKETS_S,
                 max_context_windows: int = 16,
                 reg_tokens: int = 32):
        super().__init__()
        self.reg_tokens = reg_tokens
        self.reg_target = getattr(loss, "reg_target", "pooled")
        self.signal = signal
        self.model_cfg = model
        self.loss_cfg = loss
        self.hier_cfg = hierarchical or HierarchicalConfig()
        self.fwd_cfg = forward_sampling or ForwardSamplingConfig()
        self.horizon_buckets = tuple(horizon_buckets)
        self.max_context_windows = max_context_windows
        self.use_ema_target = loss.use_ema_target

        C, P, t = signal.n_leads, signal.n_time_patches, signal.patch_samples
        D = model.encoder_embed_dim

        self.encoder = CroPAEncoder(
            embed_dim=D, depth=model.encoder_depth, num_heads=model.encoder_num_heads,
            mlp_ratio=model.mlp_ratio, qkv_bias=model.qkv_bias,
            drop_rate=model.drop_rate, attn_drop_rate=model.attn_drop_rate,
            drop_path_rate=model.drop_path_rate, init_std=model.init_std,
            pos_type=model.pos_type, use_cropa=model.use_cropa,
            n_leads=C, n_time_patches=P, patch_samples=t)

        if self.use_ema_target:
            self.target_encoder = copy.deepcopy(self.encoder)
            for p in self.target_encoder.parameters():
                p.requires_grad = False
        else:
            # SIGReg / LeJEPA arm: no second network, targets are stop-gradiented
            # outputs of the online encoder itself.
            self.target_encoder = None

        self.masked_predictor = MaskedPredictor(
            embed_dim=D, predictor_embed_dim=model.predictor_embed_dim,
            depth=model.predictor_depth, num_heads=model.predictor_num_heads,
            mlp_ratio=model.mlp_ratio, qkv_bias=model.qkv_bias,
            drop_rate=model.drop_rate, attn_drop_rate=model.attn_drop_rate,
            drop_path_rate=model.drop_path_rate, init_std=model.init_std,
            pos_type=model.pos_type, n_leads=C, n_time_patches=P)

        self.forward_predictor = ForwardPredictor(
            embed_dim=D, predictor_embed_dim=model.predictor_embed_dim,
            depth=model.predictor_depth, num_heads=model.predictor_num_heads,
            mlp_ratio=model.mlp_ratio, qkv_bias=model.qkv_bias,
            drop_rate=model.drop_rate, attn_drop_rate=model.attn_drop_rate,
            drop_path_rate=model.drop_path_rate, init_std=model.init_std,
            n_leads=C, n_time_patches=P, n_horizon_buckets=len(self.horizon_buckets),
            max_context_windows=max_context_windows,
            output_mode=model.forward_target,
            use_cropa_queries=model.predictor_cropa)

        self.regularizer = build_regularizer(loss, D)

        # v0.2: how many training samples each horizon bucket has seen. Persistent, so
        # it travels with the checkpoint; querying an untrained bucket (count 0) at
        # evaluation time now raises a warning instead of silently using a
        # random-initialised horizon embedding (the v0.1 900 s / 1800 s bug).
        self.register_buffer("horizon_counts",
                             torch.zeros(len(self.horizon_buckets), dtype=torch.long))
        self.forward_target_long = getattr(model, "forward_target_long", "tokens")

        self.level2 = None
        if self.hier_cfg.enabled:
            self.level2 = Level2Stack(
                embed_dim=D, cfg=self.hier_cfg,
                n_horizon_buckets=len(self.horizon_buckets),
                max_context_len=max(64, self.hier_cfg.l2_context_windows + 4),
                use_ema_target=self.use_ema_target)

    # ------------------------------------------------------------------ helpers
    @property
    def embed_dim(self) -> int:
        return self.model_cfg.encoder_embed_dim

    def train(self, mode: bool = True):
        """v0.2 FIX: the EMA target network is never put in train mode.

        In v0.1 ``model.train()`` also switched the target encoder to training mode,
        so with ``drop_path_rate > 0`` (the ``full`` preset uses 0.1) the *targets*
        were computed with stochastic depth -- noisy targets for a stop-gradient
        branch, which I-JEPA explicitly avoids.
        """
        super().train(mode)
        if self.target_encoder is not None:
            self.target_encoder.eval()
        if self.level2 is not None and self.level2.target_encoder is not None:
            self.level2.target_encoder.eval()
        return self

    def _note_horizons(self, h_idx: torch.Tensor):
        if self.training:
            with torch.no_grad():
                self.horizon_counts += torch.bincount(
                    h_idx.detach().to(self.horizon_counts.device).reshape(-1),
                    minlength=len(self.horizon_buckets))

    def trained_horizons(self) -> List[float]:
        """Horizon buckets (seconds) that received training signal."""
        return [b for b, c in zip(self.horizon_buckets, self.horizon_counts.tolist()) if c > 0]

    def _check_horizon(self, horizon_s: float):
        idx = int(horizon_to_bucket(torch.tensor([float(horizon_s)]),
                                    self.horizon_buckets)[0])
        if int(self.horizon_counts.sum()) > 0 and int(self.horizon_counts[idx]) == 0:
            warnings.warn(
                f"horizon {horizon_s:g}s maps to bucket {self.horizon_buckets[idx]:g}s, "
                f"which was never trained (trained: {self.trained_horizons()}); the "
                f"forecast uses a random-initialised horizon embedding.", RuntimeWarning)

    @torch.no_grad()
    def target_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """(B, C, T) -> (B, D) pooled target in exactly the space L_fwd regresses
        onto (LayerNormed target tokens, then mean). Used by forecast_skill."""
        return self._target_tokens(x).float().mean(dim=1)

    def _target_tokens(self, x: torch.Tensor) -> torch.Tensor:
        """Stop-gradiented, layer-normed target latents for a window batch."""
        with torch.no_grad():
            enc = self.target_encoder if self.target_encoder is not None else self.encoder
            h = enc.full_tokens(x)
        return normalize_target(h)

    def sample_time_mask(self, device=None) -> torch.Tensor:
        P = self.signal.n_time_patches
        if self.model_cfg.mask_type == "random":
            m = make_random_mask(P, self.model_cfg.mask_scale)
        else:
            m = make_block_mask(P, self.model_cfg.mask_scale)
        return m.to(device) if device is not None else m

    @torch.no_grad()
    def update_target(self, momentum: float):
        """EMA update: theta_bar <- m * theta_bar + (1 - m) * theta."""
        if self.target_encoder is not None:
            for q, k in zip(self.encoder.parameters(), self.target_encoder.parameters()):
                k.data.mul_(momentum).add_(q.detach().data, alpha=1.0 - momentum)
            for qb, kb in zip(self.encoder.buffers(), self.target_encoder.buffers()):
                kb.data.copy_(qb.data)
        if self.level2 is not None and self.level2.target_encoder is not None:
            for q, k in zip(self.level2.online_parameters(),
                            self.level2.target_parameters()):
                k.data.mul_(momentum).add_(q.detach().data, alpha=1.0 - momentum)

    def representation(self, x: torch.Tensor) -> torch.Tensor:
        """Frozen-encoder embedding used by every downstream probe: (B, C, T) -> (B, D)."""
        return self.encoder.representation(x)

    def _intra_window_forward(self, x: torch.Tensor) -> torch.Tensor:
        """Causal forward prediction inside a single window.

        Context = the first ``intra_window_frac`` of the time patches, target = the
        remaining (later) patches. The encoder never sees the future patches, so this
        is genuine forecasting rather than infilling: it differs from ``L_mask`` in
        that the held-out region is always a *suffix*, and the prediction is
        conditioned on a horizon token.
        """
        B, C, T = x.shape
        P = self.signal.n_time_patches
        D = self.embed_dim
        device = x.device

        k = int(round(P * self.fwd_cfg.intra_window_frac))
        k = max(1, min(k, P - 1))

        time_mask = torch.zeros(P, dtype=torch.bool)
        time_mask[k:] = True                                   # future = hidden

        # grid indices (lead-major, matching CroPAEncoder's token ordering)
        lead = torch.arange(C).unsqueeze(1)
        ctx_idx = (lead * P + torch.arange(0, k).unsqueeze(0)).reshape(-1)
        q_idx = (lead * P + torch.arange(k, P).unsqueeze(0)).reshape(-1)

        ctx_vis = self.encoder(x, time_mask=time_mask)          # (B, C*k, D)

        h_full = self._target_tokens(x).reshape(B, C, P, D)
        h_tgt = h_full[:, :, k:, :].reshape(B, -1, D)

        # horizon = centre of the target span measured from the end of the context
        patch_s = self.signal.patch_samples / self.signal.fs
        horizon = ((P - k) / 2.0) * patch_s
        h_idx = horizon_to_bucket(torch.full((B,), horizon), self.horizon_buckets).to(device)
        self._note_horizons(h_idx)

        if self.model_cfg.forward_target == "pooled":
            h_tgt = h_tgt.mean(dim=1)
            pred = self.forward_predictor(ctx_vis, h_idx, n_context_windows=1,
                                          ctx_pos_index=ctx_idx)
        else:
            pred = self.forward_predictor(ctx_vis, h_idx, n_context_windows=1,
                                          ctx_pos_index=ctx_idx, query_pos_index=q_idx)

        h_tgt = self.regularizer.transform_target(h_tgt)
        return prediction_loss(pred, h_tgt, self.loss_cfg.loss_fn)

    # ------------------------------------------------------------------ forward
    def forward(self,
                ctx_windows: torch.Tensor,
                future_window: Optional[torch.Tensor] = None,
                horizon_s: Optional[torch.Tensor] = None,
                l2_future: Optional[torch.Tensor] = None,
                l2_horizon_s: Optional[torch.Tensor] = None,
                use_mask_branch: bool = True,
                use_forward_branch: bool = True) -> Dict[str, torch.Tensor]:
        """
        Args:
            ctx_windows:   (B, W, C, T) contiguous context windows, most recent last.
            future_window: (B, C, T) target window at ``horizon_s`` after the context.
            horizon_s:     (B,) forecast horizon in seconds.
            l2_future:     (B, C, T) far-future window for the Level-2 rhythm stack.
            l2_horizon_s:  (B,) Level-2 horizon in seconds.
        Returns:
            dict with ``loss`` plus its components and ``z_ctx`` for diagnostics.
        """
        if ctx_windows.dim() == 3:
            ctx_windows = ctx_windows.unsqueeze(1)
        B, W, C, T = ctx_windows.shape
        if W > self.max_context_windows:
            raise ValueError(f"{W} context windows exceeds max_context_windows="
                             f"{self.max_context_windows}")
        device = ctx_windows.device
        D = self.embed_dim

        out: Dict[str, torch.Tensor] = {}
        zero = ctx_windows.new_zeros(())
        loss_mask = zero
        loss_fwd = zero
        loss_l2 = zero

        # ---------------- context encoding (shared by both branches) ----------------
        flat = ctx_windows.reshape(B * W, C, T)
        ctx_tokens = self.encoder.full_tokens(flat)                  # (B*W, C*P, D)
        ctx_tokens = ctx_tokens.reshape(B, W * ctx_tokens.size(1), D)

        # ---------------- (a) masked infilling: morphology --------------------------
        if use_mask_branch and self.loss_cfg.lambda_mask > 0:
            x_mask = ctx_windows[:, -1]                              # most recent window
            time_mask = self.sample_time_mask()
            h_full = self._target_tokens(x_mask)                     # (B, C*P, D)
            h_full = h_full.reshape(B, C, self.signal.n_time_patches, D)
            h_tgt = h_full[:, :, time_mask.to(h_full.device), :].reshape(B, -1, D)
            h_tgt = self.regularizer.transform_target(h_tgt)

            ctx_vis = self.encoder(x_mask, time_mask=time_mask)
            pred = self.masked_predictor(ctx_vis, time_mask)
            loss_mask = prediction_loss(pred, h_tgt, self.loss_cfg.loss_fn)

        # ---------------- (b) forward latent prediction: dynamics -------------------
        if use_forward_branch and self.loss_cfg.weight_forward > 0:
            if future_window is not None:
                # cross-window forecasting (long recordings: LTAFDB, MIT-BIH)
                if horizon_s is None:
                    horizon_s = torch.full((B,), self.horizon_buckets[0], device=device)
                h_idx = horizon_to_bucket(horizon_s.detach().cpu(),
                                          self.horizon_buckets).to(device)
                self._note_horizons(h_idx)

                h_fut = self._target_tokens(future_window)           # (B, C*P, D)
                pooled = (self.model_cfg.forward_target == "pooled"
                          or self.forward_target_long == "pooled")
                if pooled:
                    h_fut = h_fut.mean(dim=1)                        # (B, D)
                h_fut = self.regularizer.transform_target(h_fut)

                pred_fut = self.forward_predictor(ctx_tokens, h_idx, n_context_windows=W,
                                                  pooled=pooled)
                loss_fwd = prediction_loss(pred_fut, h_fut, self.loss_cfg.loss_fn)

            elif self.fwd_cfg.intra_window:
                # intra-window (causal) forecasting for 10 s clips that have no
                # "next window": predict the later time patches from the earlier ones.
                loss_fwd = self._intra_window_forward(ctx_windows[:, -1])

        # ---------------- (c) Level-2 rhythm stack (RQ2) ----------------------------
        if self.level2 is not None and l2_future is not None:
            summaries = ctx_tokens.reshape(B, W, -1, D).mean(dim=2)  # (B, W, D)
            l2_ctx = self.level2.encode(summaries)                   # (B, W, D2)

            if l2_horizon_s is None:
                l2_horizon_s = torch.full((B,), self.hier_cfg.l2_horizons_s[0],
                                          device=device)
            l2_idx = horizon_to_bucket(l2_horizon_s.detach().cpu(),
                                       self.horizon_buckets).to(device)

            # v0.2 FIX: build the target summary exactly like the online summaries
            # (mean of the encoder's own output tokens). v0.1 added an extra
            # parameter-free LayerNorm on the target side only, so online and target
            # Level-2 inputs lived on different scales.
            with torch.no_grad():
                t_enc = (self.target_encoder if self.target_encoder is not None
                         else self.encoder)
                fut_summary = t_enc.full_tokens(l2_future).mean(dim=1, keepdim=True)
                l2_tgt = self.level2.encode(fut_summary, target=True)
            l2_tgt = normalize_target(l2_tgt)
            l2_tgt = self.regularizer.transform_target(l2_tgt)

            l2_pred = self.level2.fwd_predictor(l2_ctx, l2_idx)      # (B, 1, D2)
            loss_l2 = prediction_loss(l2_pred, l2_tgt, self.loss_cfg.loss_fn)

            # Level-2 masked infilling over the coarse sequence
            if self.hier_cfg.l2_mask_weight > 0 and W > self.level2.n_mask_slots + 1:
                n_mask = self.level2.n_mask_slots
                perm = torch.randperm(W)[:n_mask].sort().values
                keep = torch.ones(W, dtype=torch.bool)
                keep[perm] = False
                l2_vis = self.level2.encode(summaries, keep=keep)
                with torch.no_grad():
                    l2_all = self.level2.encode(summaries.detach(), target=True)
                l2_masked_tgt = normalize_target(l2_all[:, perm])
                l2_masked_tgt = self.regularizer.transform_target(l2_masked_tgt)
                # v0.2 FIX: tell the predictor where the visible windows sit and which
                # windows it must fill in (v0.1 passed neither -- only a horizon id).
                keep_idx = keep.nonzero(as_tuple=True)[0].to(device)
                l2_masked_pred = self.level2.mask_predictor(
                    l2_vis, None, ctx_pos_index=keep_idx,
                    query_pos_index=perm.to(device))
                loss_l2 = loss_l2 + self.hier_cfg.l2_mask_weight * prediction_loss(
                    l2_masked_pred, l2_masked_tgt, self.loss_cfg.loss_fn)

            loss_l2 = self.hier_cfg.l2_weight * loss_l2

        # ---------------- (d) anti-collapse regulariser (RQ3) -----------------------
        # Regularise the *pooled per-sample* representation by default.
        #
        # Applying it to the flattened token matrix (B*N, D) instead measures variance
        # across samples and patch positions mixed together -- and position alone
        # supplies plenty. Measured here: token-level batch-std 0.110 while the pooled
        # per-sample std was 0.00047, a 233x gap. The hinge was satisfied by positional
        # variance while every *sample* still mapped to the same pooled vector, which is
        # why raising beta from 0.05 to 4.0 changed the outcome by <1e-4. The pooled
        # vector is also exactly what every downstream probe consumes, so it is the
        # thing that has to stay non-degenerate.
        z_pooled = ctx_tokens.reshape(B, W, -1, D).mean(dim=2).reshape(B * W, D)

        if self.reg_target in ("tokens", "both"):
            n_tok = ctx_tokens.size(1)
            if self.reg_tokens and n_tok > self.reg_tokens:
                sel = torch.randperm(n_tok, device=ctx_tokens.device)[:self.reg_tokens]
                z_tok = ctx_tokens[:, sel].reshape(-1, D)
            else:
                z_tok = ctx_tokens.reshape(-1, D)

        if self.reg_target == "pooled":
            loss_reg, reg_stats = self.regularizer(z_pooled)
        elif self.reg_target == "tokens":
            loss_reg, reg_stats = self.regularizer(z_tok)
        else:                                            # 'both'
            lp, reg_stats = self.regularizer(z_pooled)
            lt, _ = self.regularizer(z_tok)
            loss_reg = lp + 0.5 * lt

        z_ctx = z_pooled

        total = (self.loss_cfg.weight_forward * loss_fwd
                 + self.loss_cfg.lambda_mask * loss_mask
                 + self.loss_cfg.beta_reg * loss_reg
                 + loss_l2)

        out.update({
            "loss": total,
            "loss_fwd": loss_fwd.detach(),
            "loss_mask": loss_mask.detach(),
            "loss_reg": loss_reg.detach() if torch.is_tensor(loss_reg) else loss_reg,
            "loss_l2": loss_l2.detach() if torch.is_tensor(loss_l2) else loss_l2,
            "z_ctx": z_ctx.detach(),
        })
        for k, v in reg_stats.items():
            out[k] = torch.as_tensor(v)
        return out

    # ------------------------------------------------------------------ inference
    @torch.no_grad()
    def forecast_latent(self, ctx_windows: torch.Tensor, horizon_s: float) -> torch.Tensor:
        """Query the world model: predicted future latent at ``horizon_s`` seconds.

        Returns (B, D) -- mean-pooled if the predictor is in ``tokens`` mode. This is
        the feature used by the anticipation probes on top of a frozen encoder.
        """
        self.eval()
        if ctx_windows.dim() == 3:
            ctx_windows = ctx_windows.unsqueeze(1)
        B, W, C, T = ctx_windows.shape
        self._check_horizon(horizon_s)
        flat = ctx_windows.reshape(B * W, C, T)
        tok = self.encoder.full_tokens(flat)
        tok = tok.reshape(B, W * tok.size(1), self.embed_dim)
        h_idx = horizon_to_bucket(
            torch.full((B,), float(horizon_s)), self.horizon_buckets).to(ctx_windows.device)
        pred = self.forward_predictor(tok, h_idx, n_context_windows=W,
                                      pooled=self._long_pooled())
        return pred.mean(dim=1) if pred.dim() == 3 else pred

    def _long_pooled(self) -> bool:
        return (self.model_cfg.forward_target == "pooled"
                or self.forward_target_long == "pooled")

    @torch.no_grad()
    def anticipation_features(self, ctx_windows: torch.Tensor,
                              horizons_s: Sequence[float]) -> torch.Tensor:
        """[ pooled context embedding | predicted future latent per horizon ].

        This is the representation the anticipation benchmarks probe: it exposes both
        "what the heart looks like now" and "where the world model thinks it is going".
        """
        self.eval()
        if ctx_windows.dim() == 3:
            ctx_windows = ctx_windows.unsqueeze(1)
        B, W, C, T = ctx_windows.shape
        flat = ctx_windows.reshape(B * W, C, T)
        tok = self.encoder.full_tokens(flat)
        pooled = tok.reshape(B, W, -1, self.embed_dim).mean(dim=(1, 2))     # (B, D)
        feats: List[torch.Tensor] = [pooled]
        tok = tok.reshape(B, W * tok.size(1), self.embed_dim)
        for h in horizons_s:
            self._check_horizon(h)
            h_idx = horizon_to_bucket(torch.full((B,), float(h)),
                                      self.horizon_buckets).to(ctx_windows.device)
            pred = self.forward_predictor(tok, h_idx, n_context_windows=W,
                                          pooled=self._long_pooled())
            feats.append(pred.mean(dim=1) if pred.dim() == 3 else pred)
        return torch.cat(feats, dim=-1)

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/models/baselines.py
"""Baselines for Section 6.3.

  * ``STMEMBaseline``  -- masked *raw-signal* reconstruction over the same
    lead x time patch grid (ST-MEM / ECG-MAE family). Same encoder, same masking,
    but the target is the raw patch instead of a latent, which is exactly the
    "wastes capacity reconstructing high-frequency noise" limitation the proposal
    argues against. Kept architecture-matched so the comparison isolates the
    objective.
  * ``XResNet1d`` / ``Inception1d`` -- supervised-from-scratch references
    (the standard PTB-XL benchmark CNNs).

The masked-infilling ECG-JEPA baseline (Kim 2024) is not a separate class: it is
``CardioJEPA`` with ``weight_forward = 0``, so the two arms share every line of
code except the objective.
"""

from __future__ import annotations

from typing import List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from .encoder import CroPAEncoder, make_block_mask, make_random_mask
from .pos_encoding import get_2d_sincos_pos_embed
from .transformer import TransformerStack, init_weights_, trunc_normal_


class STMEMBaseline(nn.Module):
    """Masked raw-signal reconstruction with the same CroPA encoder."""

    def __init__(self, signal, model, target_norm: str = "none"):
        """
        Args:
            target_norm: v0.2 FIX. 'patch' is v0.1 behaviour (MAE ``norm_pix_loss``
                applied to 0.2 s patches of an *already per-lead z-scored* signal).
                Most 0.2 s ECG patches are iso-electric segments, and standardising
                each of them blows its residual noise up to unit variance, so the
                target is mostly white noise: the v0.1 run sat at loss 0.976 for 30
                epochs with effective rank 1 (never learned). 'none' regresses the
                z-scored signal itself (inputs are already per-lead standardised).
        """
        super().__init__()
        assert target_norm in ("none", "patch")
        self.target_norm = target_norm
        self.signal = signal
        self.model_cfg = model
        C, P, t = signal.n_leads, signal.n_time_patches, signal.patch_samples
        D = model.encoder_embed_dim
        dec_dim = model.predictor_embed_dim

        self.encoder = CroPAEncoder(
            embed_dim=D, depth=model.encoder_depth, num_heads=model.encoder_num_heads,
            mlp_ratio=model.mlp_ratio, qkv_bias=model.qkv_bias,
            drop_path_rate=model.drop_path_rate, init_std=model.init_std,
            pos_type=model.pos_type, use_cropa=model.use_cropa,
            n_leads=C, n_time_patches=P, patch_samples=t)

        self.decoder_embed = nn.Linear(D, dec_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dec_dim))
        self.register_buffer(
            "decoder_pos",
            torch.from_numpy(get_2d_sincos_pos_embed(dec_dim, C, P)).float(),
            persistent=False)
        self.decoder = TransformerStack(embed_dim=dec_dim, depth=model.predictor_depth,
                                        num_heads=model.predictor_num_heads,
                                        qkv_bias=model.qkv_bias)
        self.decoder_norm = nn.LayerNorm(dec_dim)
        self.decoder_pred = nn.Linear(dec_dim, t)

        trunc_normal_(self.mask_token, std=model.init_std)
        self.apply(lambda m: init_weights_(m, model.init_std))

    def representation(self, x):
        return self.encoder.representation(x)

    def forward(self, x: torch.Tensor):
        """x: (B, C, T) -> dict with the reconstruction loss on masked patches."""
        if x.dim() == 4:
            x = x[:, -1]
        B, C, T = x.shape
        P, t = self.signal.n_time_patches, self.signal.patch_samples
        D = self.model_cfg.encoder_embed_dim

        if self.model_cfg.mask_type == "random":
            time_mask = make_random_mask(P, self.model_cfg.mask_scale)
        else:
            time_mask = make_block_mask(P, self.model_cfg.mask_scale)
        time_mask = time_mask.to(x.device)

        vis = self.encoder(x, time_mask=time_mask)                 # (B, C*P_vis, D)
        vis = self.decoder_embed(vis)

        keep = (~time_mask).repeat(C)
        n_tokens = C * P
        full = self.mask_token.expand(B, n_tokens, vis.size(-1)).clone()
        full[:, keep] = vis.to(full.dtype)
        full = full + self.decoder_pos.unsqueeze(0)

        dec = self.decoder_pred(self.decoder_norm(self.decoder(full)))   # (B, C*P, t)

        target = x.reshape(B, C * P, t)
        sel = time_mask.repeat(C)
        tgt = target[:, sel]
        if self.target_norm == "patch":
            mu = tgt.mean(dim=-1, keepdim=True)
            sd = tgt.std(dim=-1, keepdim=True) + 1e-6
            tgt = (tgt - mu) / sd
        loss = F.mse_loss(dec[:, sel].float(), tgt.float())
        # v0.2: no second full-grid encoder pass for `z_ctx` (the engine discards it;
        # it cost a full extra forward per step in v0.1).
        return {"loss": loss, "loss_recon": loss.detach()}

    def update_target(self, momentum: float):   # API parity with CardioJEPA
        return


# --------------------------------------------------------------------------------------
# Supervised references
# --------------------------------------------------------------------------------------
def _conv(ni, nf, ks=3, stride=1):
    return nn.Conv1d(ni, nf, kernel_size=ks, stride=stride, padding=ks // 2, bias=False)


class ResBlock1d(nn.Module):
    def __init__(self, ni, nf, stride=1, ks=5):
        super().__init__()
        self.conv1 = _conv(ni, nf, ks, stride)
        self.bn1 = nn.BatchNorm1d(nf)
        self.conv2 = _conv(nf, nf, ks, 1)
        self.bn2 = nn.BatchNorm1d(nf)
        self.act = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (ni == nf and stride == 1)
                      else nn.Sequential(nn.Conv1d(ni, nf, 1, stride=stride, bias=False),
                                         nn.BatchNorm1d(nf)))

    def forward(self, x):
        idt = self.short(x)
        x = self.act(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.act(x + idt)


class XResNet1d(nn.Module):
    """Compact xresnet1d-style supervised baseline (PTB-XL benchmark family)."""

    def __init__(self, n_leads: int = 8, num_classes: int = 5,
                 widths: List[int] = (64, 128, 256, 512), blocks_per_stage: int = 2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, widths[0], 7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(widths[0]), nn.ReLU(inplace=True),
            nn.MaxPool1d(3, stride=2, padding=1))
        stages: List[nn.Module] = []
        ni = widths[0]
        for i, nf in enumerate(widths):
            for b in range(blocks_per_stage):
                stages.append(ResBlock1d(ni, nf, stride=2 if (b == 0 and i > 0) else 1))
                ni = nf
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(ni, num_classes))
        self.feature_dim = ni

    def features(self, x):
        if x.dim() == 4:
            x = x[:, -1]
        return self.stages(self.stem(x)).mean(dim=-1)

    def representation(self, x):
        return self.features(x)

    def forward(self, x):
        if x.dim() == 4:
            x = x[:, -1]
        return self.head(self.stages(self.stem(x)))


class InceptionModule1d(nn.Module):
    def __init__(self, ni, nf=32, ks=(39, 19, 9), bottleneck=32):
        super().__init__()
        self.bottleneck = (nn.Conv1d(ni, bottleneck, 1, bias=False)
                           if ni > 1 else nn.Identity())
        nb = bottleneck if ni > 1 else ni
        self.convs = nn.ModuleList([_conv(nb, nf, k) for k in ks])
        self.maxpool = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1),
                                     nn.Conv1d(ni, nf, 1, bias=False))
        self.bn = nn.BatchNorm1d(nf * (len(ks) + 1))
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        b = self.bottleneck(x)
        outs = [c(b) for c in self.convs] + [self.maxpool(x)]
        return self.act(self.bn(torch.cat(outs, dim=1)))


class Inception1d(nn.Module):
    """inception1d supervised baseline (PTB-XL benchmark family)."""

    def __init__(self, n_leads: int = 8, num_classes: int = 5, depth: int = 6, nf: int = 32):
        super().__init__()
        layers, ni = [], n_leads
        for _ in range(depth):
            layers.append(InceptionModule1d(ni, nf=nf))
            ni = nf * 4
        self.body = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(ni, num_classes))
        self.feature_dim = ni

    def representation(self, x):
        if x.dim() == 4:
            x = x[:, -1]
        return self.body(x).mean(dim=-1)

    def forward(self, x):
        if x.dim() == 4:
            x = x[:, -1]
        return self.head(self.body(x))

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/models/ssl_baselines.py
"""v0.2: self-supervised baselines that share Cardio-JEPA's exact CroPA encoder.

A Q1 reviewer will ask how a forward-predicting JEPA compares with

  * **SimCLR** -- instance discrimination with ECG-specific augmentations (the
    contrastive family; Kim 2024 reports it at 0.875 PTB-XL linear AUROC), and
  * **CPC**    -- contrastive *forward* latent prediction (van den Oord 2018; Mehari &
    Strodthoff 2022 found it the strongest SSL objective on PTB-XL). CPC is the closest
    prior art to L_fwd: it also predicts future latents, but contrastively and without
    an EMA target. Cardio-JEPA's novelty claim is only defensible against it.

Both are architecture-matched (same ``CroPAEncoder``, same width/depth, same patching),
expose ``representation(x)`` like ``CardioJEPA``, and train with ``engine.pretrain``
on the same interleaved loader, so the comparison isolates the objective.
"""

from __future__ import annotations

import math
from typing import Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from .encoder import CroPAEncoder
from .transformer import init_weights_


def _encoder_from(signal, model) -> CroPAEncoder:
    return CroPAEncoder(
        embed_dim=model.encoder_embed_dim, depth=model.encoder_depth,
        num_heads=model.encoder_num_heads, mlp_ratio=model.mlp_ratio,
        qkv_bias=model.qkv_bias, drop_rate=model.drop_rate,
        attn_drop_rate=model.attn_drop_rate, drop_path_rate=model.drop_path_rate,
        init_std=model.init_std, pos_type=model.pos_type, use_cropa=model.use_cropa,
        n_leads=signal.n_leads, n_time_patches=signal.n_time_patches,
        patch_samples=signal.patch_samples)


# --------------------------------------------------------------------------------------
# ECG augmentations (operate on per-lead z-scored (B, C, T) tensors)
# --------------------------------------------------------------------------------------
def ecg_augment(x: torch.Tensor, fs: float = 250.0,
                generator: Optional[torch.Generator] = None) -> torch.Tensor:
    """Physiologically plausible random transforms (Mehari & Strodthoff 2022 style).

    amplitude scaling per lead, additive Gaussian noise, baseline wander, a random
    circular time shift, random zeroing of one time segment, random lead dropout.
    """
    B, C, T = x.shape
    dev, dt = x.device, x.dtype
    g = generator

    def rand(*shape):
        return torch.rand(*shape, generator=g, device="cpu").to(dev, dt)

    out = x * (0.8 + 0.4 * rand(B, C, 1))                         # amplitude
    out = out + 0.05 * torch.randn(B, C, T, generator=g, device="cpu").to(dev, dt)
    t = torch.arange(T, device=dev, dtype=dt) / fs                # baseline wander
    f = 0.05 + 0.45 * rand(B, 1, 1)
    ph = 2 * math.pi * rand(B, C, 1)
    out = out + (0.1 + 0.2 * rand(B, C, 1)) * torch.sin(2 * math.pi * f * t + ph)

    shifts = (rand(B) * T).long()                                 # circular shift
    idx = (torch.arange(T, device=dev).unsqueeze(0) + shifts.unsqueeze(1)) % T
    out = torch.gather(out, 2, idx.unsqueeze(1).expand(B, C, T))

    seg = int(0.5 * fs)                                           # time cut-out
    starts = (rand(B) * max(T - seg, 1)).long()
    ar = torch.arange(T, device=dev).unsqueeze(0)
    cut = (ar >= starts.unsqueeze(1)) & (ar < (starts + seg).unsqueeze(1))
    keep_cut = rand(B, 1) < 0.5
    out = out.masked_fill((cut & keep_cut).unsqueeze(1), 0.0)

    drop = rand(B, C, 1) < 0.15                                   # lead dropout
    all_dropped = drop.all(dim=1, keepdim=True)                   # keep >= 1 lead
    drop[:, :1] = drop[:, :1] & ~all_dropped
    out = out.masked_fill(drop, 0.0)
    return out


def _no_autocast(device: torch.device):
    """Contrastive logits must be fp32: 1/temperature-scaled fp16 logits overflow."""
    return torch.autocast(device_type=device.type, enabled=False)


def _last_window(x: torch.Tensor) -> torch.Tensor:
    return x[:, -1] if x.dim() == 4 else x


# --------------------------------------------------------------------------------------
# SimCLR
# --------------------------------------------------------------------------------------
class SimCLRBaseline(nn.Module):
    """NT-Xent over two augmented views; projector discarded for evaluation."""

    def __init__(self, signal, model, proj_dim: int = 128, temperature: float = 0.1):
        super().__init__()
        self.signal = signal
        self.temperature = temperature
        self.encoder = _encoder_from(signal, model)
        D = model.encoder_embed_dim
        self.projector = nn.Sequential(nn.Linear(D, D), nn.BatchNorm1d(D),
                                       nn.ReLU(inplace=True), nn.Linear(D, proj_dim))
        self.projector.apply(lambda m: init_weights_(m, model.init_std))

    def representation(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder.representation(_last_window(x))

    def update_target(self, momentum: float):      # API parity with CardioJEPA
        return

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = _last_window(x)
        v1 = ecg_augment(x, self.signal.fs)
        v2 = ecg_augment(x, self.signal.fs)
        z = self.projector(self.encoder.representation(torch.cat([v1, v2])))
        n = x.size(0)
        with _no_autocast(z.device):
            z = F.normalize(z.float(), dim=-1)
            sim = (z @ z.T) / self.temperature
            sim = sim.masked_fill(torch.eye(2 * n, dtype=torch.bool, device=z.device),
                                  -1e9)
            target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
            loss = F.cross_entropy(sim, target)
        with torch.no_grad():
            acc = (sim.argmax(dim=1) == target).float().mean()
        return {"loss": loss, "loss_nce": loss.detach(), "nce_acc": acc}


# --------------------------------------------------------------------------------------
# CPC (intra-window, causal) -- same split as Cardio-JEPA's intra-window L_fwd
# --------------------------------------------------------------------------------------
class CPCBaseline(nn.Module):
    """Contrastive predictive coding inside a 10 s window.

    Context = the first ``ctx_frac`` of the time patches, encoded with the future
    patches *removed* (the CroPA encoder's time mask), so the context never sees the
    future. Targets = per-time-patch latents of the future part, encoded on their own
    (lead-averaged). One prediction head per future offset; InfoNCE with the other
    samples in the batch as negatives. This mirrors Cardio-JEPA's intra-window
    forward objective exactly, except contrastive instead of regression-to-EMA.
    """

    def __init__(self, signal, model, ctx_frac: float = 0.6, temperature: float = 0.1,
                 proj_dim: int = 128):
        super().__init__()
        self.signal = signal
        self.temperature = temperature
        self.encoder = _encoder_from(signal, model)
        P = signal.n_time_patches
        self.k = max(1, min(int(round(P * ctx_frac)), P - 1))
        n_future = P - self.k
        D = model.encoder_embed_dim
        self.ctx_head = nn.Linear(D, proj_dim)
        self.offset_heads = nn.Parameter(torch.randn(n_future, proj_dim, proj_dim) * 0.02)
        self.tgt_head = nn.Linear(D, proj_dim)
        self.ctx_head.apply(lambda m: init_weights_(m, model.init_std))
        self.tgt_head.apply(lambda m: init_weights_(m, model.init_std))

    def representation(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder.representation(_last_window(x))

    def update_target(self, momentum: float):
        return

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = _last_window(x)
        B, C, T = x.shape
        P = self.signal.n_time_patches
        fut_mask = torch.zeros(P, dtype=torch.bool)
        fut_mask[self.k:] = True                                   # True = hidden

        ctx = self.encoder(x, time_mask=fut_mask)                  # (B, C*k, D)
        c = self.ctx_head(ctx.mean(dim=1))                         # (B, d)

        fut = self.encoder(x, time_mask=~fut_mask)                 # (B, C*(P-k), D)
        fut = fut.reshape(B, C, P - self.k, -1).mean(dim=1)        # (B, P-k, D)
        z = self.tgt_head(fut)                                     # (B, P-k, d)

        with _no_autocast(x.device):
            z = F.normalize(z.float(), dim=-1)
            pred = torch.einsum("bd,kde->bke", c.float(), self.offset_heads.float())
            pred = F.normalize(pred, dim=-1)                       # (B, P-k, d)
            logits = torch.einsum("bkd,nkd->kbn", pred, z) / self.temperature
            target = torch.arange(B, device=x.device).unsqueeze(0).expand(P - self.k, B)
            loss = F.cross_entropy(logits.reshape(-1, B), target.reshape(-1))
        with torch.no_grad():
            acc = (logits.argmax(dim=-1) == target).float().mean()
        return {"loss": loss, "loss_nce": loss.detach(), "nce_acc": acc}

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/data/__init__.py
from .build import (build_pretrain_loaders, build_probe_splits, load_long_corpus,
                    probe_batch_from_loader)
from .cache import MemmapWindowDataset, cache_dataset, cache_probe_splits
from .discovery import FoundDataset, discover, summarize
from .longform import (AnticipationDataset, ForwardPairDataset, LongRecord, WindowSample,
                       build_arrhythmia_onset_dataset, build_burden_dataset,
                       build_onset_dataset, collate_mixed, load_long_records,
                       LongSplit, split_by_ids, split_by_record, split_long_records,
                       synthetic_long_records)
from .short import (ArrayWindowDataset, ShortIndex, ShortWindowDataset, build_index,
                    ptbxl_splits, wfdb_splits)

__all__ = [
    "discover", "summarize", "FoundDataset",
    "ShortIndex", "ShortWindowDataset", "ArrayWindowDataset", "build_index",
    "ptbxl_splits", "wfdb_splits",
    "LongRecord", "WindowSample", "ForwardPairDataset", "AnticipationDataset",
    "load_long_records", "synthetic_long_records", "collate_mixed", "split_by_record",
    "build_onset_dataset", "build_arrhythmia_onset_dataset", "build_burden_dataset",
    "build_pretrain_loaders", "build_probe_splits", "load_long_corpus",
    "probe_batch_from_loader", "LongSplit", "split_long_records", "split_by_ids",
    "cache_dataset", "cache_probe_splits", "MemmapWindowDataset",
]

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/data/wfdb_lite.py
"""Minimal dependency-free WFDB reader (headers, signals, annotations).

The default Kaggle image does not ship the ``wfdb`` package and Kaggle notebooks
are often run with internet disabled, so we cannot rely on ``pip install wfdb``.
This module implements just enough of the format to read the databases the
proposal needs:

  * ``.hea`` headers (single-segment)
  * signal formats 212 (MIT-BIH, LTAFDB), 16, 61, 80 and 32
  * ``.atr`` / ``.qrs`` annotation files, including AUX rhythm strings such as
    ``(AFIB`` -- these are what the AF-onset anticipation benchmark is built from

``read_record`` / ``read_annotation`` transparently delegate to the real ``wfdb``
package when it is importable, and fall back to this implementation otherwise.
"""

from __future__ import annotations

import math
import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

try:                                    # pragma: no cover - environment dependent
    import wfdb as _wfdb
except Exception:                       # pragma: no cover
    _wfdb = None


# MIT annotation code -> symbol (WFDB Programmer's Guide, Table 'ecgcodes')
ANN_SYMBOLS: Dict[int, str] = {
    0: " ", 1: "N", 2: "L", 3: "R", 4: "a", 5: "V", 6: "F", 7: "J", 8: "A",
    9: "S", 10: "E", 11: "j", 12: "/", 13: "Q", 14: "~", 16: "|", 18: "s",
    19: "T", 20: "*", 21: "D", 22: '"', 23: "=", 24: "p", 25: "B", 26: "^",
    27: "t", 28: "+", 29: "u", 30: "?", 31: "!", 32: "[", 33: "]", 34: "e",
    35: "n", 36: "@", 37: "x", 38: "f", 39: "(", 40: ")", 41: "r",
}

# Beat labels that count as a "normal" beat vs. an ectopic/arrhythmic beat.
NORMAL_BEATS = frozenset("NLRej")
VENTRICULAR_BEATS = frozenset("VEF")
SUPRAVENTRICULAR_BEATS = frozenset("AaJS")
BEAT_SYMBOLS = frozenset("NLRejVEFAaJS/fQ")


@dataclass
class SignalSpec:
    file_name: str
    fmt: str
    samps_per_frame: int = 1
    byte_offset: int = 0
    adc_gain: float = 200.0
    baseline: int = 0
    units: str = "mV"
    adc_res: int = 12
    adc_zero: int = 0
    init_value: int = 0
    description: str = ""


@dataclass
class Header:
    record_name: str
    n_sig: int
    fs: float
    n_samp: int
    signals: List[SignalSpec] = field(default_factory=list)
    comments: List[str] = field(default_factory=list)

    @property
    def sig_names(self) -> List[str]:
        return [s.description for s in self.signals]

    @property
    def duration_s(self) -> float:
        return self.n_samp / self.fs if self.fs else 0.0


@dataclass
class Record:
    p_signal: np.ndarray          # (n_samp, n_sig) physical units
    fs: float
    sig_name: List[str]
    record_name: str = ""


@dataclass
class Annotation:
    sample: np.ndarray            # int64 sample indices
    symbol: List[str]
    aux_note: List[str]
    fs: float
    subtype: Optional[np.ndarray] = None


# --------------------------------------------------------------------------------------
# header
# --------------------------------------------------------------------------------------
def parse_header(path: str) -> Header:
    with open(path, "r", errors="replace") as fh:
        raw_lines = fh.read().splitlines()

    comments = [ln[1:].strip() for ln in raw_lines if ln.startswith("#")]
    lines = [ln.strip() for ln in raw_lines if ln.strip() and not ln.startswith("#")]
    if not lines:
        raise ValueError(f"empty WFDB header: {path}")

    head = lines[0].split()
    record_name = head[0].split("/")[0]
    n_sig = int(head[1])
    fs = float(head[2].split("/")[0]) if len(head) > 2 else 250.0
    n_samp = int(head[3]) if len(head) > 3 else 0

    signals: List[SignalSpec] = []
    for ln in lines[1:1 + n_sig]:
        parts = ln.split()
        file_name = parts[0]

        fmt_field = parts[1]
        samps_per_frame, byte_offset = 1, 0
        if "+" in fmt_field:
            fmt_field, off = fmt_field.split("+", 1)
            byte_offset = int(off)
        if ":" in fmt_field:
            fmt_field, _skew = fmt_field.split(":", 1)
        if "x" in fmt_field:
            fmt_field, spf = fmt_field.split("x", 1)
            samps_per_frame = int(spf)
        fmt = fmt_field

        adc_gain, baseline, units = 200.0, None, "mV"
        if len(parts) > 2:
            gain_field = parts[2]
            if "/" in gain_field:
                gain_field, units = gain_field.split("/", 1)
            if "(" in gain_field:
                gain_field, base = gain_field.split("(", 1)
                baseline = int(float(base.rstrip(")")))
            adc_gain = float(gain_field) if gain_field else 200.0
            if adc_gain == 0:
                adc_gain = 200.0

        adc_res = int(parts[3]) if len(parts) > 3 and parts[3] else 12
        adc_zero = int(parts[4]) if len(parts) > 4 and parts[4] else 0
        init_value = int(float(parts[5])) if len(parts) > 5 and parts[5] else 0
        desc = " ".join(parts[8:]) if len(parts) > 8 else f"ch{len(signals)}"

        signals.append(SignalSpec(
            file_name=file_name, fmt=fmt, samps_per_frame=samps_per_frame,
            byte_offset=byte_offset, adc_gain=adc_gain,
            baseline=adc_zero if baseline is None else baseline,
            units=units, adc_res=adc_res, adc_zero=adc_zero,
            init_value=init_value, description=desc))

    return Header(record_name=record_name, n_sig=n_sig, fs=fs, n_samp=n_samp,
                  signals=signals, comments=comments)


# --------------------------------------------------------------------------------------
# signals
# --------------------------------------------------------------------------------------
def _decode_212(buf: np.ndarray, n_samples: int) -> np.ndarray:
    """Format 212: two 12-bit samples packed into three bytes."""
    n_triplets = len(buf) // 3
    b = buf[:n_triplets * 3].reshape(-1, 3).astype(np.int32)
    s0 = b[:, 0] + ((b[:, 1] & 0x0F) << 8)
    s1 = b[:, 2] + ((b[:, 1] >> 4) << 8)
    s0 = np.where(s0 > 2047, s0 - 4096, s0)
    s1 = np.where(s1 > 2047, s1 - 4096, s1)
    out = np.empty(n_triplets * 2, dtype=np.int32)
    out[0::2] = s0
    out[1::2] = s1
    return out[:n_samples]


def _decode_16(buf: np.ndarray, n_samples: int) -> np.ndarray:
    return buf.view(np.int16)[:n_samples].astype(np.int32)


def _decode_80(buf: np.ndarray, n_samples: int) -> np.ndarray:
    return buf[:n_samples].astype(np.int32) - 128


def _decode_32(buf: np.ndarray, n_samples: int) -> np.ndarray:
    return buf.view(np.int32)[:n_samples].astype(np.int32)


def _decode_61(buf: np.ndarray, n_samples: int) -> np.ndarray:
    """Format 61: 16-bit, big-endian."""
    return buf.view(">i2")[:n_samples].astype(np.int32)


def read_signal_lite(dirname: str, header: Header, sampfrom: int = 0,
                     sampto: Optional[int] = None) -> np.ndarray:
    """Return (n, n_sig) physical-unit signal.

    Assumes all channels live in a single interleaved data file (true for every
    database used here). Partial reads seek into the file so a 24-hour LTAFDB
    record never has to be materialised in full.
    """
    spec0 = header.signals[0]
    fmt = spec0.fmt
    n_sig = header.n_sig
    total = header.n_samp
    sampto = total if sampto is None else min(sampto, total)
    sampfrom = max(0, min(sampfrom, sampto))
    n_frames = sampto - sampfrom
    if n_frames <= 0:
        return np.zeros((0, n_sig), dtype=np.float32)

    path = os.path.join(dirname, spec0.file_name)

    # PhysioNet Challenge 2020/2021 mirrors (CPSC2018, Chapman, Ningbo) ship the
    # samples in a MATLAB `.mat` file holding a single (n_sig, n_samp) `val` array,
    # with the `.hea` still describing gains and baselines.
    if spec0.file_name.lower().endswith(".mat"):
        from scipy.io import loadmat
        val = loadmat(path)["val"]
        sig = np.asarray(val, dtype=np.float32).T          # -> (n_samp, n_sig)
        sig = sig[sampfrom:sampto]
        for i, spec in enumerate(header.signals[:sig.shape[1]]):
            sig[:, i] = (sig[:, i] - spec.baseline) / (spec.adc_gain or 200.0)
        return sig

    if fmt == "212":
        # 2 samples per 3 bytes over the interleaved stream
        start_sample = sampfrom * n_sig
        aligned_start = (start_sample // 2) * 2            # 3-byte boundary
        lead_in = start_sample - aligned_start
        n_needed = n_frames * n_sig + lead_in
        byte_start = spec0.byte_offset + (aligned_start // 2) * 3
        n_bytes = int(math.ceil((n_needed + 1) / 2) * 3)
        with open(path, "rb") as fh:
            fh.seek(byte_start)
            buf = np.frombuffer(fh.read(n_bytes), dtype=np.uint8)
        flat = _decode_212(buf, n_needed)[lead_in:lead_in + n_frames * n_sig]
    else:
        itemsize = {"16": 2, "61": 2, "80": 1, "32": 4, "8": 1}.get(fmt)
        if itemsize is None:
            raise NotImplementedError(
                f"WFDB format {fmt!r} not supported by wfdb_lite; install the `wfdb` "
                f"package to read this record")
        byte_start = spec0.byte_offset + sampfrom * n_sig * itemsize
        n_bytes = n_frames * n_sig * itemsize
        with open(path, "rb") as fh:
            fh.seek(byte_start)
            buf = np.frombuffer(fh.read(n_bytes), dtype=np.uint8)
        decoder = {"16": _decode_16, "61": _decode_61, "80": _decode_80,
                   "8": _decode_80, "32": _decode_32}[fmt]
        flat = decoder(buf, n_frames * n_sig)

    got = len(flat) // n_sig
    sig = flat[:got * n_sig].reshape(got, n_sig).astype(np.float32)

    for i, spec in enumerate(header.signals):
        sig[:, i] = (sig[:, i] - spec.baseline) / (spec.adc_gain or 200.0)
    return sig


# --------------------------------------------------------------------------------------
# annotations
# --------------------------------------------------------------------------------------
def read_annotation_lite(path: str, fs: float = 250.0) -> Annotation:
    """Parse a WFDB annotation file (``.atr``, ``.qrs``, ...).

    Follows the layout in the WFDB Programmer's Guide: each annotation is a
    16-bit word whose top 6 bits are the type and bottom 10 bits the interval
    since the previous annotation, optionally followed by SUB/CHN/NUM/AUX
    modifier words.
    """
    with open(path, "rb") as fh:
        raw = np.frombuffer(fh.read(), dtype=np.uint8)
    if len(raw) % 2:
        raw = raw[:-1]
    fb = raw.reshape(-1, 2)                       # [:, 0] = low byte, [:, 1] = high byte

    samples: List[int] = []
    symbols: List[str] = []
    auxes: List[str] = []
    subtypes: List[int] = []

    pos, cur = 0, 0
    n = len(fb)
    while pos < n:
        code = int(fb[pos, 1]) >> 2
        if code == 0 and int(fb[pos, 0]) == 0:
            break                                  # end of file
        if code == 59:                             # SKIP: 32-bit interval follows
            if pos + 3 >= n:
                break
            diff = (65536 * int(fb[pos + 1, 0]) + 16777216 * int(fb[pos + 1, 1])
                    + 256 * int(fb[pos + 2, 0]) + int(fb[pos + 2, 1]))
            if diff > 2147483647:
                diff -= 4294967296
            code = int(fb[pos + 3, 1]) >> 2
            pos += 4
        else:
            diff = int(fb[pos, 0]) + 256 * (int(fb[pos, 1]) & 0b11)
            pos += 1

        cur += diff
        aux, sub = "", 0
        # modifier words
        while pos < n:
            mcode = int(fb[pos, 1]) >> 2
            if mcode == 61:                        # SUB
                sub = int(fb[pos, 0])
                pos += 1
            elif mcode in (60, 62):                # NUM, CHN
                pos += 1
            elif mcode == 63:                      # AUX
                alen = int(fb[pos, 0]) + 256 * (int(fb[pos, 1]) & 0b11)
                nwords = int(math.ceil(alen / 2))
                chunk = fb[pos + 1:pos + 1 + nwords].reshape(-1)
                aux = bytes(chunk.tolist())[:alen].decode("ascii", errors="replace")
                aux = aux.rstrip("\x00").strip()
                pos += 1 + nwords
            else:
                break

        samples.append(cur)
        symbols.append(ANN_SYMBOLS.get(code, "?"))
        auxes.append(aux)
        subtypes.append(sub)

    return Annotation(sample=np.asarray(samples, dtype=np.int64), symbol=symbols,
                      aux_note=auxes, fs=fs, subtype=np.asarray(subtypes, dtype=np.int16))


# --------------------------------------------------------------------------------------
# public API (delegates to `wfdb` when available)
# --------------------------------------------------------------------------------------
def read_header(record_path: str) -> Header:
    return parse_header(record_path + ".hea")


def read_record(record_path: str, sampfrom: int = 0, sampto: Optional[int] = None,
                channels: Optional[Sequence[int]] = None, prefer_wfdb: bool = True
                ) -> Record:
    """Read a WFDB record. ``record_path`` is the path *without* extension."""
    if prefer_wfdb and _wfdb is not None:
        try:
            kw = {"sampfrom": sampfrom}
            if sampto is not None:
                kw["sampto"] = sampto
            if channels is not None:
                kw["channels"] = list(channels)
            r = _wfdb.rdrecord(record_path, **kw)
            sig = np.asarray(r.p_signal, dtype=np.float32)
            return Record(p_signal=sig, fs=float(r.fs), sig_name=list(r.sig_name),
                          record_name=os.path.basename(record_path))
        except Exception:
            pass                                   # fall through to the lite reader

    hdr = parse_header(record_path + ".hea")
    sig = read_signal_lite(os.path.dirname(record_path), hdr, sampfrom, sampto)
    names = hdr.sig_names
    if channels is not None:
        sig = sig[:, list(channels)]
        names = [names[i] for i in channels]
    return Record(p_signal=sig, fs=hdr.fs, sig_name=names,
                  record_name=os.path.basename(record_path))


def read_annotation(record_path: str, extension: str = "atr",
                    prefer_wfdb: bool = True) -> Annotation:
    fs = 250.0
    try:
        fs = parse_header(record_path + ".hea").fs
    except Exception:
        pass

    if prefer_wfdb and _wfdb is not None:
        try:
            a = _wfdb.rdann(record_path, extension)
            aux = [("" if x is None else str(x).rstrip("\x00").strip())
                   for x in (a.aux_note or [""] * len(a.sample))]
            return Annotation(sample=np.asarray(a.sample, dtype=np.int64),
                              symbol=list(a.symbol), aux_note=aux,
                              fs=float(a.fs or fs))
        except Exception:
            pass

    return read_annotation_lite(record_path + "." + extension, fs=fs)


def list_records(root: str, extensions: Sequence[str] = ("hea",),
                 limit: Optional[int] = None,
                 exclude_dirs: Sequence[str] = ()) -> List[str]:
    """Find WFDB records (paths without extension) under ``root``.

    ``exclude_dirs`` prunes sub-trees that belong to a different corpus -- real
    Kaggle mounts nest one database inside another (a PTB-XL mount that also ships
    the MIT-BIH Noise Stress database), and without pruning those records leak into
    the parent corpus's index.
    """
    out: List[str] = []
    excluded = {os.path.abspath(p) for p in exclude_dirs}
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.abspath(dirpath) in excluded:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames
                       if os.path.abspath(os.path.join(dirpath, d)) not in excluded]
        for fn in sorted(filenames):
            for ext in extensions:
                if fn.endswith("." + ext):
                    out.append(os.path.join(dirpath, fn[: -(len(ext) + 1)]))
                    break
        if limit is not None and len(out) >= limit:
            return sorted(out)[:limit]
    out = sorted(set(out))
    return out[:limit] if limit is not None else out

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/data/preprocess.py
"""Signal preprocessing shared by every dataset loader."""

from __future__ import annotations

from typing import Optional, Sequence

import numpy as np

from ..config import REDUCED_LEAD_IDX

try:
    from scipy.signal import butter, resample_poly, sosfiltfilt
    _HAVE_SCIPY = True
except Exception:                                   # pragma: no cover
    _HAVE_SCIPY = False


def resample_signal(x: np.ndarray, fs_in: float, fs_out: float) -> np.ndarray:
    """Resample along the last axis. x: (..., T)."""
    if abs(fs_in - fs_out) < 1e-6:
        return x.astype(np.float32, copy=False)
    if _HAVE_SCIPY:
        from fractions import Fraction
        frac = Fraction(float(fs_out) / float(fs_in)).limit_denominator(1000)
        return resample_poly(x, frac.numerator, frac.denominator,
                             axis=-1).astype(np.float32)
    # linear-interpolation fallback
    n_out = int(round(x.shape[-1] * fs_out / fs_in))
    src = np.linspace(0.0, 1.0, x.shape[-1], dtype=np.float64)
    dst = np.linspace(0.0, 1.0, n_out, dtype=np.float64)
    flat = x.reshape(-1, x.shape[-1])
    out = np.stack([np.interp(dst, src, row) for row in flat])
    return out.reshape(*x.shape[:-1], n_out).astype(np.float32)


def bandpass(x: np.ndarray, fs: float, low: float = 0.5, high: float = 45.0,
             order: int = 3) -> np.ndarray:
    """Zero-phase bandpass. Removes baseline wander (<0.5 Hz) and EMG/mains (>45 Hz)."""
    if not _HAVE_SCIPY:
        return x.astype(np.float32, copy=False)
    nyq = fs / 2.0
    high = min(high, nyq * 0.95)
    if low >= high or x.shape[-1] < 3 * order * 4:
        return x.astype(np.float32, copy=False)
    sos = butter(order, [low / nyq, high / nyq], btype="band", output="sos")
    return sosfiltfilt(sos, x, axis=-1).astype(np.float32)


def reduce_leads(x: np.ndarray, lead_idx: Sequence[int] = REDUCED_LEAD_IDX) -> np.ndarray:
    """12-lead -> 8 independent leads (I, II, V1..V6). x: (..., C, T)."""
    if x.shape[-2] == len(lead_idx):
        return x
    if x.shape[-2] < 12:
        raise ValueError(f"expected 12 leads to reduce, got {x.shape[-2]}")
    return np.take(x, list(lead_idx), axis=-2)


def normalize(x: np.ndarray, mode: str = "instance", eps: float = 1e-6) -> np.ndarray:
    """Per-lead standardisation. x: (..., C, T)."""
    if mode == "none":
        return x
    if mode == "instance":                          # per lead, per recording
        mu = x.mean(axis=-1, keepdims=True)
        sd = x.std(axis=-1, keepdims=True)
        return ((x - mu) / (sd + eps)).astype(np.float32)
    if mode == "global":                            # shared across leads
        mu = x.mean(axis=(-2, -1), keepdims=True)
        sd = x.std(axis=(-2, -1), keepdims=True)
        return ((x - mu) / (sd + eps)).astype(np.float32)
    raise ValueError(f"unknown normalize mode {mode!r}")


def clean(x: np.ndarray, clip: float = 20.0) -> np.ndarray:
    """Replace NaN/Inf and clip extreme values left by lead-off artefacts."""
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(x, -clip, clip).astype(np.float32)


def is_valid(x: np.ndarray, min_std: float = 1e-4) -> bool:
    """Reject flat-line / all-zero / NaN recordings (ECG-JEPA `remove_invalid_samples`)."""
    if not np.isfinite(x).all():
        return False
    if np.abs(x[..., :15]).sum() == 0:
        return False
    return float(x.std()) > min_std


def prepare_window(x: np.ndarray, fs_in: float, fs_out: float,
                   n_samples: int, reduced: bool = True,
                   do_bandpass: bool = True, norm: str = "instance") -> np.ndarray:
    """Full pipeline for one window: (C, T_in) -> (C_out, n_samples) float32."""
    x = np.asarray(x, dtype=np.float32)
    if reduced and x.shape[0] >= 12:
        x = reduce_leads(x)
    x = clean(x)
    if do_bandpass:
        x = bandpass(x, fs_in)
    x = resample_signal(x, fs_in, fs_out)
    if x.shape[-1] < n_samples:
        x = np.pad(x, ((0, 0), (0, n_samples - x.shape[-1])), mode="edge")
    x = x[..., :n_samples]
    return normalize(x, norm)


def expand_to_n_leads(x: np.ndarray, n_leads: int) -> np.ndarray:
    """Tile/truncate the lead axis so single- or two-channel data (MIT-BIH, LTAFDB)
    can be fed to an 8-lead model.

    Long-term Holter databases record 2 channels, not 12. Rather than train a
    separate model we replicate the available channels across the lead axis; the
    encoder's positional embedding still distinguishes the slots, and the lead
    dimension degenerates gracefully. Documented explicitly because it is a real
    modelling compromise, not a no-op.
    """
    c = x.shape[-2]
    if c == n_leads:
        return x
    if c > n_leads:
        return x[..., :n_leads, :]
    reps = int(np.ceil(n_leads / c))
    return np.concatenate([x] * reps, axis=-2)[..., :n_leads, :]

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/data/discovery.py
"""Auto-discovery of ECG corpora mounted under ``/kaggle/input``.

Kaggle does not use one fixed mount layout. Depending on how an input was added
you get either::

    /kaggle/input/<slug>/...                                    (classic)
    /kaggle/input/datasets/<owner>/<slug>/<subdir>/...          (current)
    /kaggle/input/models/<owner>/<model>/<fw>/<var>/<ver>/...   (models)

so we cannot just keyword-match the top-level directory names -- doing that on the
current layout matches only ``datasets`` and ``models`` and finds nothing. Instead
we walk directories (not files) to a bounded depth and match each directory *name*
against a corpus pattern, keeping the shallowest match per corpus.

Anything under a top-level ``models/`` directory is skipped: that is code and
checkpoints, not data (see ``package_source.py``).

Verified Kaggle sources (slug -> role in the proposal's Table 6.1):

    ishtiaqueahmed354/metti-dataset (MEETI)                   MIMIC-IV-ECG derived
    biswajitkrshaw/mimic-iv-ecg-diagnostic-electrocardiogram  MIMIC-IV-ECG  pretrain
    ccf002/code-15-ecg-complete | gerardolug/ecg-code-15      CODE-15       pretrain
    erarayamorenzomuten/chapmanshaoxing-12lead-ecg-database   Chapman       pretrain
    bjoernjostein/shaoxing-and-ningbo-first-hospital-database Chapman+Ningbo
    physionet/china-physiological-signal-challenge-in-2018    CPSC2018      pretrain+down
    khyeh0719/ptb-xl-dataset | omar1863/ptb-xl                PTB-XL        pretrain+down
    sahilshahare12/long-term-af-database                      LTAFDB        forward/hier
    protobioengineering/mit-bih-arrhythmia-database-modern-2023  MIT-BIH    forward/hier
"""

from __future__ import annotations

import os
import re
from collections import deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

# key -> (regex over a directory name, human label, kind)
#
# Order matters: the first pattern that matches a directory name wins, so the more
# specific patterns (noise-stress-test before mit-bih) must come first.
DATASET_PATTERNS: List[Tuple[str, str, str, str]] = [
    # MEETI is built from MIMIC-IV-ECG (784,680 12-lead 10 s ECGs, 160,597 patients),
    # so it fills the proposal's primary large-scale pretraining slot. The Kaggle
    # dataset is spelled "metti-dataset" with the payload directory "MEETI".
    ("meeti",   r"(meeti|metti)",                              "MEETI (MIMIC-IV-ECG)", "short"),
    # nstdb is *not* the MIT-BIH Arrhythmia Database: it is 12 deliberately
    # noise-corrupted records plus three pure-noise records, and it carries no AF
    # rhythm annotations. Matched before `mitbih` so it is never mistaken for it.
    ("mitbih_nst", r"(noise[-_]?stress|nstdb)",                "MIT-BIH Noise Stress", "noise"),
    ("ptbxl",   r"ptb[-_ ]?xl",                                "PTB-XL",             "short"),
    ("cpsc",    r"(cpsc|china[-_]physiological)",              "CPSC2018",           "short"),
    ("chapman", r"(chapman|shaoxing)",                         "Chapman-Shaoxing",   "short"),
    ("ningbo",  r"ningbo",                                     "Ningbo",             "short"),
    ("code15",  r"code[-_ ]?15",                               "CODE-15",            "short"),
    ("mimic",   r"mimic",                                      "MIMIC-IV-ECG",       "short"),
    ("ltaf",    r"(long[-_ ]?term[-_ ]?af|ltafdb|ltaf)",       "Long-Term AF",       "long"),
    # v0.2 FIX: the MIT-BIH *Long-Term ECG* database (ltdb, 128 Hz, 7 multi-hour
    # records) was matched by the generic `mit-bih` pattern and reported -- and read
    # -- as the MIT-BIH Arrhythmia database (360 Hz, 48 x 30 min). Matched first now.
    ("mitbih_lt", r"(mit[-_ ]?bih[-_ ]?long[-_ ]?term|long[-_ ]?term[-_ ]?ecg|ltdb)",
                                                               "MIT-BIH Long-Term ECG", "long"),
    ("mitbih",  r"(mit[-_ ]?bih|mitdb)",                       "MIT-BIH Arrhythmia", "long"),
]

# A mirror that ships Chapman and Ningbo together
COMBINED_PATTERNS = [(r"shaoxing[-_ ]and[-_ ]ningbo",
                      [("chapman", "Chapman-Shaoxing", "short"),
                       ("ningbo", "Ningbo", "short")])]

# Directory names that never contain ECG corpora
SKIP_TOP_LEVEL = {"models", "model", "kernels", "competitions"}


@dataclass
class FoundDataset:
    key: str
    label: str
    kind: str                    # 'short' | 'long' (Holter) | 'noise'
    root: str                    # directory whose name matched
    data_root: str               # directory that actually holds the signals
    fmt: str                     # 'wfdb' | 'hdf5' | 'ptbxl' | 'mitbih_csv' | 'npy'
    n_files: int = 0
    #: Sub-directories belonging to a *different* corpus nested inside this one.
    #: Real mounts do this -- `omar1863/ptb-xl` ships the MIT-BIH Noise Stress
    #: database inside the PTB-XL mount -- and without excluding them a PTB-XL
    #: index can silently swallow noise records.
    exclude_dirs: List[str] = field(default_factory=list)
    extra: Dict[str, str] = field(default_factory=dict)

    def __repr__(self) -> str:      # pragma: no cover - display only
        return (f"<{self.label} kind={self.kind} fmt={self.fmt} "
                f"n={self.n_files} at {self.data_root}>")


def _match_key(name: str) -> Optional[Tuple[str, str, str]]:
    low = name.lower()
    for key, pat, label, kind in DATASET_PATTERNS:
        if re.search(pat, low):
            return key, label, kind
    return None


def _match_combined(name: str):
    low = name.lower()
    for pat, keys in COMBINED_PATTERNS:
        if re.search(pat, low):
            return keys
    return None


def _candidate_roots(input_root: str, max_depth: int = 6,
                     max_dirs: int = 20000) -> List[Tuple[str, str, str, str, int]]:
    """BFS over *directories only*, returning (key, label, kind, path, depth).

    Files are never listed here -- a corpus directory can hold hundreds of
    thousands of them, and ``os.walk`` would stat every one just to find a name.
    """
    out: List[Tuple[str, str, str, str, int]] = []
    queue: deque = deque([(input_root, 0)])
    seen_dirs = 0

    while queue and seen_dirs < max_dirs:
        path, depth = queue.popleft()
        if depth >= max_depth:
            continue
        try:
            entries = list(os.scandir(path))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue

        for e in entries:
            try:
                if not e.is_dir(follow_symlinks=False):
                    continue
            except OSError:
                continue
            if e.name.startswith("."):
                continue
            if depth == 0 and e.name.lower() in SKIP_TOP_LEVEL:
                continue

            seen_dirs += 1
            combined = _match_combined(e.name)
            if combined:
                for key, label, kind in combined:
                    out.append((key, label, kind, e.path, depth + 1))
                continue

            m = _match_key(e.name)
            if m:
                key, label, kind = m
                out.append((key, label, kind, e.path, depth + 1))
                # keep descending: a mount can nest several corpora (e.g. a
                # "ptb-xl" dataset that also ships the noise-stress database)
            queue.append((e.path, depth + 1))

    return out


def _walk_probe(root: str, max_files: int = 20_000,
                exclude_dirs: Sequence[str] = ()) -> Dict[str, object]:
    """Single bounded walk collecting the signals needed to classify a corpus.

    v0.2: capped at 20k files (was 200k) -- the format is decided by the first few
    hundred files, and on Kaggle's network file system each 200k-file walk took many
    minutes (v0.1 spent 51 min in discovery). File counts are therefore lower bounds,
    printed with a '+'.
    """
    counts = {"hea": 0, "mat": 0, "dat": 0, "hdf5": 0, "npy": 0, "csv": 0, "atr": 0}
    first: Dict[str, str] = {}
    ptbxl_csv: Optional[str] = None
    mitbih_csv = 0
    seen = 0
    excluded = {os.path.abspath(p) for p in exclude_dirs}

    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.abspath(dirpath) in excluded:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")
                       and os.path.abspath(os.path.join(dirpath, d)) not in excluded]
        for fn in filenames:
            seen += 1
            if seen > max_files:
                break
            low = fn.lower()
            if low == "ptbxl_database.csv":
                ptbxl_csv = os.path.join(dirpath, fn)
            if low.endswith("_ekg.csv"):
                mitbih_csv += 1
            for ext in counts:
                if low.endswith("." + ext):
                    counts[ext] += 1
                    first.setdefault(ext, os.path.join(dirpath, fn))
                    break
        if seen > max_files:
            break

    return {"counts": counts, "first": first, "ptbxl_csv": ptbxl_csv,
            "mitbih_csv": mitbih_csv, "truncated": seen > max_files}


def _classify_format(probe: Dict[str, object]) -> Optional[str]:
    counts = probe["counts"]                       # type: ignore[index]
    if probe["ptbxl_csv"]:                         # type: ignore[index]
        return "ptbxl"
    if probe["mitbih_csv"]:                        # type: ignore[index]
        return "mitbih_csv"
    if counts["hdf5"] > 0:
        return "hdf5"
    if counts["hea"] > 0:
        return "wfdb"
    if counts["npy"] > 0:
        return "npy"
    return None


def _data_root(probe: Dict[str, object], fmt: str, mount: str) -> str:
    first = probe["first"]                         # type: ignore[index]
    if fmt == "ptbxl":
        return os.path.dirname(str(probe["ptbxl_csv"]))    # type: ignore[index]
    if fmt == "hdf5" and "hdf5" in first:
        return os.path.dirname(first["hdf5"])
    if fmt == "npy" and "npy" in first:
        return os.path.dirname(first["npy"])
    return mount                                   # loaders walk recursively


def discover(input_root: str = "/kaggle/input",
             only: Optional[Sequence[str]] = None,
             verbose: bool = True,
             max_depth: int = 6,
             skip: Sequence[str] = ()) -> Dict[str, FoundDataset]:
    """Return {key: FoundDataset} for every recognised corpus under ``input_root``.

    ``skip`` (v0.2): corpus keys that are not even walked -- e.g. ``("meeti",)`` while
    no reader exists for its format.
    """
    skip = set(skip or ())
    found: Dict[str, FoundDataset] = {}
    if not os.path.isdir(input_root):
        if verbose:
            print(f"[discovery] {input_root} does not exist - nothing mounted.")
        return found

    candidates = _candidate_roots(input_root, max_depth=max_depth)
    # shallowest match wins for each corpus key
    candidates.sort(key=lambda c: (c[0], c[4], c[3]))

    # One root per corpus key, resolved before probing so that a corpus nested inside
    # another (PTB-XL mounts that also ship the noise-stress database) can be excluded
    # from its parent's probe rather than polluting the parent's classification.
    chosen: Dict[str, Tuple[str, str, str]] = {}
    for key, label, kind, path, _depth in candidates:
        if key in chosen or key in skip or (only is not None and key not in only):
            continue
        chosen[key] = (label, kind, path)

    all_roots = [p for _l, _k, p in chosen.values()]

    probe_cache: Dict[Tuple[str, Tuple[str, ...]], Dict[str, object]] = {}
    for key, (label, kind, path) in chosen.items():
        nested = [r for r in all_roots
                  if r != path and os.path.abspath(r).startswith(
                      os.path.abspath(path) + os.sep)]
        # v0.2: a combined mirror (Chapman+Ningbo) maps two keys to one directory;
        # walk it once, not once per key.
        ck = (os.path.abspath(path), tuple(sorted(nested)))
        if ck not in probe_cache:
            probe_cache[ck] = _walk_probe(path, exclude_dirs=nested)
        probe = probe_cache[ck]
        fmt = _classify_format(probe)
        if fmt is None:
            if verbose:
                print(f"[discovery] {label}: no readable ECG files under {path}, skipping")
            continue

        counts = probe["counts"]                   # type: ignore[index]
        n_files = {"ptbxl": counts["dat"] or counts["hea"],
                   "wfdb": counts["hea"],
                   "hdf5": counts["hdf5"],
                   "npy": counts["npy"],
                   "mitbih_csv": probe["mitbih_csv"]}[fmt]   # type: ignore[index]

        found[key] = FoundDataset(key=key, label=label, kind=kind, root=path,
                                  data_root=_data_root(probe, fmt, path), fmt=fmt,
                                  n_files=int(n_files), exclude_dirs=nested)
        if verbose:
            approx = "+" if probe.get("truncated") else ""
            print(f"[discovery] {label:24s} fmt={fmt:10s} files~{n_files}{approx:<2s} {path}")
            for n in nested:
                print(f"             excluding nested corpus: {os.path.basename(n)}")

    if verbose:
        if not found:
            print(f"[discovery] no recognised ECG datasets under {input_root}.")
        _warn_about_gaps(found)
    return found


def _warn_about_gaps(found: Dict[str, FoundDataset]):
    """Say plainly which experiments the attached data cannot support."""
    has_long = any(d.kind == "long" and d.key in ("ltaf", "mitbih")
                   for d in found.values())
    if not has_long:
        print("\n[discovery] NOTE: no long-recording corpus with rhythm annotations "
              "(Long-Term AF / MIT-BIH Arrhythmia) was found.")
        print("            Without one there is no real 'future window' and no AF-onset "
              "labels, so:")
        print("              - L_fwd trains only via the intra-window causal path;")
        print("              - Stage 2 (anticipation) and Stage 4 (hierarchical) fall "
              "back to synthetic data.")
        print("            Attach `sahilshahare12/long-term-af-database` and/or "
              "`protobioengineering/mit-bih-arrhythmia-database-modern-2023` to enable them.")
    if "mitbih_nst" in found and "mitbih" not in found:
        print("\n[discovery] NOTE: the MIT-BIH *Noise Stress Test* database was found. "
              "It is deliberately noise-corrupted and has no AF rhythm annotations, so "
              "it is not used as an anticipation corpus. Enable it for pretraining with "
              "cfg.data.use_noise_corpora = True.")


def summarize(found: Dict[str, FoundDataset]) -> str:
    if not found:
        return "No ECG datasets discovered."
    lines = [f"{'key':12s} {'dataset':24s} {'kind':6s} {'fmt':11s} {'files':>9s}",
             "-" * 68]
    for k, d in sorted(found.items()):
        lines.append(f"{k:12s} {d.label:24s} {d.kind:6s} {d.fmt:11s} {d.n_files:>9d}")
    return "\n".join(lines)

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/data/synthetic.py
"""Synthetic ECG generator.

Purpose: make the notebook runnable end-to-end with **zero** attached datasets, so
every code path (pretraining, all four RQ3 arms, the hierarchical stack, the
anticipation benchmarks) can be smoke-tested before a GPU quota is spent.

The generator produces a P-QRS-T complex as a sum of Gaussians (the standard
McSharry-style parameterisation), with two rhythm states:

  * **sinus**  -- regular RR with respiratory sinus arrhythmia, P wave present
  * **AF**     -- irregularly irregular RR, P wave replaced by fibrillatory waves

Long synthetic records alternate between the two so the AF-onset anticipation
benchmark has something to detect. These signals are *not* a substitute for real
data: they exist to validate the pipeline, and any number computed on them should
be read as a smoke-test result, not as evidence about ECG.
"""

from __future__ import annotations

from typing import Dict, List, Optional, Tuple

import numpy as np

# (amplitude, centre in seconds relative to R, width in seconds)
_WAVES_SINUS = [
    (0.10, -0.20, 0.025),    # P
    (-0.10, -0.035, 0.008),  # Q
    (1.20, 0.000, 0.010),    # R
    (-0.25, 0.035, 0.012),   # S
    (0.30, 0.200, 0.045),    # T
]
_WAVES_AF = [
    (-0.10, -0.035, 0.008),
    (1.15, 0.000, 0.010),
    (-0.25, 0.035, 0.012),
    (0.28, 0.200, 0.045),
]

# per-lead amplitude/polarity scaling for the 8 reduced leads (I, II, V1..V6)
_LEAD_GAIN = np.array([0.7, 1.0, -0.4, -0.2, 0.5, 0.9, 1.0, 0.8], dtype=np.float32)


def _beat(t: np.ndarray, waves) -> np.ndarray:
    out = np.zeros_like(t)
    for amp, mu, sigma in waves:
        out += amp * np.exp(-0.5 * ((t - mu) / sigma) ** 2)
    return out


def _rr_intervals(duration_s: float, rhythm: str, hr: float,
                  rng: np.random.Generator) -> np.ndarray:
    base = 60.0 / hr
    rr: List[float] = []
    total = 0.0
    while total < duration_s:
        if rhythm == "af":
            # irregularly irregular: heavy-tailed multiplicative jitter
            interval = base * float(np.clip(rng.lognormal(0.0, 0.22), 0.45, 2.2))
        else:
            # sinus with respiratory modulation (~0.25 Hz) + small noise
            phase = 2 * np.pi * 0.25 * total
            interval = base * (1.0 + 0.05 * np.sin(phase)) * float(
                np.clip(rng.normal(1.0, 0.025), 0.8, 1.25))
        rr.append(interval)
        total += interval
    return np.asarray(rr, dtype=np.float64)


def synth_segment(duration_s: float, fs: float, rhythm: str = "sinus",
                  hr: float = 70.0, n_leads: int = 8,
                  rng: Optional[np.random.Generator] = None,
                  noise: float = 0.03) -> np.ndarray:
    """Return (n_leads, duration_s*fs) synthetic ECG."""
    rng = rng or np.random.default_rng()
    n = int(round(duration_s * fs))
    trace = np.zeros(n, dtype=np.float32)

    waves = _WAVES_AF if rhythm == "af" else _WAVES_SINUS
    half = 0.35                                   # seconds of beat template each side
    tt = np.arange(-half, half, 1.0 / fs)
    template = _beat(tt, waves).astype(np.float32)

    pos = 0.0
    for interval in _rr_intervals(duration_s, rhythm, hr, rng):
        centre = int(round(pos * fs))
        lo = centre - len(template) // 2
        a, b = max(0, lo), min(n, lo + len(template))
        if b > a:
            trace[a:b] += template[a - lo:b - lo]
        pos += interval

    if rhythm == "af":
        # fibrillatory baseline: narrowband 5-9 Hz wander replacing organised P waves
        t = np.arange(n) / fs
        f = rng.uniform(5.0, 9.0)
        trace += (0.05 * np.sin(2 * np.pi * f * t + rng.uniform(0, 6.28))).astype(np.float32)

    sig = np.outer(_LEAD_GAIN[:n_leads] if n_leads <= 8
                   else np.resize(_LEAD_GAIN, n_leads), trace).astype(np.float32)

    # baseline wander + white noise, per lead
    t = np.arange(n) / fs
    for c in range(sig.shape[0]):
        sig[c] += 0.08 * np.sin(2 * np.pi * rng.uniform(0.15, 0.4) * t
                                + rng.uniform(0, 6.28)).astype(np.float32)
        sig[c] += rng.normal(0.0, noise, size=n).astype(np.float32)
    return sig


def synth_short_corpus(n: int, fs: float = 250.0, window_s: float = 10.0,
                       n_leads: int = 8, af_frac: float = 0.25,
                       seed: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    """n synthetic 10 s clips. Returns (waves (n, C, T), labels (n,) 1 = AF)."""
    rng = np.random.default_rng(seed)
    waves, labels = [], []
    for _ in range(n):
        is_af = rng.random() < af_frac
        hr = rng.uniform(90, 150) if is_af else rng.uniform(52, 95)
        waves.append(synth_segment(window_s, fs, "af" if is_af else "sinus", hr,
                                   n_leads, rng))
        labels.append(int(is_af))
    return np.stack(waves), np.asarray(labels, dtype=np.int64)


def synth_long_record(minutes: float = 40.0, fs: float = 250.0, n_leads: int = 8,
                      seed: int = 0, af_burden: Optional[float] = None,
                      af_episode_s: Tuple[float, float] = (30.0, 180.0)
                      ) -> Dict[str, object]:
    """One long synthetic Holter-style record alternating sinus <-> AF.

    ``af_burden`` is the target fraction of the record spent in AF; if None it is
    drawn per record so the corpus spans everything from AF-free to AF-dominated.
    Sinus episode lengths are then set so the realised burden matches, which is what
    makes *paroxysmal* AF (rare, short bursts inside long sinus stretches) possible.

    This matters for the anticipation benchmark: with equal-length alternating
    episodes, every sinus window has an AF onset close behind it and the negative
    class is empty, so the benchmark is undefined. Long sinus stretches are what
    create genuine negatives.

    Returns dict with ``signal`` (C, T), ``fs`` and ``episodes``: a list of
    ``(start_sample, end_sample, rhythm)`` triples in the same shape the LTAFDB
    loader produces, so downstream code is identical for real and synthetic data.
    """
    rng = np.random.default_rng(seed)
    total_s = minutes * 60.0

    if af_burden is None:
        # Paroxysmal regime: low burden, so sinus stretches are long relative to the
        # forecast horizons. A record only contains *both* anticipation classes when
        # its sinus stretches exceed the horizon -- some windows then sit far from any
        # onset (negatives) and some sit just before one (positives). Sampling burden
        # uniformly over [0, 0.9] instead yields records that are all-positive or
        # all-negative, and a record-level split of those cannot produce a usable
        # test set. 10% of records are AF-free throughout.
        af_burden = 0.0 if rng.random() < 0.10 else float(0.03 + 0.25 * rng.beta(1.5, 2.5))
    af_burden = float(np.clip(af_burden, 0.0, 0.9))

    a_bar = float(np.mean(af_episode_s))
    # E[sinus] chosen so E[af] / (E[af] + E[sinus]) == af_burden
    s_bar = total_s * 2 if af_burden <= 1e-6 else a_bar * (1.0 - af_burden) / af_burden

    segments, episodes = [], []
    pos_s, cursor = 0.0, 0
    rhythm = "sinus"                       # always start in sinus

    while pos_s < total_s:
        if rhythm == "af":
            dur = float(rng.uniform(*af_episode_s))
        else:
            dur = float(rng.uniform(0.5 * s_bar, 1.5 * s_bar))
        dur = min(dur, total_s - pos_s)
        if dur <= 1.0:
            break

        hr = rng.uniform(95, 155) if rhythm == "af" else rng.uniform(50, 90)
        seg = synth_segment(dur, fs, rhythm, hr, n_leads, rng)
        segments.append(seg)
        episodes.append((cursor, cursor + seg.shape[1], "AFIB" if rhythm == "af" else "N"))
        cursor += seg.shape[1]
        pos_s += dur
        rhythm = "af" if rhythm == "sinus" else "sinus"

    signal = (np.concatenate(segments, axis=1) if segments
              else np.zeros((n_leads, int(total_s * fs)), dtype=np.float32))
    return {"signal": signal, "fs": fs, "episodes": episodes,
            "name": f"synth{seed:03d}", "n_samples": signal.shape[1],
            "target_af_burden": af_burden}


def synth_long_corpus(n_records: int, minutes: float = 40.0, fs: float = 250.0,
                      n_leads: int = 8, seed: int = 0) -> List[Dict[str, object]]:
    return [synth_long_record(minutes, fs, n_leads, seed=seed + i) for i in range(n_records)]

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/data/short.py
"""Loaders for the 10-second, 12-lead corpora.

Covers every "short" row of the proposal's Table 6.1:
MIMIC-IV-ECG, CODE-15, Chapman-Shaoxing, Ningbo, CPSC2018 and PTB-XL.

Everything is *lazy*: we build an index of record locators and decode signals in
``__getitem__``. MIMIC-IV-ECG alone is ~800k 10-second 12-lead recordings -- roughly
200 GB as float32 -- so eager loading is not an option at any session RAM size.
``DataConfig.max_records_per_dataset`` additionally caps how many locators are indexed.
"""

from __future__ import annotations

import ast
import os
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset

from ..config import DataConfig, SignalConfig
from . import wfdb_lite
from .discovery import FoundDataset
from .preprocess import expand_to_n_leads, is_valid, prepare_window

# SNOMED-CT -> label name for the nine CPSC2018 classes (used for display; the
# multi-label matrix itself is built generically from whatever Dx codes appear).
SNOMED_NAMES: Dict[str, str] = {
    "426783006": "SNR", "164889003": "AF", "164890007": "AFL",
    "270492004": "IAVB", "164909002": "LBBB", "59118001": "RBBB",
    "713427006": "CRBBB", "713426002": "IRBBB", "284470004": "PAC",
    "63593006": "SVPB", "164884008": "PVC", "427172004": "PVC",
    "429622005": "STD", "164931005": "STE", "39732003": "LAD",
    "164934002": "TAB", "111975006": "LQT", "164917005": "QAb",
    "47665007": "RAD", "251146004": "LQRSV", "446358003": "RAH",
    "365413008": "LVH", "426177001": "SB", "427084000": "STach",
    "426761007": "SVT", "17338001": "VPB",
}


@dataclass
class ShortIndex:
    """Locators for one corpus. ``fmt`` decides how a locator is decoded."""

    key: str
    label: str
    fmt: str                                  # 'wfdb' | 'ptbxl' | 'hdf5' | 'npy'
    locators: List[object] = field(default_factory=list)
    fs_hint: Optional[float] = None
    meta: Dict[str, object] = field(default_factory=dict)

    def __len__(self) -> int:
        return len(self.locators)


# --------------------------------------------------------------------------------------
# index builders
# --------------------------------------------------------------------------------------
#: Records in the MIT-BIH Noise Stress Test database that are *pure noise*, not ECG:
#: baseline wander, electrode motion and muscle artefact. Feeding these to an ECG
#: encoder would be training on noise labelled as signal.
NOISE_ONLY_RECORDS = frozenset({"bw", "em", "ma"})


def index_wfdb(ds: FoundDataset, limit: int) -> ShortIndex:
    recs = wfdb_lite.list_records(ds.data_root, ("hea",), limit=None,
                                  exclude_dirs=ds.exclude_dirs)
    if ds.kind == "noise":
        recs = [r for r in recs if os.path.basename(r).lower() not in NOISE_ONLY_RECORDS]
    return ShortIndex(key=ds.key, label=ds.label, fmt="wfdb", locators=recs[:limit])


def index_hdf5(ds: FoundDataset, limit: int) -> ShortIndex:
    """CODE-15 ships HDF5 files with a (N, 4096, 12) ``tracings`` array at 400 Hz."""
    import h5py
    files = []
    _excl = {os.path.abspath(p) for p in ds.exclude_dirs}
    for dirpath, dirnames, filenames in os.walk(ds.root):
        dirnames[:] = [d for d in dirnames
                       if os.path.abspath(os.path.join(dirpath, d)) not in _excl]
        for fn in sorted(filenames):
            if fn.lower().endswith((".hdf5", ".h5")):
                files.append(os.path.join(dirpath, fn))
    locators: List[Tuple[str, int]] = []
    for f in sorted(files):
        try:
            with h5py.File(f, "r") as h:
                key = "tracings" if "tracings" in h else next(iter(h.keys()))
                n = h[key].shape[0]
        except Exception:
            continue
        for i in range(n):
            locators.append((f, i))
            if len(locators) >= limit:
                return ShortIndex(key=ds.key, label=ds.label, fmt="hdf5",
                                  locators=locators, fs_hint=400.0)
    return ShortIndex(key=ds.key, label=ds.label, fmt="hdf5", locators=locators,
                      fs_hint=400.0)


def index_npy(ds: FoundDataset, limit: int) -> ShortIndex:
    files = []
    _excl = {os.path.abspath(p) for p in ds.exclude_dirs}
    for dirpath, dirnames, filenames in os.walk(ds.root):
        dirnames[:] = [d for d in dirnames
                       if os.path.abspath(os.path.join(dirpath, d)) not in _excl]
        for fn in sorted(filenames):
            if fn.lower().endswith(".npy"):
                files.append(os.path.join(dirpath, fn))
    locators: List[Tuple[str, int]] = []
    for f in sorted(files):
        try:
            arr = np.load(f, mmap_mode="r")
        except Exception:
            continue
        if arr.ndim == 2:                         # a single recording per file
            locators.append((f, -1))
        elif arr.ndim == 3:                       # (N, C, T) or (N, T, C)
            for i in range(arr.shape[0]):
                locators.append((f, i))
                if len(locators) >= limit:
                    break
        if len(locators) >= limit:
            break
    return ShortIndex(key=ds.key, label=ds.label, fmt="npy", locators=locators[:limit],
                      fs_hint=500.0)


def index_ptbxl(ds: FoundDataset, limit: int, use_hr: bool = False) -> ShortIndex:
    """Index PTB-XL from ``ptbxl_database.csv`` and attach superdiagnostic labels."""
    import pandas as pd

    root = ds.data_root
    db = pd.read_csv(os.path.join(root, "ptbxl_database.csv"), index_col="ecg_id")
    db.scp_codes = db.scp_codes.apply(ast.literal_eval)

    col = "filename_hr" if use_hr else "filename_lr"
    if col not in db.columns:
        col = "filename_lr" if "filename_lr" in db.columns else "filename_hr"
    # v0.2: only use the 500 Hz files if they are actually on disk in this mirror
    if col == "filename_hr":
        probe = os.path.join(root, str(db[col].iloc[0]))
        if not (os.path.exists(probe + ".hea") or os.path.exists(probe + ".dat")):
            col = "filename_lr"
    fs_hint = 500.0 if col == "filename_hr" else 100.0

    agg_path = os.path.join(root, "scp_statements.csv")
    y_cols: List[str] = []
    Y = None
    if os.path.exists(agg_path):
        agg = pd.read_csv(agg_path, index_col=0)
        agg = agg[agg.diagnostic == 1]

        def superclass(codes: dict) -> List[str]:
            out = {agg.loc[k].diagnostic_class for k in codes if k in agg.index}
            return sorted(x for x in out if isinstance(x, str))

        labels = db.scp_codes.apply(superclass)
        y_cols = sorted({c for row in labels for c in row})
        Y = np.zeros((len(db), len(y_cols)), dtype=np.int64)
        idx = {c: i for i, c in enumerate(y_cols)}
        for r, row in enumerate(labels):
            for c in row:
                Y[r, idx[c]] = 1

    keep = np.arange(len(db))
    if limit < len(db):
        rng = np.random.default_rng(0)
        keep = np.sort(rng.choice(len(db), size=limit, replace=False))

    locators = [os.path.join(root, p) for p in db[col].values[keep]]
    locators = [p[:-4] if p.endswith(".dat") else p for p in locators]

    meta: Dict[str, object] = {
        "strat_fold": db.strat_fold.values[keep] if "strat_fold" in db.columns else None,
        "patient_id": db.patient_id.values[keep] if "patient_id" in db.columns else None,
        "label_names": y_cols,
        "labels": None if Y is None else Y[keep],
        "ecg_id": db.index.values[keep],
    }
    return ShortIndex(key=ds.key, label=ds.label, fmt="ptbxl", locators=locators,
                      fs_hint=fs_hint, meta=meta)


def build_index(ds: FoundDataset, limit: int, use_hr: bool = False) -> ShortIndex:
    if ds.fmt == "ptbxl":
        return index_ptbxl(ds, limit, use_hr=use_hr)
    if ds.fmt == "hdf5":
        return index_hdf5(ds, limit)
    if ds.fmt == "npy":
        return index_npy(ds, limit)
    if ds.fmt == "wfdb":
        return index_wfdb(ds, limit)
    raise ValueError(f"cannot build a short-window index for fmt={ds.fmt!r}")


# --------------------------------------------------------------------------------------
# decoding
# --------------------------------------------------------------------------------------
def _load_locator(fmt: str, loc, fs_hint: Optional[float]) -> Tuple[np.ndarray, float]:
    """Return (C, T) raw signal in physical units and its sampling rate."""
    if fmt in ("wfdb", "ptbxl"):
        rec = wfdb_lite.read_record(str(loc))
        return np.asarray(rec.p_signal, dtype=np.float32).T, float(rec.fs)

    if fmt == "hdf5":
        import h5py
        path, i = loc
        with h5py.File(path, "r") as h:
            key = "tracings" if "tracings" in h else next(iter(h.keys()))
            arr = np.asarray(h[key][i], dtype=np.float32)
        if arr.ndim == 2 and arr.shape[0] > arr.shape[1]:
            arr = arr.T                                  # (T, C) -> (C, T)
        return arr, float(fs_hint or 400.0)

    if fmt == "npy":
        path, i = loc
        arr = np.load(path, mmap_mode="r")
        arr = np.asarray(arr if i < 0 else arr[i], dtype=np.float32)
        if arr.ndim == 2 and arr.shape[0] > arr.shape[1]:
            arr = arr.T
        return arr, float(fs_hint or 500.0)

    raise ValueError(f"unknown fmt {fmt!r}")


class ShortWindowDataset(Dataset):
    """Lazily decoded 10 s windows, shaped (1, C, T) so they stack with context strips.

    Yields a dict so a single collate function can serve short clips (no future
    window) and long-recording pairs (with one) in the same training loop.
    """

    def __init__(self, indices: Sequence[ShortIndex], signal: SignalConfig,
                 labels: Optional[np.ndarray] = None, augment: bool = False,
                 return_dict: bool = True):
        self.signal = signal
        self.return_dict = return_dict
        self.augment = augment
        self.items: List[Tuple[str, object, Optional[float]]] = []
        self.source: List[str] = []
        for ix in indices:
            for loc in ix.locators:
                self.items.append((ix.fmt, loc, ix.fs_hint))
                self.source.append(ix.key)
        self.labels = labels
        #: v0.2: failed / invalid reads used to be silently replaced by all-zero
        #: windows. They are still zero-filled (so a batch never crashes) but now
        #: counted, and `failure_report()` makes them visible. Per-process counter:
        #: with DataLoader workers use `audit_reads()` instead.
        self.n_failed = 0
        self.n_invalid = 0

    def __len__(self) -> int:
        return len(self.items)

    def failure_report(self) -> str:
        return (f"{self.n_failed} unreadable, {self.n_invalid} invalid "
                f"(flat/NaN) of {len(self)} windows read in this process")

    def audit_reads(self, n: int = 200, seed: int = 0) -> Dict[str, float]:
        """Read ``n`` random items in-process and report the failure rate."""
        rng = np.random.default_rng(seed)
        f0, i0 = self.n_failed, self.n_invalid
        idx = rng.choice(len(self), size=min(n, len(self)), replace=False)
        for i in idx:
            self._window(int(i))
        k = max(len(idx), 1)
        return {"n": float(len(idx)), "failed_frac": (self.n_failed - f0) / k,
                "invalid_frac": (self.n_invalid - i0) / k}

    def _window(self, idx: int) -> np.ndarray:
        fmt, loc, fs_hint = self.items[idx]
        try:
            raw, fs = _load_locator(fmt, loc, fs_hint)
        except Exception:
            self.n_failed += 1
            return np.zeros((self.signal.n_leads, self.signal.window_samples),
                            dtype=np.float32)
        if raw.ndim != 2 or raw.shape[1] < 8:
            self.n_failed += 1
            return np.zeros((self.signal.n_leads, self.signal.window_samples),
                            dtype=np.float32)

        need = int(round(self.signal.window_s * fs))
        if raw.shape[1] > need:
            start = (random.randint(0, raw.shape[1] - need) if self.augment
                     else (raw.shape[1] - need) // 2)
            raw = raw[:, start:start + need]

        x = prepare_window(raw, fs, self.signal.fs, self.signal.window_samples,
                           reduced=self.signal.reduced_lead)
        x = expand_to_n_leads(x, self.signal.n_leads)
        if not is_valid(x):
            self.n_invalid += 1
            x = np.zeros_like(x)
        return x

    def __getitem__(self, idx: int):
        x = torch.from_numpy(self._window(idx)).float()
        if not self.return_dict:
            if self.labels is not None:
                return x, torch.as_tensor(self.labels[idx])
            return x
        out = {
            "ctx": x.unsqueeze(0),                       # (1, C, T)
            "has_future": torch.tensor(0, dtype=torch.bool),
            "horizon_s": torch.tensor(0.0),
        }
        if self.labels is not None:
            out["label"] = torch.as_tensor(self.labels[idx])
        return out


class ArrayWindowDataset(Dataset):
    """In-memory (N, C, T) windows -- used for synthetic data and cached probes."""

    def __init__(self, waves: np.ndarray, labels: Optional[np.ndarray] = None,
                 return_dict: bool = True):
        self.waves = np.asarray(waves, dtype=np.float32)
        self.labels = labels
        self.return_dict = return_dict

    def __len__(self) -> int:
        return len(self.waves)

    def __getitem__(self, idx: int):
        x = torch.from_numpy(self.waves[idx]).float()
        if not self.return_dict:
            if self.labels is not None:
                return x, torch.as_tensor(self.labels[idx])
            return x
        out = {"ctx": x.unsqueeze(0),
               "has_future": torch.tensor(0, dtype=torch.bool),
               "horizon_s": torch.tensor(0.0)}
        if self.labels is not None:
            out["label"] = torch.as_tensor(self.labels[idx])
        return out


# --------------------------------------------------------------------------------------
# supervised splits for the downstream benchmarks
# --------------------------------------------------------------------------------------
def ptbxl_splits(ix: ShortIndex, signal: SignalConfig, test_fold: int = 10,
                 val_fold: int = 9) -> Dict[str, object]:
    """Official PTB-XL folds: 1-8 train, 9 val, 10 test (Strodthoff et al.)."""
    Y = ix.meta.get("labels")
    folds = ix.meta.get("strat_fold")
    if Y is None or folds is None:
        raise ValueError("PTB-XL index is missing labels or strat_fold")
    folds = np.asarray(folds)
    # v0.2 FIX: records with no superdiagnostic statement (~2 % of PTB-XL) are dropped,
    # as in the Strodthoff et al. benchmark. v0.1 kept them as all-zero label rows.
    has_label = np.asarray(Y).sum(axis=1) > 0
    tr = np.where((folds < val_fold) & has_label)[0]
    va = np.where((folds == val_fold) & has_label)[0]
    te = np.where((folds == test_fold) & has_label)[0]

    def subset(sel):
        sub = ShortIndex(key=ix.key, label=ix.label, fmt=ix.fmt,
                         locators=[ix.locators[i] for i in sel], fs_hint=ix.fs_hint)
        return ShortWindowDataset([sub], signal, labels=Y[sel], return_dict=False)

    pid = ix.meta.get("patient_id")
    out = {"train": subset(tr), "val": subset(va), "test": subset(te),
           "label_names": ix.meta.get("label_names"), "n_classes": Y.shape[1]}
    if pid is not None:
        pid = np.asarray(pid)
        out["groups"] = {"train": pid[tr], "val": pid[va], "test": pid[te]}
        # official folds are patient-disjoint; verify rather than assume
        overlap = (set(pid[tr]) | set(pid[va])) & set(pid[te])
        out["patient_overlap"] = len(overlap)
    return out


def wfdb_dx_labels(ix: ShortIndex, min_count: int = 30
                   ) -> Tuple[np.ndarray, List[str], np.ndarray]:
    """Multi-label matrix from the ``#Dx:`` header comments (CPSC2018 / Chapman / Ningbo).

    Returns (Y, label_names, keep_idx). Codes rarer than ``min_count`` are dropped,
    then records left with no positive label are removed.
    """
    per_record: List[List[str]] = []
    for loc in ix.locators:
        codes: List[str] = []
        try:
            hdr = wfdb_lite.parse_header(str(loc) + ".hea")
            for c in hdr.comments:
                if c.lower().startswith("dx"):
                    payload = c.split(":", 1)[1] if ":" in c else ""
                    codes = [s.strip() for s in payload.split(",") if s.strip()]
                    break
        except Exception:
            pass
        per_record.append(codes)

    counts: Dict[str, int] = {}
    for codes in per_record:
        for c in codes:
            counts[c] = counts.get(c, 0) + 1
    names = sorted([c for c, n in counts.items() if n >= min_count])
    if not names:
        names = sorted(counts, key=counts.get, reverse=True)[:10]

    idx = {c: i for i, c in enumerate(names)}
    Y = np.zeros((len(per_record), len(names)), dtype=np.int64)
    for r, codes in enumerate(per_record):
        for c in codes:
            if c in idx:
                Y[r, idx[c]] = 1
    keep = np.where(Y.sum(axis=1) > 0)[0]
    pretty = [f"{SNOMED_NAMES.get(c, c)}" for c in names]
    return Y[keep], pretty, keep


def wfdb_splits(ix: ShortIndex, signal: SignalConfig, test_frac: float = 0.2,
                seed: int = 0, min_count: int = 30, val_frac: float = 0.1
                ) -> Dict[str, object]:
    """Random record-level split for CPSC2018 / Chapman / Ningbo multi-label probing.

    v0.2: adds a validation split (for probe learning-rate selection / early stopping)
    carved from the training records; CPSC2018 has one record per patient.
    """
    Y, names, keep = wfdb_dx_labels(ix, min_count=min_count)
    locs = [ix.locators[i] for i in keep]

    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(locs))
    n_test = max(1, int(len(locs) * test_frac))
    n_val = max(1, int(len(locs) * val_frac))
    te, va, tr = perm[:n_test], perm[n_test:n_test + n_val], perm[n_test + n_val:]

    def subset(sel):
        sub = ShortIndex(key=ix.key, label=ix.label, fmt=ix.fmt,
                         locators=[locs[i] for i in sel], fs_hint=ix.fs_hint)
        return ShortWindowDataset([sub], signal, labels=Y[sel], return_dict=False)

    return {"train": subset(tr), "test": subset(te), "val": subset(va),
            "label_names": names, "n_classes": Y.shape[1]}

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/data/longform.py
"""Long-recording (Holter / patch) data: LTAFDB and MIT-BIH.

These are the corpora the proposal marks **Required** for the forward-prediction
and hierarchical objectives (Table 6.1), because they are the only ones long
enough to contain a genuine "future window".

Provides
  * ``LongRecord``          -- uniform random-access view over a long recording,
                               with rhythm episodes and beat annotations.
  * ``ForwardPairDataset``  -- (context strip, future window, horizon) pretraining pairs.
  * ``build_onset_dataset`` -- the near-term rhythm-onset anticipation benchmark
                               (proposal Section 6.2 items 2-3) with explicit
                               leakage control.
  * ``build_burden_dataset``-- AF-burden regression (Section 6.2 item 4, for H2).

A note on leads: LTAFDB and MIT-BIH are 2-channel. ``expand_to_n_leads`` tiles
those channels across the 8-lead axis so a single encoder serves both regimes;
this is a documented modelling compromise, not a claim that the tiled channels
carry 12-lead information.
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset

from ..config import SignalConfig
from . import wfdb_lite
from .discovery import FoundDataset
from .preprocess import expand_to_n_leads, is_valid, prepare_window

AF_RHYTHMS = frozenset({"AFIB", "AFL"})
# WFDB aux rhythm strings look like "(AFIB"; normalise by stripping the paren.
def _norm_rhythm(aux: str) -> str:
    return aux.strip().lstrip("(").strip().upper()


@dataclass
class LongRecord:
    """Random-access view over one long recording."""

    name: str
    fs: float
    n_samples: int
    n_channels: int
    path: Optional[str] = None                 # WFDB record path (no extension)
    memory: Optional[np.ndarray] = None        # (C, T) for synthetic/in-memory records
    episodes: List[Tuple[int, int, str]] = field(default_factory=list)
    beat_samples: Optional[np.ndarray] = None
    beat_symbols: Optional[List[str]] = None

    @property
    def duration_s(self) -> float:
        return self.n_samples / self.fs

    def read(self, start: int, n: int) -> np.ndarray:
        """Native-rate raw window, shape (C, n). Out-of-range reads are edge-padded."""
        start = max(0, min(start, max(0, self.n_samples - 1)))
        stop = min(self.n_samples, start + n)
        if self.memory is not None:
            sig = np.asarray(self.memory[:, start:stop], dtype=np.float32)
        else:
            rec = wfdb_lite.read_record(self.path, sampfrom=start, sampto=stop)
            sig = np.asarray(rec.p_signal, dtype=np.float32).T
        if sig.shape[1] < n and sig.shape[1] > 0:
            sig = np.pad(sig, ((0, 0), (0, n - sig.shape[1])), mode="edge")
        elif sig.shape[1] == 0:
            sig = np.zeros((self.n_channels, n), dtype=np.float32)
        return sig

    # ---------------------------------------------------------------- rhythm views
    def rhythm_intervals(self, rhythms: Sequence[str] = tuple(AF_RHYTHMS)
                         ) -> List[Tuple[int, int]]:
        want = {r.upper() for r in rhythms}
        return [(s, e) for s, e, r in self.episodes if r.upper() in want]

    def rhythm_at(self, sample: int) -> Optional[str]:
        for s, e, r in self.episodes:
            if s <= sample < e:
                return r
        return None

    def burden(self, rhythms: Sequence[str] = tuple(AF_RHYTHMS)) -> float:
        """Fraction of the record spent in the given rhythms (AF burden)."""
        if self.n_samples <= 0:
            return 0.0
        total = sum(e - s for s, e in self.rhythm_intervals(rhythms))
        return float(np.clip(total / self.n_samples, 0.0, 1.0))

    def beats_in(self, start: int, stop: int) -> List[str]:
        if self.beat_samples is None or self.beat_symbols is None:
            return []
        lo = int(np.searchsorted(self.beat_samples, start, "left"))
        hi = int(np.searchsorted(self.beat_samples, stop, "left"))
        return self.beat_symbols[lo:hi]


# --------------------------------------------------------------------------------------
# loading
# --------------------------------------------------------------------------------------
def _episodes_from_annotation(ann: wfdb_lite.Annotation, n_samples: int
                              ) -> List[Tuple[int, int, str]]:
    """Turn '+' rhythm-change annotations into contiguous (start, end, rhythm) spans."""
    marks: List[Tuple[int, str]] = []
    for smp, sym, aux in zip(ann.sample, ann.symbol, ann.aux_note):
        if sym == "+" and aux:
            marks.append((int(smp), _norm_rhythm(aux)))
    if not marks:
        return []
    out: List[Tuple[int, int, str]] = []
    for i, (smp, rhythm) in enumerate(marks):
        end = marks[i + 1][0] if i + 1 < len(marks) else n_samples
        if end > smp and rhythm:
            out.append((smp, end, rhythm))
    return out


def load_long_records(ds: FoundDataset, limit: int = 40,
                      want_beats: bool = True) -> List[LongRecord]:
    """Load LTAFDB / MIT-BIH records (annotations only -- signals stay on disk)."""
    records: List[LongRecord] = []
    if ds.fmt == "mitbih_csv":
        return _load_mitbih_csv(ds, limit=limit)

    paths = wfdb_lite.list_records(ds.data_root, ("hea",), limit=None)
    for p in paths:
        if len(records) >= limit:
            break
        try:
            hdr = wfdb_lite.parse_header(p + ".hea")
        except Exception:
            continue
        if hdr.n_samp <= 0 or hdr.fs <= 0:
            continue

        episodes: List[Tuple[int, int, str]] = []
        beats_s, beats_y = None, None
        for ext in ("atr", "qrs"):
            if not os.path.exists(p + "." + ext):
                continue
            try:
                ann = wfdb_lite.read_annotation(p, ext)
            except Exception:
                continue
            if ext == "atr":
                episodes = _episodes_from_annotation(ann, hdr.n_samp)
                if want_beats:
                    keep = [i for i, s in enumerate(ann.symbol)
                            if s in wfdb_lite.BEAT_SYMBOLS]
                    if keep:
                        beats_s = ann.sample[keep]
                        beats_y = [ann.symbol[i] for i in keep]
            break

        records.append(LongRecord(name=os.path.basename(p), fs=float(hdr.fs),
                                  n_samples=int(hdr.n_samp), n_channels=hdr.n_sig,
                                  path=p, episodes=episodes,
                                  beat_samples=beats_s, beat_symbols=beats_y))
    return records


#: Native sampling rates of the PhysioNet databases the CSV mirrors are made from.
CSV_DEFAULT_FS = {"mitbih": 360.0, "mitbih_lt": 128.0, "ltaf": 128.0}


def _infer_csv_fs(df, key: str, rid: str, verbose: bool = True) -> float:
    """v0.2 FIX: infer the sampling rate instead of hard-coding 360 Hz.

    v0.1 assumed 360 Hz for every ``*_ekg.csv`` mirror. The MIT-BIH *Long-Term* ECG
    mirror is 128 Hz, so its windows were read 2.8x too long and every heart rate was
    inflated ~2.8x (70 bpm looked like ~200 bpm). Priority: a time column in the CSV,
    then the corpus default.
    """
    import numpy as _np
    for c in df.columns:
        lc = str(c).lower()
        if "time" in lc or lc in ("t", "seconds", "sec", "ms"):
            try:
                t = _np.asarray(df[c].iloc[:2000], dtype=_np.float64)
                d = _np.median(_np.diff(t))
                if d > 0:
                    # milliseconds if the step is > 1 (e.g. 7.8125 ms at 128 Hz)
                    fs = 1000.0 / d if (d > 1.0 or "ms" in lc) else 1.0 / d
                    if 50.0 <= fs <= 2000.0:
                        return float(round(fs, 3))
            except Exception:
                pass
    fs = CSV_DEFAULT_FS.get(key, 360.0)
    if verbose:
        print(f"[data] {rid}: no time column, assuming the {key} default fs={fs:g} Hz")
    return fs


def _load_mitbih_csv(ds: FoundDataset, limit: int = 40,
                     verbose: bool = True) -> List[LongRecord]:
    """The ``protobioengineering`` CSV mirrors (MIT-BIH Arrhythmia *and* Long-Term).

    Layout: ``{id}_ekg.csv`` (signal columns + a ``symbol`` column) and
    ``{id}_annotations.csv``. v0.2: the sampling rate is inferred per corpus
    (360 Hz for mitdb, 128 Hz for ltdb), not hard-coded.
    """
    import pandas as pd

    ekgs: Dict[str, str] = {}
    anns: Dict[str, List[str]] = {}
    for dirpath, _d, filenames in os.walk(ds.root):
        for fn in sorted(filenames):
            full = os.path.join(dirpath, fn)
            if fn.endswith("_ekg.csv"):
                ekgs[fn[: -len("_ekg.csv")]] = full
            elif "_annotations" in fn and fn.endswith(".csv"):
                anns.setdefault(fn.split("_annotations")[0], []).append(full)

    records: List[LongRecord] = []
    for rid in sorted(ekgs)[:limit]:
        try:
            df = pd.read_csv(ekgs[rid], low_memory=False)
        except Exception:
            continue
        fs = _infer_csv_fs(df, ds.key, rid, verbose=verbose and not records)
        num = df.select_dtypes(include=[np.number])
        drop = [c for c in num.columns
                if c.lower() in ("sample", "sample #", "time", "index", "unnamed: 0")]
        drop += [c for c in num.columns if "time" in str(c).lower()]
        num = num.drop(columns=drop, errors="ignore")
        if num.shape[1] < 1:
            continue
        sig = num.to_numpy(dtype=np.float32).T                 # (C, T)

        episodes: List[Tuple[int, int, str]] = []
        beats_s, beats_y = None, None
        if rid in anns:
            try:
                adf = pd.read_csv(sorted(anns[rid])[0])
            except Exception:
                adf = None
            if adf is not None:
                scol = next((c for c in adf.columns
                             if c.lower() in ("sample", "sample #", "index")), None)
                ycol = next((c for c in adf.columns
                             if c.lower() in ("symbol", "type", "label")), None)
                acol = next((c for c in adf.columns
                             if c.lower() in ("aux", "aux_note", "auxnote")), None)
                if scol and ycol:
                    smp = adf[scol].to_numpy(dtype=np.int64)
                    sym = adf[ycol].astype(str).tolist()
                    aux = (adf[acol].fillna("").astype(str).tolist() if acol
                           else [""] * len(sym))
                    ann = wfdb_lite.Annotation(sample=smp, symbol=sym, aux_note=aux,
                                               fs=fs)
                    episodes = _episodes_from_annotation(ann, sig.shape[1])
                    keep = [i for i, s in enumerate(sym) if s in wfdb_lite.BEAT_SYMBOLS]
                    if keep:
                        beats_s = smp[keep]
                        beats_y = [sym[i] for i in keep]

        records.append(LongRecord(name=rid, fs=fs, n_samples=sig.shape[1],
                                  n_channels=sig.shape[0], memory=sig,
                                  episodes=episodes, beat_samples=beats_s,
                                  beat_symbols=beats_y))
    return records


def synthetic_long_records(n: int, minutes: float, fs: float, n_leads: int,
                           seed: int = 0) -> List[LongRecord]:
    from .synthetic import synth_long_corpus
    out: List[LongRecord] = []
    for d in synth_long_corpus(n, minutes=minutes, fs=fs, n_leads=n_leads, seed=seed):
        sig = d["signal"]
        out.append(LongRecord(name=str(d["name"]), fs=float(d["fs"]),
                              n_samples=int(sig.shape[1]), n_channels=int(sig.shape[0]),
                              memory=sig, episodes=list(d["episodes"])))
    return out


# --------------------------------------------------------------------------------------
# pretraining pairs
# --------------------------------------------------------------------------------------
class ForwardPairDataset(Dataset):
    """(context strip, future window, horizon) samples for L_fwd.

    A sample is ``W`` consecutive windows ending at time t, plus the window that
    starts ``horizon`` seconds after t. When the hierarchical stack is enabled an
    extra far-future window is drawn at the Level-2 horizon.
    """

    def __init__(self, records: Sequence[LongRecord], signal: SignalConfig,
                 context_windows: int = 1,
                 horizons_s: Sequence[float] = (10.0, 30.0, 60.0, 300.0),
                 samples_per_record: int = 200,
                 l2_horizons_s: Optional[Sequence[float]] = None,
                 seed: int = 0, resample_starts: bool = False):
        self.records = [r for r in records if r.duration_s > 0]
        #: v0.2: draw a fresh start time on every access (the horizon assignment of
        #: each plan entry is kept, so the horizon mix is unchanged). Uses the global
        #: numpy RNG, which torch seeds per DataLoader worker -> reproducible.
        self.resample_starts = resample_starts
        self.signal = signal
        self.W = max(1, context_windows)
        self.horizons = list(horizons_s)
        self.l2_horizons = list(l2_horizons_s) if l2_horizons_s else None
        self.rng = np.random.default_rng(seed)

        self.plan: List[Tuple[int, int, float, Optional[float]]] = []
        win_s = signal.window_s
        for ri, rec in enumerate(self.records):
            ctx_s = self.W * win_s
            for _ in range(samples_per_record):
                h = float(self.rng.choice(self.horizons))
                l2h = (float(self.rng.choice(self.l2_horizons))
                       if self.l2_horizons else None)
                need_s = ctx_s + max(h, l2h or 0.0) + win_s
                if rec.duration_s <= need_s:
                    # record too short for this horizon: fall back to the largest
                    # horizon it can support, or skip it entirely
                    usable = rec.duration_s - ctx_s - win_s
                    feasible = [x for x in self.horizons if x < usable]
                    if not feasible:
                        continue
                    h = float(max(feasible))
                    if self.l2_horizons:
                        # v0.2 FIX: v0.1 set l2h=None here, and collate_mixed then
                        # zero-filled that sample's Level-2 target window (training the
                        # Level-2 stack to predict a flat line). Use a feasible Level-2
                        # horizon, or skip the sample.
                        l2_ok = [x for x in self.l2_horizons if x < usable]
                        if not l2_ok:
                            continue
                        l2h = float(max(l2_ok))
                    need_s = ctx_s + max(h, l2h or 0.0) + win_s
                start_s = float(self.rng.uniform(0.0, rec.duration_s - need_s))
                self.plan.append((ri, int(start_s * rec.fs), h, l2h))
        self.horizon_counts = {float(h): sum(1 for p in self.plan if p[2] == h)
                               for h in self.horizons}

    def __len__(self) -> int:
        return len(self.plan)

    def _prep(self, rec: LongRecord, start: int) -> np.ndarray:
        n_native = int(round(self.signal.window_s * rec.fs))
        raw = rec.read(start, n_native)
        x = prepare_window(raw, rec.fs, self.signal.fs, self.signal.window_samples,
                           reduced=(rec.n_channels >= 12))
        x = expand_to_n_leads(x, self.signal.n_leads)
        return x if is_valid(x) else np.zeros_like(x)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        ri, start, h, l2h = self.plan[idx]
        rec = self.records[ri]
        win_native = int(round(self.signal.window_s * rec.fs))
        if self.resample_starts:
            need_s = self.W * self.signal.window_s + max(h, l2h or 0.0) + self.signal.window_s
            hi = max(rec.duration_s - need_s, 0.0)
            start = int(np.random.uniform(0.0, hi) * rec.fs) if hi > 0 else start

        ctx = np.stack([self._prep(rec, start + w * win_native) for w in range(self.W)])
        ctx_end = start + self.W * win_native
        fut = self._prep(rec, ctx_end + int(round(h * rec.fs)))

        out = {
            "ctx": torch.from_numpy(ctx).float(),                 # (W, C, T)
            "future": torch.from_numpy(fut).float(),              # (C, T)
            "has_future": torch.tensor(1, dtype=torch.bool),
            "horizon_s": torch.tensor(float(h)),
        }
        if l2h is not None:
            l2f = self._prep(rec, ctx_end + int(round(l2h * rec.fs)))
            out["l2_future"] = torch.from_numpy(l2f).float()
            out["l2_horizon_s"] = torch.tensor(float(l2h))
            out["has_l2"] = torch.tensor(1, dtype=torch.bool)
        return out


def collate_mixed(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    """Collate short (no future) and long (with future) items into one batch.

    Items without a future window contribute only to L_mask and the intra-window
    forward term; the loop below splits the batch by ``has_future`` so the two
    regimes never get mixed inside a single loss term.
    """
    out: Dict[str, torch.Tensor] = {}
    widths = {int(b["ctx"].shape[0]) for b in batch}
    W = max(widths)
    ctx = []
    for b in batch:
        c = b["ctx"]
        if c.shape[0] < W:                      # repeat the earliest window as padding
            c = torch.cat([c[:1].repeat(W - c.shape[0], 1, 1), c], dim=0)
        ctx.append(c)
    out["ctx"] = torch.stack(ctx)
    out["has_future"] = torch.stack([b["has_future"] for b in batch])
    out["horizon_s"] = torch.stack([b["horizon_s"] for b in batch])
    if any("future" in b for b in batch):
        zero = torch.zeros_like(batch[0]["ctx"][0])
        out["future"] = torch.stack([b.get("future", zero) for b in batch])
    if any("l2_future" in b for b in batch):
        zero = torch.zeros_like(batch[0]["ctx"][0])
        out["l2_future"] = torch.stack([b.get("l2_future", zero) for b in batch])
        out["l2_horizon_s"] = torch.stack(
            [b.get("l2_horizon_s", torch.tensor(0.0)) for b in batch])
        out["has_l2"] = torch.stack(
            [b.get("has_l2", torch.tensor(0, dtype=torch.bool)) for b in batch])
    if any("label" in b for b in batch):
        out["label"] = torch.stack([b["label"] for b in batch])
    return out


# --------------------------------------------------------------------------------------
# downstream benchmarks
# --------------------------------------------------------------------------------------
@dataclass
class WindowSample:
    record: int
    start: int                 # sample index of the context start (native rate)
    label: float


class AnticipationDataset(Dataset):
    """Frozen-encoder probe inputs for the anticipation benchmarks."""

    def __init__(self, records: Sequence[LongRecord], samples: Sequence[WindowSample],
                 signal: SignalConfig, context_windows: int = 1):
        self.records = list(records)
        self.samples = list(samples)
        self.signal = signal
        self.W = max(1, context_windows)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        s = self.samples[idx]
        rec = self.records[s.record]
        win_native = int(round(self.signal.window_s * rec.fs))
        wins = []
        for w in range(self.W):
            raw = rec.read(s.start + w * win_native, win_native)
            x = prepare_window(raw, rec.fs, self.signal.fs, self.signal.window_samples,
                               reduced=(rec.n_channels >= 12))
            wins.append(expand_to_n_leads(x, self.signal.n_leads))
        ctx = np.stack(wins)
        return torch.from_numpy(ctx).float(), torch.tensor(s.label, dtype=torch.float32)


def build_onset_dataset(records: Sequence[LongRecord], signal: SignalConfig,
                        horizon_s: float, context_windows: int = 1,
                        stride_s: float = 30.0, guard_s: float = 5.0,
                        rhythms: Sequence[str] = tuple(AF_RHYTHMS),
                        max_per_record: int = 400,
                        negatives_before_first_event_only: bool = False,
                        seed: int = 0) -> Tuple[List[WindowSample], Dict[str, float]]:
    """Near-term rhythm-onset anticipation (proposal Section 6.2, items 2-3).

    For every candidate context window that is itself entirely free of the target
    rhythm, the label is 1 if an episode of that rhythm *starts* within
    ``horizon_s`` after the window ends.

    Leakage control (proposal Section 9):
      * the context window must not overlap the target rhythm at all;
      * a ``guard_s`` band immediately after the context is excluded, so a window
        whose event begins essentially at its edge is dropped rather than being
        called a negative;
      * a negative requires a full ``horizon_s`` of *observed* signal after the
        window -- windows near the end of a record cannot be negatives, because
        the absence of an event there is unobserved rather than known;
      * ``negatives_before_first_event_only`` reproduces the strict Attia-style
        rule: once a patient has had a documented episode, later windows are no
        longer eligible as negatives.
    """
    rng = np.random.default_rng(seed)
    out: List[WindowSample] = []
    n_pos = n_neg = 0

    ctx_s = context_windows * signal.window_s
    for ri, rec in enumerate(records):
        iv = sorted(rec.rhythm_intervals(rhythms))
        if not rec.episodes:
            continue
        starts = np.array([s for s, _e in iv], dtype=np.int64)
        first_event = int(starts[0]) if len(starts) else None

        step = max(1, int(round(stride_s * rec.fs)))
        ctx_n = int(round(ctx_s * rec.fs))
        hor_n = int(round(horizon_s * rec.fs))
        guard_n = int(round(guard_s * rec.fs))

        cands = list(range(0, max(0, rec.n_samples - ctx_n - hor_n), step))
        rng.shuffle(cands)
        taken = 0
        for t0 in cands:
            if taken >= max_per_record:
                break
            t1 = t0 + ctx_n
            # (1) context must be free of the target rhythm
            if any(s < t1 and e > t0 for s, e in iv):
                continue
            # (2) guard band
            if len(starts) and np.any((starts >= t1) & (starts < t1 + guard_n)):
                continue
            # (3) event inside the horizon?
            hit = bool(len(starts) and np.any(
                (starts >= t1 + guard_n) & (starts <= t1 + hor_n)))
            if hit:
                out.append(WindowSample(ri, t0, 1.0))
                n_pos += 1
                taken += 1
            else:
                # (4) negatives need the full horizon observed
                if t1 + hor_n > rec.n_samples:
                    continue
                if (negatives_before_first_event_only and first_event is not None
                        and t0 >= first_event):
                    continue
                out.append(WindowSample(ri, t0, 0.0))
                n_neg += 1
                taken += 1

    stats = {"n_positive": float(n_pos), "n_negative": float(n_neg),
             "prevalence": float(n_pos / max(n_pos + n_neg, 1)),
             "horizon_s": float(horizon_s)}
    return out, stats


def build_burden_dataset(records: Sequence[LongRecord], signal: SignalConfig,
                         context_windows: int = 6, from_first_minutes: float = 30.0,
                         rhythms: Sequence[str] = tuple(AF_RHYTHMS),
                         samples_per_record: int = 8,
                         seed: int = 0) -> Tuple[List[WindowSample], Dict[str, float]]:
    """AF-burden regression (Section 6.2 item 4, for H2).

    Target = fraction of the *whole* record spent in AF; input = an early context
    segment drawn from the first ``from_first_minutes`` minutes. Predicting a
    whole-record property from an early window is exactly the long-horizon
    dependency H2 says the hierarchical model should capture.
    """
    rng = np.random.default_rng(seed)
    out: List[WindowSample] = []
    burdens: List[float] = []
    ctx_n_s = context_windows * signal.window_s

    for ri, rec in enumerate(records):
        if not rec.episodes or rec.duration_s < ctx_n_s * 2:
            continue
        b = rec.burden(rhythms)
        burdens.append(b)
        hi = min(from_first_minutes * 60.0, rec.duration_s - ctx_n_s)
        if hi <= 0:
            continue
        for _ in range(samples_per_record):
            t0 = float(rng.uniform(0.0, hi))
            out.append(WindowSample(ri, int(t0 * rec.fs), b))

    stats = {"n_records": float(len(burdens)),
             "mean_burden": float(np.mean(burdens)) if burdens else 0.0,
             "n_samples": float(len(out))}
    return out, stats


def build_arrhythmia_onset_dataset(records: Sequence[LongRecord], signal: SignalConfig,
                                   horizon_s: float, context_windows: int = 1,
                                   stride_s: float = 10.0, guard_s: float = 2.0,
                                   target_beats: Sequence[str] = ("V", "E", "F"),
                                   max_per_record: int = 600,
                                   seed: int = 0
                                   ) -> Tuple[List[WindowSample], Dict[str, float]]:
    """Beat-level near-term arrhythmia anticipation (MIT-BIH).

    Given a context window containing *no* beat of the target class, predict
    whether one occurs within ``horizon_s``. Same guard-band and observed-horizon
    rules as ``build_onset_dataset``.
    """
    rng = np.random.default_rng(seed)
    want = set(target_beats)
    out: List[WindowSample] = []
    n_pos = n_neg = 0

    ctx_s = context_windows * signal.window_s
    for ri, rec in enumerate(records):
        if rec.beat_samples is None or rec.beat_symbols is None:
            continue
        ev = np.asarray([s for s, y in zip(rec.beat_samples, rec.beat_symbols)
                         if y in want], dtype=np.int64)
        if len(ev) == 0:
            continue

        ctx_n = int(round(ctx_s * rec.fs))
        hor_n = int(round(horizon_s * rec.fs))
        guard_n = int(round(guard_s * rec.fs))
        step = max(1, int(round(stride_s * rec.fs)))

        cands = list(range(0, max(0, rec.n_samples - ctx_n - hor_n), step))
        rng.shuffle(cands)
        taken = 0
        for t0 in cands:
            if taken >= max_per_record:
                break
            t1 = t0 + ctx_n
            if np.any((ev >= t0) & (ev < t1)):          # context must be clean
                continue
            if np.any((ev >= t1) & (ev < t1 + guard_n)):
                continue
            hit = bool(np.any((ev >= t1 + guard_n) & (ev <= t1 + hor_n)))
            if hit:
                out.append(WindowSample(ri, t0, 1.0))
                n_pos += 1
            else:
                if t1 + hor_n > rec.n_samples:
                    continue
                out.append(WindowSample(ri, t0, 0.0))
                n_neg += 1
            taken += 1

    stats = {"n_positive": float(n_pos), "n_negative": float(n_neg),
             "prevalence": float(n_pos / max(n_pos + n_neg, 1)),
             "horizon_s": float(horizon_s)}
    return out, stats


@dataclass
class LongSplit:
    """v0.2: one fixed record-level split, decided *before* pretraining."""
    train_ids: List[int]
    test_ids: List[int]
    train_names: List[str]
    test_names: List[str]

    def train_records(self, records: Sequence[LongRecord]) -> List[LongRecord]:
        return [records[i] for i in self.train_ids]

    def test_records(self, records: Sequence[LongRecord]) -> List[LongRecord]:
        return [records[i] for i in self.test_ids]


def split_long_records(records: Sequence[LongRecord], test_frac: float = 0.3,
                       seed: int = 0, rhythms: Sequence[str] = tuple(AF_RHYTHMS)
                       ) -> LongSplit:
    """v0.2 FIX (leakage): a patient-level split fixed *before* pretraining.

    v0.1 pretrained the forward predictor on forward pairs from **all** long records
    and then evaluated anticipation / forecast skill on 30 % of those same records:
    the model had literally been trained to predict the futures it was tested on.
    It also re-drew the split per horizon (stratified on that horizon's labels), so
    each horizon had a different test set.

    Here records are stratified by whole-record AF burden (from the annotations, not
    from any horizon-specific label) and split once. Records without rhythm
    annotations always go to the training side (they cannot be evaluated).
    """
    rng = np.random.default_rng(seed)
    annotated = [i for i, r in enumerate(records) if r.episodes]
    unannotated = [i for i, r in enumerate(records) if not r.episodes]
    n_test = int(round(len(annotated) * test_frac))
    if annotated and n_test == 0 and len(annotated) >= 2:
        n_test = 1
    test_ids: List[int] = []
    if n_test > 0:
        burden = np.array([records[i].burden(rhythms) for i in annotated])
        key = burden + rng.normal(0, 1e-6, len(burden))
        order = [annotated[j] for j in np.argsort(key)]
        picks = np.unique(np.linspace(0, len(order) - 1, n_test).round().astype(int))
        test_ids = sorted(order[k] for k in picks)
    train_ids = sorted([i for i in annotated if i not in set(test_ids)] + unannotated)
    return LongSplit(train_ids=train_ids, test_ids=test_ids,
                     train_names=[records[i].name for i in train_ids],
                     test_names=[records[i].name for i in test_ids])


def split_by_ids(samples: Sequence[WindowSample], test_ids: Sequence[int]
                 ) -> Tuple[List[WindowSample], List[WindowSample]]:
    """Split benchmark samples by a *fixed* set of test record indices."""
    t = set(int(i) for i in test_ids)
    return ([s for s in samples if s.record not in t],
            [s for s in samples if s.record in t])


def split_by_record(samples: Sequence[WindowSample], n_records: int,
                    test_frac: float = 0.3, seed: int = 0, stratify: bool = True
                    ) -> Tuple[List[WindowSample], List[WindowSample]]:
    """Patient-level (record-level) split -- never split windows from one patient.

    With ``stratify=True`` records are ordered by their positive-window rate and the
    test records are drawn evenly across that ordering, so both sides span the range
    from event-free to event-heavy patients. A plain random split over a few dozen
    records can easily hand the test set only AF-heavy patients, which makes AUROC
    undefined (single class) or wildly optimistic; that is a property of the split,
    not of the model.
    """
    rng = np.random.default_rng(seed)
    n_test = max(1, min(n_records - 1, int(round(n_records * test_frac))))

    rates = np.full(n_records, np.nan)
    if stratify and samples:
        tot = np.zeros(n_records)
        pos = np.zeros(n_records)
        for s in samples:
            tot[s.record] += 1
            pos[s.record] += float(s.label)
        with np.errstate(invalid="ignore", divide="ignore"):
            rates = np.where(tot > 0, pos / np.maximum(tot, 1), np.nan)

    if stratify and np.isfinite(rates).sum() >= n_test:
        # jitter breaks ties reproducibly so equal-rate records are not always ordered
        # the same way; NaN (no samples) sorts last
        key = np.where(np.isfinite(rates), rates, np.inf) + rng.normal(0, 1e-6, n_records)
        order = np.argsort(key)
        picks = np.unique(np.linspace(0, n_records - 1, n_test).round().astype(int))
        test_ids = set(order[picks].tolist())
    else:
        test_ids = set(rng.permutation(n_records)[:n_test].tolist())

    train = [s for s in samples if s.record not in test_ids]
    test = [s for s in samples if s.record in test_ids]
    return train, test

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/data/cache.py
"""v0.2: one-time preprocessing cache for evaluation datasets.

Every probe in v0.1 re-read and re-filtered every PTB-XL / anticipation window from
Kaggle's network file system *for every model* (Stage 1 probing alone took ~19 min
for four models, Stage 2 took 54 min). The preprocessing is deterministic for
evaluation datasets (centre crop, no augmentation), so it is done once and stored as
a float16 memmap under ``cfg.data.cache_dir`` (default ``/tmp``, which does **not**
count against Kaggle's 20 GB output limit).

The cache pass also *audits* the data: windows that came back all-zero (a failed
read or an invalid/flat recording silently zero-filled by the loader) are counted
across all DataLoader workers and reported, so a broken reader can no longer hide.
"""

from __future__ import annotations

import hashlib
import json
import os
from typing import Dict, Optional

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


class MemmapWindowDataset(Dataset):
    """(x, y) pairs backed by float16 / float32 memmaps."""

    def __init__(self, x_path: str, y_path: Optional[str], meta: Dict[str, object]):
        self.meta = meta
        self.X = np.load(x_path, mmap_mode="r")
        self.Y = np.load(y_path, mmap_mode="r") if y_path else None

    def __len__(self) -> int:
        return int(self.X.shape[0])

    def __getitem__(self, idx: int):
        x = torch.from_numpy(np.asarray(self.X[idx], dtype=np.float32))
        if self.Y is None:
            return x
        return x, torch.from_numpy(np.asarray(self.Y[idx]))


def _key(name: str, n: int, extra: str = "") -> str:
    h = hashlib.md5(f"{name}|{n}|{extra}".encode()).hexdigest()[:10]
    safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in name)
    return f"{safe}_{h}"


def cache_dataset(ds: Dataset, name: str, cache_dir: str, batch_size: int = 64,
                  num_workers: int = 2, extra_key: str = "", verbose: bool = True
                  ) -> MemmapWindowDataset:
    """Materialise ``ds`` (yielding ``x`` or ``(x, y)``) once; reuse afterwards."""
    os.makedirs(cache_dir, exist_ok=True)
    key = _key(name, len(ds), extra_key)
    x_path = os.path.join(cache_dir, key + "_x.npy")
    y_path = os.path.join(cache_dir, key + "_y.npy")
    meta_path = os.path.join(cache_dir, key + "_meta.json")

    if os.path.exists(meta_path) and os.path.exists(x_path):
        with open(meta_path) as fh:
            meta = json.load(fh)
        if verbose:
            print(f"[cache] {name}: reusing {meta['n']} windows "
                  f"(zero-filled: {meta['zero_frac']:.2%})")
        return MemmapWindowDataset(x_path, y_path if meta.get("has_y") else None, meta)

    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers)
    X = Y = None
    n_zero = 0
    pos = 0
    has_y = False
    for batch in loader:
        if isinstance(batch, (tuple, list)):
            xb, yb = batch[0], batch[1]
            has_y = True
        else:
            xb, yb = batch, None
        if X is None:
            X = np.lib.format.open_memmap(x_path + ".tmp.npy", mode="w+",
                                          dtype=np.float16,
                                          shape=(len(ds),) + tuple(xb.shape[1:]))
            if yb is not None:
                Y = np.lib.format.open_memmap(y_path + ".tmp.npy", mode="w+",
                                              dtype=np.float32,
                                              shape=(len(ds),) + tuple(yb.shape[1:]))
        b = xb.shape[0]
        flat = xb.reshape(b, -1)
        n_zero += int((flat.abs().sum(dim=1) == 0).sum())
        X[pos:pos + b] = np.clip(xb.numpy(), -6e4, 6e4).astype(np.float16)
        if Y is not None:
            Y[pos:pos + b] = yb.numpy().astype(np.float32)
        pos += b
    if X is None:
        raise RuntimeError(f"[cache] {name}: dataset is empty")
    X.flush()
    del X
    os.replace(x_path + ".tmp.npy", x_path)
    if Y is not None:
        Y.flush()
        del Y
        os.replace(y_path + ".tmp.npy", y_path)
    meta = {"name": name, "n": len(ds), "has_y": has_y,
            "zero_frac": n_zero / max(len(ds), 1), "n_zero": n_zero}
    with open(meta_path, "w") as fh:
        json.dump(meta, fh)
    if verbose:
        flag = "  <-- CHECK THE READER" if meta["zero_frac"] > 0.01 else ""
        print(f"[cache] {name}: cached {len(ds)} windows, zero-filled (failed or "
              f"invalid reads): {n_zero} ({meta['zero_frac']:.2%}){flag}")
    return MemmapWindowDataset(x_path, y_path if has_y else None, meta)


def cache_probe_splits(splits: Dict[str, object], task: str, cache_dir: str,
                       num_workers: int = 2, verbose: bool = True) -> Dict[str, object]:
    """Cache the train/val/test datasets of one probe task in place (returns a copy)."""
    out = dict(splits)
    for part in ("train", "val", "test"):
        ds = splits.get(part)
        if ds is None:
            continue
        out[part] = cache_dataset(ds, f"{task}_{part}", cache_dir,
                                  num_workers=num_workers, verbose=verbose)
    return out

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/data/build.py
"""High-level wiring: discovered mounts -> ready-to-train loaders and probe splits.

This is the layer the notebook calls. It degrades gracefully: whatever subset of
the six corpora is attached gets used, and if none is attached (or a corpus fails
to parse) the synthetic generator keeps every downstream cell runnable.
"""

from __future__ import annotations

from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
from torch.utils.data import DataLoader

from ..config import ExperimentConfig
from ..engine import InterleavedLoader
from .discovery import FoundDataset, discover
from .longform import (ForwardPairDataset, LongRecord, collate_mixed, load_long_records,
                       synthetic_long_records)
from .short import (ArrayWindowDataset, ShortIndex, ShortWindowDataset, build_index,
                    ptbxl_splits, wfdb_splits)
from .synthetic import synth_short_corpus


def load_short_indices(found: Dict[str, FoundDataset], cfg: ExperimentConfig,
                       verbose: bool = True) -> Dict[str, ShortIndex]:
    out: Dict[str, ShortIndex] = {}
    seen_roots: Dict[str, str] = {}

    keys = list(cfg.data.pretrain_datasets)
    if cfg.data.use_noise_corpora:
        keys += [k for k, d in found.items() if d.kind == "noise"]
    # v0.2 FIX (leakage guard): a downstream benchmark must never be a pretraining
    # corpus unless an explicit transductive ablation asks for it.
    probe = set(getattr(cfg.data, "probe_datasets", ()))
    clash = [k for k in keys if k in probe]
    if clash and not getattr(cfg.data, "allow_probe_in_pretrain", False):
        raise ValueError(
            f"{clash} are downstream probe datasets and also in pretrain_datasets: "
            f"their test folds would be seen during pretraining. Remove them, or set "
            f"cfg.data.allow_probe_in_pretrain=True for a deliberate transductive run.")

    for key in keys:
        ds = found.get(key)
        if ds is None or ds.kind not in ("short", "noise"):
            continue
        # A combined mirror (e.g. shaoxing-and-ningbo) satisfies several corpus keys
        # from one directory. Indexing it once per key would load every record twice.
        if ds.data_root in seen_roots:
            if verbose:
                print(f"[data] {ds.label:22s} shares a directory with "
                      f"{seen_roots[ds.data_root]} - already indexed, skipping")
            continue
        seen_roots[ds.data_root] = ds.label
        try:
            ix = build_index(ds, cfg.data.max_records_per_dataset)
        except Exception as exc:
            if verbose:
                print(f"[data] {ds.label}: index failed ({type(exc).__name__}: {exc})")
            continue
        if len(ix) == 0:
            if verbose:
                print(f"[data] {ds.label}: no usable records")
            continue
        out[key] = ix
        if verbose:
            print(f"[data] {ds.label:22s} indexed {len(ix):>7d} recordings")
    return out


def load_long_corpus(found: Dict[str, FoundDataset], cfg: ExperimentConfig,
                     verbose: bool = True) -> List[LongRecord]:
    """LTAFDB + MIT-BIH records; falls back to synthetic Holter-style records."""
    records: List[LongRecord] = []
    for key in cfg.data.long_datasets:
        ds = found.get(key)
        if ds is None:
            continue
        try:
            recs = load_long_records(ds, limit=cfg.data.max_long_records)
        except Exception as exc:
            if verbose:
                print(f"[data] {ds.label}: load failed ({type(exc).__name__}: {exc})")
            continue
        with_rhythm = sum(1 for r in recs if r.episodes)
        if verbose:
            print(f"[data] {ds.label:22s} {len(recs):>4d} long records "
                  f"({with_rhythm} with rhythm annotations, "
                  f"total {sum(r.duration_s for r in recs) / 3600:.1f} h)")
        records.extend(recs)

    if not records and cfg.data.allow_synthetic_fallback:
        if verbose:
            print("[data] no long recordings found -> using synthetic Holter records "
                  "(pipeline smoke test only, not a scientific result)")
        records = synthetic_long_records(
            cfg.data.synthetic_n_long, cfg.data.synthetic_long_minutes,
            cfg.signal.fs, cfg.signal.n_leads, seed=cfg.optim.seed)
    return records


def build_pretrain_loaders(found: Dict[str, FoundDataset], cfg: ExperimentConfig,
                           long_records: Optional[Sequence[LongRecord]] = None,
                           samples_per_record: int = 200,
                           steps_per_epoch: Optional[int] = None,
                           verbose: bool = True) -> Tuple[InterleavedLoader, Dict[str, object]]:
    """Interleaved loader over short clips (L_mask + intra-window L_fwd) and long
    recordings (cross-window L_fwd, plus the Level-2 stack when enabled)."""
    info: Dict[str, object] = {}
    loaders: List[DataLoader] = []
    weights: List[float] = []

    # ---- short corpora -----------------------------------------------------------
    indices = load_short_indices(found, cfg, verbose=verbose)
    if indices:
        ds_short = ShortWindowDataset(list(indices.values()), cfg.signal, augment=True)
        info["n_short"] = len(ds_short)
        info["short_sources"] = {k: len(v) for k, v in indices.items()}
    elif cfg.data.allow_synthetic_fallback:
        if verbose:
            print("[data] no short corpora found -> synthetic 10 s clips")
        waves, _ = synth_short_corpus(cfg.data.synthetic_n_short, cfg.signal.fs,
                                      cfg.signal.window_s, cfg.signal.n_leads,
                                      seed=cfg.optim.seed)
        ds_short = ArrayWindowDataset(waves)
        info["n_short"] = len(ds_short)
        info["short_sources"] = {"synthetic": len(ds_short)}
    else:
        ds_short = None

    if ds_short is not None and len(ds_short) > 0:
        loaders.append(DataLoader(ds_short, batch_size=cfg.optim.batch_size, shuffle=True,
                                  num_workers=cfg.optim.num_workers, drop_last=True,
                                  collate_fn=collate_mixed,
                                  pin_memory=torch.cuda.is_available()))
        weights.append(1.0)

    # ---- long corpora ------------------------------------------------------------
    if long_records is None:
        long_records = load_long_corpus(found, cfg, verbose=verbose)
    if long_records:
        ds_long = ForwardPairDataset(
            long_records, cfg.signal,
            context_windows=cfg.forward.context_windows,
            horizons_s=cfg.forward.horizons_s,
            samples_per_record=samples_per_record,
            l2_horizons_s=(cfg.hierarchical.l2_horizons_s
                           if cfg.hierarchical.enabled else None),
            seed=cfg.optim.seed,
            resample_starts=getattr(cfg.forward, "resample_starts", False))
        info["n_forward_pairs"] = len(ds_long)
        info["forward_horizon_counts"] = dict(ds_long.horizon_counts)
        info["pretrain_long_record_names"] = [r.name for r in long_records]
        missing = [h for h, c in ds_long.horizon_counts.items() if c == 0]
        if missing and verbose:
            print(f"[data] WARNING: no long record supports horizons {missing} s; "
                  f"those horizon buckets will stay untrained")
        if len(ds_long) > 0:
            loaders.append(DataLoader(ds_long, batch_size=cfg.optim.batch_size,
                                      shuffle=True, num_workers=cfg.optim.num_workers,
                                      drop_last=True, collate_fn=collate_mixed,
                                      pin_memory=torch.cuda.is_available()))
            # v0.2: configurable (was a hard-coded 2.0 -> 2/3 of all batches Holter)
            weights.append(float(getattr(cfg.data, "long_loader_weight", 0.5)))
        elif verbose:
            print("[data] long records are too short for the configured horizons; "
                  "L_fwd will use the intra-window path only")

    if not loaders:
        raise RuntimeError(
            "No training data available. Attach at least one ECG dataset under "
            "/kaggle/input, or set cfg.data.allow_synthetic_fallback = True.")

    loader = InterleavedLoader(loaders, weights, steps_per_epoch=steps_per_epoch)
    info["steps_per_epoch"] = len(loader)
    info["long_records"] = long_records
    info["loader_weights"] = list(loader.weights)
    if verbose:
        share = ", ".join(f"{w:.0%}" for w in loader.weights)
        print(f"[data] {len(loaders)} loader(s) [batch share: {share}], "
              f"{len(loader)} steps/epoch, batch={cfg.optim.batch_size}")
    return loader, info


def build_probe_splits(found: Dict[str, FoundDataset], cfg: ExperimentConfig,
                       verbose: bool = True) -> Dict[str, Dict[str, object]]:
    """Supervised splits for the standard-classification sanity check."""
    out: Dict[str, Dict[str, object]] = {}

    lim = int(getattr(cfg.data, "max_probe_records", cfg.data.max_records_per_dataset))
    if "ptbxl" in found:
        try:
            ix = build_index(found["ptbxl"], lim,
                             use_hr=bool(getattr(cfg.data, "ptbxl_use_hr", False)))
            out["ptbxl"] = ptbxl_splits(ix, cfg.signal)
            if verbose:
                print(f"[probe] PTB-XL ({ix.fs_hint:g} Hz): "
                      f"{len(out['ptbxl']['train'])} train / "
                      f"{len(out['ptbxl']['val'])} val / "
                      f"{len(out['ptbxl']['test'])} test, "
                      f"{out['ptbxl']['n_classes']} superdiagnostic classes, "
                      f"patient overlap train/test = "
                      f"{out['ptbxl'].get('patient_overlap', 'n/a')}")
        except Exception as exc:
            if verbose:
                print(f"[probe] PTB-XL split failed: {type(exc).__name__}: {exc}")

    if "cpsc" in found:
        try:
            ix = build_index(found["cpsc"], lim)
            out["cpsc"] = wfdb_splits(ix, cfg.signal)
            if verbose:
                print(f"[probe] CPSC2018: {len(out['cpsc']['train'])} train / "
                      f"{len(out['cpsc']['val'])} val / "
                      f"{len(out['cpsc']['test'])} test, "
                      f"{out['cpsc']['n_classes']} classes")
        except Exception as exc:
            if verbose:
                print(f"[probe] CPSC2018 split failed: {type(exc).__name__}: {exc}")

    if not out and cfg.data.allow_synthetic_fallback:
        if verbose:
            print("[probe] no labelled corpus found -> synthetic AF/non-AF task")
        n = max(256, cfg.data.synthetic_n_short)
        w, y = synth_short_corpus(n, cfg.signal.fs, cfg.signal.window_s,
                                  cfg.signal.n_leads, seed=cfg.optim.seed + 7)
        # Single column, not a 2-column one-hot: AF vs sinus is one binary label, and
        # encoding it as two perfectly anti-correlated columns makes the macro-AUROC
        # average a score with its own near-complement, which is harder to read and
        # buys nothing.
        Y = y.reshape(-1, 1).astype(np.int64)
        cut = int(len(w) * 0.6)
        cut2 = int(len(w) * 0.75)
        out["synthetic"] = {
            "train": ArrayWindowDataset(w[:cut], Y[:cut], return_dict=False),
            "val": ArrayWindowDataset(w[cut:cut2], Y[cut:cut2], return_dict=False),
            "test": ArrayWindowDataset(w[cut2:], Y[cut2:], return_dict=False),
            "label_names": ["AF"], "n_classes": 1}
    return out


def probe_batch_from_loader(loader, n: int = 512, short_only: bool = True) -> torch.Tensor:
    """A fixed batch of windows for the per-epoch collapse diagnostics.

    v0.2: default ``n`` raised 128 -> 512. The effective rank of an (n, d) matrix is
    bounded by min(n, d); with n = 128 < d = 384 every rank number in the v0.1 RQ3
    table was capped at 128. v0.2 also draws from the 12-lead short loader only (the
    v0.1 batch was ~2/3 tiled 2-channel Holter windows), and does not advance the
    interleaving schedule of the training loader.
    """
    src = loader
    if short_only and isinstance(loader, InterleavedLoader) and loader.loaders:
        src = loader.loaders[0]
    chunks: List[torch.Tensor] = []
    total = 0
    for batch in src:
        x = batch["ctx"] if isinstance(batch, dict) else batch[0]
        x = x[:, -1] if x.dim() == 4 else x
        chunks.append(x)
        total += len(x)
        if total >= n:
            break
    if not chunks:
        raise RuntimeError("could not draw a probe batch: loader is empty")
    return torch.cat(chunks)[:n]

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/eval/__init__.py
from .benchmarks import (forecast_skill, horizon_sweep, make_feature_fn,
                         run_anticipation_probe, run_burden_probe,
                         run_classification_probe)
from .linear_probe import (LinearHead, macro_auroc, precompute_features, probe_encoder,
                           sweep_fractions, train_linear_probe)
from .robustness import (LEAD_SETS, CorruptedDataset, SupervisedEncoder,
                         load_nstdb_noise, run_robustness_probe, synthetic_noise_bank,
                         train_supervised)
from .stats import (bootstrap_auroc, bootstrap_auroc_delta, compare_models,
                    label_efficiency_table, paired_seed_test, seed_level_summary)

__all__ = [
    "run_classification_probe", "run_anticipation_probe", "run_burden_probe",
    "horizon_sweep", "make_feature_fn", "forecast_skill",
    "probe_encoder", "train_linear_probe", "precompute_features", "macro_auroc",
    "sweep_fractions", "LinearHead",
    "bootstrap_auroc", "bootstrap_auroc_delta", "compare_models",
    "label_efficiency_table", "seed_level_summary", "paired_seed_test",
    "run_robustness_probe", "load_nstdb_noise", "synthetic_noise_bank",
    "CorruptedDataset", "LEAD_SETS", "train_supervised", "SupervisedEncoder",
]

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/eval/stats.py
"""Statistical testing for Section 6.4: bootstrap CIs on AUROC differences."""

from __future__ import annotations

from typing import Dict, Optional, Sequence, Tuple

import numpy as np


def _auroc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    from sklearn.metrics import roc_auc_score
    if y_true.ndim == 1:
        if len(np.unique(y_true)) < 2:
            return float("nan")
        return float(roc_auc_score(y_true, y_score.reshape(-1)))
    per = []
    for j in range(y_true.shape[1]):
        if len(np.unique(y_true[:, j])) < 2:
            continue
        per.append(roc_auc_score(y_true[:, j], y_score[:, j]))
    return float(np.mean(per)) if per else float("nan")


def _resampler(N: int, groups: Optional[np.ndarray], rng):
    """v0.2: index generator for the (cluster) bootstrap.

    Without ``groups`` samples are resampled i.i.d. (v0.1 behaviour). With ``groups``
    (record / patient ids) whole groups are resampled -- windows from one Holter
    record are strongly correlated, so an i.i.d. window bootstrap gives CIs that are
    far too narrow and p-values of 0.000 that the data cannot support.
    """
    if groups is None:
        return lambda: rng.integers(0, N, N)
    groups = np.asarray(groups)
    uniq = np.unique(groups)
    members = [np.flatnonzero(groups == g) for g in uniq]
    G = len(uniq)

    def draw():
        pick = rng.integers(0, G, G)
        return np.concatenate([members[i] for i in pick])
    return draw


def bootstrap_auroc(y_true: np.ndarray, y_score: np.ndarray, n: int = 2000,
                    alpha: float = 0.05, seed: int = 0,
                    groups: Optional[np.ndarray] = None) -> Dict[str, float]:
    """Percentile bootstrap CI for a single model's macro AUROC (cluster bootstrap
    over ``groups`` when given)."""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    rng = np.random.default_rng(seed)
    N = len(y_true)
    draw = _resampler(N, groups, rng)
    vals = []
    for _ in range(n):
        idx = draw()
        v = _auroc(y_true[idx], y_score[idx])
        if not np.isnan(v):
            vals.append(v)
    if not vals:
        return {"auroc": float("nan"), "lo": float("nan"), "hi": float("nan")}
    vals = np.asarray(vals)
    return {"auroc": _auroc(y_true, y_score),
            "lo": float(np.percentile(vals, 100 * alpha / 2)),
            "hi": float(np.percentile(vals, 100 * (1 - alpha / 2))),
            "std": float(vals.std())}


def bootstrap_auroc_delta(y_true: np.ndarray, score_a: np.ndarray, score_b: np.ndarray,
                          n: int = 2000, alpha: float = 0.05, seed: int = 0,
                          groups: Optional[np.ndarray] = None) -> Dict[str, float]:
    """Paired bootstrap on AUROC(a) - AUROC(b).

    Resampling the *same* indices for both models keeps the comparison paired,
    which is what Section 6.4 asks for. ``p_value`` is the two-sided proportion of
    bootstrap deltas on the opposite side of zero (an achieved significance level),
    not a parametric test.
    """
    y_true = np.asarray(y_true)
    a = np.asarray(score_a)
    b = np.asarray(score_b)
    rng = np.random.default_rng(seed)
    N = len(y_true)
    draw = _resampler(N, groups, rng)
    deltas = []
    for _ in range(n):
        idx = draw()
        va = _auroc(y_true[idx], a[idx])
        vb = _auroc(y_true[idx], b[idx])
        if not (np.isnan(va) or np.isnan(vb)):
            deltas.append(va - vb)
    if not deltas:
        return {"delta": float("nan"), "lo": float("nan"), "hi": float("nan"),
                "p_value": float("nan"), "significant": False}
    d = np.asarray(deltas)
    obs = _auroc(y_true, a) - _auroc(y_true, b)
    lo = float(np.percentile(d, 100 * alpha / 2))
    hi = float(np.percentile(d, 100 * (1 - alpha / 2)))
    p = 2.0 * min(float((d <= 0).mean()), float((d >= 0).mean()))
    return {"delta": float(obs), "lo": lo, "hi": hi, "p_value": min(p, 1.0),
            "significant": bool(lo > 0 or hi < 0)}


def compare_models(y_true: np.ndarray, scores: Dict[str, np.ndarray],
                   reference: str, n: int = 2000, seed: int = 0,
                   groups: Optional[np.ndarray] = None):
    """Table of every model vs. ``reference`` with paired-bootstrap CIs."""
    import pandas as pd
    rows = []
    for name, s in scores.items():
        ci = bootstrap_auroc(y_true, s, n=n, seed=seed, groups=groups)
        row = {"model": name, "auroc": ci["auroc"],
               "ci_lo": ci["lo"], "ci_hi": ci["hi"]}
        if name != reference and reference in scores:
            d = bootstrap_auroc_delta(y_true, s, scores[reference], n=n, seed=seed,
                                      groups=groups)
            row.update({"delta_vs_ref": d["delta"], "delta_lo": d["lo"],
                        "delta_hi": d["hi"], "p_value": d["p_value"],
                        "significant": d["significant"]})
        rows.append(row)
    return pd.DataFrame(rows).sort_values("auroc", ascending=False).reset_index(drop=True)


def label_efficiency_table(results_by_model: Dict[str, Dict[float, float]]):
    """Rows = model, columns = label fraction, values = AUROC (tests H1's
    'margin grows as labels shrink' claim)."""
    import pandas as pd
    df = pd.DataFrame(results_by_model).T
    df.columns = [f"{int(c * 100)}%" for c in df.columns]
    return df


def seed_level_summary(values_by_seed: Dict[int, float]) -> Dict[str, float]:
    """v0.2: mean / std / 95% t-interval across *training seeds* (the variance the
    test-set bootstrap cannot see)."""
    v = np.asarray([x for x in values_by_seed.values() if np.isfinite(x)], dtype=float)
    if len(v) == 0:
        return {"mean": float("nan"), "std": float("nan"), "n": 0,
                "ci_lo": float("nan"), "ci_hi": float("nan")}
    if len(v) == 1:
        return {"mean": float(v[0]), "std": float("nan"), "n": 1,
                "ci_lo": float("nan"), "ci_hi": float("nan")}
    from scipy import stats as _st
    m, sd = float(v.mean()), float(v.std(ddof=1))
    h = float(_st.t.ppf(0.975, len(v) - 1) * sd / np.sqrt(len(v)))
    return {"mean": m, "std": sd, "n": int(len(v)), "ci_lo": m - h, "ci_hi": m + h}


def paired_seed_test(a: Dict[int, float], b: Dict[int, float]) -> Dict[str, float]:
    """v0.2: paired t-test over seeds shared by both arms (a - b)."""
    common = sorted(set(a) & set(b))
    d = np.asarray([a[k] - b[k] for k in common], dtype=float)
    if len(d) < 2:
        return {"delta": float(d.mean()) if len(d) else float("nan"),
                "n_seeds": int(len(d)), "p_value": float("nan")}
    from scipy import stats as _st
    t = _st.ttest_1samp(d, 0.0)
    return {"delta": float(d.mean()), "delta_std": float(d.std(ddof=1)),
            "n_seeds": int(len(d)), "p_value": float(t.pvalue)}

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/eval/linear_probe.py
"""Linear probing on a frozen encoder (the primary comparison, Section 6.4).

Linear probing removes fine-tuning confounds, so every method is compared on the
representation it actually learned. Adapted from ECG-JEPA's ``linear_probe_utils.py``
(precompute features once, then train a cheap linear head).

v0.2 changes
------------
* The head's learning rate is **selected on a validation split** (PTB-XL fold 9, or a
  record/patient-level inner split of the training data) with early stopping, as in
  Kim's protocol. v0.1 trained a fixed 40 epochs at one learning rate and never used
  the validation fold.
* Sub-100 % label fractions are **repeated** with different random subsets and
  reported as mean +/- std (v0.1 drew one subset; its 1 % AUROC was sometimes higher
  than its 10 % AUROC, i.e. noise).
* Feature extraction runs in eval mode under autocast on GPU (faster, same numbers
  to ~1e-3).
"""

from __future__ import annotations

import copy
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


@torch.no_grad()
def precompute_features(encode: Callable[[torch.Tensor], torch.Tensor],
                        loader: DataLoader, device, verbose: bool = False
                        ) -> Tuple[torch.Tensor, torch.Tensor]:
    feats: List[torch.Tensor] = []
    labels: List[torch.Tensor] = []
    dev = torch.device(device) if not isinstance(device, torch.device) else device
    use_amp = dev.type == "cuda"
    for i, batch in enumerate(loader):
        x, y = batch if isinstance(batch, (tuple, list)) else (batch, None)
        x = x.to(dev, non_blocking=True)
        with torch.autocast(device_type=dev.type, enabled=use_amp):
            f = encode(x)
        feats.append(f.detach().float().cpu())
        if y is not None:
            labels.append(y.detach().cpu())
        if verbose and i % 20 == 0:
            print(f"    features {i * loader.batch_size}...")
    F = torch.cat(feats) if feats else torch.zeros(0)
    Y = torch.cat(labels) if labels else torch.zeros(0)
    return F, Y


class LinearHead(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, batch_norm: bool = True):
        super().__init__()
        self.bn = nn.BatchNorm1d(in_dim, affine=False, eps=1e-6) if batch_norm else nn.Identity()
        self.fc = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.fc(self.bn(x))


def macro_auroc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Macro AUROC over label columns, skipping single-class columns."""
    from sklearn.metrics import roc_auc_score
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    if y_true.ndim == 1 or (y_true.ndim == 2 and y_true.shape[1] == 1):
        yt = y_true.reshape(-1)
        if len(np.unique(yt)) < 2:
            return float("nan")
        return float(roc_auc_score(yt, y_score.reshape(-1)))
    scores = []
    for j in range(y_true.shape[1]):
        if len(np.unique(y_true[:, j])) < 2:
            continue
        scores.append(roc_auc_score(y_true[:, j], y_score[:, j]))
    return float(np.mean(scores)) if scores else float("nan")


def _inner_split(n: int, groups: Optional[np.ndarray], frac: float, seed: int
                 ) -> Tuple[np.ndarray, np.ndarray]:
    """Validation carve-out from the training rows -- by group when groups exist."""
    rng = np.random.default_rng(seed + 991)
    if groups is not None:
        g = np.asarray(groups)
        uniq = np.unique(g)
        n_val = max(1, int(round(len(uniq) * frac)))
        val_g = set(rng.permutation(uniq)[:n_val].tolist())
        is_val = np.array([x in val_g for x in g])
    else:
        is_val = np.zeros(n, dtype=bool)
        is_val[rng.permutation(n)[:max(1, int(round(n * frac)))]] = True
    return np.flatnonzero(~is_val), np.flatnonzero(is_val)


def _scores(head, feats, device, multilabel: bool, n_classes: int) -> np.ndarray:
    head.eval()
    with torch.no_grad():
        logits = head(feats.to(device)).float().cpu().numpy()
    if multilabel:
        return 1.0 / (1.0 + np.exp(-logits))
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def _auroc_of(y, s, multilabel: bool, n_classes: int) -> float:
    y = np.asarray(y)
    if multilabel:
        return macro_auroc(y, s)
    return macro_auroc(np.eye(n_classes)[y.astype(int).reshape(-1)], s)


def _fit(Ftr, Ytr, Fva, Yva, n_classes, epochs, lr, weight_decay, batch_size, device,
         multilabel, seed, patience):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    ds = TensorDataset(Ftr, Ytr.float())
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, generator=g,
                    drop_last=len(ds) > batch_size)
    head = LinearHead(Ftr.shape[1], n_classes).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    crit = nn.BCEWithLogitsLoss() if multilabel else nn.CrossEntropyLoss()

    best_auc, best_state, best_ep, bad = -1.0, None, -1, 0
    for ep in range(epochs):
        head.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            if xb.size(0) < 2:
                continue
            out = head(xb)
            loss = crit(out, yb if multilabel else yb.long().reshape(-1))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        sched.step()
        if Fva is not None:
            auc = _auroc_of(Yva.numpy(), _scores(head, Fva, device, multilabel, n_classes),
                            multilabel, n_classes)
            if np.isfinite(auc) and auc > best_auc + 1e-5:
                best_auc, best_state, best_ep, bad = auc, copy.deepcopy(head.state_dict()), ep, 0
            else:
                bad += 1
                if patience and bad >= patience:
                    break
    if best_state is not None:
        head.load_state_dict(best_state)
    return head, best_auc, best_ep


def train_linear_probe(train_feats: torch.Tensor, train_labels: torch.Tensor,
                       test_feats: torch.Tensor, test_labels: torch.Tensor,
                       n_classes: int, epochs: int = 40, lr: float = 1e-3,
                       weight_decay: float = 1e-4, batch_size: int = 128,
                       device=None, multilabel: bool = True,
                       label_fraction: float = 1.0, seed: int = 0,
                       verbose: bool = False,
                       val_feats: Optional[torch.Tensor] = None,
                       val_labels: Optional[torch.Tensor] = None,
                       lr_grid: Optional[Sequence[float]] = None,
                       patience: Optional[int] = 8,
                       train_groups: Optional[np.ndarray] = None,
                       inner_val_frac: float = 0.15) -> Dict[str, object]:
    """Fit a linear head on frozen features; returns metrics and test scores.

    With ``lr_grid`` (v0.2), every learning rate is trained with early stopping on the
    validation AUROC and the best one is used for the test set. If no validation
    features are passed, a validation split is carved out of the (sub-sampled)
    training rows -- by ``train_groups`` (patients / records) when given.
    """
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    g = torch.Generator().manual_seed(seed)
    groups = None if train_groups is None else np.asarray(train_groups)

    if label_fraction < 1.0:
        n = max(n_classes * 2, int(round(len(train_feats) * label_fraction)))
        n = min(n, len(train_feats))
        idx = torch.randperm(len(train_feats), generator=g)[:n]
        train_feats, train_labels = train_feats[idx], train_labels[idx]
        if groups is not None:
            groups = groups[idx.numpy()]

    if lr_grid and val_feats is None and len(train_feats) >= 20:
        tr_i, va_i = _inner_split(len(train_feats), groups, inner_val_frac, seed)
        if len(va_i) and len(tr_i):
            val_feats, val_labels = train_feats[va_i], train_labels[va_i]
            train_feats, train_labels = train_feats[tr_i], train_labels[tr_i]

    use_val = lr_grid and val_feats is not None and len(val_feats) > 1
    grid = list(lr_grid) if use_val else [lr]
    best = (None, -1.0, -1, lr)
    for cand in grid:
        head, vauc, vep = _fit(train_feats, train_labels,
                               val_feats if use_val else None,
                               val_labels if use_val else None,
                               n_classes, epochs, cand, weight_decay, batch_size,
                               device, multilabel, seed, patience if use_val else None)
        if best[0] is None or vauc > best[1]:
            best = (head, vauc, vep, cand)
    head, val_auc, best_ep, best_lr = best

    scores = _scores(head, test_feats, device, multilabel, n_classes)
    y_true = test_labels.numpy()
    auroc = _auroc_of(y_true, scores, multilabel, n_classes)

    out: Dict[str, object] = {"macro_auroc": auroc, "y_true": y_true,
                              "y_score": scores, "n_train": len(train_feats),
                              "label_fraction": label_fraction,
                              "best_lr": best_lr, "val_auroc": val_auc,
                              "best_epoch": best_ep, "head": head.cpu()}
    if multilabel and y_true.ndim == 2 and y_true.shape[1] > 1:
        from sklearn.metrics import f1_score
        out["macro_f1"] = float(f1_score(y_true, (scores >= 0.5).astype(int),
                                         average="macro", zero_division=0))
        from sklearn.metrics import roc_auc_score
        out["per_class_auroc"] = [
            float(roc_auc_score(y_true[:, j], scores[:, j]))
            if len(np.unique(y_true[:, j])) > 1 else float("nan")
            for j in range(y_true.shape[1])]
    if verbose:
        print(f"    linear probe: macro AUROC = {auroc:.4f} "
              f"(n_train={len(train_feats)}, frac={label_fraction}, lr={best_lr:g})")
    return out


def sweep_fractions(Ftr, Ytr, Fte, Yte, n_classes: int, device, fractions,
                    repeats: int = 3, Fva=None, Yva=None, multilabel: bool = True,
                    epochs: int = 40, lr: float = 1e-3, lr_grid=None, patience=8,
                    train_groups=None, seed: int = 0, verbose: bool = False
                    ) -> Dict[float, Dict[str, object]]:
    """Label-efficiency sweep: every fraction < 1 repeated ``repeats`` times."""
    out: Dict[float, Dict[str, object]] = {}
    for frac in fractions:
        reps = 1 if frac >= 1.0 else max(1, repeats)
        runs = [train_linear_probe(Ftr, Ytr, Fte, Yte, n_classes=n_classes,
                                   epochs=epochs, lr=lr, device=device,
                                   multilabel=multilabel, label_fraction=float(frac),
                                   seed=seed + 1000 * r, verbose=verbose,
                                   val_feats=Fva, val_labels=Yva, lr_grid=lr_grid,
                                   patience=patience, train_groups=train_groups)
                for r in range(reps)]
        vals = [r["macro_auroc"] for r in runs]
        rec = dict(runs[0])
        rec["macro_auroc"] = float(np.nanmean(vals)) if vals else float("nan")
        rec["macro_auroc_std"] = float(np.nanstd(vals)) if len(vals) > 1 else 0.0
        rec["macro_auroc_runs"] = [float(v) for v in vals]
        out[float(frac)] = rec
    return out


def probe_encoder(encode: Callable[[torch.Tensor], torch.Tensor],
                  train_ds, test_ds, n_classes: int, device,
                  batch_size: int = 64, epochs: int = 40, lr: float = 1e-3,
                  multilabel: bool = True,
                  label_fractions: Sequence[float] = (1.0,),
                  num_workers: int = 0, seed: int = 0,
                  verbose: bool = True, val_ds=None,
                  lr_grid: Optional[Sequence[float]] = None, patience: int = 8,
                  repeats: int = 1, train_groups=None,
                  return_features: bool = False) -> Dict[str, object]:
    """Precompute features once, then sweep label fractions (label-efficiency curve)."""
    tr_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False,
                           num_workers=num_workers)
    te_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                           num_workers=num_workers)
    if verbose:
        print(f"    extracting features: {len(train_ds)} train / {len(test_ds)} test"
              + (f" / {len(val_ds)} val" if val_ds is not None else ""))
    Ftr, Ytr = precompute_features(encode, tr_loader, device)
    Fte, Yte = precompute_features(encode, te_loader, device)
    Fva = Yva = None
    if val_ds is not None:
        va_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers)
        Fva, Yva = precompute_features(encode, va_loader, device)

    results: Dict[str, object] = {"feature_dim": int(Ftr.shape[1]) if len(Ftr) else 0}
    results["by_fraction"] = sweep_fractions(
        Ftr, Ytr, Fte, Yte, n_classes, device, label_fractions, repeats=repeats,
        Fva=Fva, Yva=Yva, multilabel=multilabel, epochs=epochs, lr=lr,
        lr_grid=lr_grid, patience=patience, train_groups=train_groups, seed=seed,
        verbose=verbose)
    full = results["by_fraction"].get(1.0) or list(results["by_fraction"].values())[-1]
    results["macro_auroc"] = full["macro_auroc"]
    results["y_true"] = full["y_true"]
    results["y_score"] = full["y_score"]
    results["macro_f1"] = full.get("macro_f1")
    results["per_class_auroc"] = full.get("per_class_auroc")
    results["best_lr"] = full.get("best_lr")
    if return_features:
        results["features"] = {"train": (Ftr, Ytr), "test": (Fte, Yte),
                               "val": (Fva, Yva)}
    return results

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/eval/benchmarks.py
"""Downstream benchmarks from Section 6.2.

  1. ``run_classification_probe``  -- PTB-XL / CPSC2018 multi-label (sanity check)
  2. ``run_anticipation_probe``    -- near-term rhythm/arrhythmia onset (H1)
  3. ``run_burden_probe``          -- AF-burden regression (H2)
  4. ``forecast_skill``            -- is the forward predictor forecasting at all?

Every probe runs on a **frozen** encoder. For Cardio-JEPA the feature vector can
optionally be extended with the predictor's forecast latents
(``feature_mode="world"``), which is the operational form of the world-model
claim: the anticipation head sees not only the current cardiac state but where
the model thinks that state is heading.

v0.2 changes are marked inline; the important ones:
  * anticipation CIs are a **record-level cluster bootstrap** (windows from one Holter
    record are not independent);
  * the probe's learning rate is chosen on a record-level inner validation split;
  * ``forecast_skill`` measures prediction and persistence in the *same* target space
    the model was trained on, adds a horizon-shuffle control, and must be run on
    records held out from pretraining.
"""

from __future__ import annotations

from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
from torch.utils.data import DataLoader

from ..config import EvalConfig, SignalConfig
from ..data.longform import AnticipationDataset, LongRecord, WindowSample
from ..data.preprocess import expand_to_n_leads, prepare_window
from .linear_probe import macro_auroc, probe_encoder, sweep_fractions, train_linear_probe
from .stats import bootstrap_auroc


def make_feature_fn(model, horizons_s: Sequence[float] = (),
                    mode: str = "context") -> Callable[[torch.Tensor], torch.Tensor]:
    """Frozen-encoder feature extractor.

    mode='context': mean-pooled encoder embedding only (works for every model,
                    including the ST-MEM, SimCLR, CPC and supervised baselines).
                    For multi-window inputs (B, W, C, T) v0.2 pools *all* context
                    windows (v0.1 used only the last one, so 'context' saw 10 s while
                    'world' saw 20 s -- an unfair comparison).
    mode='world':   [context embedding | predicted future latent per horizon],
                    available only for models with a forward predictor.
    """
    if mode == "world" and hasattr(model, "anticipation_features") and horizons_s:
        def fn(x: torch.Tensor) -> torch.Tensor:
            return model.anticipation_features(x, horizons_s)
        return fn

    def fn(x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 4:
            B, W, C, T = x.shape
            return model.representation(x.reshape(B * W, C, T)).reshape(B, W, -1).mean(1)
        return model.representation(x)
    return fn


@torch.no_grad()
def _extract(feature_fn, ds, device, batch_size: int = 64, num_workers: int = 0
             ) -> Tuple[torch.Tensor, torch.Tensor]:
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    dev = torch.device(device) if not isinstance(device, torch.device) else device
    feats, labels = [], []
    for x, y in loader:
        with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
            f = feature_fn(x.to(dev))
        feats.append(f.detach().float().cpu())
        labels.append(y.detach().cpu())
    if not feats:
        return torch.zeros(0), torch.zeros(0)
    return torch.cat(feats), torch.cat(labels)


# --------------------------------------------------------------------------------------
# 1. standard diagnostic classification
# --------------------------------------------------------------------------------------
def run_classification_probe(model, splits: Dict[str, object], device,
                             eval_cfg: EvalConfig, batch_size: int = 64,
                             num_workers: int = 0, verbose: bool = True,
                             return_features: bool = False) -> Dict[str, object]:
    model.eval()
    fn = make_feature_fn(model, mode="context")
    groups = (splits.get("groups") or {}).get("train") if isinstance(
        splits.get("groups"), dict) else None
    res = probe_encoder(fn, splits["train"], splits["test"],
                        n_classes=int(splits["n_classes"]), device=device,
                        batch_size=batch_size, epochs=eval_cfg.linear_probe_epochs,
                        lr=eval_cfg.linear_probe_lr, multilabel=True,
                        label_fractions=eval_cfg.label_fractions,
                        num_workers=num_workers, verbose=verbose,
                        val_ds=splits.get("val"),
                        lr_grid=getattr(eval_cfg, "linear_probe_lr_grid", None),
                        patience=getattr(eval_cfg, "linear_probe_patience", 8),
                        repeats=getattr(eval_cfg, "label_fraction_repeats", 1),
                        train_groups=groups, return_features=return_features)
    res["label_names"] = splits.get("label_names")
    return res


# --------------------------------------------------------------------------------------
# 2. anticipation
# --------------------------------------------------------------------------------------
def run_anticipation_probe(model, records: Sequence[LongRecord],
                           train_samples: Sequence[WindowSample],
                           test_samples: Sequence[WindowSample],
                           signal: SignalConfig, device, eval_cfg: EvalConfig,
                           context_windows: int = 1,
                           feature_mode: str = "world",
                           horizons_s: Sequence[float] = (),
                           batch_size: int = 64, num_workers: int = 0,
                           verbose: bool = True,
                           train_ds=None, test_ds=None) -> Dict[str, object]:
    """Binary anticipation probe on a frozen encoder.

    v0.2: returns ``groups`` (record id per test window) and a record-level cluster
    bootstrap CI; the head's lr is chosen on a record-level inner validation split.
    ``train_ds`` / ``test_ds`` may be passed pre-cached (see ``data.cache``).
    """
    model.eval()
    if not train_samples or not test_samples:
        return {"auroc": float("nan"), "note": "insufficient samples"}

    tr_ds = train_ds or AnticipationDataset(records, train_samples, signal, context_windows)
    te_ds = test_ds or AnticipationDataset(records, test_samples, signal, context_windows)
    fn = make_feature_fn(model, horizons_s, mode=feature_mode)

    Ftr, Ytr = _extract(fn, tr_ds, device, batch_size, num_workers)
    Fte, Yte = _extract(fn, te_ds, device, batch_size, num_workers)
    if len(Ftr) == 0 or len(Fte) == 0 or len(np.unique(Yte.numpy())) < 2:
        return {"auroc": float("nan"), "note": "degenerate split (single-class test set)"}

    tr_groups = np.asarray([s.record for s in train_samples])
    te_groups = np.asarray([s.record for s in test_samples])
    by = sweep_fractions(Ftr, Ytr.reshape(-1, 1), Fte, Yte.reshape(-1, 1), 1, device,
                         eval_cfg.label_fractions,
                         repeats=getattr(eval_cfg, "label_fraction_repeats", 1),
                         multilabel=True, epochs=eval_cfg.linear_probe_epochs,
                         lr=eval_cfg.linear_probe_lr,
                         lr_grid=getattr(eval_cfg, "linear_probe_lr_grid", None),
                         patience=getattr(eval_cfg, "linear_probe_patience", 8),
                         train_groups=tr_groups)
    best = by.get(1.0) or list(by.values())[-1]
    by_frac = {f: r["macro_auroc"] for f, r in by.items()}

    y_true = Yte.numpy().reshape(-1)
    y_score = np.asarray(best["y_score"]).reshape(-1)
    ci = bootstrap_auroc(y_true, y_score, n=eval_cfg.bootstrap_n, groups=te_groups)
    try:
        from sklearn.metrics import average_precision_score
        auprc = float(average_precision_score(y_true, y_score))
    except Exception:
        auprc = float("nan")

    out = {"auroc": ci["auroc"], "ci_lo": ci["lo"], "ci_hi": ci["hi"],
           "auprc": auprc, "by_fraction": by_frac,
           "by_fraction_std": {f: r.get("macro_auroc_std", 0.0) for f, r in by.items()},
           "y_true": y_true, "y_score": y_score, "groups": te_groups,
           "n_train": len(train_samples), "n_test": len(test_samples),
           "n_test_records": int(len(np.unique(te_groups))),
           "prevalence": float(y_true.mean()), "feature_mode": feature_mode,
           "feature_dim": int(Ftr.shape[1]), "best_lr": best.get("best_lr")}
    if verbose:
        print(f"    anticipation [{feature_mode}]: AUROC = {ci['auroc']:.4f} "
              f"[{ci['lo']:.4f}, {ci['hi']:.4f}] (record-level CI)  "
              f"AUPRC = {auprc:.4f}  n_test={len(test_samples)} "
              f"({out['n_test_records']} records)  prev={out['prevalence']:.3f}")
    return out


# --------------------------------------------------------------------------------------
# 3. AF-burden regression
# --------------------------------------------------------------------------------------
def run_burden_probe(model, records: Sequence[LongRecord],
                     train_samples: Sequence[WindowSample],
                     test_samples: Sequence[WindowSample],
                     signal: SignalConfig, device, context_windows: int = 6,
                     feature_mode: str = "world", horizons_s: Sequence[float] = (),
                     batch_size: int = 32, num_workers: int = 0,
                     alpha: float = 1.0, verbose: bool = True) -> Dict[str, object]:
    """Ridge regression from a frozen embedding to whole-record AF burden.

    v0.2: alpha chosen by grouped (record-level) cross-validation on the training
    records; metrics also reported at the record level (mean prediction per record),
    since burden is a per-record target.
    """
    from sklearn.linear_model import Ridge
    from sklearn.metrics import mean_absolute_error, r2_score
    from sklearn.model_selection import GroupKFold

    model.eval()
    if not train_samples or not test_samples:
        return {"r2": float("nan"), "note": "insufficient samples"}

    tr_ds = AnticipationDataset(records, train_samples, signal, context_windows)
    te_ds = AnticipationDataset(records, test_samples, signal, context_windows)
    fn = make_feature_fn(model, horizons_s, mode=feature_mode)

    Ftr, Ytr = _extract(fn, tr_ds, device, batch_size, num_workers)
    Fte, Yte = _extract(fn, te_ds, device, batch_size, num_workers)
    if len(Ftr) == 0 or len(Fte) == 0:
        return {"r2": float("nan"), "note": "no features"}

    Xtr, ytr = Ftr.numpy(), Ytr.numpy()
    mu, sd = Xtr.mean(0, keepdims=True), Xtr.std(0, keepdims=True) + 1e-6
    Xtr, Xte = (Xtr - mu) / sd, (Fte.numpy() - mu) / sd
    gtr = np.asarray([s.record for s in train_samples])
    best_alpha, best_err = alpha, np.inf
    n_splits = min(5, len(np.unique(gtr)))
    if n_splits >= 2:
        for a in (0.1, 1.0, 10.0, 100.0, 1000.0):
            errs = []
            for tri, vai in GroupKFold(n_splits=n_splits).split(Xtr, ytr, gtr):
                m = Ridge(alpha=a).fit(Xtr[tri], ytr[tri])
                errs.append(mean_absolute_error(ytr[vai], np.clip(m.predict(Xtr[vai]), 0, 1)))
            if np.mean(errs) < best_err:
                best_alpha, best_err = a, float(np.mean(errs))
    reg = Ridge(alpha=best_alpha).fit(Xtr, ytr)
    pred = np.clip(reg.predict(Xte), 0.0, 1.0)
    y = Yte.numpy()
    gte = np.asarray([s.record for s in test_samples])
    rec_ids = np.unique(gte)
    y_rec = np.array([y[gte == r].mean() for r in rec_ids])
    p_rec = np.array([pred[gte == r].mean() for r in rec_ids])

    out: Dict[str, object] = {
        "r2": float(r2_score(y, pred)) if len(np.unique(y)) > 1 else float("nan"),
        "mae": float(mean_absolute_error(y, pred)),
        "r2_record": float(r2_score(y_rec, p_rec)) if len(rec_ids) > 2 else float("nan"),
        "mae_record": float(mean_absolute_error(y_rec, p_rec)),
        "n_test_records": int(len(rec_ids)), "alpha": best_alpha,
        "y_true": y, "y_pred": pred, "feature_mode": feature_mode,
        "n_train": len(train_samples), "n_test": len(test_samples)}
    try:
        from scipy.stats import spearmanr
        out["spearman"] = float(spearmanr(y, pred).correlation)
        out["spearman_record"] = (float(spearmanr(y_rec, p_rec).correlation)
                                  if len(rec_ids) > 2 else float("nan"))
    except Exception:
        out["spearman"] = out["spearman_record"] = float("nan")
    if verbose:
        print(f"    AF burden [{feature_mode}]: R2 = {out['r2']:.3f}  "
              f"MAE = {out['mae']:.3f}  rho = {out['spearman']:.3f}  "
              f"(record-level R2 {out['r2_record']:.3f}, n={len(rec_ids)})")
    return out


# --------------------------------------------------------------------------------------
# 4. forecast skill: is the predictor doing anything beyond copying the present?
# --------------------------------------------------------------------------------------
@torch.no_grad()
def forecast_skill(model, records: Sequence[LongRecord], signal: SignalConfig, device,
                   horizons_s: Sequence[float] = (10.0, 60.0, 300.0, 600.0),
                   n_samples: int = 256, context_windows: int = 1, seed: int = 0,
                   verbose: bool = True, n_boot: int = 500):
    """Compare the predictor against persistence and a horizon-shuffle control.

    For each (context, true future) pair, all in the *target space L_fwd is trained
    on* (LayerNormed target-encoder tokens, mean-pooled; v0.1 compared raw target
    tokens against raw online-encoder tokens, i.e. two different spaces):

      * ``cos_pred``        cos( g_phi(z_ctx, h),            target(x_future) )
      * ``cos_persistence`` cos( target(x_last_context_window), target(x_future) )
      * ``cos_shuffled``    cos( g_phi(z_ctx, h_wrong),      target(x_future) )

    ``skill = cos_pred - cos_persistence``; ``horizon_gain = cos_pred - cos_shuffled``.
    skill <= 0 means the predictor adds nothing over copying the present;
    horizon_gain ~ 0 means the horizon token is ignored (one forecast for every H).
    Embeddings are batch-centred (a shared offset otherwise pins every cosine near 1).

    **Pass only records held out from pretraining** -- on training records this
    measures memorisation, not forecasting (the v0.1 notebook used all records).
    """
    import pandas as pd
    from ..models.forward_predictor import horizon_to_bucket

    model.eval()
    if not hasattr(model, "forward_predictor") or not hasattr(model, "target_embedding"):
        return pd.DataFrame()

    rng = np.random.default_rng(seed)
    usable = [r for r in records if r.duration_s > 0]
    if not usable:
        return pd.DataFrame()
    trained = set(model.trained_horizons()) if hasattr(model, "trained_horizons") else set()

    def centred(a):
        return (a - a.mean(0, keepdim=True)).float()

    def cosv(a, b):
        return torch.nn.functional.cosine_similarity(centred(a), centred(b), dim=-1)

    rows = []
    for H in horizons_s:
        ctxs, futs = [], []
        need_s = context_windows * signal.window_s + H + signal.window_s
        cand = [r for r in usable if r.duration_s > need_s]
        if not cand:
            if verbose:
                print(f"  horizon {H:.0f}s: no record long enough, skipping")
            continue
        for _ in range(n_samples):
            rec = cand[int(rng.integers(len(cand)))]
            t0 = float(rng.uniform(0.0, rec.duration_s - need_s))
            start = int(t0 * rec.fs)
            win_n = int(round(signal.window_s * rec.fs))

            def prep(s):
                raw = rec.read(s, win_n)
                x = prepare_window(raw, rec.fs, signal.fs, signal.window_samples,
                                   reduced=(rec.n_channels >= 12))
                return expand_to_n_leads(x, signal.n_leads)

            ctxs.append(np.stack([prep(start + w * win_n) for w in range(context_windows)]))
            futs.append(prep(start + context_windows * win_n + int(round(H * rec.fs))))
        if len(ctxs) < 8:
            continue

        ctx = torch.from_numpy(np.stack(ctxs)).float().to(device)
        fut = torch.from_numpy(np.stack(futs)).float().to(device)

        pred = model.forecast_latent(ctx, H).float()
        true = model.target_embedding(fut)
        persist = model.target_embedding(ctx[:, -1])
        # horizon-shuffle control: the trained bucket farthest (in log space) from H
        others = [h for h in model.horizon_buckets
                  if (not trained or h in trained)
                  and int(horizon_to_bucket(torch.tensor([h]), model.horizon_buckets)[0])
                  != int(horizon_to_bucket(torch.tensor([H]), model.horizon_buckets)[0])]
        if others:
            h_wrong = max(others, key=lambda h: abs(np.log(h) - np.log(H)))
            shuffled = model.forecast_latent(ctx, h_wrong).float()
        else:
            h_wrong, shuffled = float("nan"), pred

        cp, cq, cs = cosv(pred, true), cosv(persist, true), cosv(shuffled, true)
        skill = (cp - cq).cpu().numpy()
        gain = (cp - cs).cpu().numpy()
        b_rng = np.random.default_rng(seed + 7)
        boots = [skill[b_rng.integers(0, len(skill), len(skill))].mean()
                 for _ in range(n_boot)]
        rows.append({"horizon_s": H, "cos_pred": float(cp.mean()),
                     "cos_persistence": float(cq.mean()),
                     "cos_shuffled": float(cs.mean()), "h_wrong_s": h_wrong,
                     "skill": float(skill.mean()),
                     "skill_lo": float(np.percentile(boots, 2.5)),
                     "skill_hi": float(np.percentile(boots, 97.5)),
                     "horizon_gain": float(gain.mean()),
                     "horizon_trained": (not trained) or any(
                         int(horizon_to_bucket(torch.tensor([H]), model.horizon_buckets)[0])
                         == int(horizon_to_bucket(torch.tensor([t]), model.horizon_buckets)[0])
                         for t in trained),
                     "n": len(ctxs)})
        if verbose:
            r = rows[-1]
            print(f"  horizon {H:6.0f}s  cos_pred={r['cos_pred']:+.4f}  "
                  f"persistence={r['cos_persistence']:+.4f}  "
                  f"skill={r['skill']:+.4f} [{r['skill_lo']:+.4f},{r['skill_hi']:+.4f}]  "
                  f"horizon_gain={r['horizon_gain']:+.4f}"
                  + ("" if r["horizon_trained"] else "  (UNTRAINED BUCKET)"))
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# horizon sweep (H2)
# --------------------------------------------------------------------------------------
def horizon_sweep(model_fns: Dict[str, object], records: Sequence[LongRecord],
                  signal: SignalConfig, device, eval_cfg: EvalConfig,
                  horizons_s: Sequence[float], builder, context_windows: int = 1,
                  test_frac: float = 0.3, seed: int = 0, verbose: bool = True,
                  test_ids: Optional[Sequence[int]] = None):
    """Run an anticipation benchmark at several horizons for several models.

    H2 predicts the hierarchical-vs-single-scale gap *widens monotonically* with
    the horizon, so the sweep is the actual test, not a nice-to-have.

    v0.2: pass ``test_ids`` (the fixed pre-pretraining holdout) so every horizon uses
    the same test records, none of which were seen in pretraining.
    """
    import pandas as pd
    from ..data.longform import split_by_ids, split_by_record

    rows = []
    for H in horizons_s:
        samples, stats = builder(records, signal, horizon_s=H,
                                 context_windows=context_windows)
        if stats["n_positive"] < 5 or stats["n_negative"] < 5:
            if verbose:
                print(f"  horizon {H:.0f}s: too few samples "
                      f"(pos={stats['n_positive']:.0f}, neg={stats['n_negative']:.0f}), skipping")
            continue
        if test_ids is not None:
            tr, te = split_by_ids(samples, test_ids)
        else:
            tr, te = split_by_record(samples, len(records), test_frac=test_frac, seed=seed)
        for name, spec in model_fns.items():
            model = spec["model"] if isinstance(spec, dict) else spec
            mode = spec.get("feature_mode", "context") if isinstance(spec, dict) else "context"
            r = run_anticipation_probe(model, records, tr, te, signal, device, eval_cfg,
                                       context_windows=context_windows,
                                       feature_mode=mode, horizons_s=[H],
                                       verbose=False)
            rows.append({"horizon_s": H, "model": name, "auroc": r.get("auroc"),
                         "ci_lo": r.get("ci_lo"), "ci_hi": r.get("ci_hi"),
                         "auprc": r.get("auprc"),
                         "prevalence": r.get("prevalence"), "n_test": r.get("n_test")})
            if verbose:
                print(f"  horizon {H:6.0f}s  {name:28s} AUROC={r.get('auroc', float('nan')):.4f}")
    return pd.DataFrame(rows)

In [ ]:
%%writefile cardio_jepa_v3_src/cardio_jepa/eval/robustness.py
"""v0.2: robustness and supervised reference evaluations.

* ``run_robustness_probe`` -- the linear head is trained on *clean* features, then
  evaluated on test ECGs corrupted with **real** noise from the MIT-BIH Noise Stress
  Test database (baseline wander ``bw``, electrode motion ``em``, muscle artefact
  ``ma``) at fixed SNRs, and on reduced lead sets (limb leads I+II; single lead II,
  the wearable / patch setting). Falls back to synthetic noise when NSTDB is absent.
* ``train_supervised`` -- the same CroPA encoder trained from scratch with labels:
  the reference every SSL method must be compared with at low label fractions.
"""

from __future__ import annotations

import copy
import os
from typing import Dict, Optional, Sequence

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from ..data import wfdb_lite
from ..data.preprocess import bandpass, resample_signal
from .linear_probe import macro_auroc, precompute_features, train_linear_probe

#: lead indices inside the 8 reduced leads (I, II, V1..V6)
LEAD_SETS: Dict[str, Optional[Sequence[int]]] = {
    "all8": None, "limb_I_II": (0, 1), "single_II": (1,)}


# --------------------------------------------------------------------------------------
# noise bank
# --------------------------------------------------------------------------------------
def load_nstdb_noise(root: Optional[str], fs: float, filter_like_pipeline: bool = True,
                     verbose: bool = True) -> Optional[Dict[str, np.ndarray]]:
    """Read the three pure-noise NSTDB records (bw/em/ma) -> {name: (C, N) at fs}."""
    if not root or not os.path.isdir(root):
        return None
    paths = wfdb_lite.list_records(root, ("hea",))
    bank: Dict[str, np.ndarray] = {}
    for p in paths:
        name = os.path.basename(p).lower()
        if name not in ("bw", "em", "ma"):
            continue
        try:
            rec = wfdb_lite.read_record(p)
        except Exception as exc:
            if verbose:
                print(f"[robustness] could not read {p}: {exc}")
            continue
        x = np.asarray(rec.p_signal, dtype=np.float32).T            # (C, N)
        x = np.nan_to_num(x)
        if filter_like_pipeline:
            x = bandpass(x, rec.fs)
        x = resample_signal(x, rec.fs, fs)
        x = x - x.mean(axis=1, keepdims=True)
        bank[name] = x.astype(np.float32)
    if verbose and bank:
        print(f"[robustness] NSTDB noise records: {sorted(bank)} "
              f"({min(v.shape[1] for v in bank.values()) / fs / 60:.0f} min each)")
    return bank or None


def synthetic_noise_bank(fs: float, minutes: float = 10.0, seed: int = 0
                         ) -> Dict[str, np.ndarray]:
    """Fallback noise: baseline wander, electrode-motion-like bursts, EMG-like noise."""
    rng = np.random.default_rng(seed)
    n = int(minutes * 60 * fs)
    t = np.arange(n) / fs
    bw = sum(rng.uniform(0.3, 1.0) * np.sin(2 * np.pi * rng.uniform(0.05, 0.5) * t
                                            + rng.uniform(0, 6.28)) for _ in range(4))
    em = np.cumsum(rng.normal(0, 1, n)) * 0.02
    em = em - np.convolve(em, np.ones(int(fs)) / fs, mode="same")
    em = em + (rng.random(n) < 2.0 / fs) * rng.normal(0, 3, n)
    ma = rng.normal(0, 1, n)
    out = {}
    for k, v in (("bw", bw), ("em", em), ("ma", ma)):
        v = np.stack([v, np.roll(v, n // 3)]).astype(np.float32)
        out[k] = bandpass(v, fs) if k != "bw" else v
    return out


class CorruptedDataset(Dataset):
    """Wrap an (x, y) dataset; add noise at ``snr_db`` and/or keep only ``lead_keep``.

    Deterministic per item (seeded by index), so every model sees the *same*
    corrupted test set.
    """

    def __init__(self, base: Dataset, noise: Optional[np.ndarray] = None,
                 snr_db: Optional[float] = None,
                 lead_keep: Optional[Sequence[int]] = None, seed: int = 0):
        self.base = base
        self.noise = noise
        self.snr_db = snr_db
        self.lead_keep = None if lead_keep is None else list(lead_keep)
        self.seed = seed

    def __len__(self) -> int:
        return len(self.base)

    def __getitem__(self, idx: int):
        x, y = self.base[idx]
        x = x.clone().float()
        C, T = x.shape[-2], x.shape[-1]
        if self.noise is not None and self.snr_db is not None:
            rng = np.random.default_rng(self.seed * 1_000_003 + idx)
            nz = np.zeros((C, T), dtype=np.float32)
            N = self.noise.shape[1]
            for c in range(C):
                ch = c % self.noise.shape[0]
                off = int(rng.integers(0, max(N - T, 1)))
                seg = self.noise[ch, off:off + T]
                if seg.shape[0] < T:
                    seg = np.pad(seg, (0, T - seg.shape[0]))
                nz[c] = seg
            nz = torch.from_numpy(nz)
            ps = (x ** 2).mean(dim=-1, keepdim=True).clamp_min(1e-8)
            pn = (nz ** 2).mean(dim=-1, keepdim=True).clamp_min(1e-8)
            scale = torch.sqrt(ps / (pn * (10.0 ** (self.snr_db / 10.0))))
            x = x + nz * scale
        if self.lead_keep is not None:
            mask = torch.zeros(C, 1)
            mask[self.lead_keep] = 1.0
            x = x * mask
        return x, y


def run_robustness_probe(model, splits: Dict[str, object], device, eval_cfg,
                         noise_bank: Dict[str, np.ndarray],
                         snrs_db: Sequence[float] = (24.0, 12.0, 6.0, 0.0),
                         lead_sets: Optional[Dict[str, Optional[Sequence[int]]]] = None,
                         batch_size: int = 64, num_workers: int = 0, seed: int = 0,
                         verbose: bool = True):
    """Clean-trained linear head, evaluated under noise and missing leads."""
    import pandas as pd
    lead_sets = LEAD_SETS if lead_sets is None else lead_sets
    model.eval()
    enc = model.representation
    n_cls = int(splits["n_classes"])

    def feats(ds):
        return precompute_features(enc, DataLoader(ds, batch_size=batch_size,
                                                   shuffle=False,
                                                   num_workers=num_workers), device)

    Ftr, Ytr = feats(splits["train"])
    Fva, Yva = feats(splits["val"]) if splits.get("val") is not None else (None, None)
    Fte, Yte = feats(splits["test"])
    head_res = train_linear_probe(Ftr, Ytr, Fte, Yte, n_classes=n_cls,
                                  epochs=eval_cfg.linear_probe_epochs,
                                  lr=eval_cfg.linear_probe_lr, device=device,
                                  val_feats=Fva, val_labels=Yva,
                                  lr_grid=getattr(eval_cfg, "linear_probe_lr_grid", None),
                                  patience=getattr(eval_cfg, "linear_probe_patience", 8),
                                  seed=seed)
    # the head is fit once on clean features and then only *applied* to corrupted ones
    head = head_res["head"].to(device).eval()
    rows = [{"condition": "clean", "noise": "none", "snr_db": np.inf, "leads": "all8",
             "macro_auroc": head_res["macro_auroc"]}]

    def eval_on(ds):
        Fc, Yc = feats(ds)
        with torch.no_grad():
            s = torch.sigmoid(head(Fc.to(device)).float()).cpu().numpy()
        return macro_auroc(Yc.numpy(), s)

    for nname, arr in noise_bank.items():
        for snr in snrs_db:
            ds = CorruptedDataset(splits["test"], arr, snr, None, seed=seed)
            a = eval_on(ds)
            rows.append({"condition": f"{nname}@{snr:g}dB", "noise": nname,
                         "snr_db": float(snr), "leads": "all8", "macro_auroc": a})
            if verbose:
                print(f"    {nname:3s} {snr:5.1f} dB  AUROC={a:.4f}")
    for lname, keep in lead_sets.items():
        if keep is None:
            continue
        ds = CorruptedDataset(splits["test"], None, None, keep, seed=seed)
        a = eval_on(ds)
        rows.append({"condition": f"leads:{lname}", "noise": "none", "snr_db": np.inf,
                     "leads": lname, "macro_auroc": a})
        if verbose:
            print(f"    leads {lname:10s} AUROC={a:.4f}")
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# supervised-from-scratch reference
# --------------------------------------------------------------------------------------
class SupervisedEncoder(nn.Module):
    def __init__(self, encoder: nn.Module, n_classes: int):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.embed_dim, n_classes)

    def representation(self, x):
        if x.dim() == 4:
            x = x[:, -1]
        return self.encoder.representation(x)

    def forward(self, x):
        return self.head(self.representation(x))


def train_supervised(signal, model_cfg, splits: Dict[str, object], device,
                     label_fraction: float = 1.0, max_epochs: int = 20,
                     min_steps: int = 300, lr: float = 3e-4, batch_size: int = 64,
                     num_workers: int = 2, seed: int = 0, patience: int = 4,
                     verbose: bool = True) -> Dict[str, object]:
    """Train CroPA encoder + linear head end-to-end on (a fraction of) the labels.

    Model selection on the validation split (best epoch); test AUROC of that epoch.
    """
    from ..models.ssl_baselines import _encoder_from
    torch.manual_seed(seed)
    np.random.seed(seed)
    dev = torch.device(device) if not isinstance(device, torch.device) else device
    n_cls = int(splits["n_classes"])
    model = SupervisedEncoder(_encoder_from(signal, model_cfg), n_cls).to(dev)

    tr = splits["train"]
    if label_fraction < 1.0:
        g = np.random.default_rng(seed)
        n = max(n_cls * 4, int(round(len(tr) * label_fraction)))
        idx = np.sort(g.choice(len(tr), size=min(n, len(tr)), replace=False))
        tr = torch.utils.data.Subset(tr, idx.tolist())
    steps_per_epoch = max(1, len(tr) // batch_size)
    epochs = int(min(max_epochs * 10, max(max_epochs, np.ceil(min_steps / steps_per_epoch))))
    dl = DataLoader(tr, batch_size=batch_size, shuffle=True, drop_last=len(tr) > batch_size,
                    num_workers=num_workers,
                    generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    total = epochs * max(1, len(dl))
    warm = max(1, int(0.1 * total))

    def _lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        p = (step - warm) / max(1, total - warm)
        return 0.5 * (1.0 + np.cos(np.pi * min(p, 1.0)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda)
    use_amp = dev.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    crit = nn.BCEWithLogitsLoss()

    def scores(ds):
        model.eval()
        out_s, out_y = [], []
        with torch.no_grad():
            for x, y in DataLoader(ds, batch_size=128, shuffle=False,
                                   num_workers=num_workers):
                with torch.autocast(device_type=dev.type, enabled=use_amp):
                    out_s.append(torch.sigmoid(model(x.to(dev)).float()).cpu())
                out_y.append(y)
        return torch.cat(out_y).numpy(), torch.cat(out_s).numpy()

    best_auc, best_state, bad = -1.0, None, 0
    for ep in range(epochs):
        model.train()
        for x, y in dl:
            x, y = x.to(dev), y.float().to(dev)
            with torch.autocast(device_type=dev.type, enabled=use_amp):
                loss = crit(model(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
        if splits.get("val") is not None:
            yv, sv = scores(splits["val"])
            auc = macro_auroc(yv, sv)
            if auc > best_auc + 1e-4:
                best_auc, best_state, bad = auc, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
            if verbose:
                print(f"    [supervised {label_fraction:.0%}] epoch {ep:02d} val AUROC={auc:.4f}")
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    yt, st = scores(splits["test"])
    return {"model": model, "macro_auroc": macro_auroc(yt, st), "y_true": yt,
            "y_score": st, "val_auroc": best_auc, "label_fraction": label_fraction,
            "n_train": len(tr), "epochs_run": ep + 1}

In [ ]:
%%writefile cardio_jepa_v3_src/tests/test_smoke.py
"""End-to-end smoke tests. Run with: python -m pytest tests/ -q"""

from __future__ import annotations

import os

import numpy as np
import torch

import cardio_jepa as cj
from cardio_jepa.config import smoke_config
from cardio_jepa.data.build import build_pretrain_loaders, build_probe_splits, probe_batch_from_loader
from cardio_jepa.data.longform import (build_arrhythmia_onset_dataset, build_burden_dataset,
                                       build_onset_dataset, split_by_record,
                                       synthetic_long_records)
from cardio_jepa.data.synthetic import synth_short_corpus
from cardio_jepa.diagnostics import detect_collapse_signature, effective_rank
from cardio_jepa.eval.benchmarks import (run_anticipation_probe, run_burden_probe,
                                         run_classification_probe)
from cardio_jepa.eval.stats import bootstrap_auroc_delta
from cardio_jepa.models.baselines import Inception1d, STMEMBaseline, XResNet1d
from cardio_jepa.models.cardio_jepa import CardioJEPA
from cardio_jepa.models.forward_predictor import horizon_to_bucket


def _cfg():
    cfg = smoke_config()
    cfg.data.allow_synthetic_fallback = True
    cfg.data.input_root = "/nonexistent-input"
    return cfg


def _model(cfg, **loss_kw):
    for k, v in loss_kw.items():
        setattr(cfg.loss, k, v)
    return CardioJEPA(cfg.signal, cfg.model, cfg.loss, cfg.hierarchical, cfg.forward)


# --------------------------------------------------------------------------- units
def test_horizon_buckets_are_monotone():
    idx = horizon_to_bucket([1.0, 10.0, 60.0, 3600.0], cj.HORIZON_BUCKETS_S)
    assert idx.tolist() == sorted(idx.tolist())
    assert idx[0] == 0 and idx[-1] == len(cj.HORIZON_BUCKETS_S) - 1


def test_cropa_mask_structure():
    cfg = _cfg()
    enc = cj.models.CroPAEncoder(embed_dim=32, depth=1, num_heads=4, n_leads=4,
                                 n_time_patches=6, patch_samples=10)
    m = enc.cropa_mask
    assert m.shape == (24, 24)
    # patch (lead 0, t 0) may see (lead 0, t 3) [same lead] and (lead 2, t 0) [same time]
    assert m[0, 3] == 1
    assert m[0, 2 * 6 + 0] == 1
    # but not (lead 2, t 3): neither the same lead nor the same time column
    assert m[0, 2 * 6 + 3] == 0


def test_effective_rank_detects_collapse():
    good = torch.randn(256, 64)
    collapsed = torch.ones(256, 64) + 1e-6 * torch.randn(256, 64)
    assert effective_rank(good) > 20.0
    # complete collapse: every row identical -> rank 1 (only visible uncentred)
    assert effective_rank(collapsed) < 5.0
    assert effective_rank(collapsed, center=True) > 20.0

    # dimensional collapse: half the directions die -> centred erank halves
    partial = torch.randn(256, 64)
    partial[:, 32:] = 0.0
    assert effective_rank(partial, center=True) < 40.0


def test_collapse_signature_detection():
    losses = [1.0, 0.5, 0.02, 0.01, 0.8, 0.7]     # drop to ~0 then recover (ER-JEPA)
    sig = detect_collapse_signature(losses)
    assert sig["collapsed_to_zero"] and sig["recovered"]
    assert not detect_collapse_signature([1.0, 0.9, 0.8, 0.7])["collapsed_to_zero"]


def test_masked_branch_matches_kim_shapes():
    cfg = _cfg()
    m = _model(cfg)
    x = torch.randn(2, 1, cfg.signal.n_leads, cfg.signal.window_samples)
    out = m(x, use_forward_branch=False)
    assert out["loss_mask"] > 0 and float(out["loss_fwd"]) == 0.0


def test_intra_window_forward_runs_without_a_future_window():
    cfg = _cfg()
    m = _model(cfg)
    x = torch.randn(2, 1, cfg.signal.n_leads, cfg.signal.window_samples)
    out = m(x)
    assert float(out["loss_fwd"]) > 0.0, "intra-window L_fwd should be active"


def test_cross_window_forward_and_gradients():
    cfg = _cfg()
    m = _model(cfg)
    C, T = cfg.signal.n_leads, cfg.signal.window_samples
    out = m(torch.randn(2, 2, C, T), future_window=torch.randn(2, C, T),
            horizon_s=torch.tensor([60.0, 300.0]))
    out["loss"].backward()
    assert any(p.grad is not None and torch.isfinite(p.grad).all()
               for p in m.forward_predictor.parameters())
    # the EMA target encoder must never receive gradients
    assert all(p.grad is None for p in m.target_encoder.parameters())


def test_ema_update_moves_target_towards_online():
    cfg = _cfg()
    m = _model(cfg)
    before = next(m.target_encoder.parameters()).clone()
    with torch.no_grad():
        for p in m.encoder.parameters():
            p.add_(1.0)
    m.update_target(0.5)
    after = next(m.target_encoder.parameters())
    assert not torch.allclose(before, after)


def test_all_four_anti_collapse_arms():
    C = _cfg().signal.n_leads
    T = _cfg().signal.window_samples
    for arm in ("ema", "vicreg", "vq", "sigreg"):
        cfg = _cfg()
        cfg.loss.anti_collapse = arm
        cfg.loss.use_ema_target = arm != "sigreg"
        m = _model(cfg)
        out = m(torch.randn(4, 1, C, T), future_window=torch.randn(4, C, T),
                horizon_s=torch.tensor([60.0] * 4))
        assert torch.isfinite(out["loss"]), f"{arm} produced a non-finite loss"
        out["loss"].backward()
        if arm == "sigreg":
            assert m.target_encoder is None, "sigreg arm should drop the target network"


def test_hierarchical_level2():
    cfg = _cfg()
    cfg.hierarchical.enabled = True
    m = _model(cfg)
    C, T = cfg.signal.n_leads, cfg.signal.window_samples
    out = m(torch.randn(2, 4, C, T), future_window=torch.randn(2, C, T),
            horizon_s=torch.tensor([60.0, 60.0]),
            l2_future=torch.randn(2, C, T), l2_horizon_s=torch.tensor([600.0, 600.0]))
    assert float(out["loss_l2"]) > 0.0
    out["loss"].backward()


def test_forecast_and_anticipation_features():
    cfg = _cfg()
    m = _model(cfg)
    C, T = cfg.signal.n_leads, cfg.signal.window_samples
    x = torch.randn(3, 2, C, T)
    assert m.forecast_latent(x, 300.0).shape == (3, cfg.model.encoder_embed_dim)
    f = m.anticipation_features(x, [60.0, 300.0])
    assert f.shape == (3, 3 * cfg.model.encoder_embed_dim)


def test_baselines_run():
    cfg = _cfg()
    C, T = cfg.signal.n_leads, cfg.signal.window_samples
    st = STMEMBaseline(cfg.signal, cfg.model)
    out = st(torch.randn(2, 1, C, T))
    out["loss"].backward()
    assert st.representation(torch.randn(2, C, T)).shape[1] == cfg.model.encoder_embed_dim
    for net in (XResNet1d(C, 5, widths=(16, 32)), Inception1d(C, 5, depth=2, nf=8)):
        assert net(torch.randn(2, C, T)).shape == (2, 5)


# ------------------------------------------------------------------------- pipeline
def test_wfdb_lite_roundtrip(tmp_path):
    """Write a format-16 record + annotation by hand and read it back."""
    from cardio_jepa.data import wfdb_lite

    fs, n = 250, 500
    raw = (np.sin(np.arange(n) / 8.0) * 1000).astype(np.int16)
    data = np.stack([raw, raw // 2], axis=1).astype("<i2")
    (tmp_path / "r.dat").write_bytes(data.tobytes())
    (tmp_path / "r.hea").write_text(
        f"r 2 {fs} {n}\nr.dat 16 200(0)/mV 16 0 0 0 0 ch0\nr.dat 16 200(0)/mV 16 0 0 0 0 ch1\n")

    rec = wfdb_lite.read_record(str(tmp_path / "r"), prefer_wfdb=False)
    assert rec.p_signal.shape == (n, 2)
    np.testing.assert_allclose(rec.p_signal[:, 0], raw / 200.0, atol=1e-4)

    part = wfdb_lite.read_record(str(tmp_path / "r"), sampfrom=100, sampto=200,
                                 prefer_wfdb=False)
    assert part.p_signal.shape == (100, 2)
    np.testing.assert_allclose(part.p_signal[:, 0], raw[100:200] / 200.0, atol=1e-4)


def test_wfdb_lite_annotation_parsing(tmp_path):
    """Encode two beats and one AFIB rhythm marker, then parse them back."""
    from cardio_jepa.data import wfdb_lite

    words = []

    def ann(code, interval):
        words.append((code << 10) | interval)

    ann(1, 100)                     # 'N' beat at sample 100
    ann(28, 50)                     # '+' rhythm change at 150
    aux = b"(AFIB"
    words.append((63 << 10) | len(aux))
    padded = aux + (b"\x00" if len(aux) % 2 else b"")
    for i in range(0, len(padded), 2):
        words.append(padded[i] | (padded[i + 1] << 8))
    ann(5, 60)                      # 'V' beat at 210
    ann(0, 0)                       # EOF

    (tmp_path / "r.atr").write_bytes(np.array(words, dtype="<u2").tobytes())
    a = wfdb_lite.read_annotation_lite(str(tmp_path / "r.atr"), fs=250.0)

    assert a.sample.tolist() == [100, 150, 210]
    assert a.symbol == ["N", "+", "V"]
    assert a.aux_note[1] == "(AFIB"


def test_synthetic_long_records_and_onset_labels():
    cfg = _cfg()
    recs = synthetic_long_records(4, minutes=20.0, fs=cfg.signal.fs,
                                  n_leads=cfg.signal.n_leads, seed=1)
    assert all(r.episodes for r in recs)
    assert 0.0 < np.mean([r.burden() for r in recs]) < 1.0

    samples, stats = build_onset_dataset(recs, cfg.signal, horizon_s=120.0,
                                         stride_s=10.0, max_per_record=200)
    assert stats["n_positive"] > 0 and stats["n_negative"] > 0

    # leakage control: no context window may overlap an AF episode
    for s in samples:
        rec = recs[s.record]
        t0, t1 = s.start, s.start + int(cfg.signal.window_s * rec.fs)
        assert not any(a < t1 and b > t0 for a, b in rec.rhythm_intervals())

    tr, te = split_by_record(samples, len(recs), test_frac=0.5, seed=0)
    assert tr and te
    assert not (set(s.record for s in tr) & set(s.record for s in te))


def test_burden_dataset():
    cfg = _cfg()
    recs = synthetic_long_records(3, minutes=20.0, fs=cfg.signal.fs,
                                  n_leads=cfg.signal.n_leads, seed=2)
    samples, stats = build_burden_dataset(recs, cfg.signal, context_windows=2,
                                          from_first_minutes=5.0, samples_per_record=4)
    assert len(samples) > 0 and 0.0 <= stats["mean_burden"] <= 1.0


def test_full_pretrain_and_probe_pipeline():
    cfg = _cfg()
    cfg.hierarchical.enabled = True
    cfg.optim.epochs = 1

    loader, info = build_pretrain_loaders({}, cfg, samples_per_record=16, verbose=False)
    probe = probe_batch_from_loader(loader, n=16)
    assert probe.shape[1:] == (cfg.signal.n_leads, cfg.signal.window_samples)

    model = _model(cfg)
    res = cj.pretrain(model, loader, cfg, arm_name="ema", probe_batch=probe,
                      verbose=False, log_every=0)
    assert len(res.monitor.history) == 1
    assert np.isfinite(res.monitor.history[0]["effective_rank"])

    splits = build_probe_splits({}, cfg, verbose=False)
    task = next(iter(splits.values()))
    out = run_classification_probe(model, task, torch.device("cpu"), cfg.evaluation,
                                   batch_size=16, verbose=False)
    assert 0.0 <= out["macro_auroc"] <= 1.0

    recs = info["long_records"]
    samples, stats = build_onset_dataset(recs, cfg.signal, horizon_s=120.0, stride_s=20.0,
                                         max_per_record=60)
    tr, te = split_by_record(samples, len(recs), test_frac=0.4, seed=0)
    if tr and te:
        r = run_anticipation_probe(model, recs, tr, te, cfg.signal, torch.device("cpu"),
                                   cfg.evaluation, feature_mode="world",
                                   horizons_s=[120.0], batch_size=16, verbose=False)
        assert np.isnan(r["auroc"]) or 0.0 <= r["auroc"] <= 1.0

    bs, _ = build_burden_dataset(recs, cfg.signal, context_windows=2,
                                 from_first_minutes=4.0, samples_per_record=4)
    btr, bte = split_by_record(bs, len(recs), test_frac=0.4, seed=0)
    if btr and bte:
        rb = run_burden_probe(model, recs, btr, bte, cfg.signal, torch.device("cpu"),
                              context_windows=2, feature_mode="context",
                              batch_size=8, verbose=False)
        assert "mae" in rb


def test_bootstrap_delta_is_paired():
    rng = np.random.default_rng(0)
    y = rng.integers(0, 2, 400)
    good = y + rng.normal(0, 0.5, 400)
    bad = rng.normal(0, 1, 400)
    d = bootstrap_auroc_delta(y, good, bad, n=200)
    assert d["delta"] > 0 and d["significant"]


def test_synthetic_short_corpus_is_separable():
    """Sanity check on the fallback data: AF vs sinus must be learnable at all."""
    w, y = synth_short_corpus(200, fs=250.0, window_s=10.0, n_leads=8, seed=3)
    assert w.shape == (200, 8, 2500) and set(np.unique(y)) <= {0, 1}
    # AF clips have faster, more variable rhythm -> higher zero-crossing dispersion
    zc = np.array([np.diff(np.signbit(x[1])).sum() for x in w])
    assert zc[y == 1].mean() != zc[y == 0].mean()


def test_forecast_skill_runs_and_is_signed():
    """An untrained predictor should show no meaningful skill over persistence."""
    from cardio_jepa.eval.benchmarks import forecast_skill

    cfg = _cfg()
    m = _model(cfg)
    recs = synthetic_long_records(3, minutes=15.0, fs=cfg.signal.fs,
                                  n_leads=cfg.signal.n_leads, seed=5)
    df = forecast_skill(m, recs, cfg.signal, torch.device("cpu"),
                        horizons_s=(10.0, 60.0), n_samples=16, verbose=False)
    assert set(["horizon_s", "cos_pred", "cos_persistence", "skill"]) <= set(df.columns)
    assert len(df) == 2
    assert np.isfinite(df["skill"]).all()
    # skill is a difference of cosines, so it must stay in [-2, 2]
    assert df["skill"].abs().max() <= 2.0


def test_discovery_handles_nested_kaggle_layout(tmp_path):
    """Kaggle's current layout nests corpora under datasets/<owner>/<slug>/<subdir>."""
    from cardio_jepa.data.discovery import discover

    root = tmp_path / "input"
    def mk(rel, files=()):
        d = root / rel
        d.mkdir(parents=True, exist_ok=True)
        for f in files:
            (d / f).write_text("")
        return d

    mk("datasets/erarayamorenzomuten/chapmanshaoxing-12lead-ecg-database/WFDB_ChapmanShaoxing",
       ["JS00001.hea", "JS00001.mat"])
    mk("datasets/organizations/physionet/china-physiological-signal-challenge-in-2018/Training_WFDB",
       ["A0001.hea", "A0001.mat"])
    mk("datasets/ishtiaqueahmed354/metti-dataset/MEETI", ["rec0.npy"])
    mk("datasets/omar1863/ptb-xl/PTB XL/ptb-xl-1.0.1", ["ptbxl_database.csv"])
    mk("datasets/omar1863/ptb-xl/mit-bih-noise-stress-test-database-1.0.0",
       ["118e00.hea", "118e00.dat", "bw.hea", "bw.dat"])
    # models/ holds code, never data -- must not be scanned as a corpus
    mk("models/someone/cardio-jepa/pytorch/default/1", ["ptbxl_database.csv"])

    found = discover(str(root), verbose=False)
    assert {"chapman", "cpsc", "meeti", "ptbxl"} <= set(found)
    assert found["ptbxl"].fmt == "ptbxl"
    assert found["meeti"].kind == "short"
    # the noise-stress database must be its own key, never mistaken for MIT-BIH
    assert "mitbih_nst" in found and found["mitbih_nst"].kind == "noise"
    assert "mitbih" not in found
    # nothing may be discovered inside models/
    assert all("/models/" not in d.root for d in found.values())


def test_noise_only_records_are_excluded():
    """bw/em/ma are pure noise, not ECG -- they must never enter a training index."""
    from cardio_jepa.data.discovery import FoundDataset
    from cardio_jepa.data.short import NOISE_ONLY_RECORDS
    assert NOISE_ONLY_RECORDS == {"bw", "em", "ma"}


def test_combined_mirror_is_indexed_once(tmp_path):
    """A Chapman+Ningbo mirror satisfies two keys from one directory."""
    from cardio_jepa.data.build import load_short_indices
    from cardio_jepa.data.discovery import discover

    root = tmp_path / "input"
    d = root / "datasets/bjoernjostein/shaoxing-and-ningbo-first-hospital-database/WFDB"
    d.mkdir(parents=True)
    for i in range(3):
        (d / f"JS{i:05d}.hea").write_text("")

    found = discover(str(root), verbose=False)
    assert {"chapman", "ningbo"} <= set(found)
    assert found["chapman"].data_root == found["ningbo"].data_root

    cfg = _cfg()
    idx = load_short_indices(found, cfg, verbose=False)
    assert len(idx) == 1, "the shared directory must be indexed once, not twice"


def test_warm_start_tolerates_width_mismatch(tmp_path):
    import cardio_jepa as cj

    cfg = _cfg()
    src = _model(cfg)
    p = tmp_path / "ck.pth"
    torch.save({"model": src.state_dict(), "config": cfg.to_dict()}, p)

    same = _model(_cfg())
    r = cj.load_weights(same, str(p), verbose=False)
    assert not r.missing and r.n_loaded > 0

    wide = _cfg()
    wide.model.encoder_embed_dim = 128
    wide.model.encoder_num_heads = 4
    r2 = cj.load_weights(_model(wide), str(p), verbose=False)
    assert r2.shape_mismatch, "a width change must be reported, not silently ignored"
    assert 0 < r2.n_loaded < len(r.loaded)


def test_find_package_root(tmp_path):
    import cardio_jepa as cj

    holder = tmp_path / "models" / "o" / "cardio-jepa" / "pytorch" / "default" / "1"
    (holder / "cardio_jepa").mkdir(parents=True)
    (holder / "cardio_jepa" / "__init__.py").write_text("")
    assert cj.find_package_root([str(tmp_path)]) == str(holder)
    assert cj.find_package_root([str(tmp_path / "nope")]) is None


def test_nested_corpus_is_excluded_from_its_parent(tmp_path):
    """`omar1863/ptb-xl` really does ship the noise-stress DB inside the PTB-XL mount.

    Without exclusion the PTB-XL index swallows the noise records.
    """
    from cardio_jepa.data.build import load_short_indices
    from cardio_jepa.data.discovery import discover

    root = tmp_path / "input"
    ptb = root / "datasets/omar1863/ptb-xl/PTB XL/ptb-xl-1.0.1"
    ptb.mkdir(parents=True)
    for i in range(4):
        (ptb / f"rec{i}.hea").write_text("")
    nst = root / "datasets/omar1863/ptb-xl/mit-bih-noise-stress-test-database-1.0.0"
    nst.mkdir(parents=True)
    for nm in ("118e00", "bw", "em", "ma"):
        (nst / f"{nm}.hea").write_text("")

    found = discover(str(root), verbose=False)
    assert "ptbxl" in found and "mitbih_nst" in found
    assert found["mitbih_nst"].root in found["ptbxl"].exclude_dirs

    cfg = _cfg()
    # v0.2: PTB-XL is a probe dataset and no longer pretrained on by default; this test
    # is about nested-corpus exclusion, so opt in explicitly.
    cfg.data.pretrain_datasets = tuple(cfg.data.pretrain_datasets) + ("ptbxl",)
    cfg.data.allow_probe_in_pretrain = True
    idx = load_short_indices(found, cfg, verbose=False)
    ptb_records = [os.path.basename(p) for p in idx["ptbxl"].locators]
    assert len(ptb_records) == 4
    for noise in ("118e00", "bw", "em", "ma"):
        assert noise not in ptb_records, f"{noise} leaked into the PTB-XL index"


def test_find_checkpoints_prefers_root_order_over_size(tmp_path):
    """The curated model-input checkpoint must beat a bigger scratch one.

    Sorting globally by size let a mid-run file in runs/ silently outrank the
    checkpoint deliberately shipped with the model input.
    """
    import cardio_jepa as cj

    model_in = tmp_path / "models" / "o" / "cardio-jepa" / "pytorch" / "default" / "1"
    model_in.mkdir(parents=True)
    (model_in / "pretrained_dual.pth").write_bytes(b"x" * 1000)

    runs = tmp_path / "working" / "runs"
    runs.mkdir(parents=True)
    (runs / "stage4_hierarchical.pth").write_bytes(b"x" * 5000)   # bigger, but scratch

    found = cj.find_checkpoints([str(model_in), str(runs)])
    assert os.path.basename(found[0]) == "pretrained_dual.pth", (
        f"model input must win regardless of size, got {found}")

    # reversing the priority order reverses the winner
    found2 = cj.find_checkpoints([str(runs), str(model_in)])
    assert os.path.basename(found2[0]) == "stage4_hierarchical.pth"


def test_find_checkpoints_newest_first_within_a_root(tmp_path):
    import time
    import cardio_jepa as cj

    d = tmp_path / "runs"
    d.mkdir()
    (d / "old.pth").write_bytes(b"x" * 9000)
    time.sleep(0.01)
    (d / "new.pth").write_bytes(b"x" * 100)
    os.utime(d / "old.pth", (1, 1))

    found = cj.find_checkpoints([str(d)])
    assert os.path.basename(found[0]) == "new.pth"


# --------------------------------------------------------- collapse guard / budget
def _forward_only_cfg(**over):
    cfg = _cfg()
    cfg.model.encoder_embed_dim = 64
    cfg.model.encoder_depth = 2
    cfg.model.encoder_num_heads = 4
    cfg.model.predictor_embed_dim = 48
    cfg.model.predictor_depth = 1
    cfg.optim.epochs = 3
    cfg.optim.batch_size = 8
    cfg.optim.warmup_epochs = 0
    cfg.optim.lr = 1e-3
    cfg.data.synthetic_n_short = 64
    cfg.data.synthetic_n_long = 0
    cfg.loss.weight_forward, cfg.loss.lambda_mask = 1.0, 0.0
    for k, v in over.items():
        obj, _, attr = k.rpartition(".")
        setattr(getattr(cfg, obj) if obj else cfg, attr, v)
    return cfg


def test_check_collapse_detects_the_trivial_solution():
    """Unit-test the detector itself, on both of its signals."""
    from cardio_jepa.engine import check_collapse

    cfg = _cfg().optim

    # signal 1: loss far below its own starting value
    assert check_collapse(None, None, torch.device("cpu"),
                          loss_now=0.001, loss_ref=1.0, cfg=cfg)
    assert check_collapse(None, None, torch.device("cpu"),
                          loss_now=0.5, loss_ref=1.0, cfg=cfg) is None

    # signal 2: degenerate embeddings, even when the loss looks unremarkable
    class _Const:
        training = False

        def eval(self):
            pass

        def representation(self, x):
            return torch.ones(x.size(0), 64) * 3.0

    reason = check_collapse(_Const(), torch.randn(32, 8, 2500), torch.device("cpu"),
                            loss_now=0.5, loss_ref=1.0, cfg=cfg)
    assert reason and "dead" in reason

    # ...but a *healthy* model must not trip the embedding gate. The pooled
    # representation here is offset-dominated even when fine (cos_sim ~ 0.9999),
    # so a loose threshold would abort good runs.
    class _Healthy:
        training = False

        def eval(self):
            pass

        def representation(self, x):
            g = torch.Generator().manual_seed(0)
            return 3.0 + 0.004 * torch.randn(x.size(0), 64, generator=g)

    assert check_collapse(_Healthy(), torch.randn(32, 8, 2500), torch.device("cpu"),
                          loss_now=0.5, loss_ref=1.0, cfg=cfg) is None


def test_collapse_guard_stops_the_run():
    """When the detector fires, training must abandon the run rather than continue."""
    cfg = _forward_only_cfg()
    cfg.optim.on_collapse = "stop"
    cfg.optim.collapse_check_every = 2
    cfg.optim.collapse_loss_frac = 10.0    # force: any loss under 10x the ref trips it

    loader, _ = build_pretrain_loaders({}, cfg, verbose=False)
    res = cj.pretrain(_model(cfg), loader, cfg, arm_name="forced",
                      verbose=False, log_every=0)
    assert res.collapsed and res.stopped_early
    assert res.collapse_step is not None
    assert len(res.monitor.history) < cfg.optim.epochs


def test_collapse_guard_can_observe_instead_of_aborting():
    """RQ3 needs to *measure* collapse, so 'warn' must run to completion."""
    cfg = _forward_only_cfg()
    cfg.optim.on_collapse = "warn"
    cfg.optim.collapse_check_every = 2
    cfg.optim.collapse_loss_frac = 10.0

    loader, _ = build_pretrain_loaders({}, cfg, verbose=False)
    res = cj.pretrain(_model(cfg), loader, cfg, arm_name="warned",
                      verbose=False, log_every=0)
    assert res.collapsed and not res.stopped_early
    assert len(res.monitor.history) == cfg.optim.epochs


def test_max_steps_per_epoch_caps_the_budget():
    cfg = _forward_only_cfg()
    cfg.optim.on_collapse = "ignore"
    cfg.optim.epochs = 2
    loader, _ = build_pretrain_loaders({}, cfg, verbose=False)
    assert len(loader) > 3, "need a loader longer than the cap for this test"

    cfg.optim.max_steps_per_epoch = 3
    res = cj.pretrain(_model(cfg), loader, cfg, arm_name="capped",
                      verbose=False, log_every=0)
    assert len(res.monitor.history) == 2
    # seconds/epoch should reflect 3 steps, not len(loader)
    assert res.history[0]["seconds"] < 1e6


def test_reg_target_pooled_is_measured_on_the_pooled_representation():
    """The token target is satisfied by positional variance alone, which is why the
    pooled target exists."""
    cfg = _cfg()
    cfg.loss.anti_collapse = "vicreg"
    C, T = cfg.signal.n_leads, cfg.signal.window_samples
    x = torch.randn(4, 1, C, T)

    cfg.loss.reg_target = "pooled"
    m_pooled = _model(cfg)
    r_pooled = float(m_pooled(x)["loss_reg"])

    cfg.loss.reg_target = "tokens"
    m_tok = _model(cfg)
    r_tok = float(m_tok(x)["loss_reg"])

    assert r_pooled > 0 and r_tok > 0
    # pooling 400 LayerNormed tokens shrinks per-sample spread, so the hinge on the
    # pooled representation must be the stricter of the two
    assert r_pooled > r_tok

    cfg.loss.reg_target = "both"
    assert float(_model(cfg)(x)["loss_reg"]) > 0


def test_estimate_runtime_projects_sanely():
    from cardio_jepa.engine import budget_steps_per_epoch, estimate_runtime

    cfg = _forward_only_cfg()
    cfg.optim.epochs = 10
    loader, _ = build_pretrain_loaders({}, cfg, verbose=False)
    est = estimate_runtime(_model(cfg), loader, cfg, torch.device("cpu"),
                           n_steps=3, n_models=4)
    assert est["s_per_step"] > 0
    assert est["hours_total"] == est["hours_per_model"] * 4
    # the suggested cap must actually bring the projection inside the budget
    fits = budget_steps_per_epoch(est["s_per_step"], cfg.optim.epochs, 4, 1.0)
    assert fits * cfg.optim.epochs * 4 * est["s_per_step"] <= 1.0 * 3600 * 1.05


def test_real_ltafdb_header_and_annotation_shapes(tmp_path):
    """Regression guard built from a real LTAFDB record (ltafdb 1.0.0, record 00).

    LTAFDB is format 16 at 128 Hz with two channels and a gain that has no
    parenthesised baseline -- a header shape the synthetic fixtures did not cover.
    Rhythm changes arrive as '+' annotations whose AUX text is '(AFIB' / '(VT' / '(N'.
    """
    from cardio_jepa.data import wfdb_lite
    from cardio_jepa.data.longform import _episodes_from_annotation

    (tmp_path / "00.hea").write_text(
        "00 2 128 9661440  9:30:00 31/01/2003\n"
        "00.dat 16 166.945/mV 0 0 -1 -8202 0 ECG\n"
        "00.dat 16 173.01/mV 0 0 3 6311 0 ECG\n")
    h = wfdb_lite.parse_header(str(tmp_path / "00.hea"))
    assert h.n_sig == 2 and h.fs == 128.0 and h.n_samp == 9661440
    assert h.signals[0].fmt == "16"
    assert abs(h.signals[0].adc_gain - 166.945) < 1e-6
    assert h.signals[0].baseline == 0        # no "(baseline)" -> falls back to adc_zero

    # a '+' rhythm marker carrying '(AFIB' must open an AF episode
    ann = wfdb_lite.Annotation(
        sample=np.array([100, 200, 5000], dtype=np.int64),
        symbol=["N", "+", "+"], aux_note=["", "(AFIB", "(N"], fs=128.0)
    eps = _episodes_from_annotation(ann, n_samples=10000)
    assert eps == [(200, 5000, "AFIB"), (5000, 10000, "N")]

In [ ]:
%%writefile cardio_jepa_v3_src/tests/test_v02.py
"""Regression tests for every v0.2 fix. Run with: python -m pytest tests/ -q"""

from __future__ import annotations

import os
import warnings

import numpy as np
import pandas as pd
import pytest
import torch

import cardio_jepa as cj
from cardio_jepa.config import smoke_config
from cardio_jepa.data.build import build_pretrain_loaders, build_probe_splits, probe_batch_from_loader
from cardio_jepa.data.cache import cache_dataset
from cardio_jepa.data.discovery import discover
from cardio_jepa.data.longform import (LongRecord, build_onset_dataset, split_by_ids,
                                       split_long_records, synthetic_long_records)
from cardio_jepa.data.short import ArrayWindowDataset
from cardio_jepa.data.synthetic import synth_short_corpus
from cardio_jepa.eval.stats import bootstrap_auroc, paired_seed_test, seed_level_summary
from cardio_jepa.models import CardioJEPA, CPCBaseline, SimCLRBaseline, STMEMBaseline
from cardio_jepa.models import transformer as tr
from cardio_jepa.regularizers import SoftVQ


def _cfg():
    cfg = smoke_config()
    cfg.data.allow_synthetic_fallback = True
    cfg.data.input_root = "/nonexistent-input"
    return cfg


def _model(cfg):
    return CardioJEPA(cfg.signal, cfg.model, cfg.loss, cfg.hierarchical, cfg.forward)


# ------------------------------------------------------------------- memory / numerics
def test_sdpa_attention_matches_explicit_softmax():
    torch.manual_seed(0)
    att = tr.Attention(32, num_heads=4, qkv_bias=True).eval()
    x = torch.randn(3, 20, 32)
    mask = (torch.rand(20, 20) > 0.5).float()
    mask.fill_diagonal_(1.0)
    old = tr.USE_SDPA
    try:
        tr.USE_SDPA = True
        a = att(x, mask)
        tr.USE_SDPA = False
        b = att(x, mask)
    finally:
        tr.USE_SDPA = old
    assert torch.allclose(a, b, atol=1e-5)


# ------------------------------------------------------------------- leakage guards
def test_probe_dataset_in_pretrain_raises():
    cfg = _cfg()
    cfg.data.pretrain_datasets = ("chapman", "ptbxl")
    with pytest.raises(ValueError, match="probe datasets"):
        build_pretrain_loaders({}, cfg, verbose=False)


def test_long_holdout_is_disjoint_and_used_by_split_by_ids():
    cfg = _cfg()
    recs = synthetic_long_records(10, 30.0, cfg.signal.fs, cfg.signal.n_leads, seed=0)
    sp = split_long_records(recs, test_frac=0.3, seed=0)
    assert set(sp.train_ids).isdisjoint(sp.test_ids)
    assert len(sp.test_ids) == 3
    assert sorted(sp.train_ids + sp.test_ids) == list(range(10))
    samples, _ = build_onset_dataset(recs, cfg.signal, horizon_s=60.0, stride_s=30.0,
                                     max_per_record=40)
    trs, tes = split_by_ids(samples, sp.test_ids)
    assert {s.record for s in tes} <= set(sp.test_ids)
    assert not ({s.record for s in trs} & set(sp.test_ids))


def test_pretraining_never_sees_holdout_records():
    cfg = _cfg()
    recs = synthetic_long_records(6, 30.0, cfg.signal.fs, cfg.signal.n_leads, seed=0)
    sp = split_long_records(recs, test_frac=0.34, seed=0)
    _, info = build_pretrain_loaders({}, cfg, long_records=sp.train_records(recs),
                                     samples_per_record=8, verbose=False)
    assert not (set(info["pretrain_long_record_names"]) & set(sp.test_names))


# ------------------------------------------------------------------- EMA / target
def test_target_encoder_stays_in_eval_mode():
    cfg = _cfg()
    cfg.model.drop_path_rate = 0.2
    m = _model(cfg).train()
    assert m.encoder.training
    assert not m.target_encoder.training


def test_horizon_counts_track_training_and_warn_on_untrained_bucket():
    cfg = _cfg()
    m = _model(cfg).train()
    B, W = 2, cfg.forward.context_windows
    ctx = torch.randn(B, W, cfg.signal.n_leads, cfg.signal.window_samples)
    fut = torch.randn(B, cfg.signal.n_leads, cfg.signal.window_samples)
    m(ctx, future_window=fut, horizon_s=torch.tensor([60.0, 60.0]))
    assert 60.0 in m.trained_horizons()
    with pytest.warns(RuntimeWarning, match="never trained"):
        m.forecast_latent(ctx, 1800.0)


def test_cross_window_target_is_pooled_by_default():
    cfg = _cfg()
    m = _model(cfg).eval()
    ctx = torch.randn(2, 1, cfg.signal.n_leads, cfg.signal.window_samples)
    z = m.forecast_latent(ctx, 60.0)
    assert z.shape == (2, cfg.model.encoder_embed_dim)


def test_level2_mask_predictor_gets_positions():
    cfg = _cfg()
    cfg.hierarchical.enabled = True
    m = _model(cfg).train()
    W = cfg.hierarchical.l2_context_windows
    ctx = torch.randn(2, W, cfg.signal.n_leads, cfg.signal.window_samples)
    fut = torch.randn(2, cfg.signal.n_leads, cfg.signal.window_samples)
    out = m(ctx, future_window=fut, horizon_s=torch.tensor([60.0, 60.0]),
            l2_future=fut, l2_horizon_s=torch.tensor([300.0, 300.0]))
    assert torch.isfinite(out["loss"])
    out["loss"].backward()


# ------------------------------------------------------------------- collapse arms
def test_soft_vq_targets_keep_their_spread():
    """v0.1 settings (T=0.5, random codebook) squash distinct d=384 targets onto
    almost one vector (the VQ-arm collapse); v0.2 (T=0.1, data-seeded codes) keeps
    their spread."""
    import torch.nn.functional as F
    torch.manual_seed(0)
    cent = torch.randn(16, 384)
    h = F.layer_norm(cent[torch.randint(0, 16, (2048,))] + 0.5 * torch.randn(2048, 384),
                     (384,))
    true_spread = float(h.std(0).mean())
    old = SoftVQ(384, 512, temperature=0.5).train()
    old.initialized.fill_(1)                              # v0.1: random codebook
    new = SoftVQ(384, 512, temperature=0.1).train()
    assert float(old.transform_target(h).std(0).mean()) < 0.3 * true_spread
    assert float(new.transform_target(h).std(0).mean()) > 0.8 * true_spread
    loss, stats = new(h)
    assert torch.isfinite(loss) and "vq_perplexity" in stats


def test_collapse_verdict_flags_a_collapsed_embedding():
    mon = cj.CollapseMonitor(arm="x")
    base = torch.randn(1, 64)
    # everyone points the same way and the residual lives in 2 directions (the v0.1
    # VQ-arm picture: cos 0.94, low covariance rank)
    z = base + 0.05 * torch.randn(512, 2) @ torch.randn(2, 64)
    mon.log(0, 1.0, z)
    mon.log(1, 0.5, z)
    mon.log(2, 0.4, z)
    s = mon.summary()
    assert s["representation_collapse"] and s["verdict"] == "collapsed"
    mon2 = cj.CollapseMonitor(arm="y")
    for e in range(3):
        mon2.log(e, 1.0 - 0.1 * e, torch.randn(512, 64))
    assert mon2.summary()["verdict"] == "healthy"


# ------------------------------------------------------------------- baselines
def _xbatch(cfg, n=16, seed=1):
    from cardio_jepa.data.preprocess import prepare_window
    w, _ = synth_short_corpus(n, cfg.signal.fs, cfg.signal.window_s, cfg.signal.n_leads,
                              seed=seed)
    w = np.stack([prepare_window(x, cfg.signal.fs, cfg.signal.fs,
                                 cfg.signal.window_samples, reduced=False) for x in w])
    return torch.from_numpy(w).float()


def test_stmem_fixed_target_learns_while_patch_norm_does_not():
    cfg = _cfg()
    X = _xbatch(cfg, 32)
    res = {}
    for norm in ("patch", "none"):
        torch.manual_seed(0)
        m = STMEMBaseline(cfg.signal, cfg.model, target_norm=norm)
        opt = torch.optim.AdamW(m.parameters(), lr=5e-4)
        ls = []
        for _ in range(40):
            idx = torch.randint(0, len(X), (8,))
            loss = m(X[idx])["loss"]
            opt.zero_grad(); loss.backward(); opt.step()
            ls.append(float(loss.detach()))
        res[norm] = (np.mean(ls[:5]), np.mean(ls[-5:]))
    assert res["none"][1] < 0.95 * res["none"][0]
    assert res["patch"][0] > 0.9                         # ~white-noise target


@pytest.mark.parametrize("cls", [SimCLRBaseline, CPCBaseline])
def test_ssl_baselines_train(cls):
    cfg = _cfg()
    torch.manual_seed(0)
    m = cls(cfg.signal, cfg.model)
    X = _xbatch(cfg, 16)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    first = last = None
    for i in range(15):
        out = m(X.unsqueeze(1))                           # (B, 1, C, T) like the loader
        opt.zero_grad(); out["loss"].backward(); opt.step()
        first = first if first is not None else float(out["loss"].detach())
        last = float(out["loss"].detach())
    assert np.isfinite(last) and last < first
    assert m.representation(X).shape == (16, cfg.model.encoder_embed_dim)


# ------------------------------------------------------------------- data
def test_mitbih_csv_sampling_rate_is_inferred(tmp_path):
    from cardio_jepa.data.discovery import FoundDataset
    from cardio_jepa.data.longform import load_long_records
    d = tmp_path / "mit-bih-long-term-ecg-database"
    d.mkdir()
    n = 128 * 60
    t_ms = np.arange(n) * (1000.0 / 128.0)
    pd.DataFrame({"time_ms": t_ms, "ECG1": np.random.randn(n),
                  "ECG2": np.random.randn(n)}).to_csv(d / "14046_ekg.csv", index=False)
    pd.DataFrame({"ECG1": np.random.randn(n), "ECG2": np.random.randn(n)}
                 ).to_csv(d / "14134_ekg.csv", index=False)
    ds = FoundDataset(key="mitbih_lt", label="MIT-BIH Long-Term ECG", kind="long",
                      root=str(d), data_root=str(d), fmt="mitbih_csv")
    recs = load_long_records(ds, limit=10)
    fs = {r.name: r.fs for r in recs}
    assert abs(fs["14046"] - 128.0) < 0.5            # from the time column
    assert fs["14134"] == 128.0                       # corpus default, not 360
    assert all(r.n_channels == 2 for r in recs)       # time column not read as a lead


def test_discovery_separates_mitbih_long_term(tmp_path):
    root = tmp_path / "input"
    lt = root / "datasets/protobioengineering/mit-bih-long-term-ecg-database"
    lt.mkdir(parents=True)
    (lt / "14046_ekg.csv").write_text("a,b\n1,2\n")
    found = discover(str(root), verbose=False)
    assert "mitbih_lt" in found and "mitbih" not in found


def test_discovery_skip_list(tmp_path):
    root = tmp_path / "input"
    m = root / "datasets/x/meeti-final-dataset"
    m.mkdir(parents=True)
    (m / "a.hea").write_text("")
    assert "meeti" in discover(str(root), verbose=False)
    assert "meeti" not in discover(str(root), verbose=False, skip=("meeti",))


def test_cache_roundtrip_and_zero_audit(tmp_path):
    w = np.random.randn(10, 8, 2500).astype(np.float32)
    w[3] = 0.0
    y = np.eye(2)[np.arange(10) % 2].astype(np.int64)
    ds = ArrayWindowDataset(w, y, return_dict=False)
    c = cache_dataset(ds, "t", str(tmp_path), num_workers=0, verbose=False)
    assert len(c) == 10 and c.meta["n_zero"] == 1
    x0, y0 = c[0]
    assert torch.allclose(x0, torch.from_numpy(w[0]), atol=1e-2)
    c2 = cache_dataset(ds, "t", str(tmp_path), num_workers=0, verbose=False)
    assert c2.meta == c.meta


def test_interleaved_loader_reset_gives_identical_schedule():
    cfg = _cfg()
    loader, _ = build_pretrain_loaders({}, cfg, samples_per_record=4, verbose=False)

    def schedule():
        loader.reset()
        return [tuple(b["ctx"].shape) for b in loader]
    assert schedule() == schedule()


def test_probe_batch_is_short_and_large():
    cfg = _cfg()
    loader, _ = build_pretrain_loaders({}, cfg, samples_per_record=4, verbose=False)
    pb = probe_batch_from_loader(loader, n=40)
    assert pb.shape[0] == 40 and pb.dim() == 3


# ------------------------------------------------------------------- statistics
def test_cluster_bootstrap_is_wider_than_iid():
    rng = np.random.default_rng(0)
    groups = np.repeat(np.arange(20), 50)
    eff = rng.normal(0, 1.0, 20)[groups]                 # strong within-record correlation
    y = (eff + rng.normal(0, 0.5, len(groups)) > 0).astype(int)
    s = eff + rng.normal(0, 1.0, len(groups))
    iid = bootstrap_auroc(y, s, n=300)
    clu = bootstrap_auroc(y, s, n=300, groups=groups)
    assert (clu["hi"] - clu["lo"]) > (iid["hi"] - iid["lo"])


def test_seed_level_statistics():
    s = seed_level_summary({0: 0.80, 1: 0.82, 2: 0.81})
    assert s["n"] == 3 and s["ci_lo"] < 0.81 < s["ci_hi"]
    t = paired_seed_test({0: 0.82, 1: 0.83, 2: 0.84}, {0: 0.80, 1: 0.80, 2: 0.81})
    assert t["delta"] > 0 and t["n_seeds"] == 3


# ------------------------------------------------------------------- evaluation
def test_probe_uses_validation_and_repeats():
    cfg = _cfg()
    splits = build_probe_splits({}, cfg, verbose=False)["synthetic"]
    m = _model(cfg)
    from cardio_jepa.eval import run_classification_probe
    res = run_classification_probe(m, splits, "cpu", cfg.evaluation, batch_size=32,
                                   verbose=False)
    assert res["best_lr"] in cfg.evaluation.linear_probe_lr_grid
    small = res["by_fraction"][0.1]
    assert len(small["macro_auroc_runs"]) == cfg.evaluation.label_fraction_repeats


def test_robustness_and_supervised_reference_run():
    from cardio_jepa.eval import run_robustness_probe, synthetic_noise_bank, train_supervised
    cfg = _cfg()
    splits = build_probe_splits({}, cfg, verbose=False)["synthetic"]
    m = _model(cfg)
    bank = synthetic_noise_bank(cfg.signal.fs, minutes=1.0)
    df = run_robustness_probe(m, splits, "cpu", cfg.evaluation, bank, snrs_db=(6.0,),
                              batch_size=32, verbose=False)
    assert {"clean", "bw@6dB", "leads:single_II"} <= set(df.condition)
    sup = train_supervised(cfg.signal, cfg.model, splits, "cpu", max_epochs=1,
                           min_steps=2, batch_size=32, num_workers=0, verbose=False)
    assert 0.0 <= sup["macro_auroc"] <= 1.0


def test_forecast_skill_reports_controls():
    from cardio_jepa.eval import forecast_skill
    cfg = _cfg()
    recs = synthetic_long_records(3, 30.0, cfg.signal.fs, cfg.signal.n_leads, seed=0)
    m = _model(cfg)
    df = forecast_skill(m, recs, cfg.signal, "cpu", horizons_s=(10.0, 60.0),
                        n_samples=12, verbose=False, n_boot=20)
    assert {"skill", "skill_lo", "horizon_gain", "cos_shuffled"} <= set(df.columns)

In [ ]:
# --- 2. import the package written above -------------------------------------------------
SRC_ROOT = os.path.join(WORK, "cardio_jepa_v3_src")
sys.path.insert(0, SRC_ROOT)
for _m in [m for m in list(sys.modules) if m == "cardio_jepa" or m.startswith("cardio_jepa.")]:
    del sys.modules[_m]                 # never pick up an older attached cardio_jepa package
import cardio_jepa as cj
assert cj.__version__ == "0.2.0", f"wrong package version {cj.__version__} at {cj.__file__}"
print(f"cardio_jepa {cj.__version__} from {os.path.dirname(cj.__file__)}")

from cardio_jepa.config import configure_arm, kaggle_config, smoke_config, full_config
from cardio_jepa.data import (build_onset_dataset, build_burden_dataset, discover, summarize,
                              split_long_records, split_by_ids, cache_dataset,
                              cache_probe_splits, AnticipationDataset)
from cardio_jepa.data.build import (build_pretrain_loaders, build_probe_splits,
                                    load_long_corpus, probe_batch_from_loader)
from cardio_jepa.eval import (run_classification_probe, run_anticipation_probe,
                              run_burden_probe, forecast_skill, horizon_sweep,
                              run_robustness_probe, load_nstdb_noise, synthetic_noise_bank,
                              train_supervised, bootstrap_auroc_delta, seed_level_summary,
                              paired_seed_test)
from cardio_jepa.models import CardioJEPA, STMEMBaseline, SimCLRBaseline, CPCBaseline
from cardio_jepa.engine import estimate_runtime, load_train_result, free_cuda
from cardio_jepa.models import transformer as _tr

import pandas as pd
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 60)
DEVICE = cj.pick_device()
print("device:", DEVICE, "| fused SDPA attention:", _tr.USE_SDPA)

## 3. Configuration

* `SESSION` picks which models this run **trains** (see the plan below). Models from earlier
  sessions are **loaded** from the attached previous output and are not retrained.
* The step budget (`EPOCHS × MAX_STEPS`) is **fixed for every model in every session**.
  v0.1 auto-shrank it to fit one session, which would make arms trained in different
  sessions incomparable; if a session does not fit, move jobs to another session instead.


In [ ]:
# --- 3. configuration ----------------------------------------------------------------
RUN_MODE = "kaggle"        # "kaggle" (T4/P100) | "smoke" (synthetic, minutes, CPU ok)
SESSION = "A"              # "A" | "B" | "C" | "D" | "ALL"  (ALL is meant for smoke mode)
VERSION_TAG = "v3"
INPUT_ROOT = "/kaggle/input"
SKIP_DATASETS = ("meeti",)  # not even walked by discovery (no reader for MEETI's format)
SPLIT_SEED = 0              # fixes the long-record holdout for ALL sessions -- do not change
BUDGET_HOURS = 11.0
FORCE_REEVAL = False        # re-run evaluations already present in the results ledger
ZIP_CHECKPOINTS = False     # .pth files stay in /kaggle/working/runs (saved by Kaggle anyway)
PREVIOUS_OUTPUT_DIRS = "auto"   # or a list of directories holding runs/ and results/

CFG = {"kaggle": kaggle_config, "smoke": smoke_config, "full": full_config}[RUN_MODE]()
CFG.data.input_root = INPUT_ROOT
CFG.data.skip_datasets = SKIP_DATASETS
CFG.out_dir = RUNS_DIR
CFG.forward.context_windows = 2           # 20 s of context for the forward objective
CFG.hierarchical.enabled = False          # switched on for Stage 4 only
if RUN_MODE == "kaggle":
    CFG.optim.epochs = 30
    CFG.optim.max_steps_per_epoch = 200   # 6 000 steps per model, every model, every session
    CFG.optim.num_workers = 2
    SAMPLES_PER_RECORD = 200
    MAX_PER_RECORD = 300                  # anticipation windows per record
    N_SKILL = 256
    PROBE_N = 512
    SUP_MAX_EPOCHS = 20
    RQ3_SEEDS = (0,)
else:                                     # smoke
    CFG.optim.epochs = 2
    CFG.optim.max_steps_per_epoch = 6
    CFG.optim.num_workers = 0
    SAMPLES_PER_RECORD = 8
    MAX_PER_RECORD = 40
    N_SKILL = 16
    PROBE_N = 48
    SUP_MAX_EPOCHS = 1
    RQ3_SEEDS = (0,)

CM_FACTOR = 1.5    # masked_only_cm gets 1.5x steps: dual costs ~1.5x per step (compute match)

# Which models each session TRAINS. (arm, seed). Everything else is loaded if available.
SESSION_PLAN = {
    "A": {"train": [("masked_only", 0), ("forward_only", 0), ("dual", 0),
                    ("simclr", 0), ("cpc", 0)],
          "supervised": [(1.0, 0), (0.1, 0), (0.01, 0)],
          "stages": ["s1", "robust", "s2"]},
    "B": {"train": [("masked_only", 1), ("dual", 1), ("masked_only", 2), ("dual", 2),
                    ("st_mem", 0), ("masked_only_cm", 0)],
          "supervised": [],
          "stages": ["s1", "s2"]},
    "C": {"train": [("rq3_ema", 0), ("rq3_vq", 0), ("rq3_sigreg", 0),
                    ("simclr", 1), ("simclr", 2), ("cpc", 1), ("cpc", 2)],
          "supervised": [(1.0, 1), (1.0, 2)],
          "stages": ["s1", "s3"]},
    "D": {"train": [("stage4_hier", 0), ("stage4_single", 0),
                    ("forward_only", 1), ("forward_only", 2)],
          "supervised": [],
          "stages": ["s1", "s3", "s4"]},
}
SESSION_PLAN["ALL"] = {
    "train": sum((SESSION_PLAN[k]["train"] for k in "ABCD"), []),
    "supervised": sum((SESSION_PLAN[k]["supervised"] for k in "ABCD"), []),
    "stages": ["s1", "robust", "s2", "s3", "s4"]}
if RUN_MODE == "smoke" and SESSION == "ALL":   # keep the smoke run short: one seed per arm
    SESSION_PLAN["ALL"]["train"] = [j for j in SESSION_PLAN["ALL"]["train"] if j[1] == 0]
    SESSION_PLAN["ALL"]["supervised"] = [(1.0, 0), (0.1, 0)]
PLAN = SESSION_PLAN[SESSION]

cj.set_seed(CFG.optim.seed, CFG.optim.deterministic)
print(f"mode={RUN_MODE} session={SESSION} | encoder {CFG.model.encoder_embed_dim}d x "
      f"{CFG.model.encoder_depth}L | {CFG.optim.epochs} epochs x {CFG.optim.max_steps_per_epoch} "
      f"steps, batch {CFG.optim.batch_size}")
print("trains:", [f"{a}_s{s}" for a, s in PLAN["train"]],
      "| supervised:", PLAN["supervised"], "| stages:", PLAN["stages"])
print("forward horizons (s):", CFG.forward.horizons_s,
      "| anticipation horizons (s):", CFG.evaluation.anticipation_horizons_s)

In [ ]:
# --- 4. results ledger, checkpoint reuse, export -------------------------------------
LEDGER_PATH = os.path.join(RESULTS_DIR, "ledger.csv")
LEDGER_COLS = ["version", "session", "time", "model_id", "arm", "family", "seed", "task",
               "setting", "metric", "value", "extra"]
_ROWS = []

def family_of(arm):
    if arm.startswith("rq3_"): return "rq3"
    if arm.startswith("stage4_"): return "stage4"
    if arm in ("masked_only", "forward_only", "dual", "masked_only_cm"): return "jepa"
    if arm.startswith("supervised"): return "supervised"
    return "baseline"

def log(model_id, arm, seed, task, metric, value, setting="", extra=None):
    try:
        value = float(value)
    except (TypeError, ValueError):
        value = float("nan")
    _ROWS.append({"version": VERSION_TAG, "session": SESSION,
                  "time": datetime.datetime.utcnow().isoformat(timespec="seconds"),
                  "model_id": model_id, "arm": arm, "family": family_of(arm),
                  "seed": seed, "task": task, "setting": str(setting), "metric": metric,
                  "value": value, "extra": json.dumps(extra, default=str) if extra else ""})

def _bfs_dirs(root, max_depth, skip):
    out, stack = [], [(root, 0)]
    while stack:
        p, d = stack.pop()
        if d > max_depth or any(os.path.abspath(p).startswith(s) for s in skip):
            continue
        out.append(p)
        try:
            ents = [e for e in os.scandir(p) if e.is_dir(follow_symlinks=False)]
        except OSError:
            continue
        if len(ents) > 2000:
            continue
        stack.extend((e.path, d + 1) for e in ents if not e.name.startswith("."))
    return out

def find_previous_outputs():
    if PREVIOUS_OUTPUT_DIRS != "auto":
        return list(PREVIOUS_OUTPUT_DIRS)
    if not os.path.isdir(INPUT_ROOT):
        return []
    skip = [os.path.abspath(d.root) for d in globals().get("FOUND", {}).values()]
    hits = []
    for p in _bfs_dirs(INPUT_ROOT, 6, skip):
        if (os.path.isfile(os.path.join(p, "results", "ledger.csv"))
                or glob.glob(os.path.join(p, "runs", "*_diagnostics.json"))):
            hits.append(p)
    return hits

def previous_ledgers():
    frames = []
    for d in PREV_DIRS:
        f = os.path.join(d, "results", "ledger.csv")
        if os.path.isfile(f):
            frames.append(pd.read_csv(f, keep_default_na=False, na_values=[""]))
    return frames

def ledger_df():
    frames = previous_ledgers() + [pd.DataFrame(_ROWS, columns=LEDGER_COLS)]
    df = pd.concat([f for f in frames if len(f)], ignore_index=True) if any(
        len(f) for f in frames) else pd.DataFrame(columns=LEDGER_COLS)
    if len(df):
        df["setting"] = df["setting"].fillna("").astype(str)
        df = (df.sort_values("time")
                .drop_duplicates(["version", "model_id", "task", "setting", "metric"], keep="last"))
    return df

def save_ledger():
    df = ledger_df()
    df.to_csv(LEDGER_PATH, index=False)
    return df

def already_done(model_id, task, metric="macro_auroc"):
    if FORCE_REEVAL:
        return False
    df = ledger_df()
    if not len(df):
        return False
    m = (df.version == VERSION_TAG) & (df.model_id == model_id) & (df.task == task) & (df.metric == metric)
    return bool(m.any())

def checkpoint_index():
    idx = {}
    for d in PREV_DIRS:                         # earlier sessions (first match wins)
        for p in glob.glob(os.path.join(d, "runs", "*.pth")):
            idx.setdefault(os.path.basename(p), p)
    for p in glob.glob(os.path.join(RUNS_DIR, "*.pth")):   # this session overrides
        idx[os.path.basename(p)] = p
    return idx

def save_scores(name, **arrays):
    np.savez_compressed(os.path.join(SCORE_DIR, name + ".npz"), **arrays)

def load_scores(name):
    for d in [WORK] + PREV_DIRS:
        f = os.path.join(d, "results", "scores", name + ".npz")
        if os.path.isfile(f):
            return dict(np.load(f, allow_pickle=False))
    return None

PREV_DIRS = []           # filled after discovery (needs the dataset roots to skip them)
REGISTRY = {}            # model_id -> {arm, seed, model, res, cfg}
print("ledger helpers ready")

## 5–6. Discovery, fixed holdout, loaders, probe caches, leakage checks

In [ ]:
# --- 5. discover datasets ----------------------------------------------------------------
t0 = time.time()
FOUND = discover(INPUT_ROOT, verbose=True, skip=CFG.data.skip_datasets)
print(); print(summarize(FOUND))
print(f"discovery took {time.time() - t0:.0f}s (v0.1: 3066 s)")
for k in ("ptbxl", "cpsc"):
    if k in FOUND and k in CFG.data.pretrain_datasets:
        raise RuntimeError(f"{k} must not be a pretraining corpus")
PREV_DIRS = find_previous_outputs()
CKPTS = checkpoint_index()
print(f"previous outputs found: {PREV_DIRS or 'none'} | checkpoints available: {len(CKPTS)}")

In [ ]:
# --- 6. data: fixed long-record holdout FIRST, then loaders, probe splits, caches ----------
t0 = time.time()
LONG_ALL = load_long_corpus(FOUND, CFG, verbose=True)
HOLDOUT = split_long_records(LONG_ALL, test_frac=CFG.evaluation.long_test_frac, seed=SPLIT_SEED)
LONG_TRAIN = HOLDOUT.train_records(LONG_ALL)
LONG_TEST = HOLDOUT.test_records(LONG_ALL)
print(f"long records: {len(LONG_ALL)} | pretrain {len(LONG_TRAIN)} | held out {len(LONG_TEST)} "
      f"(annotated, burden-stratified, fixed by SPLIT_SEED={SPLIT_SEED})")

LOADER, DATA_INFO = build_pretrain_loaders(FOUND, CFG, long_records=LONG_TRAIN,
                                           samples_per_record=SAMPLES_PER_RECORD, verbose=True)
PROBE_BATCH = probe_batch_from_loader(LOADER, n=PROBE_N)

# ---- leakage checks (all must pass; recorded in the ledger) ----
CHECKS = {}
CHECKS["probe_sets_not_pretrained"] = not (set(DATA_INFO.get("short_sources", {})) &
                                           set(CFG.data.probe_datasets))
CHECKS["holdout_not_in_forward_pairs"] = not (set(DATA_INFO.get("pretrain_long_record_names", []))
                                              & set(HOLDOUT.test_names))
from cardio_jepa.models.forward_predictor import horizon_to_bucket
fh = DATA_INFO.get("forward_horizon_counts", {})
_tb = {int(horizon_to_bucket([t], cj.HORIZON_BUCKETS_S)[0]) for t, n in fh.items() if n > 0}
_nb = {int(horizon_to_bucket([h], cj.HORIZON_BUCKETS_S)[0]) for h in CFG.evaluation.anticipation_horizons_s}
CHECKS["eval_horizons_are_trained_buckets"] = (not fh) or _nb <= _tb

RAW_SPLITS = build_probe_splits(FOUND, CFG, verbose=True)
PROBE_SPLITS = {}
for task, sp in RAW_SPLITS.items():
    PROBE_SPLITS[task] = cache_probe_splits(sp, task, CFG.data.cache_dir,
                                            num_workers=CFG.optim.num_workers)
    if "patient_overlap" in sp:
        CHECKS[f"{task}_patient_disjoint"] = sp["patient_overlap"] == 0
    zf = max(getattr(PROBE_SPLITS[task][p], "meta", {}).get("zero_frac", 0.0)
             for p in ("train", "val", "test") if PROBE_SPLITS[task].get(p) is not None)
    CHECKS[f"{task}_reads_ok(<1% zero-filled)"] = zf < 0.01

short_ds = getattr(LOADER.loaders[0], "dataset", None)
if hasattr(short_ds, "audit_reads"):
    aud = short_ds.audit_reads(n=200 if RUN_MODE == "kaggle" else 20)
    CHECKS["pretrain_reads_ok(<2% failed)"] = aud["failed_frac"] < 0.02
    print(f"pretraining read audit: {aud}")

for k, v in CHECKS.items():
    log("data", "data", -1, "checks", k, float(bool(v)))
print("\nLEAKAGE / DATA CHECKS")
for k, v in CHECKS.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")

rows = [{"corpus": k, "label": d.label, "kind": d.kind, "fmt": d.fmt, "files>=": d.n_files,
         "role": ("probe" if k in CFG.data.probe_datasets else
                  "pretrain" if k in CFG.data.pretrain_datasets else
                  "long (pretrain/holdout)" if d.kind == "long" else "noise / robustness")}
        for k, d in FOUND.items()]
DATA_REPORT = pd.DataFrame(rows)
DATA_REPORT.to_csv(os.path.join(RESULTS_DIR, "data_report.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "holdout_records.json"), "w") as fh_:
    json.dump({"test": HOLDOUT.test_names, "train": HOLDOUT.train_names}, fh_, indent=1)
print(f"\nshort clips {DATA_INFO.get('n_short', 0)} | forward pairs {DATA_INFO.get('n_forward_pairs', 0)}"
      f" | batch share {DATA_INFO.get('loader_weights')} | probe tasks {list(PROBE_SPLITS)}"
      f" | ready in {time.time() - t0:.0f}s")
save_ledger()

## 7–8. Arms, train-or-load, runtime projection

In [ ]:
# --- 7. arms, models, train-or-load ---------------------------------------------------
def arm_cfg(arm, seed):
    c = copy.deepcopy(CFG)
    c.optim.seed = seed
    c.name = f"{arm}_s{seed}"
    if arm in ("masked_only", "masked_only_cm"):
        c.loss.weight_forward, c.loss.lambda_mask = 0.0, 1.0
        if arm == "masked_only_cm":
            c.optim.max_steps_per_epoch = int(round(c.optim.max_steps_per_epoch * CM_FACTOR))
    elif arm == "forward_only":
        c.loss.weight_forward, c.loss.lambda_mask = 1.0, 0.0
    elif arm == "dual":
        c.loss.weight_forward, c.loss.lambda_mask = 1.0, 1.0
    elif arm.startswith("rq3_"):
        c = configure_arm(c, arm[4:])
        c.optim.seed, c.name = seed, f"{arm}_s{seed}"
        c.loss.weight_forward, c.loss.lambda_mask = 1.0, 1.0
        c.optim.on_collapse = "warn"            # RQ3 observes collapse instead of aborting
    elif arm.startswith("stage4_"):
        c.hierarchical.enabled = arm == "stage4_hier"
        c.forward.context_windows = c.hierarchical.l2_context_windows
        c.loss.weight_forward, c.loss.lambda_mask = 1.0, 1.0
    return c

def make_model(arm, c):
    if arm == "simclr": return SimCLRBaseline(c.signal, c.model)
    if arm == "cpc":    return CPCBaseline(c.signal, c.model)
    if arm == "st_mem": return STMEMBaseline(c.signal, c.model)
    return CardioJEPA(c.signal, c.model, c.loss, c.hierarchical, c.forward)

def log_training(model_id, arm, seed, res):
    s = res.monitor.summary()
    losses = [h.get("loss") for h in res.history if h.get("loss") is not None]
    first, last = (losses[0], losses[-1]) if losses else (float("nan"), float("nan"))
    drop = (first - last) / abs(first) if losses and first else float("nan")
    for k, v in {"first_loss": first, "final_loss": last, "loss_drop": drop,
                 "effective_rank": s.get("final_effective_rank"),
                 "cov_erank_ratio": s.get("final_cov_erank_ratio"),
                 "std_mean": s.get("final_std_mean"), "cos_sim": s.get("final_cos_sim"),
                 "collapsed_flag": float(bool(res.collapsed)),
                 "loss_to_zero": float(bool(s.get("collapsed_to_zero"))),
                 "seconds": res.seconds}.items():
        log(model_id, arm, seed, "train", k, v)
    log(model_id, arm, seed, "train", "verdict_collapsed",
        float(s.get("verdict") == "collapsed"), extra={"verdict": s.get("verdict")})

def get_model(arm, seed, loader=None, allow_train=False):
    model_id = f"{arm}_s{seed}"
    if model_id in REGISTRY:
        return REGISTRY[model_id]
    c = arm_cfg(arm, seed)
    path = CKPTS.get(model_id + ".pth")
    if path and os.path.isfile(os.path.splitext(path)[0] + "_diagnostics.json"):
        m = make_model(arm, c)
        rep = cj.load_weights(m, path, strict=False, verbose=False)
        if not rep.shape_mismatch and len(rep.missing) == 0:
            res = load_train_result(path)
            print(f"[{model_id}] loaded from {path}")
            REGISTRY[model_id] = {"arm": arm, "seed": seed, "model": m, "res": res, "cfg": c}
            return REGISTRY[model_id]
        print(f"[{model_id}] checkpoint does not match this config ({len(rep.missing)} missing, "
              f"{len(rep.shape_mismatch)} mismatched) -> " + ("retraining" if allow_train else "skipped"))
    if not allow_train:
        return None
    print(f"\n=== training [{model_id}] ===")
    cj.set_seed(seed, c.optim.deterministic)
    m = make_model(arm, c)
    n_par = sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6
    print(f"    {n_par:.1f}M trainable parameters | budget {c.optim.epochs} x {c.optim.max_steps_per_epoch} steps")
    res = cj.pretrain(m, loader or LOADER, c, device=DEVICE, arm_name=model_id,
                      probe_batch=PROBE_BATCH, save_path=os.path.join(RUNS_DIR, model_id + ".pth"),
                      log_every=0 if RUN_MODE == "smoke" else 200)
    log_training(model_id, arm, seed, res)
    CKPTS[model_id + ".pth"] = os.path.join(RUNS_DIR, model_id + ".pth")
    m.to("cpu"); free_cuda()
    REGISTRY[model_id] = {"arm": arm, "seed": seed, "model": m, "res": res, "cfg": c}
    save_ledger()
    return REGISTRY[model_id]

class RandomInit(torch.nn.Module):
    # floor: the same encoder, never trained
    def __init__(self):
        super().__init__()
        cj.set_seed(0)
        self.inner = CardioJEPA(CFG.signal, CFG.model, CFG.loss, CFG.hierarchical, CFG.forward)
    def representation(self, x):
        return self.inner.representation(x)
print("model helpers ready")

In [ ]:
# --- 8. runtime projection (informational; the budget is NOT changed automatically) ----
jobs = [j for j in PLAN["train"] if f"{j[0]}_s{j[1]}.pth" not in CKPTS
        and not j[0].startswith("stage4_")]
if jobs and RUN_MODE == "kaggle":
    _m = CardioJEPA(CFG.signal, CFG.model, CFG.loss, CFG.hierarchical, CFG.forward)
    c_ = arm_cfg("dual", 0)
    EST = estimate_runtime(_m, LOADER, c_, DEVICE, n_steps=8)
    del _m; free_cuda()
    cost = {"masked_only": 0.7, "forward_only": 0.75, "dual": 1.0, "masked_only_cm": 1.05,
            "simclr": 0.8, "cpc": 0.7, "st_mem": 0.6, "rq3_ema": 1.0, "rq3_vq": 1.0,
            "rq3_sigreg": 1.0}
    hours = sum(cost.get(a, 1.0) for a, _ in jobs) * EST["s_per_step"] * \
        CFG.optim.epochs * CFG.optim.max_steps_per_epoch / 3600
    hours += 0.6 * len(PLAN["supervised"]) + 1.5      # supervised runs + evaluation
    print(f"dual: {EST['s_per_step']:.3f} s/step -> this session ~{hours:.1f} h "
          f"for {len(jobs)} pretraining job(s) + evaluation (budget {BUDGET_HOURS} h)")
    if hours > BUDGET_HOURS:
        print(">>> over budget: move some jobs from PLAN['train'] to a later session "
              "(do not shrink MAX_STEPS -- all arms must share one budget)")
    log("session", "session", -1, "runtime", "projected_hours", hours)
else:
    print("nothing new to pretrain in this session (or smoke mode)")

## Stage 1 — objective ablation + baselines (PTB-XL / CPSC2018 linear probe)

| arm | `w_fwd` | `λ` | role |
|---|---|---|---|
| `masked_only` | 0 | 1 | Kim-style masked latent infilling (same encoder, our budget) |
| `forward_only` | 1 | 0 | forward objective alone |
| `dual` | 1 | 1 | **Cardio-JEPA** |
| `masked_only_cm` | 0 | 1 | compute-matched (1.5× steps) |
| `simclr`, `cpc`, `st_mem` | — | — | contrastive / contrastive-forward / reconstruction baselines |
| `supervised_f*` | — | — | same encoder, trained with labels (1 / 10 / 100 %) |
| `random_init` | — | — | untrained encoder (floor) |

All arms use VICReg on the pooled representation (β = 0.1) except the RQ3 arms. Linear
probe: frozen encoder, head learning rate chosen on fold 9, test on fold 10.


In [ ]:
# --- Stage 1a: train (or load) this session's models --------------------------------------
for arm, seed in PLAN["train"]:
    if arm.startswith("stage4_"):
        continue                      # Stage 4 needs its own wide-context loader (below)
    try:
        get_model(arm, seed, allow_train=True)
    except torch.cuda.OutOfMemoryError as exc:
        print(f"[{arm}_s{seed}] OOM: {exc}"); free_cuda()
    save_ledger()

# load every model that earlier sessions trained (for summaries / reuse), without training
for sess in ("A", "B", "C", "D"):
    for arm, seed in SESSION_PLAN[sess]["train"]:
        if not arm.startswith("stage4_"):
            get_model(arm, seed, allow_train=False)
print("\navailable models:", sorted(REGISTRY))

In [ ]:
# --- Stage 1b: linear probes (+ supervised references) -----------------------------------
def eval_probe(model_id, arm, seed, model):
    for task, splits in PROBE_SPLITS.items():
        if already_done(model_id, f"probe_{task}"):
            continue
        out = run_classification_probe(model.to(DEVICE), splits, DEVICE, CFG.evaluation,
                                       batch_size=64, num_workers=CFG.optim.num_workers,
                                       verbose=False)
        log(model_id, arm, seed, f"probe_{task}", "macro_auroc", out["macro_auroc"], "1.0",
            extra={"best_lr": out.get("best_lr")})
        log(model_id, arm, seed, f"probe_{task}", "macro_f1", out.get("macro_f1"), "1.0")
        for j, a in enumerate(out.get("per_class_auroc") or []):
            log(model_id, arm, seed, f"probe_{task}", "class_auroc",
                a, (splits.get("label_names") or [str(j)] * 99)[j])
        for frac, r in out["by_fraction"].items():
            log(model_id, arm, seed, f"labeleff_{task}", "macro_auroc", r["macro_auroc"], frac)
            log(model_id, arm, seed, f"labeleff_{task}", "macro_auroc_std",
                r.get("macro_auroc_std", 0.0), frac)
        save_scores(f"{model_id}__{task}", y_true=np.asarray(out["y_true"]),
                    y_score=np.asarray(out["y_score"]))
        print(f"  {model_id:22s} {task:8s} macro AUROC={out['macro_auroc']:.4f} "
              f"(lr {out.get('best_lr')})")
    model.to("cpu"); free_cuda()

if "s1" in PLAN["stages"] and PROBE_SPLITS:
    print("### linear probes")
    if SESSION in ("A", "ALL") and not already_done("random_init_s0", f"probe_{next(iter(PROBE_SPLITS))}"):
        eval_probe("random_init_s0", "random_init", 0, RandomInit())
    for mid, e in sorted(REGISTRY.items()):
        if not e["arm"].startswith("stage4_"):
            eval_probe(mid, e["arm"], e["seed"], e["model"])
    for frac, seed in PLAN["supervised"]:
        mid = f"supervised_f{frac:g}_s{seed}"
        for task, splits in PROBE_SPLITS.items():
            if already_done(mid, f"probe_{task}"):
                continue
            sup = train_supervised(CFG.signal, CFG.model, splits, DEVICE, label_fraction=frac,
                                   max_epochs=SUP_MAX_EPOCHS, seed=seed,
                                   num_workers=CFG.optim.num_workers, verbose=False)
            log(mid, f"supervised_f{frac:g}", seed, f"probe_{task}", "macro_auroc",
                sup["macro_auroc"], "1.0", extra={"epochs": sup["epochs_run"]})
            log(mid, f"supervised_f{frac:g}", seed, f"labeleff_{task}", "macro_auroc",
                sup["macro_auroc"], frac)
            save_scores(f"{mid}__{task}", y_true=sup["y_true"], y_score=sup["y_score"])
            print(f"  {mid:22s} {task:8s} macro AUROC={sup['macro_auroc']:.4f} "
                  f"({sup['n_train']} labelled ECGs)")
            if frac == 1.0 and seed == 0 and task == next(iter(PROBE_SPLITS)):
                REGISTRY[mid] = {"arm": "supervised_f1", "seed": 0, "model": sup["model"].cpu(),
                                 "res": None, "cfg": CFG}
            free_cuda()
    save_ledger()
else:
    print("Stage 1 probes skipped in this session")

In [ ]:
# --- Stage 1c: seed-level tables and the go/no-go test ------------------------------------
def probe_table(task):
    df = ledger_df()
    df = df[(df.version == VERSION_TAG) & (df.task == f"probe_{task}")]
    if not len(df):
        return pd.DataFrame()
    au = df[df.metric == "macro_auroc"]; f1 = df[df.metric == "macro_f1"]
    rows = []
    for arm, g in au.groupby("arm"):
        s = seed_level_summary(dict(zip(g.seed.astype(int), g.value)))
        f = f1[f1.arm == arm].value
        rows.append({"arm": arm, "family": g.family.iloc[0], "n_seeds": s["n"],
                     "macro_auroc_mean": s["mean"], "macro_auroc_std": s["std"],
                     "ci_lo": s["ci_lo"], "ci_hi": s["ci_hi"],
                     "macro_f1_mean": f.mean() if len(f) else np.nan})
    return pd.DataFrame(rows).sort_values("macro_auroc_mean", ascending=False)

TESTS = []
for task in PROBE_SPLITS:
    t = probe_table(task)
    if not len(t):
        continue
    t.to_csv(os.path.join(RESULTS_DIR, f"table_linear_probe_{task}.csv"), index=False)
    print(f"\n### {task}: linear probe (mean over seeds)"); display(t.round(4))
    df = ledger_df()
    au = df[(df.version == VERSION_TAG) & (df.task == f"probe_{task}") & (df.metric == "macro_auroc")]
    by = {a: dict(zip(g.seed.astype(int), g.value)) for a, g in au.groupby("arm")}
    for a, b in (("dual", "masked_only"), ("dual", "masked_only_cm"), ("dual", "forward_only"),
                 ("dual", "cpc"), ("dual", "simclr"), ("dual", "st_mem")):
        if a in by and b in by:
            p = paired_seed_test(by[a], by[b])
            sa, sb = load_scores(f"{a}_s0__{task}"), load_scores(f"{b}_s0__{task}")
            boot = (bootstrap_auroc_delta(sa["y_true"], sa["y_score"], sb["y_score"],
                                          n=CFG.evaluation.bootstrap_n)
                    if sa is not None and sb is not None else {})
            TESTS.append({"task": task, "a": a, "b": b, "seed_delta": p["delta"],
                          "n_seeds": p["n_seeds"], "seed_p": p["p_value"],
                          "boot_delta_s0": boot.get("delta"), "boot_lo": boot.get("lo"),
                          "boot_hi": boot.get("hi")})
if TESTS:
    TESTS_DF = pd.DataFrame(TESTS)
    TESTS_DF.to_csv(os.path.join(RESULTS_DIR, "table_tests.csv"), index=False)
    print("\n### paired tests (seed-level t-test; bootstrap on seed 0 test set)")
    display(TESTS_DF.round(4))
    g = TESTS_DF[(TESTS_DF.a == "dual") & (TESTS_DF.b == "masked_only")]
    if len(g):
        r = g.iloc[0]
        if r.n_seeds >= 3:
            verdict = ("GO: dual better" if r.seed_delta > 0 and r.seed_p < 0.05 else
                       "NO-GO: dual significantly worse" if r.seed_delta < 0 and r.seed_p < 0.05 else
                       "TIE: no significant difference over seeds")
        else:
            verdict = f"PRELIMINARY ({int(r.n_seeds)} seed(s)) -- run sessions B-D for 3 seeds"
        print(f"\nGo/no-go (dual vs masked_only): delta={r.seed_delta:+.4f} -> {verdict}")

## Stage 1d — robustness to real noise and missing leads

The linear head is trained on clean PTB-XL features and applied to test ECGs with NSTDB
noise (baseline wander, electrode motion, muscle artefact) at 24 / 12 / 6 / 0 dB SNR, and to
reduced lead sets (I+II; single lead II, as in patch/wearable ECG). The noise is band-passed
like the ECG pipeline (0.5–45 Hz), i.e. it is the part of the noise that survives standard
filtering.


In [ ]:
# --- Stage 1d: robustness ------------------------------------------------------------------
if "robust" in PLAN["stages"] and PROBE_SPLITS:
    task = "ptbxl" if "ptbxl" in PROBE_SPLITS else next(iter(PROBE_SPLITS))
    nst = FOUND.get("mitbih_nst")
    BANK = load_nstdb_noise(nst.data_root if nst else None, CFG.signal.fs) or \
        synthetic_noise_bank(CFG.signal.fs, minutes=10.0 if RUN_MODE == "kaggle" else 1.0)
    snrs = (24.0, 12.0, 6.0, 0.0) if RUN_MODE == "kaggle" else (6.0,)
    targets = [m for m in sorted(REGISTRY) if m.endswith("_s0") and
               REGISTRY[m]["arm"] in ("masked_only", "forward_only", "dual", "simclr", "cpc",
                                      "st_mem", "supervised_f1")]
    for mid in targets:
        e = REGISTRY[mid]
        if already_done(mid, f"robust_{task}", "macro_auroc_clean"):
            continue
        print(f"[{mid}]")
        df_r = run_robustness_probe(e["model"].to(DEVICE), PROBE_SPLITS[task], DEVICE,
                                    CFG.evaluation, BANK, snrs_db=snrs, batch_size=64,
                                    num_workers=CFG.optim.num_workers, verbose=True)
        for _, r in df_r.iterrows():
            log(mid, e["arm"], e["seed"], f"robust_{task}",
                "macro_auroc_clean" if r.condition == "clean" else "macro_auroc",
                r.macro_auroc, r.condition)
        e["model"].to("cpu"); free_cuda(); save_ledger()
else:
    print("robustness skipped in this session")

## Stage 2 — H1: AF-onset anticipation on **held-out** records + forecast skill

The test records were removed from pretraining (section 6), so the forward predictor never
saw their futures. Features: `context` (pooled encoder embedding of the 20 s context) and
`world` (`context` ‖ predicted future latent at H). **Read `world` gains against the
`masked_only/world` control**: its forward predictor is never trained, so it adds a random
projection of the context of the same size. CIs are record-level cluster bootstraps.

Forecast skill = cos(prediction, true future) − cos(present, true future) in the trained
target space; `horizon_gain` = cos(prediction at H) − cos(prediction at a wrong horizon).


In [ ]:
# --- Stage 2: anticipation + forecast skill -------------------------------------------------
CW = CFG.forward.context_windows
annotated_test = [r for r in LONG_TEST if r.episodes]
if "s2" in PLAN["stages"] and annotated_test:
    jepa = [m for m in sorted(REGISTRY) if REGISTRY[m]["arm"] in
            ("masked_only", "forward_only", "dual", "masked_only_cm")]
    others = [m for m in sorted(REGISTRY) if REGISTRY[m]["arm"] in ("simclr", "cpc", "st_mem")
              and m.endswith("_s0")]
    for H in CFG.evaluation.anticipation_horizons_s:
        samples, stats = build_onset_dataset(
            LONG_ALL, CFG.signal, horizon_s=H, context_windows=CW,
            stride_s=CFG.evaluation.anticipation_stride_s,
            guard_s=CFG.evaluation.anticipation_guard_s, max_per_record=MAX_PER_RECORD)
        tr, te = split_by_ids(samples, HOLDOUT.test_ids)
        n_pos_te = sum(s.label for s in te)
        print(f"\nH={H:.0f}s: {stats['n_positive']:.0f} pos / {stats['n_negative']:.0f} neg | "
              f"train {len(tr)} / test {len(te)} windows ({n_pos_te:.0f} test positives)")
        if n_pos_te < 5 or len(te) - n_pos_te < 5 or not tr:
            print("  too few test events at this horizon -> skipped"); continue
        tr_ds = cache_dataset(AnticipationDataset(LONG_ALL, tr, CFG.signal, CW), f"onset{H:.0f}_tr",
                              CFG.data.cache_dir, num_workers=CFG.optim.num_workers)
        te_ds = cache_dataset(AnticipationDataset(LONG_ALL, te, CFG.signal, CW), f"onset{H:.0f}_te",
                              CFG.data.cache_dir, num_workers=CFG.optim.num_workers)
        for mid in jepa + others:
            e = REGISTRY[mid]
            modes = ("context", "world") if hasattr(e["model"], "forward_predictor") else ("context",)
            for mode in modes:
                key = f"onset_{mode}"
                if already_done(mid, key, f"auroc@{H:.0f}"):
                    continue
                r = run_anticipation_probe(e["model"].to(DEVICE), LONG_ALL, tr, te, CFG.signal,
                                           DEVICE, CFG.evaluation, context_windows=CW,
                                           feature_mode=mode, horizons_s=[H], batch_size=64,
                                           train_ds=tr_ds, test_ds=te_ds, verbose=False)
                if not np.isfinite(r.get("auroc", np.nan)):
                    print(f"  {mid}/{mode}: {r.get('note')}"); continue
                for k in ("auroc", "ci_lo", "ci_hi", "auprc", "prevalence"):
                    log(mid, e["arm"], e["seed"], key, f"{k}@{H:.0f}", r[k], H)
                for f, v in r["by_fraction"].items():
                    log(mid, e["arm"], e["seed"], f"onset_labeleff_{mode}", f"auroc@{H:.0f}", v, f)
                save_scores(f"{mid}__onset{H:.0f}_{mode}", y_true=r["y_true"],
                            y_score=r["y_score"], groups=r["groups"])
                print(f"  {mid:20s} {mode:7s} AUROC={r['auroc']:.4f} [{r['ci_lo']:.4f},{r['ci_hi']:.4f}]"
                      f"  AUPRC={r['auprc']:.4f}")
            e["model"].to("cpu"); free_cuda()
        save_ledger()

    # forecast skill on held-out records only
    HS = tuple(h for h in (10.0, 60.0, 300.0, 600.0, 1800.0) if RUN_MODE == "kaggle" or h <= 300)
    for mid in jepa:
        e = REGISTRY[mid]
        if e["arm"] == "masked_only_cm" or already_done(mid, "forecast_skill", f"skill@{HS[0]:.0f}"):
            continue
        print(f"[{mid}] forecast skill on {len(LONG_TEST)} held-out records")
        df_s = forecast_skill(e["model"].to(DEVICE), LONG_TEST, CFG.signal, DEVICE, horizons_s=HS,
                              n_samples=N_SKILL, context_windows=CW, seed=0, verbose=True)
        for _, r in df_s.iterrows():
            for k in ("skill", "skill_lo", "skill_hi", "horizon_gain", "cos_pred", "cos_persistence"):
                log(mid, e["arm"], e["seed"], "forecast_skill", f"{k}@{r.horizon_s:.0f}", r[k],
                    r.horizon_s)
        e["model"].to("cpu"); free_cuda()
    save_ledger()
else:
    print("Stage 2 skipped (not in this session, or no annotated held-out records)")

In [ ]:
# --- Stage 2 summary: world gain vs the random-predictor control --------------------------
df = ledger_df()
an = df[(df.version == VERSION_TAG) & df.task.isin(["onset_context", "onset_world"])
        & df.metric.str.startswith("auroc@")]
if len(an):
    an = an.assign(H=an.metric.str.split("@").str[1].astype(float),
                   mode=an.task.str.replace("onset_", ""))
    tab = an.pivot_table(index=["arm", "H"], columns="mode", values="value", aggfunc="mean")
    if "world" in tab.columns and "context" in tab.columns:
        tab["world_gain"] = tab["world"] - tab["context"]
        ctrl = tab.xs("masked_only", level="arm")["world_gain"] if "masked_only" in tab.index.get_level_values(0) else None
        if ctrl is not None:
            tab["gain_over_random_predictor"] = [
                g - ctrl.get(h, np.nan) for (a, h), g in tab["world_gain"].items()]
    tab = tab.reset_index()
    tab.to_csv(os.path.join(RESULTS_DIR, "table_anticipation.csv"), index=False)
    print("AUROC (mean over seeds); 'gain_over_random_predictor' is the honest world-model effect")
    display(tab.round(4))
sk = df[(df.version == VERSION_TAG) & (df.task == "forecast_skill")]
if len(sk):
    sk = sk.assign(H=sk.metric.str.split("@").str[1].astype(float), m=sk.metric.str.split("@").str[0])
    skt = sk.pivot_table(index=["arm", "H"], columns="m", values="value", aggfunc="mean").reset_index()
    skt.to_csv(os.path.join(RESULTS_DIR, "table_forecast_skill.csv"), index=False)
    display(skt.round(4))

## Stage 3 — RQ3: anti-collapse mechanisms for the dual objective

`ema` (β = 0, EMA + stop-gradient only — now actually trained), `vicreg` (= Stage-1 `dual`,
reused only if its loss config is identical), `vq` (fixed soft codebook), `sigreg`
(LeJEPA, no EMA teacher). Verdict per arm: healthy / dimensional / collapsed, on a
512-window 12-lead probe batch.


In [ ]:
# --- Stage 3: RQ3 -------------------------------------------------------------------------
if "s3" in PLAN["stages"]:
    task = next(iter(PROBE_SPLITS)) if PROBE_SPLITS else None
    rows = []
    for arm in ("rq3_ema", "rq3_vicreg", "rq3_vq", "rq3_sigreg"):
        for seed in RQ3_SEEDS:
            if arm == "rq3_vicreg":
                _wo = lambda c: {k: v for k, v in vars(c.loss).items()
                                 if k not in ("weight_forward", "lambda_mask")}
                same = _wo(configure_arm(CFG, "vicreg")) == _wo(arm_cfg("dual", seed))
                e = get_model("dual", seed) if same else get_model(arm, seed, allow_train=True)
                if same:
                    print(f"rq3_vicreg_s{seed}: identical loss config to dual_s{seed} -> reused")
            else:
                e = get_model(arm, seed, allow_train=(arm, seed) in PLAN["train"])
            if e is None or e["res"] is None:
                continue
            s = e["res"].monitor.summary()
            row = {"arm": arm, "seed": seed, "verdict": s.get("verdict"),
                   "final_loss": s.get("final_loss"), "loss_to_zero": s.get("collapsed_to_zero"),
                   "erank": s.get("final_effective_rank"), "cov_erank_ratio": s.get("final_cov_erank_ratio"),
                   "std_mean": s.get("final_std_mean"), "cos_sim": s.get("final_cos_sim")}
            mid = f"{e['arm']}_s{seed}"
            if task:
                if not already_done(mid, f"probe_{task}"):
                    eval_probe(mid, e["arm"], seed, e["model"])
                d = ledger_df()
                v = d[(d.model_id == mid) & (d.task == f"probe_{task}") & (d.metric == "macro_auroc")].value
                row["macro_auroc"] = float(v.iloc[-1]) if len(v) else np.nan
            rows.append(row)
    if rows:
        RQ3 = pd.DataFrame(rows)
        RQ3.to_csv(os.path.join(RESULTS_DIR, "table_rq3.csv"), index=False)
        display(RQ3.round(4))
        print("Pareto rule: prefer the highest AUROC among arms whose verdict is 'healthy'.")
    save_ledger()
else:
    print("Stage 3 skipped in this session")

## Stage 4 — H2: hierarchical vs single-scale across horizons

Both arms train on the same 12-window (2 min) context loader and differ only in the
Level-2 stack. The batch size is reduced automatically if the GPU runs out of memory (the
v0.1 crash); both arms always use the same batch and learning rate.


In [ ]:
# --- Stage 4: hierarchical stack -----------------------------------------------------------
if "s4" in PLAN["stages"] and any(r.episodes for r in LONG_TRAIN):
    L2W = CFG.hierarchical.l2_context_windows
    want = [(a, s) for a, s in PLAN["train"] if a.startswith("stage4_")]
    hcfg = arm_cfg("stage4_hier", 0)
    S4_LOADER = None
    need_train = [j for j in want if f"{j[0]}_s{j[1]}.pth" not in CKPTS]
    bs = hcfg.optim.batch_size
    while need_train and bs >= 2:
        hcfg.optim.batch_size = bs
        S4_LOADER, _ = build_pretrain_loaders(FOUND, hcfg, long_records=LONG_TRAIN,
                                              samples_per_record=SAMPLES_PER_RECORD, verbose=False)
        probe_m, fits = None, True
        try:
            probe_m = CardioJEPA(hcfg.signal, hcfg.model, hcfg.loss, hcfg.hierarchical, hcfg.forward)
            estimate_runtime(probe_m, S4_LOADER, hcfg, DEVICE, n_steps=2)
        except torch.cuda.OutOfMemoryError:
            fits = False
        del probe_m; free_cuda()
        if fits:
            break
        print(f"  batch {bs} does not fit -> {bs // 2}"); bs //= 2
    S4_BATCH = bs
    print(f"Stage 4: context {L2W} x {CFG.signal.window_s:.0f}s, batch {S4_BATCH}")
    for arm, seed in want:
        c = arm_cfg(arm, seed)
        c.optim.batch_size = S4_BATCH
        c.optim.lr = CFG.optim.lr * S4_BATCH / CFG.optim.batch_size   # linear lr scaling
        mid = f"{arm}_s{seed}"
        if mid not in REGISTRY and mid + ".pth" not in CKPTS:
            print(f"\n=== training [{mid}] ===")
            cj.set_seed(seed, c.optim.deterministic)
            m = make_model(arm, c)
            res = cj.pretrain(m, S4_LOADER, c, device=DEVICE, arm_name=mid, probe_batch=PROBE_BATCH,
                              save_path=os.path.join(RUNS_DIR, mid + ".pth"),
                              log_every=0 if RUN_MODE == "smoke" else 200)
            log_training(mid, arm, seed, res); m.to("cpu"); free_cuda()
            REGISTRY[mid] = {"arm": arm, "seed": seed, "model": m, "res": res, "cfg": c}
            CKPTS[mid + ".pth"] = os.path.join(RUNS_DIR, mid + ".pth")
        else:
            get_model(arm, seed)
    s4 = {k: {"model": REGISTRY[k]["model"].to(DEVICE), "feature_mode": "world"}
          for k in REGISTRY if REGISTRY[k]["arm"].startswith("stage4_")}
    if len(s4) == 2:
        HS4 = (300.0, 600.0, 1800.0) if RUN_MODE == "kaggle" else (60.0, 300.0)
        def _builder(records, signal, horizon_s, context_windows):
            return build_onset_dataset(records, signal, horizon_s=horizon_s,
                                       context_windows=context_windows,
                                       stride_s=CFG.evaluation.anticipation_stride_s,
                                       guard_s=CFG.evaluation.anticipation_guard_s,
                                       max_per_record=MAX_PER_RECORD)
        sweep = horizon_sweep(s4, LONG_ALL, CFG.signal, DEVICE, CFG.evaluation, horizons_s=HS4,
                              builder=_builder, context_windows=L2W, test_ids=HOLDOUT.test_ids)
        for _, r in sweep.iterrows():
            e = REGISTRY[r.model]
            log(r.model, e["arm"], e["seed"], "stage4_sweep", f"auroc@{r.horizon_s:.0f}", r.auroc,
                r.horizon_s, extra={"ci": [r.ci_lo, r.ci_hi]})
        bs_, bst = build_burden_dataset(LONG_ALL, CFG.signal, context_windows=L2W,
                                        from_first_minutes=30.0, samples_per_record=8, seed=0)
        btr, bte = split_by_ids(bs_, HOLDOUT.test_ids)
        for k, spec in s4.items():
            r = run_burden_probe(spec["model"], LONG_ALL, btr, bte, CFG.signal, DEVICE,
                                 context_windows=L2W, feature_mode="world",
                                 horizons_s=[1800.0 if RUN_MODE == "kaggle" else 300.0],
                                 batch_size=16, verbose=True)
            for m_ in ("r2", "mae", "spearman", "r2_record", "spearman_record"):
                log(k, REGISTRY[k]["arm"], 0, "stage4_burden", m_, r.get(m_))
        if len(sweep):
            piv = sweep.pivot(index="horizon_s", columns="model", values="auroc")
            display(piv.round(4)); piv.to_csv(os.path.join(RESULTS_DIR, "table_stage4.csv"))
    for k in s4: REGISTRY[k]["model"].to("cpu")
    free_cuda(); save_ledger()
else:
    print("Stage 4 skipped in this session")

## Figures for the paper
Built from the merged results ledger, so the last session draws every figure.

In [ ]:
# --- figures for the paper (PNG + PDF; built from the merged ledger) -------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), bbox_inches="tight")
    plt.show(); plt.close(fig)

DF = ledger_df()
DF = DF[DF.version == VERSION_TAG] if len(DF) else DF
ORDER = ["random_init", "st_mem", "simclr", "cpc", "masked_only", "masked_only_cm",
         "forward_only", "dual", "supervised_f0.01", "supervised_f0.1", "supervised_f1"]

# F2 training curves
hist = {m: e["res"] for m, e in REGISTRY.items() if e.get("res") is not None and e["res"].history}
if hist:
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
    for m, r in sorted(hist.items()):
        h = pd.DataFrame(r.history); mon = pd.DataFrame(r.monitor.history)
        ax[0].plot(h.epoch, h.loss, label=m)
        if "loss_fwd" in h: ax[1].plot(h.epoch, h.loss_fwd, label=m)
        if "cov_effective_rank" in mon: ax[2].plot(mon.epoch, mon.cov_effective_rank, label=m)
    for a, t in zip(ax, ("total loss", "L_fwd", "covariance effective rank (512 windows)")):
        a.set_title(t); a.set_xlabel("epoch")
    ax[2].legend(fontsize=6, ncol=2); savefig(fig, "fig_training_curves")

# F3 linear probe bars
for task in PROBE_SPLITS:
    t = probe_table(task)
    if len(t):
        t = (t.assign(_o=t.arm.map({a: i for i, a in enumerate(ORDER)}).fillna(99))
              .sort_values("_o").drop(columns="_o"))
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.bar(t.arm, t.macro_auroc_mean, yerr=t.macro_auroc_std.fillna(0), capsize=3,
               color=["#c44e52" if a == "dual" else "#4c72b0" for a in t.arm])
        ax.set_ylim(max(0.5, t.macro_auroc_mean.min() - 0.05), 1.0)
        ax.set_ylabel("macro AUROC"); ax.set_title(f"{task}: frozen-encoder linear probe (mean ± sd over seeds)")
        plt.xticks(rotation=35, ha="right"); savefig(fig, f"fig_linear_probe_{task}")

# F4 label efficiency
for task in PROBE_SPLITS:
    le = DF[(DF.task == f"labeleff_{task}") & (DF.metric == "macro_auroc")] if len(DF) else DF
    if len(le):
        le = le.assign(frac=le.setting.astype(float))
        g = le.groupby(["arm", "frac"]).value.agg(["mean", "std"]).reset_index()
        g.to_csv(os.path.join(RESULTS_DIR, f"table_label_efficiency_{task}.csv"), index=False)
        fig, ax = plt.subplots(figsize=(5.5, 3.5))
        for arm, gg in g.groupby("arm"):
            if arm.startswith("supervised"):
                continue
            ax.errorbar(gg.frac * 100, gg["mean"], yerr=gg["std"].fillna(0), marker="o", label=arm, capsize=2)
        sup = g[g.arm.str.startswith("supervised")]
        if len(sup):
            ax.plot(sup.frac * 100, sup["mean"], "k--", marker="s", label="supervised")
        ax.set_xscale("log"); ax.set_xlabel("% labelled training ECGs"); ax.set_ylabel("macro AUROC")
        ax.legend(fontsize=7); ax.set_title(f"label efficiency ({task})"); savefig(fig, f"fig_label_efficiency_{task}")

# F5 robustness
rb = DF[DF.task.str.startswith("robust_")] if len(DF) else DF
if len(rb):
    rb.to_csv(os.path.join(RESULTS_DIR, "table_robustness.csv"), index=False)
    noise = rb[rb.setting.str.contains("dB")]
    if len(noise):
        noise = noise.assign(kind=noise.setting.str.split("@").str[0],
                             snr=noise.setting.str.extract(r"@([-\d.]+)dB")[0].astype(float))
        kinds = sorted(noise.kind.unique())
        fig, ax = plt.subplots(1, len(kinds), figsize=(4 * len(kinds), 3), squeeze=False)
        for a, k in zip(ax[0], kinds):
            for arm, g in noise[noise.kind == k].groupby("arm"):
                g = g.sort_values("snr", ascending=False)
                a.plot(g.snr, g.value, marker="o", label=arm)
            a.invert_xaxis(); a.set_title(f"noise: {k}"); a.set_xlabel("SNR (dB)"); a.set_ylabel("macro AUROC")
        ax[0][-1].legend(fontsize=6); savefig(fig, "fig_robustness_noise")

# F6/F7 anticipation + forecast skill
ant = DF[DF.task.isin(["onset_context", "onset_world"]) & DF.metric.str.startswith("auroc@")] if len(DF) else DF
if len(ant):
    ant = ant.assign(H=ant.metric.str.split("@").str[1].astype(float))
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for (arm, task), g in ant.groupby(["arm", "task"]):
        g = g.groupby("H").value.mean()
        ax.plot(g.index / 60, g.values, marker="o", ls="-" if task.endswith("world") else ":",
                label=f"{arm}/{task.split('_')[1]}")
    ax.set_xlabel("horizon (min)"); ax.set_ylabel("AUROC (held-out records)"); ax.legend(fontsize=6, ncol=2)
    savefig(fig, "fig_anticipation")
sk = DF[(DF.task == "forecast_skill")] if len(DF) else DF
if len(sk):
    sk = sk.assign(H=sk.metric.str.split("@").str[1].astype(float), m=sk.metric.str.split("@").str[0])
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    for arm, g in sk.groupby("arm"):
        p = g.pivot_table(index="H", columns="m", values="value", aggfunc="mean")
        if "skill" in p:
            lo = np.abs(p["skill"] - p.get("skill_lo", p["skill"]))
            hi = np.abs(p.get("skill_hi", p["skill"]) - p["skill"])
            ax[0].errorbar(p.index, p["skill"], yerr=[lo, hi], marker="o", label=arm, capsize=2)
        if "horizon_gain" in p:
            ax[1].plot(p.index, p["horizon_gain"], marker="o", label=arm)
    for a, t in zip(ax, ("forecast skill vs persistence", "horizon gain (right vs wrong horizon)")):
        a.axhline(0, color="k", lw=0.8); a.set_xscale("log"); a.set_xlabel("horizon (s)"); a.set_title(t)
    ax[0].legend(fontsize=7); savefig(fig, "fig_forecast_skill")

# F8 RQ3
rq = [e for m, e in REGISTRY.items() if (e["arm"].startswith("rq3_") or m == "dual_s0") and e.get("res")]
if rq:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    for e in rq:
        mon = pd.DataFrame(e["res"].monitor.history)
        lab = "rq3_vicreg(=dual)" if e["arm"] == "dual" else e["arm"]
        if "cov_effective_rank" in mon: ax[0].plot(mon.epoch, mon.cov_effective_rank, label=lab)
        ax[1].plot(mon.epoch, mon.loss, label=lab)
    ax[0].set_title("covariance effective rank"); ax[1].set_title("training loss"); ax[1].set_yscale("log")
    ax[0].legend(fontsize=7); savefig(fig, "fig_rq3")
print("figures written to", FIG_DIR, ":", sorted(os.listdir(FIG_DIR)))

## Loop engineering: exit criteria, loop log, comparison with earlier versions
Each session is one loop iteration: *analyse → fix → run → evaluate → compare → log*. Criteria 1–5 check correctness; criterion 6 is the scientific result.

In [ ]:
# --- exit criteria, loop log, comparison with v0.1 and the previous version ----------------
def repro_check(steps=6):
    # same seed twice -> same losses (tiny model, real loader)
    c = smoke_config(); c.signal = CFG.signal; c.optim.max_steps_per_epoch = steps
    c.optim.epochs = 1; c.optim.amp = CFG.optim.amp; c.optim.num_workers = 0
    c.loss.anti_collapse, c.loss.beta_reg = "vicreg", 0.1
    c.optim.deterministic = True      # keep the global cuDNN flags deterministic
    sub_loader, _ = build_pretrain_loaders({}, c, samples_per_record=4, verbose=False)
    losses = []
    for _ in range(2):
        cj.set_seed(123, True)
        m = CardioJEPA(c.signal, c.model, c.loss, c.hierarchical, c.forward)
        r = cj.pretrain(m, sub_loader, c, device=DEVICE, arm_name="repro", verbose=False)
        losses.append(r.history[-1]["loss"])
    return abs(losses[0] - losses[1]) <= 1e-3 * max(1.0, abs(losses[0])), losses

DF = ledger_df(); DFv = DF[DF.version == VERSION_TAG] if len(DF) else DF
crit = {}
chk = DFv[DFv.task == "checks"] if len(DFv) else DFv
crit["1_leakage_and_data_checks"] = bool(len(chk)) and bool((chk.value == 1).all())
tr = DFv[DFv.task == "train"] if len(DFv) else DFv
main = tr[~tr.arm.str.startswith("rq3_")] if len(tr) else tr
coll = main[main.metric == "verdict_collapsed"] if len(main) else main
crit["2_no_collapse_in_reported_arms"] = bool(len(coll)) and bool((coll.value == 0).all())
# fixed-target baselines must reduce their loss; JEPA losses may legitimately rise as the
# EMA target becomes more informative, so JEPA arms are judged by the collapse verdict (2)
drops = tr[(tr.metric == "loss_drop") & tr.arm.isin(["st_mem", "simclr", "cpc"])] if len(tr) else tr
crit["3_every_baseline_trained(loss_drop>=10%)"] = (bool((drops.value >= 0.10).all())
                                                    if len(drops) else None)
task0 = next(iter(PROBE_SPLITS), None)
seeds_ok = std_ok = False
if task0:
    t = probe_table(task0)
    if len(t) and {"dual", "masked_only"} <= set(t.arm):
        sub = t[t.arm.isin(["dual", "masked_only"])]
        seeds_ok = bool((sub.n_seeds >= 3).all())
        std_ok = bool((sub.macro_auroc_std.fillna(1) <= 0.01).all())
crit["4_three_seeds_and_stable(std<=0.01)"] = seeds_ok and std_ok
ok, rl = repro_check() if RUN_MODE == "kaggle" or SESSION == "ALL" else (None, [])
crit["5_reproducible(same_seed_same_loss)"] = None if ok is None else bool(ok)
better = None
if task0 and len(TESTS):
    T = pd.DataFrame(TESTS); T = T[T.task == task0]
    dm = T[(T.a == "dual") & (T.b == "masked_only")]
    if len(dm) and dm.iloc[0].n_seeds >= 3:
        tab = probe_table(task0).set_index("arm").macro_auroc_mean
        best_ssl = max([tab.get(a, -1) for a in ("simclr", "cpc", "st_mem")])
        better = bool(dm.iloc[0].seed_delta > 0 and dm.iloc[0].seed_p < 0.05 and tab.get("dual", 0) >= best_ssl)
crit["6_dual_beats_masked_and_ssl_baselines"] = better
EXIT_CRITERIA = crit
print("EXIT CRITERIA (1-5 = correctness, must pass; 6 = the scientific result)")
for k, v in crit.items():
    print(f"  {'PASS' if v else ('n/a ' if v is None else 'FAIL')}  {k}")
if rl: print(f"  repro losses: {rl}")
done = all(crit[k] is not False for k in list(crit)[:5]) and crit["1_leakage_and_data_checks"]
print("\nLOOP STATUS:", "exit criteria 1-5 met" if done else
      "continue the loop: run the next session / fix the failing checks")

# v0.1 reference numbers (leaky, single seed -- for orientation only)
V01 = {"probe_ptbxl": {"dual": 0.8177, "masked_only": 0.7675, "forward_only": 0.7176, "st_mem": 0.6500}}
if task0 == "ptbxl":
    t = probe_table("ptbxl")
    if len(t):
        cmp_ = t[["arm", "macro_auroc_mean"]].assign(v01=t.arm.map(V01["probe_ptbxl"]))
        cmp_["delta_vs_v01"] = cmp_.macro_auroc_mean - cmp_.v01
        print("\nvs v0.1 (v0.1 pretrained on PTB-XL -> NOT comparable; shown only to track the loop)")
        display(cmp_.round(4))
prev = previous_ledgers()
if prev:
    p = pd.concat(prev); p = p[(p.metric == "macro_auroc") & p.task.str.startswith("probe_")]
    q = DFv[(DFv.metric == "macro_auroc") & DFv.task.str.startswith("probe_")] if len(DFv) else DFv
    if len(p) and len(q):
        j = q.merge(p, on=["model_id", "task", "setting"], suffixes=("", "_prev"))
        j = j[j.version_prev != j.version]
        if len(j):
            print("\nchange vs previous ledger version:"); display(j[["model_id", "task", "value", "value_prev", "version_prev"]])

with open(os.path.join(RESULTS_DIR, "exit_criteria.json"), "w") as fh:
    json.dump({k: (None if v is None else bool(v)) for k, v in crit.items()}, fh, indent=1)
with open(os.path.join(RESULTS_DIR, "loop_log.jsonl"), "a") as fh:
    fh.write(json.dumps({"time": datetime.datetime.utcnow().isoformat(timespec="seconds"),
                         "version": VERSION_TAG, "session": SESSION, "run_mode": RUN_MODE,
                         "models": sorted(REGISTRY), "criteria": {k: (None if v is None else bool(v))
                                                                   for k, v in crit.items()}}) + "\n")
save_ledger()

In [ ]:
# --- documents written into the output zip (also shown in the Appendix) ---
DOCS = {
"BUG_REPORT.md": "# Cardio-JEPA — bug report for v0.1 (the `cardio-jepa-v-2a-corrected` run)\n\nEvery item lists **where** it was, **why** it is wrong, the **evidence** from the v0.1 Kaggle\nlog, and the **fix** in v0.2. Items B01–B11 change or invalidate published-looking numbers;\nfix them before any result goes into a paper. Each fix has a regression test in\n`tests/test_v02.py`.\n\n## Critical — results were invalid or misleading\n\n| ID | Where | What was wrong | Evidence in the v0.1 log | Fix (v0.2) |\n|---|---|---|---|---|\n| B01 | `config.DataConfig.pretrain_datasets`, `data/build.py` | **Test-set leakage.** PTB-XL (incl. test fold 10) was a *pretraining* corpus, and the PTB-XL fold-10 linear probe was then reported. Kim's ECG-JEPA protocol pretrains on Chapman/Ningbo/CODE-15 only. | `[data] PTB-XL indexed 20000 recordings` inside the pretraining loader | PTB-XL and CPSC2018 are probe-only (`probe_datasets`); `build_pretrain_loaders` raises if a probe set is in the pretraining list (opt-in flag for a deliberate transductive ablation). |\n| B02 | notebook Stage 2 / 2c / 4, `ForwardPairDataset` | **Forecasting leakage.** Forward pairs were drawn from *all* LTAFDB records, then anticipation and forecast skill were tested on 30 % of the *same* records — the model had been trained to predict the exact futures it was scored on. The test split was also re-drawn per horizon. | `forward pairs: 18200` from all 91 long records; `split_by_record(...)` called again in every stage | `split_long_records()` fixes a burden-stratified record holdout **before** pretraining; pretraining gets only the train records; every benchmark uses `split_by_ids(..., HOLDOUT)`; the notebook asserts no overlap. |\n| B03 | `ForwardSamplingConfig.horizons_s`, notebook Stage 2 | **Untrained horizon embeddings at test time.** Training horizons were (10, 30, 60, 300) s, but anticipation used H = 900 s (→ 600 s bucket) and forecast skill 900 / 1800 s — buckets whose embeddings never received a gradient (random init). | forecast skill flat at +0.05 for 10 s … 1800 s; `dual/world` ≈ `dual/context` | Training horizons now (10, 30, 60, 300, 600, 1800); evaluation horizons are trained buckets (300, 600, 1800); a persistent `horizon_counts` buffer warns whenever an untrained bucket is queried. |\n| B04 | `data/longform._load_mitbih_csv`, `data/discovery.py` | **Sampling-rate mismatch + wrong corpus label.** The attached *MIT-BIH Long-Term ECG* mirror (128 Hz) matched the generic `mit-bih` pattern, was reported as \"MIT-BIH Arrhythmia\", and was read at a hard-coded 360 Hz: every \"10 s\" window held 28 s of ECG, so heart rates looked ~2.8× too fast. | `MIT-BIH Arrhythmia … 7 long records (0 with rhythm annotations, total 52.4 h)` — the Arrhythmia DB has 48 × 30 min records | New `mitbih_lt` corpus key; fs inferred from a time column, else the corpus default (128 Hz ltdb / 360 Hz mitdb); time column no longer read as an ECG lead. |\n| B05 | notebook Stage 3 | **RQ3 had no EMA-only arm.** The `ema` row reused Stage-1 `dual`, but `kaggle_config` runs Stage 1 with VICReg (`reg=0.51` in the log). So \"ema\" and \"vicreg\" were two runs of the *same* configuration, and the key RQ3 condition was never trained. | Stage-1 `dual` log: `reg=0.5132`; Stage 3 prints `reusing Stage 1 'dual'` | The EMA-only arm (β = 0) is trained; Stage-1 `dual` is reused only for the `vicreg` arm and only when the loss configs are identical (checked in code). |\n| B06 | `models/baselines.STMEMBaseline` | **ST-MEM baseline never trained.** Targets were per-patch-normalised 0.2 s patches of an already z-scored signal — mostly iso-electric segments whose noise was blown up to unit variance, i.e. white-noise targets. | `st_mem` loss 0.985 → 0.975 over 30 epochs, effective rank 1.0, probe AUROC 0.650 | `target_norm=\"none\"` (regress the z-scored signal). Reproduced offline: old target stays at 0.98 (exactly the Kaggle value), new target learns (`test_stmem_fixed_target_learns_while_patch_norm_does_not`). Removed a wasted second encoder pass. |\n| B07 | `regularizers.SoftVQ`, `config.LossConfig.vq_*` | **The VQ-arm \"collapse\" was an implementation artefact.** Temperature 0.5 on *cosine* logits with a random 384-d codebook gives near-uniform assignments, so every target became the same (rescaled) codebook average: a constant target, trivially predicted. Only the last of two assignment sets was regularised. | `vq` loss 0.29 → 0.009 by epoch 1; std 0.07; cos-sim 0.94 | T = 0.1, codebook seeded from real targets on the first step, commitment weight 1.0, every assignment regularised. Measured target spread: 0.148 (v0.1) vs 0.948 (v0.2) against a true 0.963. |\n| B08 | `models/transformer.Attention`, notebook Stage 4 | **Stage-4 OOM.** Attention materialised a full fp32 (B·W, heads, 400, 400) score matrix per layer for backward (1.38 GiB/layer at 12 windows), while seven trained models still sat on the GPU. | `OutOfMemoryError: Tried to allocate 1.38 GiB … 11.72 GiB allocated` | `F.scaled_dot_product_attention` (memory-efficient kernel, works on T4); `free_cuda()` between stages; Stage 4 retries with a halved batch on OOM. |\n| B09 | notebook export cell | **All results lost on a crash.** Tables were written only by the last cell; the Stage-4 OOM meant nothing reached `/kaggle/working/results/`. | run ends with `OutOfMemoryError`; export cell `pending` | A results **ledger** (CSV) is written after every stage, plus the zip. |\n| B10 | `eval/stats.py`, notebook Stage 1/2 | **Over-confident statistics.** Anticipation CIs bootstrapped individual windows although windows from one Holter record are strongly correlated; the Stage-1 GO decision used one training seed. | `p_value 0.0` for every Stage-2 comparison | Record-level **cluster bootstrap**; ≥3 seeds with a paired seed-level t-test and seed CIs. |\n| B11 | notebook Stage 2 (`world` features) | **Confounded \"world-model\" evidence.** `masked_only/world` (forward predictor *never trained*, i.e. a random projection of the context) beat `masked_only/context` by +0.016 (p < 0.05). So `world > context` partly measures extra feature dimensions, not forecasting. `context` also used only the last window while `world` used both. | `masked_only/world 0.6219` vs `masked_only/context 0.6062` | Gains are reported **relative to the random-predictor control**; `context` features now pool all context windows. |\n\n## High — biased comparisons or wasted compute\n\n| ID | Where | What was wrong | Fix |\n|---|---|---|---|\n| B12 | `data/build.py` (`weights.append(2.0)`) | 2 of every 3 training batches — for **every** arm, including the masked-only \"Kim reproduction\" — were 2-channel Holter windows tiled to 8 leads. Large domain shift from the 12-lead downstream tasks; also explains part of the gap to Kim's 0.906. | `cfg.data.long_loader_weight = 0.5` (≈33 %), configurable for an ablation. |\n| B13 | `engine.InterleavedLoader` | The interleaving RNG was seeded by an epoch counter never reset, so sequentially trained arms saw different short/long batch orders. | `reset()`; called by `pretrain` and `estimate_runtime`. |\n| B14 | `probe_batch_from_loader(n=128)` | Effective rank of 128 samples is capped at 128 < d = 384: every RQ3 rank number was bounded by the probe batch. The batch was also mostly Holter. | n = 512, drawn from the 12-lead short loader. |\n| B15 | `diagnostics.CollapseMonitor.summary` | Collapse flags were so strict that nothing was flagged — not even the VQ arm above. | `representation_collapse`, wider `dimensional_collapse`, and a single `verdict` (healthy / dimensional / collapsed). |\n| B16 | `eval/benchmarks.forecast_skill` | Prediction (normalised target space) was compared with persistence in the raw *online* space; training records were sampled; no control for an ignored horizon token. | Both in the trained target space; held-out records only; horizon-shuffle control (`horizon_gain`); bootstrap CI on skill. |\n| B17 | `eval/linear_probe.py` | Fixed lr and 40 epochs; the PTB-XL validation fold was never used; one random draw per label fraction (1 % sometimes beat 10 %). | lr chosen on the validation fold with early stopping; 3 repeats per fraction (mean ± std). |\n| B18 | `data/short.index_ptbxl`, `build_probe_splits` | Probe set randomly subsampled to 20 000 of 21 799 records (test fold 2 010 instead of ~2 160); records without a superclass kept as all-negative rows; 100 Hz files only. | Full PTB-XL; unlabelled rows dropped (Strodthoff protocol); 500 Hz when present; patient overlap train/test checked. |\n| B19 | `models/forward_predictor.SequencePredictor` | Level-2 masked infilling was never told which windows were masked (fixed query slots), and context positions shifted after masking: an ill-posed task. | Query and context position indices passed explicitly. |\n| B20 | `models/cardio_jepa.py` Level 2 | Target summaries got an extra LayerNorm that the online summaries did not. | Same construction on both sides. |\n| B21 | `CardioJEPA` | `model.train()` also put the EMA target encoder in train mode (stochastic depth in targets with the `full` preset). | `train()` override keeps targets in eval mode. |\n| B22 | `engine.estimate_runtime` | Ignored `max_steps_per_epoch`, so the \"fixed\" projection was unchanged (104 h). | Uses the capped step count. |\n| B23 | `data/short.ShortWindowDataset` | Read failures silently became all-zero windows (e.g. MEETI: no readable files). | Failures counted; the evaluation cache audits the zero-filled fraction across workers. |\n| B24 | `data/discovery.py` | Discovery took 51 min: up to 200 k files walked per corpus, the Chapman+Ningbo mirror walked twice, the unreadable MEETI mirror walked in full. | 20 k-file cap, per-path probe cache, `skip=` list (`meeti`). |\n| B25 | forward objective design | Long-horizon (≥10 s) forecasting regressed the **per-patch** token grid of a future window; beat phase minutes ahead is unpredictable, so the optimum is a per-position mean (noise to fit). | Cross-window target is the pooled window latent (`forward_target_long=\"pooled\"`; `\"tokens\"` kept as an ablation). |\n\n## Notebook-level issues\n\n| ID | What was wrong | Fix |\n|---|---|---|\n| B26 | `WARM_START=\"auto\"` would initialise *every* ablation arm from the same checkpoint found in a later session, contaminating the comparison. | Off by default; multi-session runs reuse a model's **own** checkpoint instead. |\n| B27 | (older notebook) `os.environ[\"CUDA_LAUNCH_BLOCKING\"]=\"1\"` makes every CUDA kernel synchronous (much slower); `!pip install flash_attn xformers triton` fails with internet off and is not needed (T4 has no FlashAttention-2; SDPA's memory-efficient kernel is used). | Removed. |\n| B28 | Stage 3 read `res` left over from the Stage-1 loop for the reused arm (correct only by accident of loop order). | Uses the arm's own result object. |\n| B29 | No determinism settings (cuDNN autotuning, cuBLAS workspace). | `CUBLAS_WORKSPACE_CONFIG`, `cudnn.deterministic`, per-arm `set_seed`. |\n| B30 | 11-h single session for 9 models (projection said 104 h; the cap brought it to ~11 h with zero margin, then Stage 4 crashed at hour 9.8). | Four-session plan (A–D) with checkpoint reuse and a results ledger. |\n\n## Checked and **not** a bug\n\n* PTB-XL official folds are patient-disjoint (now verified in code: `patient_overlap = 0`).\n* The Chapman+Ningbo combined mirror and the separate Chapman-Shaoxing mirror are not double-counted: discovery picks one root per corpus key and indexes a shared root once.\n* EMA update direction and stop-gradient on targets are correct (`_target_tokens` runs under `no_grad`; `update_target` is θ̄ ← mθ̄ + (1−m)θ, m: 0.996 → 1.0).\n* \"ECG Heartbeat Categorization\" is not picked up by discovery (single beats at 125 Hz, unusable for 10 s windows). **Detach it**: its beats come from MIT-BIH Arrhythmia and PTB Diagnostic, so it would leak if either were later used for evaluation.\n",
"CHANGELOG.md": "# Changelog\n\n## 0.2.0 — corrected and extended framework (v3 notebook)\n\nBug IDs refer to `docs/BUG_REPORT.md`.\n\n### Fixed — leakage and validity\n- PTB-XL / CPSC2018 removed from pretraining; guard raises if a probe dataset is pretrained on (B01).\n- Long-record holdout fixed before pretraining (`split_long_records`, `split_by_ids`); forward pairs only from training records (B02).\n- Forward horizons now cover every evaluated horizon; `horizon_counts` buffer + warning on untrained buckets (B03).\n- MIT-BIH Long-Term ECG: own corpus key, sampling rate inferred (128 Hz, was read as 360 Hz) (B04).\n- RQ3: a real EMA-only arm; Stage-1 reuse only for an identical VICReg config (B05).\n- ST-MEM target fixed (it never trained) (B06).\n- Soft-VQ: temperature, data-seeded codebook, commitment weight, all assignments regularised (B07).\n- Cluster (record-level) bootstrap; multi-seed paired tests (B10); random-predictor control for \"world\" features (B11).\n\n### Fixed — training and data\n- Fused SDPA attention (fixes Stage-4 OOM, faster); GPU freed between stages (B08).\n- Holter share of batches 67 % → 33 % (configurable) (B12); identical batch schedule for every arm (B13).\n- Diagnostics on 512 short 12-lead windows (B14); stricter collapse verdict (B15).\n- PTB-XL: full dataset, 500 Hz, unlabelled rows dropped, patient-overlap check (B18).\n- Level-2 masked predictor gets positions; Level-2 target consistent (B19, B20).\n- Target encoder always in eval mode (B21); runtime estimate honours the step cap (B22).\n- Failed reads counted and audited (B23); discovery 20 k-file cap, probe cache, skip list (B24).\n- Cross-window forward target pooled (phase-invariant) by default (B25).\n- Deterministic seeding (cuDNN flags, cuBLAS workspace) (B29).\n\n### Added\n- `SimCLRBaseline` and `CPCBaseline` (same CroPA encoder) — CPC is the closest prior art to `L_fwd`.\n- `train_supervised` (supervised-from-scratch reference, incl. label fractions) and a random-init floor.\n- `run_robustness_probe`: real NSTDB noise (bw / em / ma) at 24 / 12 / 6 / 0 dB, and lead subsets (I+II, single lead II).\n- Linear probe: validation-selected learning rate + early stopping; repeated label fractions.\n- `forecast_skill`: same-space persistence, horizon-shuffle control, bootstrap CI.\n- Burden probe: grouped-CV ridge alpha, record-level metrics.\n- `data.cache`: one-time float16 memmap cache of evaluation sets in `/tmp` (+ zero-fill audit).\n- `load_train_result`, `free_cuda`; `seed_level_summary`, `paired_seed_test`.\n- 24 new regression tests (`tests/test_v02.py`); all 59 tests pass on CPU.\n\n### Notebook (v3)\n- Self-contained: writes the corrected package from the notebook (no model re-upload needed).\n- Four-session plan (A–D) with automatic reuse of checkpoints from earlier sessions.\n- Results ledger saved after every stage; exit-criteria checker; loop log; comparison with v0.1.\n- Paper figures (PNG + PDF) and all docs written to `/kaggle/working`, zipped at the end.\n- Removed `CUDA_LAUNCH_BLOCKING`, `pip install` of flash-attn/xformers/triton, warm-start-by-default.\n\n### Changed defaults (may change numbers vs v0.1)\n`pretrain_datasets`, `long_loader_weight=0.5`, `forward.horizons_s`, `forward_target_long=\"pooled\"`,\n`vq_temperature=0.1`, `vq_commit_weight=1.0`, `anticipation_horizons_s=(300, 600, 1800)`,\n`max_records_per_dataset=46000` (all of Chapman+Ningbo), `ptbxl_use_hr=True`, `cache_dir=/tmp/...`.\n\n## 0.1.0\nInitial implementation (see the v0.1 Kaggle run for its results).\n",
"RESEARCH_PLAN.md": "# Research plan — making Cardio-JEPA a defensible Q1 paper\n\n## 1. What the v0.1 run actually showed (after reading the logs)\n\n| Question | v0.1 result | Can it be trusted? |\n|---|---|---|\n| Does `L_fwd` hurt classification? (Stage 1) | dual 0.818 > masked-only 0.768 > forward-only 0.718 > ST-MEM 0.650 (PTB-XL macro AUROC) | **No.** PTB-XL test fold was in pretraining (B01); one seed (B10); ST-MEM never trained (B06); 2/3 of batches were tiled Holter (B12). All numbers are far below the published ECG-JEPA linear probe (0.906, Kim 2024), so the models are under-trained. The *direction* (dual ≥ masked) is promising and must be re-tested. |\n| Does forward pretraining help anticipation? (H1) | No: dual 0.525 / masked-only 0.622 AUROC at 15 min | **No** in either direction: tested on records the forward predictor was trained on (B02), with an untrained 600 s horizon embedding (B03), window-level CIs (B10). |\n| Is the predictor forecasting? | skill ≈ +0.05 at every horizon (flat) | Flat skill is the signature of an ignored / untrained horizon token (B03) measured in-sample (B02, B16). |\n| What prevents collapse? (RQ3) | \"ema\" ≈ vicreg > vq ≈ sigreg | **No:** there was no EMA-only arm (B05); the VQ collapse was a bug (B07); ranks capped at 128 (B14). |\n\n**Bottom line:** v0.1 does not yet support any claim. v0.2 removes the leakage and bugs, so the\nnext run gives the first valid numbers. Plan the paper around what v0.2 shows, not around\nthe v0.1 table.\n\n## 2. Related work and the real research gap\n\n* **Masked latent prediction for ECG.** Kim's ECG-JEPA ([arXiv:2410.08559](https://arxiv.org/abs/2410.08559)), Weimann & Conrad's I-JEPA for ECG ([arXiv:2410.13867](https://arxiv.org/abs/2410.13867)) and the hierarchical ER-JEPA ([arXiv:2607.01145](https://arxiv.org/abs/2607.01145)) all predict *masked* latents inside one ~10 s recording.\n* **Forward latent prediction already exists — contrastively.** CPC predicts future latents; on 12-lead ECG it was the strongest self-supervised objective in Mehari & Strodthoff ([arXiv:2103.12676](https://arxiv.org/abs/2103.12676), PTB-XL linear-probe macro AUROC 0.927 vs SimCLR 0.883), and a 2026 systematic study found CPC slightly ahead of JEPA for transfer ([arXiv:2605.12241](https://arxiv.org/abs/2605.12241)).\n* **Non-contrastive forward prediction in time series.** Eidos ([arXiv:2602.14024](https://arxiv.org/abs/2602.14024)) predicts future latents autoregressively with a stop-gradient target; SC-JEPA/MTS-JEPA ([arXiv:2602.04643](https://arxiv.org/abs/2602.04643)) forecasts the next window in latent space for anomaly *anticipation* with a soft codebook + EMA (close to our VQ arm); TS-JEPA ([arXiv:2509.25449](https://arxiv.org/abs/2509.25449)). None is on multi-lead ECG or clinical rhythm anticipation.\n* **World models / anti-collapse.** V-JEPA 2 ([arXiv:2506.09985](https://arxiv.org/abs/2506.09985)) uses latent future prediction as a world model; LeJEPA/SIGReg ([arXiv:2511.08544](https://arxiv.org/abs/2511.08544)) removes the EMA teacher.\n* **Clinical anticipation.** AF-onset prediction from long RR series reaches high AUROC with ~1 h of context (AF-Mamba, [arXiv:2609.06984](https://arxiv.org/abs/2609.06984)); IRIDIA-AF ([Sci Data 2023](https://pmc.ncbi.nlm.nih.gov/articles/PMC10584890)) provides 167 Holter records (152 patients, 2 leads, 200 Hz, 388 AF episodes, CC-BY 4.0).\n\n**So the honest gap is narrower than \"nobody predicts forward in time\":**\n1. Nobody has *combined* masked infilling (morphology) with **horizon-conditioned, non-contrastive** forward prediction (dynamics) for **multi-lead ECG**, and tested whether the forward term is what makes CPC strong.\n2. ECG \"world-model\" claims are not evaluated with a **leakage-safe forecasting protocol** (held-out records, persistence and horizon-shuffle controls, record-level CIs).\n3. No ECG-JEPA reports **robustness to real ambulatory noise and missing leads**.\n\nFrame the paper as: *\"Do JEPAs need dynamics? Unifying masked and forward latent prediction for\nECG, with a leakage-safe anticipation benchmark.\"* That is defensible even if the anticipation\nresult is modest, because the controls make it a clean answer.\n\n## 3. Proposed contributions (realistic on Kaggle T4 sessions)\n\n| # | Contribution | What to do | Real-world impact |\n|---|---|---|---|\n| C1 | **Dual objective vs strong SSL baselines** | dual vs masked-only vs forward-only vs CPC vs SimCLR vs ST-MEM vs supervised vs random-init; same encoder; 3 seeds; compute-matched masked-only control | Shows whether adding dynamics gives better general ECG features → fewer labels needed for diagnosis models |\n| C2 | **Leakage-safe anticipation + forecast-skill protocol** | pre-pretraining record holdout, persistence & horizon-shuffle controls, random-predictor control, record-level cluster bootstrap, AUROC + AUPRC | A reusable evaluation standard; prevents inflated \"early warning\" claims in AF screening |\n| C3 | **Label efficiency** | 1 / 10 / 50 / 100 % labels, 3 repeats, vs supervised from scratch | Hospitals have few expert-labelled ECGs; quantifies annotation savings |\n| C4 | **Robustness to real noise and missing leads** | NSTDB baseline wander / electrode motion / muscle noise at 24→0 dB; I+II and single lead II | Holter, patch and smartwatch ECGs are noisy and have 1–2 leads; tells clinicians when outputs stay reliable |\n| C5 | **Cross-dataset generalisation** | pretrain Chapman+Ningbo (+CODE-15) → probe PTB-XL (Germany) and CPSC2018 (China) as external sites; leave-one-dataset-out pretraining; harmonise SNOMED↔SCP labels for shared classes (AF, RBBB, LBBB, 1AVB, PVC) | Models must work in a different hospital and country than they were trained in |\n| C6 | **ECG-specific masking** | random vs temporal-block (Kim) vs lead-wise vs **beat-aligned** (mask whole beats from R-peaks) | Learns beat-level morphology instead of interpolating smooth segments |\n| C7 | **Anti-collapse recipe for forward prediction (RQ3)** | EMA-only vs VICReg vs VQ vs SIGReg, with stability–accuracy Pareto, 2 seeds | A practical recipe others can reuse for JEPA-style forecasting |\n| C8 | **Clinical horizons (stretch)** | external AF-onset test on IRIDIA-AF (upload as a Kaggle dataset); hierarchical stack with ≥ 30 min context; compare with an RR/HRV logistic-regression baseline | Early warning of paroxysmal AF from Holter/patch monitoring |\n\n**Priority.** Must have for Q1: C1, C2, C3, C4, plus C5 on PTB-XL + CPSC2018. Strongly\nrecommended: C6, C7. Stretch: C8 (needs IRIDIA-AF on Kaggle and longer context).\n\n## 4. Expected outcomes and how to write them\n\n* If **dual > masked-only (3 seeds, p < 0.05) and ≥ CPC**: headline claim \"forward dynamics improve JEPA ECG representations\"; C1 + C3 + C4 carry the paper.\n* If **dual ≈ masked-only but forecast skill > 0 with positive horizon gain**: claim a *free* world-model capability (forecasting without losing diagnostic quality); C2 carries the paper.\n* If **forecast skill ≤ 0 on held-out records**: report it — the controls are the contribution; pivot the paper to C2 + C4 + C5 (benchmark + robustness study of ECG JEPAs).\n\n## 5. Suggested venues (Q1)\nIEEE Journal of Biomedical and Health Informatics; Computers in Biology and Medicine;\nBiomedical Signal Processing and Control; Artificial Intelligence in Medicine. Check each\njournal's current quartile in your target index before submitting.\n\n## 6. Risks\n* **Compute:** a T4 session gives ~6 000 steps per model. Report this budget honestly and compare all methods at equal steps; do not compare raw numbers with papers trained 100 epochs on 180 k ECGs.\n* **2-lead Holter vs 12-lead:** tiling is a compromise. Ablate `long_loader_weight` (0.25 / 0.5 / 1.0) and report it.\n* **MEETI** has no reader yet (no readable files found). Add one only if its format is waveforms; otherwise detach it.\n",
"EXPERIMENT_PLAN.md": "# Experiment and ablation plan (v3 notebook)\n\nAll models: CroPA encoder 384-d × 8 layers, predictor 192-d × 4, batch 32, 30 epochs ×\n200 steps (6 000 steps per model; same budget for every method), AdamW + cosine, AMP.\nPretraining: Chapman + Ningbo (+ CODE-15 / MIMIC if attached) short clips + **training**\nLTAFDB / MIT-BIH-LT records (holdout fixed first). Probes: PTB-XL superdiagnostic\n(folds 1–8 / 9 / 10) and CPSC2018 if attached.\n\n## Session schedule (one Kaggle T4 session ≈ 9–11 h each)\n\n| Session | Trains | Evaluates | ≈ GPU h |\n|---|---|---|---|\n| **A** | seed 0: masked_only, forward_only, dual, SimCLR, CPC; supervised (100/10/1 %) | Stage 1 probes, robustness, Stage 2 anticipation + forecast skill | 8–10 |\n| **B** | seeds 1, 2: masked_only, dual; seed 0: ST-MEM, masked_only_cm (compute-matched) | Stage 1 seed table + go/no-go | 7–9 |\n| **C** | RQ3: ema, vq, sigreg (seed 0; vicreg = Stage-1 dual); seeds 1, 2: SimCLR, CPC | Stage 3 Pareto, Stage 1 baselines with seeds | 8–10 |\n| **D** | Stage 4: hierarchical, single_scale; seeds 1, 2: forward_only | Stage 4 sweep + burden, final tables, exit criteria | 6–9 |\n\nAttach the previous session's notebook output as an input before starting the next session;\nthe notebook finds the checkpoints and the results ledger automatically. Weekly Kaggle GPU\nquota is ~30 h, so A–D take about one week.\n\n## Main ablation matrix\n\n| Ablation | Arms | Seeds | Answers |\n|---|---|---|---|\n| Objective | masked_only / forward_only / dual | 3 | Does `L_fwd` help, hurt or add nothing? |\n| Compute match | masked_only_cm (≈1.5× steps) vs dual | 1→3 | Is the gain just more gradient signal per step? |\n| SSL family | SimCLR / CPC / ST-MEM / dual | 3 | JEPA+dynamics vs contrastive vs contrastive-forward vs reconstruction |\n| Supervision | supervised (1/10/100 %) / random-init | 1–3 | Floor and label-efficiency reference |\n| Anti-collapse (RQ3) | ema / vicreg / vq / sigreg | 1–2 | Which mechanism keeps the forward objective healthy? |\n| Hierarchy (H2) | hierarchical / single_scale (same 12-window loader) | 1 | Does a rhythm-level stack help at long horizons? |\n| Optional (config flags, 1 seed each) | `forward_target_long` pooled vs tokens; `long_loader_weight` 0.25 / 0.5 / 1.0; `intra_window_frac` 0.5 / 0.6 / 0.75; `mask_type` block vs random; horizon-shuffle at train time | 1 | Design choices reviewers ask about |\n\n## Tables for the paper\n\n| Table | Content | Produced by |\n|---|---|---|\n| T1 | Datasets: role, records, patients, fs, leads, split, leakage controls | `results/data_report.csv` |\n| T2 | PTB-XL (and CPSC2018) linear probe, macro AUROC mean ± std (3 seeds) and macro-F1, all methods; published ECG-JEPA 0.906 as an external reference (different budget) | `results/table_linear_probe.csv` |\n| T3 | Label efficiency 1 / 10 / 50 / 100 % (mean ± std over repeats and seeds) incl. supervised | `results/table_label_efficiency.csv` |\n| T4 | Robustness: AUROC per noise type × SNR, and lead subsets | `results/table_robustness.csv` |\n| T5 | Anticipation at 5 / 10 / 30 min: context vs world vs random-predictor control, AUROC + AUPRC with record-level CIs | `results/table_anticipation.csv` |\n| T6 | Forecast skill per horizon: skill [CI], horizon gain | `results/table_forecast_skill.csv` |\n| T7 | RQ3: verdict, effective rank, cov-rank ratio, std, AUROC | `results/table_rq3.csv` |\n| T8 | Stage 4: AUROC by horizon (hierarchical vs single-scale) + AF-burden R² / ρ | `results/table_stage4.csv` |\n| T9 | Statistical tests: paired seed t-tests and bootstrap deltas | `results/table_tests.csv` |\n\n## Figures for the paper\n\n| Figure | Content | File |\n|---|---|---|\n| F1 | Method overview (masked branch + forward branch + horizon token + regulariser) — draw by hand from `models/cardio_jepa.py` | — |\n| F2 | Training curves: total / L_mask / L_fwd / L_reg and effective rank per arm | `figures/fig_training_curves.pdf` |\n| F3 | Linear-probe AUROC bars with seed error bars | `figures/fig_linear_probe.pdf` |\n| F4 | Label-efficiency curves | `figures/fig_label_efficiency.pdf` |\n| F5 | Robustness: AUROC vs SNR per noise type + lead-subset bars | `figures/fig_robustness.pdf` |\n| F6 | Anticipation AUROC: context / world / control per horizon | `figures/fig_anticipation.pdf` |\n| F7 | Forecast skill vs horizon with CI, and horizon gain | `figures/fig_forecast_skill.pdf` |\n| F8 | RQ3 stability–accuracy scatter + effective-rank trajectories | `figures/fig_rq3.pdf` |\n| F9 | Stage-4 horizon sweep | `figures/fig_stage4.pdf` |\n\n## Exit criteria (checked automatically by the notebook)\n\n1. Leakage checks pass (no probe dataset in pretraining; no holdout record in forward pairs; PTB-XL patient overlap = 0).\n2. No collapse in any reported main or baseline arm (verdict ≠ collapsed).\n3. Every baseline trained (loss fell ≥ 10 % from the first epoch).\n4. ≥ 3 seeds for masked_only and dual, seed std of PTB-XL AUROC ≤ 0.01.\n5. Determinism on (fixed seeds, deterministic cuDNN) and a repeat-run check passes.\n6. dual ≥ masked_only (paired seed test) and dual ≥ the best SSL baseline (mean AUROC).\n\nCriteria 1–5 are about correctness and must pass. Criterion 6 is the scientific result: if it\nfails after 3 seeds, the result is negative and should be reported as such (see the\nresearch plan, section 4).\n"
}
print('docs:', list(DOCS))

## Export

In [ ]:
# --- export: docs, config, source, ledger, tables, figures -> one zip ----------------------
import zipfile
with open(os.path.join(RESULTS_DIR, "config.json"), "w") as fh:
    json.dump({"run_mode": RUN_MODE, "session": SESSION, "plan": PLAN, "config": CFG.to_dict(),
               "datasets": {k: v.label for k, v in FOUND.items()},
               "holdout_test_records": HOLDOUT.test_names,
               "torch": torch.__version__, "package": cj.__version__}, fh, indent=1, default=str)
for name, text in DOCS.items():
    with open(os.path.join(WORK, name), "w") as fh:
        fh.write(text)
save_ledger()
ZIP = os.path.join(WORK, "cardio_jepa_v3_outputs.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for base in ("results", "cardio_jepa_v3_src"):
        for root, _d, files in os.walk(os.path.join(WORK, base)):
            for f in files:
                p = os.path.join(root, f); z.write(p, os.path.relpath(p, WORK))
    for f in glob.glob(os.path.join(RUNS_DIR, "*")):
        if f.endswith(".pth") and not ZIP_CHECKPOINTS:
            continue
        z.write(f, os.path.relpath(f, WORK))
    for name in DOCS:
        z.write(os.path.join(WORK, name), name)
print(f"wrote {ZIP} ({os.path.getsize(ZIP) / 1e6:.1f} MB)")
print("next step:", "attach this version's output and run the next SESSION" if SESSION in "ABC"
      else "all sessions done -- tables are in results/, figures in results/figures/")

# Appendix

# Cardio-JEPA — bug report for v0.1 (the `cardio-jepa-v-2a-corrected` run)

Every item lists **where** it was, **why** it is wrong, the **evidence** from the v0.1 Kaggle
log, and the **fix** in v0.2. Items B01–B11 change or invalidate published-looking numbers;
fix them before any result goes into a paper. Each fix has a regression test in
`tests/test_v02.py`.

## Critical — results were invalid or misleading

| ID | Where | What was wrong | Evidence in the v0.1 log | Fix (v0.2) |
|---|---|---|---|---|
| B01 | `config.DataConfig.pretrain_datasets`, `data/build.py` | **Test-set leakage.** PTB-XL (incl. test fold 10) was a *pretraining* corpus, and the PTB-XL fold-10 linear probe was then reported. Kim's ECG-JEPA protocol pretrains on Chapman/Ningbo/CODE-15 only. | `[data] PTB-XL indexed 20000 recordings` inside the pretraining loader | PTB-XL and CPSC2018 are probe-only (`probe_datasets`); `build_pretrain_loaders` raises if a probe set is in the pretraining list (opt-in flag for a deliberate transductive ablation). |
| B02 | notebook Stage 2 / 2c / 4, `ForwardPairDataset` | **Forecasting leakage.** Forward pairs were drawn from *all* LTAFDB records, then anticipation and forecast skill were tested on 30 % of the *same* records — the model had been trained to predict the exact futures it was scored on. The test split was also re-drawn per horizon. | `forward pairs: 18200` from all 91 long records; `split_by_record(...)` called again in every stage | `split_long_records()` fixes a burden-stratified record holdout **before** pretraining; pretraining gets only the train records; every benchmark uses `split_by_ids(..., HOLDOUT)`; the notebook asserts no overlap. |
| B03 | `ForwardSamplingConfig.horizons_s`, notebook Stage 2 | **Untrained horizon embeddings at test time.** Training horizons were (10, 30, 60, 300) s, but anticipation used H = 900 s (→ 600 s bucket) and forecast skill 900 / 1800 s — buckets whose embeddings never received a gradient (random init). | forecast skill flat at +0.05 for 10 s … 1800 s; `dual/world` ≈ `dual/context` | Training horizons now (10, 30, 60, 300, 600, 1800); evaluation horizons are trained buckets (300, 600, 1800); a persistent `horizon_counts` buffer warns whenever an untrained bucket is queried. |
| B04 | `data/longform._load_mitbih_csv`, `data/discovery.py` | **Sampling-rate mismatch + wrong corpus label.** The attached *MIT-BIH Long-Term ECG* mirror (128 Hz) matched the generic `mit-bih` pattern, was reported as "MIT-BIH Arrhythmia", and was read at a hard-coded 360 Hz: every "10 s" window held 28 s of ECG, so heart rates looked ~2.8× too fast. | `MIT-BIH Arrhythmia … 7 long records (0 with rhythm annotations, total 52.4 h)` — the Arrhythmia DB has 48 × 30 min records | New `mitbih_lt` corpus key; fs inferred from a time column, else the corpus default (128 Hz ltdb / 360 Hz mitdb); time column no longer read as an ECG lead. |
| B05 | notebook Stage 3 | **RQ3 had no EMA-only arm.** The `ema` row reused Stage-1 `dual`, but `kaggle_config` runs Stage 1 with VICReg (`reg=0.51` in the log). So "ema" and "vicreg" were two runs of the *same* configuration, and the key RQ3 condition was never trained. | Stage-1 `dual` log: `reg=0.5132`; Stage 3 prints `reusing Stage 1 'dual'` | The EMA-only arm (β = 0) is trained; Stage-1 `dual` is reused only for the `vicreg` arm and only when the loss configs are identical (checked in code). |
| B06 | `models/baselines.STMEMBaseline` | **ST-MEM baseline never trained.** Targets were per-patch-normalised 0.2 s patches of an already z-scored signal — mostly iso-electric segments whose noise was blown up to unit variance, i.e. white-noise targets. | `st_mem` loss 0.985 → 0.975 over 30 epochs, effective rank 1.0, probe AUROC 0.650 | `target_norm="none"` (regress the z-scored signal). Reproduced offline: old target stays at 0.98 (exactly the Kaggle value), new target learns (`test_stmem_fixed_target_learns_while_patch_norm_does_not`). Removed a wasted second encoder pass. |
| B07 | `regularizers.SoftVQ`, `config.LossConfig.vq_*` | **The VQ-arm "collapse" was an implementation artefact.** Temperature 0.5 on *cosine* logits with a random 384-d codebook gives near-uniform assignments, so every target became the same (rescaled) codebook average: a constant target, trivially predicted. Only the last of two assignment sets was regularised. | `vq` loss 0.29 → 0.009 by epoch 1; std 0.07; cos-sim 0.94 | T = 0.1, codebook seeded from real targets on the first step, commitment weight 1.0, every assignment regularised. Measured target spread: 0.148 (v0.1) vs 0.948 (v0.2) against a true 0.963. |
| B08 | `models/transformer.Attention`, notebook Stage 4 | **Stage-4 OOM.** Attention materialised a full fp32 (B·W, heads, 400, 400) score matrix per layer for backward (1.38 GiB/layer at 12 windows), while seven trained models still sat on the GPU. | `OutOfMemoryError: Tried to allocate 1.38 GiB … 11.72 GiB allocated` | `F.scaled_dot_product_attention` (memory-efficient kernel, works on T4); `free_cuda()` between stages; Stage 4 retries with a halved batch on OOM. |
| B09 | notebook export cell | **All results lost on a crash.** Tables were written only by the last cell; the Stage-4 OOM meant nothing reached `/kaggle/working/results/`. | run ends with `OutOfMemoryError`; export cell `pending` | A results **ledger** (CSV) is written after every stage, plus the zip. |
| B10 | `eval/stats.py`, notebook Stage 1/2 | **Over-confident statistics.** Anticipation CIs bootstrapped individual windows although windows from one Holter record are strongly correlated; the Stage-1 GO decision used one training seed. | `p_value 0.0` for every Stage-2 comparison | Record-level **cluster bootstrap**; ≥3 seeds with a paired seed-level t-test and seed CIs. |
| B11 | notebook Stage 2 (`world` features) | **Confounded "world-model" evidence.** `masked_only/world` (forward predictor *never trained*, i.e. a random projection of the context) beat `masked_only/context` by +0.016 (p < 0.05). So `world > context` partly measures extra feature dimensions, not forecasting. `context` also used only the last window while `world` used both. | `masked_only/world 0.6219` vs `masked_only/context 0.6062` | Gains are reported **relative to the random-predictor control**; `context` features now pool all context windows. |

## High — biased comparisons or wasted compute

| ID | Where | What was wrong | Fix |
|---|---|---|---|
| B12 | `data/build.py` (`weights.append(2.0)`) | 2 of every 3 training batches — for **every** arm, including the masked-only "Kim reproduction" — were 2-channel Holter windows tiled to 8 leads. Large domain shift from the 12-lead downstream tasks; also explains part of the gap to Kim's 0.906. | `cfg.data.long_loader_weight = 0.5` (≈33 %), configurable for an ablation. |
| B13 | `engine.InterleavedLoader` | The interleaving RNG was seeded by an epoch counter never reset, so sequentially trained arms saw different short/long batch orders. | `reset()`; called by `pretrain` and `estimate_runtime`. |
| B14 | `probe_batch_from_loader(n=128)` | Effective rank of 128 samples is capped at 128 < d = 384: every RQ3 rank number was bounded by the probe batch. The batch was also mostly Holter. | n = 512, drawn from the 12-lead short loader. |
| B15 | `diagnostics.CollapseMonitor.summary` | Collapse flags were so strict that nothing was flagged — not even the VQ arm above. | `representation_collapse`, wider `dimensional_collapse`, and a single `verdict` (healthy / dimensional / collapsed). |
| B16 | `eval/benchmarks.forecast_skill` | Prediction (normalised target space) was compared with persistence in the raw *online* space; training records were sampled; no control for an ignored horizon token. | Both in the trained target space; held-out records only; horizon-shuffle control (`horizon_gain`); bootstrap CI on skill. |
| B17 | `eval/linear_probe.py` | Fixed lr and 40 epochs; the PTB-XL validation fold was never used; one random draw per label fraction (1 % sometimes beat 10 %). | lr chosen on the validation fold with early stopping; 3 repeats per fraction (mean ± std). |
| B18 | `data/short.index_ptbxl`, `build_probe_splits` | Probe set randomly subsampled to 20 000 of 21 799 records (test fold 2 010 instead of ~2 160); records without a superclass kept as all-negative rows; 100 Hz files only. | Full PTB-XL; unlabelled rows dropped (Strodthoff protocol); 500 Hz when present; patient overlap train/test checked. |
| B19 | `models/forward_predictor.SequencePredictor` | Level-2 masked infilling was never told which windows were masked (fixed query slots), and context positions shifted after masking: an ill-posed task. | Query and context position indices passed explicitly. |
| B20 | `models/cardio_jepa.py` Level 2 | Target summaries got an extra LayerNorm that the online summaries did not. | Same construction on both sides. |
| B21 | `CardioJEPA` | `model.train()` also put the EMA target encoder in train mode (stochastic depth in targets with the `full` preset). | `train()` override keeps targets in eval mode. |
| B22 | `engine.estimate_runtime` | Ignored `max_steps_per_epoch`, so the "fixed" projection was unchanged (104 h). | Uses the capped step count. |
| B23 | `data/short.ShortWindowDataset` | Read failures silently became all-zero windows (e.g. MEETI: no readable files). | Failures counted; the evaluation cache audits the zero-filled fraction across workers. |
| B24 | `data/discovery.py` | Discovery took 51 min: up to 200 k files walked per corpus, the Chapman+Ningbo mirror walked twice, the unreadable MEETI mirror walked in full. | 20 k-file cap, per-path probe cache, `skip=` list (`meeti`). |
| B25 | forward objective design | Long-horizon (≥10 s) forecasting regressed the **per-patch** token grid of a future window; beat phase minutes ahead is unpredictable, so the optimum is a per-position mean (noise to fit). | Cross-window target is the pooled window latent (`forward_target_long="pooled"`; `"tokens"` kept as an ablation). |

## Notebook-level issues

| ID | What was wrong | Fix |
|---|---|---|
| B26 | `WARM_START="auto"` would initialise *every* ablation arm from the same checkpoint found in a later session, contaminating the comparison. | Off by default; multi-session runs reuse a model's **own** checkpoint instead. |
| B27 | (older notebook) `os.environ["CUDA_LAUNCH_BLOCKING"]="1"` makes every CUDA kernel synchronous (much slower); `!pip install flash_attn xformers triton` fails with internet off and is not needed (T4 has no FlashAttention-2; SDPA's memory-efficient kernel is used). | Removed. |
| B28 | Stage 3 read `res` left over from the Stage-1 loop for the reused arm (correct only by accident of loop order). | Uses the arm's own result object. |
| B29 | No determinism settings (cuDNN autotuning, cuBLAS workspace). | `CUBLAS_WORKSPACE_CONFIG`, `cudnn.deterministic`, per-arm `set_seed`. |
| B30 | 11-h single session for 9 models (projection said 104 h; the cap brought it to ~11 h with zero margin, then Stage 4 crashed at hour 9.8). | Four-session plan (A–D) with checkpoint reuse and a results ledger. |

## Checked and **not** a bug

* PTB-XL official folds are patient-disjoint (now verified in code: `patient_overlap = 0`).
* The Chapman+Ningbo combined mirror and the separate Chapman-Shaoxing mirror are not double-counted: discovery picks one root per corpus key and indexes a shared root once.
* EMA update direction and stop-gradient on targets are correct (`_target_tokens` runs under `no_grad`; `update_target` is θ̄ ← mθ̄ + (1−m)θ, m: 0.996 → 1.0).
* "ECG Heartbeat Categorization" is not picked up by discovery (single beats at 125 Hz, unusable for 10 s windows). **Detach it**: its beats come from MIT-BIH Arrhythmia and PTB Diagnostic, so it would leak if either were later used for evaluation.


# Changelog

## 0.2.0 — corrected and extended framework (v3 notebook)

Bug IDs refer to `docs/BUG_REPORT.md`.

### Fixed — leakage and validity
- PTB-XL / CPSC2018 removed from pretraining; guard raises if a probe dataset is pretrained on (B01).
- Long-record holdout fixed before pretraining (`split_long_records`, `split_by_ids`); forward pairs only from training records (B02).
- Forward horizons now cover every evaluated horizon; `horizon_counts` buffer + warning on untrained buckets (B03).
- MIT-BIH Long-Term ECG: own corpus key, sampling rate inferred (128 Hz, was read as 360 Hz) (B04).
- RQ3: a real EMA-only arm; Stage-1 reuse only for an identical VICReg config (B05).
- ST-MEM target fixed (it never trained) (B06).
- Soft-VQ: temperature, data-seeded codebook, commitment weight, all assignments regularised (B07).
- Cluster (record-level) bootstrap; multi-seed paired tests (B10); random-predictor control for "world" features (B11).

### Fixed — training and data
- Fused SDPA attention (fixes Stage-4 OOM, faster); GPU freed between stages (B08).
- Holter share of batches 67 % → 33 % (configurable) (B12); identical batch schedule for every arm (B13).
- Diagnostics on 512 short 12-lead windows (B14); stricter collapse verdict (B15).
- PTB-XL: full dataset, 500 Hz, unlabelled rows dropped, patient-overlap check (B18).
- Level-2 masked predictor gets positions; Level-2 target consistent (B19, B20).
- Target encoder always in eval mode (B21); runtime estimate honours the step cap (B22).
- Failed reads counted and audited (B23); discovery 20 k-file cap, probe cache, skip list (B24).
- Cross-window forward target pooled (phase-invariant) by default (B25).
- Deterministic seeding (cuDNN flags, cuBLAS workspace) (B29).

### Added
- `SimCLRBaseline` and `CPCBaseline` (same CroPA encoder) — CPC is the closest prior art to `L_fwd`.
- `train_supervised` (supervised-from-scratch reference, incl. label fractions) and a random-init floor.
- `run_robustness_probe`: real NSTDB noise (bw / em / ma) at 24 / 12 / 6 / 0 dB, and lead subsets (I+II, single lead II).
- Linear probe: validation-selected learning rate + early stopping; repeated label fractions.
- `forecast_skill`: same-space persistence, horizon-shuffle control, bootstrap CI.
- Burden probe: grouped-CV ridge alpha, record-level metrics.
- `data.cache`: one-time float16 memmap cache of evaluation sets in `/tmp` (+ zero-fill audit).
- `load_train_result`, `free_cuda`; `seed_level_summary`, `paired_seed_test`.
- 24 new regression tests (`tests/test_v02.py`); all 59 tests pass on CPU.

### Notebook (v3)
- Self-contained: writes the corrected package from the notebook (no model re-upload needed).
- Four-session plan (A–D) with automatic reuse of checkpoints from earlier sessions.
- Results ledger saved after every stage; exit-criteria checker; loop log; comparison with v0.1.
- Paper figures (PNG + PDF) and all docs written to `/kaggle/working`, zipped at the end.
- Removed `CUDA_LAUNCH_BLOCKING`, `pip install` of flash-attn/xformers/triton, warm-start-by-default.

### Changed defaults (may change numbers vs v0.1)
`pretrain_datasets`, `long_loader_weight=0.5`, `forward.horizons_s`, `forward_target_long="pooled"`,
`vq_temperature=0.1`, `vq_commit_weight=1.0`, `anticipation_horizons_s=(300, 600, 1800)`,
`max_records_per_dataset=46000` (all of Chapman+Ningbo), `ptbxl_use_hr=True`, `cache_dir=/tmp/...`.

## 0.1.0
Initial implementation (see the v0.1 Kaggle run for its results).


# Research plan — making Cardio-JEPA a defensible Q1 paper

## 1. What the v0.1 run actually showed (after reading the logs)

| Question | v0.1 result | Can it be trusted? |
|---|---|---|
| Does `L_fwd` hurt classification? (Stage 1) | dual 0.818 > masked-only 0.768 > forward-only 0.718 > ST-MEM 0.650 (PTB-XL macro AUROC) | **No.** PTB-XL test fold was in pretraining (B01); one seed (B10); ST-MEM never trained (B06); 2/3 of batches were tiled Holter (B12). All numbers are far below the published ECG-JEPA linear probe (0.906, Kim 2024), so the models are under-trained. The *direction* (dual ≥ masked) is promising and must be re-tested. |
| Does forward pretraining help anticipation? (H1) | No: dual 0.525 / masked-only 0.622 AUROC at 15 min | **No** in either direction: tested on records the forward predictor was trained on (B02), with an untrained 600 s horizon embedding (B03), window-level CIs (B10). |
| Is the predictor forecasting? | skill ≈ +0.05 at every horizon (flat) | Flat skill is the signature of an ignored / untrained horizon token (B03) measured in-sample (B02, B16). |
| What prevents collapse? (RQ3) | "ema" ≈ vicreg > vq ≈ sigreg | **No:** there was no EMA-only arm (B05); the VQ collapse was a bug (B07); ranks capped at 128 (B14). |

**Bottom line:** v0.1 does not yet support any claim. v0.2 removes the leakage and bugs, so the
next run gives the first valid numbers. Plan the paper around what v0.2 shows, not around
the v0.1 table.

## 2. Related work and the real research gap

* **Masked latent prediction for ECG.** Kim's ECG-JEPA ([arXiv:2410.08559](https://arxiv.org/abs/2410.08559)), Weimann & Conrad's I-JEPA for ECG ([arXiv:2410.13867](https://arxiv.org/abs/2410.13867)) and the hierarchical ER-JEPA ([arXiv:2607.01145](https://arxiv.org/abs/2607.01145)) all predict *masked* latents inside one ~10 s recording.
* **Forward latent prediction already exists — contrastively.** CPC predicts future latents; on 12-lead ECG it was the strongest self-supervised objective in Mehari & Strodthoff ([arXiv:2103.12676](https://arxiv.org/abs/2103.12676), PTB-XL linear-probe macro AUROC 0.927 vs SimCLR 0.883), and a 2026 systematic study found CPC slightly ahead of JEPA for transfer ([arXiv:2605.12241](https://arxiv.org/abs/2605.12241)).
* **Non-contrastive forward prediction in time series.** Eidos ([arXiv:2602.14024](https://arxiv.org/abs/2602.14024)) predicts future latents autoregressively with a stop-gradient target; SC-JEPA/MTS-JEPA ([arXiv:2602.04643](https://arxiv.org/abs/2602.04643)) forecasts the next window in latent space for anomaly *anticipation* with a soft codebook + EMA (close to our VQ arm); TS-JEPA ([arXiv:2509.25449](https://arxiv.org/abs/2509.25449)). None is on multi-lead ECG or clinical rhythm anticipation.
* **World models / anti-collapse.** V-JEPA 2 ([arXiv:2506.09985](https://arxiv.org/abs/2506.09985)) uses latent future prediction as a world model; LeJEPA/SIGReg ([arXiv:2511.08544](https://arxiv.org/abs/2511.08544)) removes the EMA teacher.
* **Clinical anticipation.** AF-onset prediction from long RR series reaches high AUROC with ~1 h of context (AF-Mamba, [arXiv:2609.06984](https://arxiv.org/abs/2609.06984)); IRIDIA-AF ([Sci Data 2023](https://pmc.ncbi.nlm.nih.gov/articles/PMC10584890)) provides 167 Holter records (152 patients, 2 leads, 200 Hz, 388 AF episodes, CC-BY 4.0).

**So the honest gap is narrower than "nobody predicts forward in time":**
1. Nobody has *combined* masked infilling (morphology) with **horizon-conditioned, non-contrastive** forward prediction (dynamics) for **multi-lead ECG**, and tested whether the forward term is what makes CPC strong.
2. ECG "world-model" claims are not evaluated with a **leakage-safe forecasting protocol** (held-out records, persistence and horizon-shuffle controls, record-level CIs).
3. No ECG-JEPA reports **robustness to real ambulatory noise and missing leads**.

Frame the paper as: *"Do JEPAs need dynamics? Unifying masked and forward latent prediction for
ECG, with a leakage-safe anticipation benchmark."* That is defensible even if the anticipation
result is modest, because the controls make it a clean answer.

## 3. Proposed contributions (realistic on Kaggle T4 sessions)

| # | Contribution | What to do | Real-world impact |
|---|---|---|---|
| C1 | **Dual objective vs strong SSL baselines** | dual vs masked-only vs forward-only vs CPC vs SimCLR vs ST-MEM vs supervised vs random-init; same encoder; 3 seeds; compute-matched masked-only control | Shows whether adding dynamics gives better general ECG features → fewer labels needed for diagnosis models |
| C2 | **Leakage-safe anticipation + forecast-skill protocol** | pre-pretraining record holdout, persistence & horizon-shuffle controls, random-predictor control, record-level cluster bootstrap, AUROC + AUPRC | A reusable evaluation standard; prevents inflated "early warning" claims in AF screening |
| C3 | **Label efficiency** | 1 / 10 / 50 / 100 % labels, 3 repeats, vs supervised from scratch | Hospitals have few expert-labelled ECGs; quantifies annotation savings |
| C4 | **Robustness to real noise and missing leads** | NSTDB baseline wander / electrode motion / muscle noise at 24→0 dB; I+II and single lead II | Holter, patch and smartwatch ECGs are noisy and have 1–2 leads; tells clinicians when outputs stay reliable |
| C5 | **Cross-dataset generalisation** | pretrain Chapman+Ningbo (+CODE-15) → probe PTB-XL (Germany) and CPSC2018 (China) as external sites; leave-one-dataset-out pretraining; harmonise SNOMED↔SCP labels for shared classes (AF, RBBB, LBBB, 1AVB, PVC) | Models must work in a different hospital and country than they were trained in |
| C6 | **ECG-specific masking** | random vs temporal-block (Kim) vs lead-wise vs **beat-aligned** (mask whole beats from R-peaks) | Learns beat-level morphology instead of interpolating smooth segments |
| C7 | **Anti-collapse recipe for forward prediction (RQ3)** | EMA-only vs VICReg vs VQ vs SIGReg, with stability–accuracy Pareto, 2 seeds | A practical recipe others can reuse for JEPA-style forecasting |
| C8 | **Clinical horizons (stretch)** | external AF-onset test on IRIDIA-AF (upload as a Kaggle dataset); hierarchical stack with ≥ 30 min context; compare with an RR/HRV logistic-regression baseline | Early warning of paroxysmal AF from Holter/patch monitoring |

**Priority.** Must have for Q1: C1, C2, C3, C4, plus C5 on PTB-XL + CPSC2018. Strongly
recommended: C6, C7. Stretch: C8 (needs IRIDIA-AF on Kaggle and longer context).

## 4. Expected outcomes and how to write them

* If **dual > masked-only (3 seeds, p < 0.05) and ≥ CPC**: headline claim "forward dynamics improve JEPA ECG representations"; C1 + C3 + C4 carry the paper.
* If **dual ≈ masked-only but forecast skill > 0 with positive horizon gain**: claim a *free* world-model capability (forecasting without losing diagnostic quality); C2 carries the paper.
* If **forecast skill ≤ 0 on held-out records**: report it — the controls are the contribution; pivot the paper to C2 + C4 + C5 (benchmark + robustness study of ECG JEPAs).

## 5. Suggested venues (Q1)
IEEE Journal of Biomedical and Health Informatics; Computers in Biology and Medicine;
Biomedical Signal Processing and Control; Artificial Intelligence in Medicine. Check each
journal's current quartile in your target index before submitting.

## 6. Risks
* **Compute:** a T4 session gives ~6 000 steps per model. Report this budget honestly and compare all methods at equal steps; do not compare raw numbers with papers trained 100 epochs on 180 k ECGs.
* **2-lead Holter vs 12-lead:** tiling is a compromise. Ablate `long_loader_weight` (0.25 / 0.5 / 1.0) and report it.
* **MEETI** has no reader yet (no readable files found). Add one only if its format is waveforms; otherwise detach it.


# Experiment and ablation plan (v3 notebook)

All models: CroPA encoder 384-d × 8 layers, predictor 192-d × 4, batch 32, 30 epochs ×
200 steps (6 000 steps per model; same budget for every method), AdamW + cosine, AMP.
Pretraining: Chapman + Ningbo (+ CODE-15 / MIMIC if attached) short clips + **training**
LTAFDB / MIT-BIH-LT records (holdout fixed first). Probes: PTB-XL superdiagnostic
(folds 1–8 / 9 / 10) and CPSC2018 if attached.

## Session schedule (one Kaggle T4 session ≈ 9–11 h each)

| Session | Trains | Evaluates | ≈ GPU h |
|---|---|---|---|
| **A** | seed 0: masked_only, forward_only, dual, SimCLR, CPC; supervised (100/10/1 %) | Stage 1 probes, robustness, Stage 2 anticipation + forecast skill | 8–10 |
| **B** | seeds 1, 2: masked_only, dual; seed 0: ST-MEM, masked_only_cm (compute-matched) | Stage 1 seed table + go/no-go | 7–9 |
| **C** | RQ3: ema, vq, sigreg (seed 0; vicreg = Stage-1 dual); seeds 1, 2: SimCLR, CPC | Stage 3 Pareto, Stage 1 baselines with seeds | 8–10 |
| **D** | Stage 4: hierarchical, single_scale; seeds 1, 2: forward_only | Stage 4 sweep + burden, final tables, exit criteria | 6–9 |

Attach the previous session's notebook output as an input before starting the next session;
the notebook finds the checkpoints and the results ledger automatically. Weekly Kaggle GPU
quota is ~30 h, so A–D take about one week.

## Main ablation matrix

| Ablation | Arms | Seeds | Answers |
|---|---|---|---|
| Objective | masked_only / forward_only / dual | 3 | Does `L_fwd` help, hurt or add nothing? |
| Compute match | masked_only_cm (≈1.5× steps) vs dual | 1→3 | Is the gain just more gradient signal per step? |
| SSL family | SimCLR / CPC / ST-MEM / dual | 3 | JEPA+dynamics vs contrastive vs contrastive-forward vs reconstruction |
| Supervision | supervised (1/10/100 %) / random-init | 1–3 | Floor and label-efficiency reference |
| Anti-collapse (RQ3) | ema / vicreg / vq / sigreg | 1–2 | Which mechanism keeps the forward objective healthy? |
| Hierarchy (H2) | hierarchical / single_scale (same 12-window loader) | 1 | Does a rhythm-level stack help at long horizons? |
| Optional (config flags, 1 seed each) | `forward_target_long` pooled vs tokens; `long_loader_weight` 0.25 / 0.5 / 1.0; `intra_window_frac` 0.5 / 0.6 / 0.75; `mask_type` block vs random; horizon-shuffle at train time | 1 | Design choices reviewers ask about |

## Tables for the paper

| Table | Content | Produced by |
|---|---|---|
| T1 | Datasets: role, records, patients, fs, leads, split, leakage controls | `results/data_report.csv` |
| T2 | PTB-XL (and CPSC2018) linear probe, macro AUROC mean ± std (3 seeds) and macro-F1, all methods; published ECG-JEPA 0.906 as an external reference (different budget) | `results/table_linear_probe.csv` |
| T3 | Label efficiency 1 / 10 / 50 / 100 % (mean ± std over repeats and seeds) incl. supervised | `results/table_label_efficiency.csv` |
| T4 | Robustness: AUROC per noise type × SNR, and lead subsets | `results/table_robustness.csv` |
| T5 | Anticipation at 5 / 10 / 30 min: context vs world vs random-predictor control, AUROC + AUPRC with record-level CIs | `results/table_anticipation.csv` |
| T6 | Forecast skill per horizon: skill [CI], horizon gain | `results/table_forecast_skill.csv` |
| T7 | RQ3: verdict, effective rank, cov-rank ratio, std, AUROC | `results/table_rq3.csv` |
| T8 | Stage 4: AUROC by horizon (hierarchical vs single-scale) + AF-burden R² / ρ | `results/table_stage4.csv` |
| T9 | Statistical tests: paired seed t-tests and bootstrap deltas | `results/table_tests.csv` |

## Figures for the paper

| Figure | Content | File |
|---|---|---|
| F1 | Method overview (masked branch + forward branch + horizon token + regulariser) — draw by hand from `models/cardio_jepa.py` | — |
| F2 | Training curves: total / L_mask / L_fwd / L_reg and effective rank per arm | `figures/fig_training_curves.pdf` |
| F3 | Linear-probe AUROC bars with seed error bars | `figures/fig_linear_probe.pdf` |
| F4 | Label-efficiency curves | `figures/fig_label_efficiency.pdf` |
| F5 | Robustness: AUROC vs SNR per noise type + lead-subset bars | `figures/fig_robustness.pdf` |
| F6 | Anticipation AUROC: context / world / control per horizon | `figures/fig_anticipation.pdf` |
| F7 | Forecast skill vs horizon with CI, and horizon gain | `figures/fig_forecast_skill.pdf` |
| F8 | RQ3 stability–accuracy scatter + effective-rank trajectories | `figures/fig_rq3.pdf` |
| F9 | Stage-4 horizon sweep | `figures/fig_stage4.pdf` |

## Exit criteria (checked automatically by the notebook)

1. Leakage checks pass (no probe dataset in pretraining; no holdout record in forward pairs; PTB-XL patient overlap = 0).
2. No collapse in any reported main or baseline arm (verdict ≠ collapsed).
3. Every baseline trained (loss fell ≥ 10 % from the first epoch).
4. ≥ 3 seeds for masked_only and dual, seed std of PTB-XL AUROC ≤ 0.01.
5. Determinism on (fixed seeds, deterministic cuDNN) and a repeat-run check passes.
6. dual ≥ masked_only (paired seed test) and dual ≥ the best SSL baseline (mean AUROC).

Criteria 1–5 are about correctness and must pass. Criterion 6 is the scientific result: if it
fails after 3 seeds, the result is negative and should be reported as such (see the
research plan, section 4).
